# Pressure-Level PWV Analysis

This notebook explores pressure-level and surface-pressure datasets, computes precipitable water vapor (PWV), and generates comparative analyses and figures for candidate observing sites.

## Notes for reuse

- Confirm that all referenced data files are available at the paths used in the code.
- Run cells from top to bottom because later sections depend on variables defined earlier.
- Review hard-coded site names, coordinates, and output paths before adapting the notebook for a different project.

## Project setup and imports

### Step 1

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
import matplotlib.pyplot as plt 
import matplotlib as mpl
import pandas as pd
import collections
import numpy as np
import glob
import matplotlib.pyplot as plt
# from mpl_toolkits.basemap import Basemap
import re
from pathlib import Path
from scipy.interpolate import griddata
from scipy.interpolate import interp2d
from scipy.stats import gaussian_kde
from scipy.stats import spearmanr
import xarray as xr
import pandas as pd
import glob
# import seaborn as sns
# from sklearn.metrics import mean_squared_error
import os


## Data loading and inspection

### Step 2

This cell loads dataset(s) `global_pressure_level24_25.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
era_24_25 = xr.open_dataset('global_pressure_level24_25.nc')


## Analysis workflow

### Step 3

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
print(era_24_25)


### Step 4

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
print(era_24_25['expver'])


### Step 5

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
era_24_25['expver']


### Step 6

This cell runs a notebook command to check the current execution environment or working directory.

In [ ]:
%pwd


## Data loading and inspection

### Step 7

This cell loads dataset(s) `pressure_level_er5.nc`, `pressure_level_er5.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
# era5 = xr.open_dataset('data/pressure_level_er5.nc')
era5 = xr.open_dataset('/Volumes/LaCie/THz/Tanmay Singh/THz_Mac/data/pressure_level_er5.nc')


## Interpolation and extraction

### Step 8

This cell subsets the dataset to a selected time, level, region, or site so the next step works with a focused slice of the data.

In [ ]:
# cut era5 values until 2024 the end (no lower limit)
era5 = era5.sel(time=slice(None,'2023-12-31'))


## Analysis workflow

### Step 9

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
print(era5)


### Combining these two datasets

## Interpolation and extraction

### Step 10

This cell defines reusable helper function(s) `combine_era5_datasets_efficient`, `combine_era5_simple_approach`, `combine_using_dask_delayed`, `process_era5` so later sections can apply the same processing logic consistently.

In [ ]:
import xarray as xr
import numpy as np

def combine_era5_datasets_efficient(era5_path, era_24_25_path):
    """
    Efficiently combine two ERA5 datasets using lazy operations and minimal memory usage.
    """
    
    print("Opening datasets with optimal chunking...")
    
    # Open datasets with chunking but don't load into memory
    era5 = xr.open_dataset(era5_path, chunks={
        'time': 100,
        'level': 37, 
        'latitude': 200,
        'longitude': 200
    })
    
    era_24_25 = xr.open_dataset(era_24_25_path, chunks={
        'valid_time': 16,
        'pressure_level': 37,
        'latitude': 200, 
        'longitude': 200
    })
    
    print("Preparing ERA5 dataset (expver=1 only)...")
    era5_subset = era5.sel(expver=1, drop=True)
    
    print("Preparing era_24_25 dataset...")
    era_24_25_renamed = era_24_25.rename({
        'valid_time': 'time',
        'pressure_level': 'level'
    })
    
    if 'expver' in era_24_25_renamed.coords:
        era_24_25_renamed = era_24_25_renamed.drop_vars('expver')
    
    era_24_25_final = era_24_25_renamed.expand_dims('expver').assign_coords(expver=[1])
    
    print("Combining datasets...")
    combined = xr.concat([era5_subset, era_24_25_final], dim='time', 
                        coords='minimal', compat='override')
    combined = combined.sortby('time')
    
    print("Dataset combination complete!")
    return combined

def combine_era5_simple_approach():
    """
    Simple approach with proper level coordinate alignment
    """
    print("Simple approach - opening datasets...")
    
    era5 = xr.open_dataset('data/pressure_level_er5.nc')
    era5 = era5.sel(time=slice(None,'2023-12-31'))
    era_24_25 = xr.open_dataset('global_pressure_level24_25.nc')
    
    print("Checking level coordinates...")
    print(f"ERA5 levels: {era5.level.values}")
    print(f"ERA_24_25 levels: {era_24_25.pressure_level.values}")
    
    era5_clean = era5.sel(expver=1, drop=True)
    
    print("Aligning level coordinates...")
    era_24_25_sorted = era_24_25.sortby('pressure_level')
    
    era_24_25_renamed = era_24_25_sorted.rename({
        'valid_time': 'time', 
        'pressure_level': 'level'
    })
    
    era_24_25_renamed = era_24_25_renamed.assign_coords(
        level=era_24_25_renamed.level.astype('int32')
    )
    
    era5_levels = set(era5_clean.level.values)
    era_24_25_levels = set(era_24_25_renamed.level.values)
    
    print(f"ERA5 levels: {sorted(era5_levels)}")
    print(f"ERA_24_25 levels: {sorted(era_24_25_levels)}")
    print(f"Common levels: {sorted(era5_levels & era_24_25_levels)}")
    print(f"ERA5 only: {sorted(era5_levels - era_24_25_levels)}")
    print(f"ERA_24_25 only: {sorted(era_24_25_levels - era5_levels)}")
    
    common_levels = sorted(era5_levels & era_24_25_levels)
    if len(common_levels) > 0:
        print(f"Using {len(common_levels)} common levels...")
        era5_common = era5_clean.sel(level=common_levels)
        era_24_25_common = era_24_25_renamed.sel(level=common_levels)
    else:
        print("No common levels found! Using reindexing...")
        era5_common = era5_clean
        era_24_25_common = era_24_25_renamed.reindex(level=era5_clean.level, method='nearest')
    
    era_24_25_final = era_24_25_common.expand_dims('expver').assign_coords(expver=[1])
    
    print("Concatenating datasets...")
    result = xr.concat([era5_common, era_24_25_final], dim='time')
    
    return result.sortby('time')

def combine_using_dask_delayed():
    """
    Using dask.delayed for even more control over lazy evaluation
    """
    import dask
    
    @dask.delayed
    def process_era5():
        ds = xr.open_dataset('/Volumes/LaCie/THz/Tanmay Singh/THz_Mac/data/pressure_level_er5.nc', chunks={'time': 50})
        return ds.sel(expver=1, drop=True)
    
    @dask.delayed  
    def process_era_24_25():
        ds = xr.open_dataset('global_pressure_level24_25.nc', chunks={'valid_time': 16})
        return (ds.rename({'valid_time': 'time', 'pressure_level': 'level'})
                 .drop_vars('expver', errors='ignore')
                 .expand_dims('expver')
                 .assign_coords(expver=[1]))
    
    era5_processed = process_era5()
    era_24_25_processed = process_era_24_25()
    
    era5_result = era5_processed.compute()
    era_24_25_result = era_24_25_processed.compute()
    
    return xr.concat([era5_result, era_24_25_result], dim='time').sortby('time')

def minimal_memory_approach():
    """
    Absolute minimal memory approach with proper level handling and correct time slicing.
    """
    print("Minimal memory approach...")
    
    era5_full = xr.open_dataset('/Volumes/LaCie/THz/Tanmay Singh/THz_Mac/data/pressure_level_er5.nc', 
                           chunks={'time': 50, 'latitude': 100, 'longitude': 100},engine="netcdf4")
    era_24_25_full = xr.open_dataset('global_pressure_level24_25.nc',
                               chunks={'valid_time': 10, 'latitude': 100, 'longitude': 100},engine="netcdf4")

    print("Slicing datasets lazily to the correct time ranges...")
    era5 = era5_full.sel(time=slice(None, '2023-12-31'))
    era_24_25 = era_24_25_full.sel(valid_time=slice('2024-01-01', None))
    
    print("Processing ERA5...")
    era5_subset = era5.sel(expver=1, drop=True)
    
    print("Processing ERA_24_25 - sorting and renaming...")
    era_24_25_sorted = era_24_25.sortby('pressure_level')
    era_24_25_renamed = era_24_25_sorted.rename({
        'valid_time': 'time',
        'pressure_level': 'level'
    })
    
    print("Converting level coordinate types...")
    era_24_25_typed = era_24_25_renamed.assign_coords(
        level=era_24_25_renamed.level.astype('int32')
    )
    
    print("Handling level coordinate differences...")
    era5_levels = era5_subset.level.values
    era_24_25_levels = era_24_25_typed.level.values
    
    common_levels = np.intersect1d(era5_levels, era_24_25_levels)
    
    if len(common_levels) == 0:
        raise ValueError("No common pressure levels found between the two datasets!")
    else:
        print(f"Found {len(common_levels)} common levels...")
        era5_final = era5_subset.sel(level=common_levels)
        era_24_25_final_levels = era_24_25_typed.sel(level=common_levels)
    
    print("Adding expver dimension...")
    era_24_25_clean = era_24_25_final_levels.drop_vars('expver', errors='ignore')
    era_24_25_final = era_24_25_clean.expand_dims('expver').assign_coords(expver=[1])
    
    print("Final concatenation...")
    result = xr.concat([era5_final, era_24_25_final], 
                      dim='time', 
                      coords='minimal',
                      compat='override',
                      join='override')
    
    return result.sortby('time')

def combine_your_datasets():
    """
    Try different approaches in order of efficiency
    """
    methods = [
        ("Simple approach", combine_era5_simple_approach),
        ("Minimal memory approach", minimal_memory_approach),
        ("Efficient approach", lambda: combine_era5_datasets_efficient('data/pressure_level_er5.nc', 'global_pressure_level24_25.nc'))
    ]
    
    for method_name, method_func in methods:
        try:
            print(f"\nTrying {method_name}...")
            result = method_func()
            print(f"✓ {method_name} successful!")
            print(f"Combined dataset shape: {result.sizes}")
            print(f"Time range: {result.time.min().values} to {result.time.max().values}")
            return result
        except Exception as e:
            print(f"✗ {method_name} failed: {e}")
            continue
    
    print("All methods failed!")
    return None

if __name__ == "__main__":
    print("Attempting to combine datasets using the minimal memory approach...")
    try:
        combined_dataset = minimal_memory_approach()
        print("\n✓ Minimal memory approach successful!")
        print(f"Combined dataset shape: {combined_dataset.sizes}")
        print(f"Time range: {combined_dataset.time.min().values} to {combined_dataset.time.max().values}")
        print("\nCombined dataset info (lazy representation):")
        print(combined_dataset)
    except Exception as e:
        print(f"✗ Minimal memory approach failed: {e}")


## Analysis workflow

### Step 11

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
combined_dataset


## Data loading and inspection

### Step 12

This cell loads dataset(s) `pressure_all.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
surface_pressure_data = xr.open_dataset('pressure_all.nc')


## Analysis workflow

### Step 13

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
print(surface_pressure_data)


## Site definitions and metadata

### Step 14

This cell loads dataset(s) `pressure_all.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

# --- Load dataset ---
ds = xr.open_dataset("pressure_all.nc")

# --- Select time range ---
ds = ds.sel(time=slice("2010-01-01", "2025-04-30"))

# --- Define sites ---
sites = {
    "IAO-Hanle": {"lat": 32.7789, "lon": 78.9650},
    "Merak": {"lat": 33.7828, "lon": 78.57782},
    "Site A": {"lat": 34.25, "lon": 78.75},
    "Site B": {"lat": 32.5, "lon": 79.0},
}

# --- Compute mean surface pressure (Pa) for each site using interpolation ---
for name, loc in sites.items():
    sp_site = ds["sp"].interp(latitude=loc["lat"], longitude=loc["lon"])
    mean_sp = float(sp_site.mean().values)
    print(f"{name:8s}  mean surface pressure = {mean_sp/100:.1f} hPa")


## Pressure and PWV calculations

### Step 15

This cell defines reusable helper function(s) `calculate_pwv` so later sections can apply the same processing logic consistently.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator

def calculate_pwv(q, levels, pressure_threshold):
    """
    Calculate Precipitable Water Vapor (PWV).
    
    Parameters:
    q (xarray.DataArray): Specific humidity data.
    levels (xarray.DataArray): Pressure levels in Pa.
    pressure_threshold (int): Pressure threshold in Pa.
    
    Returns:
    float: Calculated PWV.
    """
    pc = pressure_threshold 
    rho_w = 1 
    g = 9.81

    levels = levels * 100  # Convert from hPa to Pa 
    q = q.sel(expver=1)

    print(f"levels: {levels}")
    # Ensure levels and q are aligned
    q_levels_pa = q.level.values * 100  # Convert q levels to Pa
    common_levels = np.intersect1d(levels, q_levels_pa)
    common_levels = np.unique(np.sort(common_levels))  # Ensure sorted and unique

    print(f"common_levels: {common_levels}")
    if len(common_levels) < 2:
        raise ValueError("Not enough common pressure levels for interpolation.")
    
    # Ensure q_values corresponds to a single time point or specific slice
    try:
        q_values = q.sel(level=common_levels / 100, method='nearest').isel(time=0).values
    except ValueError:
        raise ValueError("Multiple time points found. Please select a single time point or specific slice.")
    #q_values = q.sel(level=common_levels / 100, method='nearest').values

    print(f"common_levels length: {len(common_levels)}, q_values length: {len(q_values)}")
    if len(common_levels) != len(q_values):
        raise ValueError("The lengths of common_levels and q_values do not match.")
    
    pchip = PchipInterpolator(common_levels, q_values)  # Create PCHIP interpolator
    interpolated_q = pchip(pc)  # Interpolated q value at pc

    print(f"interpolated_q: {interpolated_q}")
    mask = levels <= pc  
    q = q.where(mask, drop=True)
    levels = levels.where(mask, drop=True)

    dp = np.diff(levels)  # Ensure dp has the same length as q_avg
    dp = xr.DataArray(dp, dims=['level'], coords={'level': levels[:-1]/100})
    print(f"dp: {dp}")
    q_avg = xr.concat([(q.isel(level=i) + q.isel(level=i+1)) / 2 for i in range(len(levels) - 1)], dim='level')
    q_avg['level'] = levels[:-1]/100   

    print(f"q_avg: {q_avg}")
    integral = (q_avg * dp).sum(dim='level')

    first_level_pwv = q.sel(level=levels[0]/100, method='nearest') * levels[0]
    pwv_corr = (integral + interpolated_q * (pc - levels[-1]) + first_level_pwv) / (rho_w * g)
    print(f' Pc - levels[-1]: {pc - levels[-1]}')
    return pwv_corr

# Example usage
lat_hanle = 32.7789
lon_hanle = 78.965
ds_point = era5.sel(latitude=lat_hanle, longitude=lon_hanle, method='nearest')
#ds_point = era5.interp(latitude=lat_hanle, longitude=lon_hanle,method='linear')
# era5_patch = era5[["q","level"]].sel(
#     latitude=slice(lat_hanle+0.25, lat_hanle-0.25),
#     longitude=slice(lon_hanle-0.25, lon_hanle+0.25)
# ).compute()

# era5_pt = era5_patch.interp(latitude=lat_hanle, longitude=lon_hanle)
ds_point = ds_point.sel(time=slice("1998-01-01", "2017-12-31"))

pressure_threshold = surface_pressure_data['sp'].sel(latitude=lat_hanle, longitude=lon_hanle, method='nearest')
pressure_threshold = pressure_threshold.sel(time=slice("1998-01-01", "2017-12-31"))

pwv_corr_old = calculate_pwv(ds_point.q, ds_point.level, pressure_threshold = 55700)

# Plotting results
plt.figure(figsize=(12, 6))
pwv_corr_old.plot(color='red', label='PWV Corrected 1', linestyle='--', alpha=0.9, marker='o')
plt.xlabel('Time')
plt.ylabel('Precipitable Water Vapor (mm)')
plt.grid(True)
plt.legend()
plt.show()


## Mapping and visualization

### Step 16

This cell subsets the dataset to a selected time, level, region, or site so the next step works with a focused slice of the data.

In [ ]:
# pick a time step 
data = ds_point.isel(time=2)

print(data)


# first plot the q values with respect to pressure levels
plt.figure(figsize=(12, 6))
q = data.q.sel(expver=1)    
plt.plot(data.level,q, marker='o', linestyle='--')

plt.xlabel('Pressure (Pa)')
plt.ylabel('Specific Humidity (kg/kg)')
plt.grid(True)
plt.show()


### With surface pressure values

## Project setup and imports

### Step 17

This cell defines reusable helper function(s) `calculate_pwv_pressure` so later sections can apply the same processing logic consistently.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator

def calculate_pwv_pressure(q, levels_imp, pressure_thresholds):
    rho_w = 1 
    g = 9.81
    levels = levels_imp * 100  # Convert from hPa to Pa
    if 'expver' in q.dims:
        
        q = q.sel(expver=1)

    pwv_list = []

    for time_step in q.time:
        levels = levels_imp * 100 
        pc = pressure_thresholds.sel(time=time_step).values.item()
        print(f'pc: {pc}')
        q_at_time = q.sel(time=time_step)

        # Ensure levels and q_at_time are aligned
        q_levels_pa = q_at_time.level.values * 100  # Convert q levels to Pa
        common_levels = np.intersect1d(levels, q_levels_pa)
        common_levels = np.unique(np.sort(common_levels))  # Ensure sorted and unique

        #print(f"common_levels: {common_levels}")
        if len(common_levels) < 2:
            raise ValueError("Not enough common pressure levels for interpolation.")
        
        # Convert common_levels to hPa for direct matching --- Block added
        # Ensure it's just 1D
        
        common_levels_hPa = common_levels / 100
        common_levels_hPa_int = common_levels_hPa.astype(int)
        q_values = q_at_time.sel(level=common_levels_hPa_int).values

        q_at_time = q_at_time.squeeze()

        if len(q_values) != len(common_levels):
            print(f"Mismatch in lengths after sel(): {len(common_levels)} vs {len(q_values)}")
            print(f"q_levels: {q_at_time.level.values}")
            print(f"common_levels_hPa: {common_levels_hPa}")
            raise ValueError("Mismatch between common_levels and q_values after sel().")
        
        
        
        # # Ensure q_values corresponds to a single time point or specific slice
        # q_values = q_at_time.sel(level=common_levels / 100, method='nearest').values

        #print(f"common_levels length: {len(common_levels)}, q_values length: {len(q_values)}")
        if len(common_levels) != len(q_values):
            raise ValueError("The lengths of common_levels and q_values do not match.")
        

        # Filter out NaN/Inf before interpolation  << ADD THIS BLOCK
        finite = np.isfinite(q_values)
        q_values        = q_values[finite]
        common_levels   = common_levels[finite]
        if len(common_levels) < 2:
            pwv_list.append(np.nan)       # or continue
            continue


        pchip = PchipInterpolator(common_levels, q_values)  # Create PCHIP interpolator
        interpolated_q = pchip(pc)  # Interpolated q value at pc

        #print(f"interpolated_q: {interpolated_q}")
        mask = levels <= pc
        q_at_time = q_at_time.where(mask, drop=True)
        levels_at_time = levels.where(mask, drop=True)
        #print(f'levels_at_time: {levels_at_time}')

        dp = np.diff(levels_at_time)
        dp = xr.DataArray(dp, dims=['level'], coords={'level': levels_at_time[:-1]/100})

        #print(f'dp: {dp}')
        #print(f'length of dp: {len(dp)}')

        q_avg = xr.concat([(q_at_time.isel(level=i) + q_at_time.isel(level=i + 1)) / 2 for i in range(len(levels_at_time) - 1)], dim='level')
        q_avg['level'] = levels_at_time[:-1] / 100
        #q_avg = (q_at_time[1:] + q_at_time[:-1]) / 2
        #print(f'q_avg: {q_avg}')
        #print(f'length of q_avg level dimension: {q_avg.sizes["level"]}')

        #integral = np.sum(q_avg * dp,axis = 1)  # Ensure dp[:-1] matches the length of q_avg
        integral = (q_avg * dp).sum(dim='level')
        #print(f'integral: {integral}')

        first_level_pwv = q_at_time.sel(level=levels_at_time[0] / 100, method='nearest') * levels_at_time[0]
        #print(f'first_level_pwv: {first_level_pwv}')
        pwv_corr = (integral + interpolated_q * (pc - levels_at_time[-1]) + first_level_pwv) / (rho_w * g)
        # print(f'Contribution from interpolated q: {(interpolated_q * (pc - levels_at_time[-1]) / (rho_w * g)).values.round(2)}')
        # print(f'Pressure slab for last interpolated level: {pc} to {levels_at_time[-1]} Pa')
        # print(f'Fraction of the contribution from the last interpolated level: {(interpolated_q * (pc - levels_at_time[-1]) / (rho_w * g)).values / pwv_corr.values * 100:.2f}%')
        # print(f'Fraction of the contribution from the first level: {(first_level_pwv / pwv_corr).values * 100:.2f}%')
        # print(f'PWV Value at time {time_step.values}: {pwv_corr.values.round(2)} mm')
        
        # # write a loop to print the contribution of each level as a fraction of the total PWV and print the pressure slab value as well 
        # # print first and last interpolated level contributions separately
        # for i in range(len(levels_at_time) - 1):
        #     level_contribution = (q_avg.isel(level=i) * dp.isel(level=i)).sum() / (rho_w * g)
        #     # add text as P1 : P2 and dont substract
        #     pressure_slab = levels_at_time[i + 1] - levels_at_time[i]
        #     fraction = (level_contribution / pwv_corr).values * 100
        #     print(f'Level {i} contribution: {level_contribution.values.round(2)} mm, Pressure slab: {levels_at_time[i + 1]} to {levels_at_time[i]} Pa, Fraction: {fraction:.2f}%')
        # Extract scalar values from xarray objects to get clean numbers
        pwv_total_mm = pwv_corr.item()
        last_level_pressure_pa = levels_at_time[-1].item()
        time_str = np.datetime_as_string(time_step.values, unit='s')

        # # --- Main Summary ---
        # print("\n" + "="*60)
        # print(f"PWV Analysis for Timestamp: {time_str}")
        # print(f"Total Calculated PWV: {pwv_total_mm:.2f} mm")
        # print("="*60)

        # # --- Contribution from the most important layers ---
        # print("Breakdown of PWV Contribution (from bottom of atmosphere up):")

        # 1. Contribution from the interpolated surface layer (the most sensitive part)
        surface_layer_contrib_mm = (interpolated_q * (pc - last_level_pressure_pa) / (rho_w * g)).item()
        surface_layer_fraction = (surface_layer_contrib_mm / pwv_total_mm) * 100 if pwv_total_mm > 0 else 0

        # print(f"\n")
        # print(f"  - Contribution: {surface_layer_contrib_mm:.3f} mm")
        # print(f"  - Pressure Slab: From {last_level_pressure_pa:.0f} Pa down to surface at {pc:.0f} Pa")
        # print(f"  - Percentage of Total PWV: {surface_layer_fraction:.1f}%")

        # 2. Contribution from the integrated atmospheric layers
        # print("\n")
        # total_integrated_fraction = 0
        # # Loop backwards to show contributions from the bottom up
        # for i in range(len(levels_at_time) - 2, -1, -1):
        #     p_upper = levels_at_time[i].item()
        #     p_lower = levels_at_time[i+1].item()
            
        #     level_contribution_mm = (q_avg.isel(level=i) * dp.isel(level=i)).item() / (rho_w * g)
        #     fraction = (level_contribution_mm / pwv_total_mm) * 100 if pwv_total_mm > 0 else 0
        #     total_integrated_fraction += fraction
            
        #     # Only print layers that contribute meaningfully to avoid clutter
        #     if fraction > 0.1:
        #         print(f"  - Layer between {p_lower:.0f} & {p_upper:.0f} Pa: {level_contribution_mm:.3f} mm ({fraction:.1f}%)")

        # print("-" * 60)
        # --- End of New Block ---
    
        pwv_list.append(pwv_corr)

    pwv = xr.concat(pwv_list, dim=q.time)
    return pwv


### New Calculate PWV Pressure.

## Project setup and imports

### Step 18

This cell defines reusable helper function(s) `calculate_pwv_pressure` so later sections can apply the same processing logic consistently.

In [ ]:
import numpy as np
import xarray as xr
from scipy.interpolate import PchipInterpolator

def calculate_pwv_pressure(q, levels_imp, pressure_thresholds):
    """
    Robust PWV integrator that tolerates float/int level mismatches.

    Parameters
    ----------
    q : xarray.DataArray  – specific humidity with dims (time, level)
    levels_imp : xarray.DataArray or np.ndarray  – pressure levels [hPa]
    pressure_thresholds : xarray.DataArray       – surface pressure [Pa]

    Returns
    -------
    xarray.DataArray  – PWV time series (same time coordinate as `q`)
    """
    rho_w = 1.0
    g     = 9.81

    # ensure pressure-level coordinate is 1-D and monotonic increasing
    levels_pa = xr.DataArray(levels_imp.astype("float64") * 100.0,
                             dims=["level"], coords={"level": levels_imp})

    q = q.sel(expver=1)

    pwv_frames = []

    for t in q.time:
        pc  = float(pressure_thresholds.sel(time=t))       # Pa
        q_t = q.sel(time=t)

        # 1. intersect levels (in Pa) – keep unique, sorted values
        q_levels_pa  = q_t.level.values.astype("float64") * 100.0
        common_pa    = np.intersect1d(levels_pa.values, q_levels_pa)

        if common_pa.size < 2:
            pwv_frames.append(xr.DataArray(np.nan, coords={"time": t}))
            continue

        # 2. retrieve q at those levels using *nearest* lookup
        common_hpa   = common_pa / 100.0
        q_vals       = q_t.sel(level=xr.DataArray(common_hpa, dims="level"),
                               method="nearest").values.astype("float64")

# squeeze everything to 1-D first
        q_vals     = np.squeeze(q_vals)
        common_pa  = np.squeeze(common_pa)

        mask       = np.isfinite(q_vals)
        q_vals     = q_vals[mask]
        common_pa  = common_pa[mask]

        if q_vals.size < 2:
            pwv_frames.append(xr.DataArray(np.nan, coords={"time": t}))
            continue

        # 3. PCHIP interpolate q at surface pressure pc
        pchip        = PchipInterpolator(common_pa, q_vals, extrapolate=False)
        q_pc         = float(pchip(pc))

        # 4. integrate q from pc to TOA (simple trapezoid rule)
        in_column    = levels_pa.where(levels_pa <= pc, drop=True)
        q_trunc      = q_t.sel(level=in_column.level, method="nearest")
        dp           = np.diff(in_column.values)
        q_mid        = (q_trunc.values[1:] + q_trunc.values[:-1]) / 2.0
        integral     = np.sum(q_mid * dp)

        first_level  = q_trunc.isel(level=0) * in_column.values[0]
        pwv_corr     = (integral + q_pc * (pc - in_column.values[-1]) + first_level) / (rho_w * g)

        pwv_frames.append(xr.DataArray(pwv_corr, coords={"time": t}))

    return xr.concat(pwv_frames, dim="time")


### New New Method

## Pressure and PWV calculations

### Step 19

This cell defines reusable helper function(s) `calculate_pwv_pressure`, `one_profile` so later sections can apply the same processing logic consistently.

In [ ]:
# ----------------------------------------------------------------------
# 4. PWV CALCULATION  (exactly the same routine you already use)
# ----------------------------------------------------------------------
def calculate_pwv_pressure(q_da: xr.DataArray,
                           levels_da: xr.DataArray,
                           psurf_da: xr.DataArray) -> xr.DataArray:
    """
    Your original trapezoidal PWV integrator (q in kg/kg, P in Pa, g = 9.81).
    """
    g = 9.81
    p_levels_pa = levels_da * 100.0  # hPa → Pa

    def one_profile(q_prof, p_prof, p_sfc_val):
        finite = np.isfinite(q_prof)
        if finite.sum() < 2:
            return np.nan

        p, qv = p_prof[finite], q_prof[finite]
        sorter = np.argsort(p)
        p, qv = p[sorter], qv[sorter]

        # keep layers above surface
        keep = p >= p_sfc_val
        p, qv = p[keep], qv[keep]
        if p.size < 2:
            return np.nan

        # add point at Psfc using PCHIP interpolation
        q_sfc = PchipInterpolator(p, qv, extrapolate=True)(p_sfc_val)
        p_full = np.append(p, p_sfc_val)
        q_full = np.append(qv, q_sfc)
        sorter2 = np.argsort(p_full)
        return np.trapz(q_full[sorter2], p_full[sorter2]) / g   # kg m⁻² ≈ mm

    return xr.apply_ufunc(
        one_profile,
        q_da, p_levels_pa, psurf_da,
        input_core_dims=[['level'], ['level'], []],
        output_core_dims=[[]],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[q_da.dtype]
    )


### Method from Valeria+2024

## Pressure and PWV calculations

### Step 20

This cell defines reusable helper function(s) `calculate_pwv_pressure_valeria`, `qv_to_pwv_layer` so later sections can apply the same processing logic consistently.

In [ ]:
# --- NEW FUNCTION BASED ON VALERIA ET AL. WORKFLOW ---

def calculate_pwv_pressure_valeria(q, levels_imp, pressure_thresholds):
    """
    Calculates PWV using the 'Integrate-Then-Interpolate' method with linear interpolation.

    This function is designed to replicate the methodology found in the Valeria et al.
    (2024) paper's associated code. It accepts the same arguments as the original
    function for direct comparison.

    Args:
        q (xr.DataArray): Time series of specific humidity profiles [kg/kg].
                          Must have 'time' and 'level' dimensions.
        levels_imp (np.ndarray): Array of the pressure levels [hPa].
        pressure_thresholds (xr.DataArray): Time series of surface pressure [Pa].

    Returns:
        xr.DataArray: A time series of the calculated Precipitable Water Vapor [mm].
    """
    rho_w = 1.0
    g = 9.81

    # This helper function calculates PWV for a single layer
    def qv_to_pwv_layer(dp_hpa, qv_kg_kg):
        # The formula is q * (dp / g). The *100 converts dp from hPa to Pascals.
        return qv_kg_kg * (dp_hpa * 100) / g

    # Handle the 'expver' dimension if it exists in ERA5 data
    if 'expver' in q.dims:
        q = q.sel(expver=1)

    pwv_list = []

    # This loop iterates through each time step to perform the calculation
    for time_step in q.time:
        # Get the specific surface pressure for this exact time
        pc_pa = pressure_thresholds.sel(time=time_step).values.item()
        
        # Select the vertical profile of humidity for this time
        q_at_time = q.sel(time=time_step)
        
        # Extract the pressure levels and specific humidity values
        pl_hpa = q_at_time.level.values
        qv_values = q_at_time.values

        # Calculate the thickness of each pressure layer (DELP)
        # We approximate DELP by taking the difference between adjacent pressure levels
        delp_hpa = np.diff(pl_hpa, prepend=0)

        # --- Step 1: Calculate PWV per layer ---
        pwv_per_layer = qv_to_pwv_layer(delp_hpa, qv_values)

        # --- Step 2: Create a Cumulative PWV Profile ---
        # The cumulative sum gives the total integrated PWV from the top of the
        # atmosphere down to each respective pressure level.
        # We convert pressure levels to Pascals for the interpolation.
        pl_pa = pl_hpa * 100
        
        # Filter out any non-finite values to prevent interpolation errors
        finite_mask = np.isfinite(pwv_per_layer) & np.isfinite(pl_pa)
        
        pwv_cumulative = np.cumsum(pwv_per_layer[finite_mask])
        pl_pa_finite = pl_pa[finite_mask]

        # --- Step 3: Interpolate the Result ---
        # Perform a linear interpolation on the cumulative PWV profile to find
        # the total PWV at the exact surface pressure.
        # The bounds_error=False and fill_value=np.nan handle cases where the
        # surface pressure is outside the range of the model's pressure levels.
        total_pwv = np.interp(pc_pa, pl_pa_finite, pwv_cumulative, left=np.nan, right=np.nan)

        pwv_list.append(total_pwv)

    # Combine all calculated values into a final time series
    pwv = xr.DataArray(pwv_list, dims=['time'], coords={'time': q.time})
    return pwv


## Analysis workflow

### Step 21

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
surface_pressure_data


## Pressure and PWV calculations

### Step 22

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:

# Example usage
lat_hanle = 32.7789
lon_hanle = 78.965
# lat_hanle = 19.8230
# lon_hanle = 204.5306

#ds_point = era5.sel(latitude=lat_hanle, longitude=lon_hanle, method='nearest')
ds_point = era5.interp(latitude=lat_hanle, longitude=lon_hanle,method='linear')
ds_point = ds_point.sel(time=slice("2015-01-01", "2017-12-31"))

pressure_threshold = surface_pressure_data['sp'].interp(latitude=lat_hanle, longitude=lon_hanle, method='linear')
pressure_threshold = pressure_threshold.sel(time=slice("1998-01-01", "2017-12-31"))
#pressure_threshold = xr.full_like(pressure_threshold,58500)  # Set a constant pressure threshold for testing
print(f'pressure_threshold: {pressure_threshold.mean().values} Pa')


print(ds_point.q, ds_point.level)

pwv_corr_new = calculate_pwv_pressure(ds_point.q, ds_point.level, pressure_thresholds=pressure_threshold)
pwv_corr_new_valeria = calculate_pwv_pressure_valeria(ds_point.q, ds_point.level, pressure_thresholds=pressure_threshold)

# print the number of months below 1 mm
print(f'Number of months with PWV below 1 mm: {np.sum(pwv_corr_new < 1)}')
print(f'Number of months with PWV below 1 mm: {np.sum(pwv_corr_new_valeria < 1)}')

# Plotting results
plt.figure(figsize=(12, 6))
pwv_corr_new.plot(color='black', label='PWV Corrected', linestyle='-', alpha=0.7, marker='x')
pwv_corr_new_valeria.plot(color='green', label='PWV Corrected Valeria', linestyle='-', alpha=0.7, marker='x')
#pwv_corr_old.plot(color='blue', label='PWV Corrected', linestyle='--', alpha=0.9, marker='o')
plt.xlabel('Time')
plt.ylabel('Precipitable Water Vapor (mm)')
plt.grid(True)
plt.legend()
plt.show()


## Site definitions and metadata

### Step 23

This cell loads dataset(s) `mauna_kea_pressure.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
mauna_kea_surface_pressure = xr.open_dataset('mauna_kea_pressure.nc')
mauna_kea_surface_pressure


### Step 24

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
mauna_kea_surface_pressure['sp']


## Pressure and PWV calculations

### Step 25

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:

# Example usage

lat_mauna_kea = 19.8230
lon_mauna_kea = 204.5306  # Convert to -180 to 180 range


#ds_point = era5.sel(latitude=lat_hanle, longitude=lon_hanle, method='nearest')
ds_point = era5.interp(latitude=lat_mauna_kea, longitude=lon_mauna_kea,method='linear')
ds_point = ds_point.sel(time=slice("1998-01-01", "2017-12-31"))

pressure_threshold = mauna_kea_surface_pressure['sp'].interp(latitude=lat_mauna_kea, longitude=lon_mauna_kea-360, method='linear')
pressure_threshold = pressure_threshold.sel(time=slice("1998-01-01", "2017-12-31"))
pressure_threshold = xr.full_like(pressure_threshold,61690)  # Set a constant pressure threshold for testing
print(f'pressure_threshold: {pressure_threshold.mean().values} Pa')




pwv_corr_new = calculate_pwv_pressure(ds_point.q, ds_point.level, pressure_thresholds=pressure_threshold)
pwv_corr_new_valeria = calculate_pwv_pressure_valeria(ds_point.q, ds_point.level, pressure_thresholds=pressure_threshold)

# print the number of months below 1 mm
print(f'Number of months with PWV below 1 mm: {np.sum(pwv_corr_new < 1)}')
print(f'Number of months with PWV below 1 mm: {np.sum(pwv_corr_new_valeria < 1)}')

# Plotting results
plt.figure(figsize=(12, 6))
pwv_corr_new.plot(color='black', label='PWV Corrected New', linestyle='-', alpha=0.7, marker='x')
pwv_corr_new_valeria.plot(color='green', label='PWV Corrected Valeria', linestyle='-', alpha=0.7, marker='x')
#pwv_corr_old.plot(color='blue', label='PWV Corrected', linestyle='--', alpha=0.9, marker='o')
plt.xlabel('Time')
plt.ylabel('Precipitable Water Vapor (mm)')
plt.grid(True)
plt.legend()
plt.show()


## Project setup and imports

### Step 26

This cell defines reusable helper function(s) `calculate_pressure` so later sections can apply the same processing logic consistently.

In [ ]:
import math

def calculate_pressure(P_b, T_b, L_b, h, h_b, R_star=8.3144598, g_0=9.80665, M=0.028964425278793993):
    T_h = T_b - L_b * (h - h_b)
    exponent = (g_0 * M) / (R_star * L_b)
    pressure = P_b * (T_h / T_b) ** exponent
    return pressure

# Example usage:
P_b = 101325  # reference pressure at sea level in Pa
T_b = 297  # reference temperature at sea level in K
L_b = 0.0065  # temperature lapse rate in K/m
h = 5107     # height at which pressure is calculated in m
h_b = 00       # height of reference level in m

pressure = calculate_pressure(P_b, T_b, L_b, h, h_b)
print(f"Pressure at {h} meters: {pressure:.2f} Pa")


#### Elevation Data

## Data loading and inspection

### Step 27

This cell loads dataset(s) `fractional_land.0.25-deg.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
elevation_data = xr.open_dataset('fractional_land.0.25-deg.nc')


## Analysis workflow

### Step 28

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
elevation_data


#### Global Comparison

## Site definitions and metadata

### Step 29

This cell reads tabular metadata that is later used for site lookup, filtering, or summary reporting.

In [ ]:
# read the excel file with site details
site_details = pd.read_excel('sites_new.xlsx',header=0)

# print the first 5 rows of the data
#print(site_details)
# drop the last row
#site_details.drop(site_details.tail(1).index,inplace=True)

# take only first site 
#site_details = site_details.iloc[0:1]
#site_details = site_details.tail(1)
site_details


### Step 30

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
# drop the 6th row
site_details = site_details.drop(site_details.index[5])


### Step 31

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
site_details


### Data Var Definition

## Data loading and inspection

### Step 32

This cell loads dataset(s) `world_pressure_t2m_new_new.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
world_pressure_t2m = xr.open_dataset('world_pressure_t2m_new_new.nc')


## Analysis workflow

### Step 33

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
surface_pressure_data = world_pressure_t2m


## Interpolation and extraction

### Step 34

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
surface_pressure_data = surface_pressure_data.rename({'valid_time': 'time'})
# remove the dimension 'expver' and 'number'
surface_pressure_data


## Mapping and visualization

### Step 35

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
# plot the values of mean sea level pressure (msl) and sst for Hanle Coordinate accrooss all the years
hanle_surface_pressure = surface_pressure_data['msl'].interp(latitude=32.78,
                                                            longitude=78.96, method='linear')

hanle_sst = surface_pressure_data['sst'].interp(latitude=32.78,
                                                longitude=78.96, method='linear')

# plot the data
plt.figure(figsize=(12, 6))
plt.plot(hanle_surface_pressure.time, hanle_surface_pressure, label='Mean Sea Level Pressure (MSL)', color='blue')
plt.ylabel('Pressure (hPa)')
plt.xlabel('Time')
plt.title('Mean Sea Level Pressure at Hanle')
plt.hlines(y=P_b, xmin=hanle_surface_pressure.time.min(), xmax=hanle_surface_pressure.time.max(),
           colors='red', linestyles='dashed', label='Pressure Threshold')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(hanle_sst.time, hanle_sst, label='Sea Surface Temperature (SST)', color='red')
plt.ylabel('Temperature (K)')
plt.xlabel('Time')
plt.title('Sea Surface Temperature at Hanle')
plt.legend()
plt.grid()
plt.show()


### Step 36

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np

# --- Instructions ---
# This script assumes your 'surface_pressure_data' xarray.Dataset is already loaded.

# --- Site-Specific Parameters for Hanle ---
hanle_lat = 32.78
hanle_lon = 78.96
hanle_elevation = 4500  # Elevation in meters

# --- Constants ---
L_b = 0.0065  # Standard temperature lapse rate (K/m)

# --- Create a dummy xarray Dataset for demonstration ---
# This section ensures the script is runnable, but it will use your real data
# if 'surface_pressure_data' is already loaded in your environment.
# if 'surface_pressure_data' not in locals():
#     print("Creating a dummy 'surface_pressure_data' dataset for demonstration.")
#     time_dummy = np.arange('1998-01-01', '2025-06-01', dtype='datetime64[M]')
#     latitude_dummy = np.linspace(90, -90, 721)
#     longitude_dummy = np.linspace(0, 359.75, 1440)
#     # Simulate a realistic temperature range for a high-altitude location
#     t2m_dummy_data = 260 + 15 * np.sin(np.arange(len(time_dummy)) * 2 * np.pi / 12)[:, np.newaxis, np.newaxis] + np.random.randn(len(time_dummy), len(latitude_dummy), len(longitude_dummy))
    
#     surface_pressure_data = xr.Dataset(
#         {'t2m': (('time', 'latitude', 'longitude'), t2m_dummy_data.astype(np.float32))},
#         coords={'time': time_dummy, 'latitude': latitude_dummy, 'longitude': longitude_dummy}
#     )
# --- End of dummy data creation ---


try:
    # 1. Extract the 2m temperature (t2m) time series for Hanle
    t2m_hanle = surface_pressure_data['t2m'].interp(
        latitude=hanle_lat, 
        longitude=hanle_lon, 
        method='linear'
    )
    # Convert to Celsius for more intuitive plotting
    t2m_hanle_c = t2m_hanle - 273.15

    # 2. Calculate the Sea Level Temperature (Tb)
    # Formula: Tb = t2m + L_b * elevation
    Tb_hanle = t2m_hanle + L_b * hanle_elevation
    # Convert to Celsius for plotting
    Tb_hanle_c = Tb_hanle - 273.15

    # 3. Create the plot
    plt.figure(figsize=(15, 7))
    
    # Plot the actual 2m temperature at Hanle
    plt.plot(t2m_hanle.time, t2m_hanle_c, label=f'Actual 2m Temperature at Hanle ({hanle_elevation}m)', color='dodgerblue', linewidth=2)
    
    # Plot the calculated sea-level temperature
    plt.plot(Tb_hanle.time, Tb_hanle_c, label='Calculated Sea Level Temperature (Tb)', color='red', linestyle='--', linewidth=2)
    
    # 4. Add plot details for clarity
    plt.title('Actual vs. Calculated Sea Level Temperature for Hanle', fontsize=16)
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('Temperature (°C)', fontsize=12)
    plt.legend(fontsize=11)
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    
    # Show the temperature difference
    temp_diff = L_b * hanle_elevation
    print(f"Elevation of Hanle: {hanle_elevation} m")
    print(f"Lapse Rate (L_b): {L_b} K/m")
    print(f"Calculated Temperature Offset: {temp_diff:.2f} °C")
    
    plt.show()

except Exception as e:
    print(f"An error occurred: {e}")
    print("Please ensure your 'surface_pressure_data' dataset is loaded and contains the 't2m' variable.")


## Analysis workflow

### Step 37

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
world_pressure_t2m


## Site definitions and metadata

### Step 38

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
site_details


### Global Comparison

## Project setup and imports

### Step 39

This cell defines reusable helper function(s) `_ensure_time_dim`, `debug_calc`, `wrapper` so later sections can apply the same processing logic consistently.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  Robust debug decorator for calculate_pwv_pressure
# ────────────────────────────────────────────────────────────────────────────
from functools import wraps
import numpy as np
import xarray as xr

def _ensure_time_dim(arr: xr.DataArray, t):
    """Return arr that *has* a time dimension; re-expand if necessary."""
    if 'time' in arr.dims:
        return arr
    # convert 0-D → 1-D with the requested timestamp
    return arr.expand_dims({'time': [t]})

def debug_calc(func):
    @wraps(func)
    def wrapper(q, levels_imp, p_sfc, *a, **kw):
        try:
            return func(q, levels_imp, p_sfc, *a, **kw)
        except ValueError as e:
            print("\n─── PWV DEBUG REPORT ───────────────────────────────────")
            print("Original ValueError :", e, "\n")

            bad_ts = []
            for t in q.time.values:
                q1      = _ensure_time_dim(q.sel(time=t, drop=False), t)
                p_sfc1  = _ensure_time_dim(p_sfc.sel(time=t, drop=False), t)
                try:
                    func(q1, levels_imp, p_sfc1, *a, **kw)
                except ValueError:
                    bad_ts.append(t)

            print(f"Total timesteps examined : {q.sizes['time']}")
            print(f"Problematic timesteps    : {len(bad_ts)}")
            if bad_ts:
                sample = bad_ts[:5]
                print("Sample bad dates        :",
                      [np.datetime_as_string(x, unit='s') for x in sample])

                # Detailed snapshot of the first failing date
                t0 = bad_ts[0]
                q0     = q.sel(time=t0, drop=False)
                p0     = p_sfc.sel(time=t0, drop=False)
                print("\nDetailed inspection for", t0)
                print("  q  levels :", list(q0['level'].values))
                if 'level' in p0.dims:
                    print("  p_sfc levels :", list(p0['level'].values))
                else:
                    print("  p_sfc has *no* level dimension")

            print("────────────────────────────────────────────────────────\n")
            raise                  # comment out to let the script keep running
    return wrapper

# Attach the decorator once
calculate_pwv_pressure = debug_calc(calculate_pwv_pressure)


## Analysis workflow

### Step 40

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
surface_pressure_data


## Interpolation and extraction

### Step 41

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
# print the median surface pressure for each site from the time period of 2010 to April 2025 using sp variable in surface_pressure_data
for index, row in site_details.iterrows():
    lat,lon = row['lat'], row['lon']
    site_name = row['Site Name']
    
    sp_median = surface_pressure_data['sp'].interp(latitude=lat, longitude=lon, method='linear').sel(time=slice('2010-01-01', '2025-04-30')).median().values
    print(f"Median surface pressure for {site_name} ({lat}, {lon}) from 2010 to April 2025 is {sp_median:.2f} hPa.")


## Site definitions and metadata

### Step 42

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
# now print the same for indian sites

indian_sites = [HANLE,MERAK,SITE_A,SITE_B]

for site in indian_sites:
    lat, lon = site['lat'], site['lon']
    site_name = site['name']
    
    sp_median = surface_pressure_data['sp'].interp(latitude=lat, longitude=lon, method ='linear').sel(time=slice('2010-01-01', '2025-04-30')).median().values
    print(f"Median surface pressure for {site_name} ({lat}, {lon}) from 2010 to April 2025 is {sp_median/100:.6f} hPa.")


## Pressure and PWV calculations

### Step 43

This cell defines reusable helper function(s) `calculate_pwv_vectorized`, `integrate_profile`, `_init_plot`, `_rename_valid_time` so later sections can apply the same processing logic consistently.

In [ ]:
#!/usr/bin/env python3
# ──────────────────────────────────────────────────────────────────────────────
# PWV comparison – two processing variants (Refactored for robustness)
#
# Variant A (“dynamic”) : barometric-formula surface pressure for *every* site
# Variant B (“static”)  : ERA5 surface-pressure field for *every* site, with
#                         special handling for IAO-Hanle using observed data.
#
# External objects expected in the current namespace
#   • site_details            – pandas DataFrame loaded from sites.xlsx
#   • combined_dataset        – xarray cube with specific humidity on p-levels
#   • world_pressure_t2m      – xarray cube with 2-m temperature
#   • surface_pressure_data   – xarray cube with ERA5 surface pressure
#   • calculate_pressure(...) – helper for barometric formula
#   • constants  P_b, L_b, h_b
# ──────────────────────────────────────────────────────────────────────────────

from __future__ import annotations
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator

# ╭──────────────── GLOBAL MATPLOTLIB / LaTeX STYLE ─────────────────────────╮
plt.rcParams.update({
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{amsmath}",   # so \text{} works
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "font.size": 20,
    "axes.labelsize":   32,
    "axes.linewidth":    2.8,
    "xtick.labelsize":  32,
    "ytick.labelsize":  32,
    "xtick.major.size": 12,  "xtick.minor.size":  8,
    "ytick.major.size": 12,  "ytick.minor.size":  8,
    "xtick.major.width": 2.5, "xtick.minor.width": 1.0,
    "ytick.major.width": 2.5, "ytick.minor.width": 1.0,
    "legend.fontsize":  22,
})

_MARKERS = ['s', 'p', 'P', '*', 'X', 'D', 'o']
_COLORS  = ['black', 'orange', 'green', 'blue', 'red', 'cyan', 'brown']


# ╭──────────────── 1. VECTORIZED PWV CALCULATION ───────────────────────────╮
def calculate_pwv_vectorized(q: xr.DataArray, p_sfc: xr.DataArray) -> xr.DataArray:
    """Calculates Precipitable Water Vapor (PWV) using a vectorized approach."""
    g = 9.81  # gravity in m/s^2

    p_levels_pa = q.level * 100
    p_levels_pa.attrs['units'] = 'Pa'

    def integrate_profile(q_profile, p_profile, sfc_p_value):
        finite_mask = np.isfinite(q_profile)
        if finite_mask.sum() < 2: return np.nan

        p, q_ = p_profile[finite_mask], q_profile[finite_mask]
        sort_idx = np.argsort(p)
        p, q_ = p[sort_idx], q_[sort_idx]

        valid_levels = p <= sfc_p_value
        p, q_ = p[valid_levels], q_[valid_levels]
        if len(p) < 2: return np.nan

        interpolator = PchipInterpolator(p, q_, extrapolate=True)
        q_sfc = interpolator(sfc_p_value)

        p_full = np.append(p, sfc_p_value)
        q_full = np.append(q_, q_sfc)

        sort_idx_full = np.argsort(p_full)
        p_full, q_full = p_full[sort_idx_full], q_full[sort_idx_full]

        return np.trapz(q_full, p_full) / g

    pwv = xr.apply_ufunc(
        integrate_profile, q, p_levels_pa, p_sfc,
        input_core_dims=[['level'], ['level'], []],
        output_core_dims=[[]], vectorize=True, dask="parallelized", output_dtypes=[q.dtype]
    )

    pwv.attrs.update(units="mm", long_name="Precipitable Water Vapor")
    return pwv


# ╭──────────────── 2. PLOT SETUP HELPER ────────────────────────────────────╮
def _init_plot(title: str) -> None:
    plt.figure(figsize=(16, 8))
    ax = plt.gca()
    ax.axhline(y=1, color='black', linestyle='--', linewidth=3, alpha=0.8, zorder=0)
    ax.set_xlabel(r'Year',  fontsize=40, labelpad=10)
    ax.set_ylabel(r'Precipitable Water Vapor (mm)', fontsize=30)
    ax.set_ylim(-0.3, 12.25)
    ax.minorticks_on()
    ax.set_facecolor('white')
    for spine in ax.spines.values():
        spine.set_linewidth(2.5)
        spine.set_edgecolor('black')
    ax.tick_params(axis='both', which='both', direction='in', top=True, right=True, width=2, length=8, labelsize=32)


# ╭──────────────── 3. SURFACE-PRESSURE CALCULATORS ─────────────────────────╮
def _rename_valid_time(arr: xr.DataArray | xr.Dataset):
    return arr.rename({'valid_time': 'time'}) if 'valid_time' in arr.dims else arr

def pressure_dynamic(lat: float, lon: float, elev: float) -> xr.DataArray:
    T = world_pressure_t2m['t2m'].interp(latitude=lat, longitude=lon)
    # T = xr.full_like(T, fill_value=288.15)  # Set a default temperature
    # Assuming calculate_pressure is defined elsewhere
    p_vals = calculate_pressure(P_b, T, L_b, elev, h_b)
    return _rename_valid_time(xr.DataArray(p_vals, coords=T.coords, dims=T.dims, name='sp'))

def pressure_static(lat: float, lon: float, elev: float) -> xr.DataArray:
    p_da = surface_pressure_data['sp'].interp(latitude=lat, longitude=lon)
    return _rename_valid_time(p_da)


# ╭──────────────── 4. LOAD HANLE OBSERVATIONAL DATA ────────────────────────╮
# Load the surface pressure data for Hanle from the provided CSV file.
try:
    hanle_df = pd.read_csv('hanle_monthly_surface_data.csv')

    # Create the average pressure (Day + Night) / 2
    # The pressure is in hPa, so multiply by 100 to convert to Pascals (Pa)
    hanle_df['avg_pressure_pa'] = hanle_df[['Pressure_Day_Mean', 'Pressure_Night_Mean']].mean(axis=1) * 100

    # Map month abbreviations to month numbers (1-12)
    month_map = {
        'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
        'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
    }
    hanle_df['month_num'] = hanle_df['Month'].map(month_map)

    # Create a lookup series (index=month_num, value=pressure_pa) for easy access
    hanle_monthly_pressure_lookup = hanle_df.set_index('month_num')['avg_pressure_pa']
    print("Successfully loaded and processed IAO-Hanle observational pressure data.")

except FileNotFoundError:
    print("Warning: 'hanle_monthly_surface_data.csv' not found. Will use ERA5 data for all sites.")
    hanle_monthly_pressure_lookup = None


# ── 5. PWV TIMESERIES HELPER (with re-chunking fix & Hanle modification) ────
def pwv_timeseries(row: pd.Series, pres_func) -> xr.DataArray:
    """Return PWV time-series, ensuring inputs are de-duplicated, aligned, and correctly chunked."""
    site = row['Site Name'].strip()
    lat, lon, elev = row['lat'], row['lon'], row['Elevation']
    print(f"Processing site: {site}...")

    # Get data for the site and time range
    q_site_raw = combined_dataset['q'].interp(latitude=lat, longitude=lon).sel(time=slice('2010-01-01', '2025-04-01'))

    # ▼▼▼ MODIFICATION FOR IAO-HANLE ▼▼▼
    # Check if this is the IAO-Hanle site AND we are using the 'static' pressure method
    # AND the Hanle data was loaded successfully.
    is_hanle_static_case = (
        site == 'IAO-Hanle' and
        pres_func.__name__ == 'pressure_static' and
        hanle_monthly_pressure_lookup is not None
    )

    if is_hanle_static_case:
        print("  → Using observed monthly surface pressure for IAO-Hanle.")
        # Get the month number for each timestamp in the humidity data
        time_coords = q_site_raw.time
        months = time_coords.dt.month
        # Map each month to its corresponding pressure value from our lookup table
        pressure_values_pa = months.to_series().map(hanle_monthly_pressure_lookup)
        # Create the surface pressure DataArray
        p_sfc_raw = xr.DataArray(
            pressure_values_pa.values,
            coords={'time': time_coords},
            dims=['time'],
            name='sp'
        )
    else:
        # Original behavior for all other sites or if Hanle file is missing
        p_sfc_raw = pres_func(lat, lon, elev)
    # ▲▲▲ END OF MODIFICATION ▲▲▲

    # De-duplicate both datasets to prevent alignment errors
    _, q_unique_indices = np.unique(q_site_raw['time'], return_index=True)
    q_site_unique = q_site_raw.isel(time=q_unique_indices)

    _, p_unique_indices = np.unique(p_sfc_raw['time'], return_index=True)
    p_sfc_unique = p_sfc_raw.isel(time=p_unique_indices)

    # Align the clean, de-duplicated datasets
    q_aligned, p_sfc_aligned = xr.align(q_site_unique, p_sfc_unique, join='inner')

    # Drop the 'expver' dimension if it exists
    if 'expver' in q_aligned.dims:
        q_aligned = q_aligned.squeeze('expver', drop=True)

    # --- FINAL FIX: Re-chunk the data for the calculation ---
    # The `apply_ufunc` requires the core dimension ('level') to be in a single chunk.
    # The `-1` tells xarray to combine all chunks along this dimension into one.
    print("  Rechunking data for vertical integration...")
    q_rechunked = q_aligned.chunk({"level": -1})
    # --------------------------------------------------------

    # Call the robust, vectorized calculation function
    return calculate_pwv_vectorized(q_rechunked, p_sfc_aligned)


# ╭──────────────── 6. MASTER PLOT ROUTINE ─────────────────────────────────╮
def make_plot(variant_name: str, pres_func) -> None:
    title_tex = rf'\textbf{{PWV comparison – {variant_name}}}'
    _init_plot(title_tex)

    for idx, row in site_details.iterrows():
        pwv = pwv_timeseries(row, pres_func)
        thin = (row['Site Name'].strip() == 'DomeA')

        plt.plot(
            pwv.time, pwv,
            label=row['Site Name'].strip().replace(' ', r'\,'),
            color=_COLORS[idx % len(_COLORS)],
            marker=_MARKERS[idx % len(_MARKERS)],
            markersize=4 if thin else 8,
            linewidth=1 if thin else 3,
            linestyle='-',
            markerfacecolor='white',
            markeredgewidth=2,
            markeredgecolor=_COLORS[idx % len(_COLORS)],
            alpha=0.75,
        )

    plt.legend(
        ncol=6, loc='lower center', bbox_to_anchor=(0.5, 1.01),
        fontsize=22, frameon=True,
        facecolor='white', edgecolor='black', framealpha=0.95,
        borderpad=0.3, columnspacing=1.0, handletextpad=0.4,
    )
    plt.tight_layout(rect=[0.04, 0.04, 1, 0.98])
    fname = f'pwv_{variant_name.lower().replace(" ", "_")}.pdf'
    plt.savefig(fname, dpi=500)
    print(f'→ Saved figure as {fname}')
    plt.show()


# ╭──────────────── 7. DRIVER ───────────────────────────────────────────────╮
if __name__ == "__main__":
    # Ensure all required external objects are loaded before running this.
    # For example:
    # site_details = pd.read_excel("sites.xlsx")
    # combined_dataset = xr.open_dataset(...)
    # ... etc.

    make_plot(r'Dynamic–pressure method', pressure_dynamic)   # Variant A
    #make_plot(r'Static–pressure method',  pressure_static)    # Variant B (with special Hanle case)


## Site definitions and metadata

### Step 44

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
site_details


### Step 45

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
# drop Merak site from the site_details DataFrame
site_details = site_details[site_details['Site Name'] != 'Merak']
site_details


#### Using Geopotential

## Pressure and PWV calculations

### Step 46

This cell defines reusable helper function(s) `calculate_pwv_vectorized`, `integrate_profile`, `_init_plot`, `_rename_valid_time` so later sections can apply the same processing logic consistently.

In [ ]:
#!/usr/bin/env python3
# ──────────────────────────────────────────────────────────────────────────────
# PWV comparison – three processing variants
#
# Variant A (“constant”)  : constant surface pressure per site obtained from
#                           ERA5 geopotential + site elevation
# Variant B (“dynamic”)   : barometric-formula surface pressure (world T2m)
# Variant C (“static”)    : ERA5 surface-pressure field, with Hanle override
#
# External objects expected in the current namespace
#   • geopotential_data       – xarray Dataset with variable 'z' [m² s⁻²]
#   • site_details            – pandas DataFrame loaded from sites.xlsx
#   • combined_dataset        – xarray cube with specific humidity on p-levels
#   • world_pressure_t2m      – xarray cube with 2-m temperature
#   • surface_pressure_data   – xarray cube with ERA5 surface pressure (sp)
#   • calculate_pressure(...) – helper for barometric formula
#   • constants  P_b, L_b, h_b
# ──────────────────────────────────────────────────────────────────────────────

from __future__ import annotations

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d, PchipInterpolator
from typing import Union

# ╭──────────────── GLOBAL MATPLOTLIB / LaTeX STYLE ─────────────────────────╮
plt.rcParams.update({
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{amsmath}",
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "font.size": 20,
    "axes.labelsize":   32,
    "axes.linewidth":    2.8,
    "xtick.labelsize":  32,
    "ytick.labelsize":  32,
    "xtick.major.size": 12,  "xtick.minor.size":  8,
    "ytick.major.size": 12,  "ytick.minor.size":  8,
    "xtick.major.width": 2.5, "xtick.minor.width": 1.0,
    "ytick.major.width": 2.5, "ytick.minor.width": 1.0,
    "legend.fontsize":  22,
})

_MARKERS = ['s', 'p', 'P', '*', 'X', 'D', 'o']
_COLORS  = ['black', 'orange', 'green', 'blue', 'red', 'cyan', 'brown']


# ╭──────────────── 1. VECTORIZED PWV CALCULATION ───────────────────────────╮
def calculate_pwv_vectorized(q: xr.DataArray, p_sfc: xr.DataArray) -> xr.DataArray:
    """Precipitable Water Vapour (PWV) by vertically integrating q/g."""
    g = 9.81  # m s⁻²
    p_levels_pa = q.level * 100  # hPa → Pa
    p_levels_pa.attrs['units'] = 'Pa'

    def integrate_profile(q_profile, p_profile, sfc_p_value):
        # remove NaNs
        finite_mask = np.isfinite(q_profile)
        if finite_mask.sum() < 2:
            return np.nan

        p, q_ = p_profile[finite_mask], q_profile[finite_mask]
        sort_idx = np.argsort(p)          # ensure monotonic
        p, q_ = p[sort_idx], q_[sort_idx]

        # keep only levels ≤ surface-pressure
        valid = p <= sfc_p_value
        p, q_ = p[valid], q_[valid]
        if p.size < 2:
            return np.nan

        # add synthetic level at p_sfc to close the column
        q_sfc = PchipInterpolator(p, q_, extrapolate=True)(sfc_p_value)
        p_full  = np.append(p,  sfc_p_value)
        q_full  = np.append(q_, q_sfc)
        sort    = np.argsort(p_full)
        return np.trapz(q_full[sort], p_full[sort]) / g  # kg m⁻² ≈ mm

    pwv = xr.apply_ufunc(
        integrate_profile, q, p_levels_pa, p_sfc,
        input_core_dims=[['level'], ['level'], []],
        output_core_dims=[[]],
        vectorize=True, dask='parallelized', output_dtypes=[q.dtype]
    )
    pwv.attrs.update(units='mm', long_name='Precipitable Water Vapour')
    return pwv


# ╭──────────────── 2. PLOT SETUP HELPER ────────────────────────────────────╮
def _init_plot(title: str) -> None:
    plt.figure(figsize=(16, 8))
    ax = plt.gca()
    ax.axhline(y=1, color='black', linestyle='--', linewidth=3, alpha=0.8, zorder=0)
    ax.set_xlabel(r'Year',  fontsize=40, labelpad=10)
    ax.set_ylabel(r'Precipitable Water Vapor (mm)', fontsize=30)
    ax.set_ylim(-0.3, 12.75)
    ax.minorticks_on()
    ax.set_facecolor('white')
    # set major and minor ticks
    for spine in ax.spines.values():
        spine.set_linewidth(2.5)
        spine.set_edgecolor('black')
    ax.tick_params(axis='both', which='both', direction='in', top=True,
                   right=True, width=2, length=8, labelsize=32)


# ╭──────────────── 3. SURFACE-PRESSURE CALCULATORS ─────────────────────────╮
def _rename_valid_time(arr: xr.DataArray | xr.Dataset):
    """ERA5 hindcast reanalyses sometimes carry 'valid_time' → rename to 'time'."""
    return arr.rename({'valid_time': 'time'}) if 'valid_time' in arr.dims else arr


def pressure_constant_from_geopotential(lat: float, lon: float, elev_m: float) -> float:
    """
    Return a *constant* surface-pressure (Pa) for the given site, obtained by:
       1. Interpolating the ERA5 geopotential field 'z' (m² s⁻²) to the
          site lat/lon and converting to geometric height (m).
       2. Interpolating the P(z) profile to the requested elevation.
    The first time slice nearest to 2020-08-01 is used as reference.
    """
    # 1. Extract height vs pressure profile at the site
    z_da = geopotential_data['z'].interp(latitude=lat, longitude=lon,
                                         method='linear') / 9.80665  # → metres
    z_profile = z_da.sel(time='2020-08-01', method='nearest')

    z_values = z_profile.values             # metres
    p_levels_hpa = z_profile['level'].values  # hPa
    

    # 2. Interpolate P(z) → P(elevation)
    interp_func = interp1d(z_values, p_levels_hpa,
                           bounds_error=False, fill_value='extrapolate')
    p_at_elev_pa = float(interp_func(elev_m)) * 100.0  # hPa → Pa
    
    print(f"Pressure at {lat:.2f}°N, {lon:.2f}°E, elev={elev_m}m: {p_at_elev_pa/100:.2f} hPa")
    return p_at_elev_pa


def pressure_dynamic(lat: float, lon: float, elev_m: float) -> xr.DataArray:
    """Barometric-formula surface pressure derived from 2-m temperature."""
    T = world_pressure_t2m['t2m'].interp(latitude=lat, longitude=lon)
    p_vals = calculate_pressure(P_b, T, L_b, elev_m, h_b)
    return _rename_valid_time(xr.DataArray(p_vals, coords=T.coords,
                                           dims=T.dims, name='sp'))


def pressure_static(lat: float, lon: float, elev_m: float) -> xr.DataArray:
    """ERA5 analysed surface-pressure field (sp)."""
    p_da = surface_pressure_data['sp'].interp(latitude=lat, longitude=lon)
    return _rename_valid_time(p_da)


# ╭──────────────── 4. HANLE OBSERVATIONAL PRESSURE ─────────────────────────╮
try:
    hanle_df = pd.read_csv('hanle_monthly_surface_data.csv')
    hanle_df['avg_pressure_pa'] = hanle_df[['Pressure_Day_Mean',
                                            'Pressure_Night_Mean']].mean(axis=1) * 100
    month_map = {m: i+1 for i, m in enumerate(
                 ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                  'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])}
    hanle_df['month_num'] = hanle_df['Month'].map(month_map)
    hanle_monthly_pressure_lookup = hanle_df.set_index('month_num')['avg_pressure_pa']
    print("✔ IAO-Hanle observational pressure loaded.")
except FileNotFoundError:
    print("⚠ 'hanle_monthly_surface_data.csv' not found → using ERA5 for Hanle.")
    hanle_monthly_pressure_lookup = None


# ╭──────────────── 5. PWV TIMESERIES HELPER ────────────────────────────────╮
def pwv_timeseries(row: pd.Series,
                   pres_func) -> xr.DataArray:
    """Return PWV time-series for a single site."""
    site = row['Site Name'].strip()
    lat, lon, elev = row['lat'], row['lon'], row['Elevation']
    print(f"Processing {site} …")

    q_site_raw = (combined_dataset['q']
                  .interp(latitude=lat, longitude=lon)
                  .sel(time=slice('2010-01-01', '2025-04-01')))

    # ── choose/construct surface-pressure series ───────────────────────────
    # 1. Hanle override (only when using the ERA5 static-pressure variant)
    is_hanle_static = (site == 'IAO-Hanle' and
                       pres_func.__name__ == 'pressure_static' and
                       hanle_monthly_pressure_lookup is not None)

    if is_hanle_static:
        print("   → Using observed monthly surface-pressure for Hanle.")
        months = q_site_raw.time.dt.month
        p_vals = months.to_series().map(hanle_monthly_pressure_lookup).values
        p_sfc_raw = xr.DataArray(p_vals, coords={'time': q_site_raw.time},
                                 dims=['time'], name='sp')

    else:
        # a) call the chosen pressure function
        p_out: Union[xr.DataArray, float] = pres_func(lat, lon, elev)

        # b) convert scalar → DataArray(time) for constant-pressure variant
        if not isinstance(p_out, xr.DataArray):
            p_sfc_raw = xr.DataArray(
                np.full(q_site_raw.time.size, p_out),
                coords={'time': q_site_raw.time},
                dims=['time'], name='sp'
            )
        else:
            p_sfc_raw = p_out

    # ── de-duplicate & align ───────────────────────────────────────────────
    q_unique = q_site_raw.isel(time=np.unique(q_site_raw['time'],
                                              return_index=True)[1])
    p_unique = p_sfc_raw.isel(time=np.unique(p_sfc_raw['time'],
                                             return_index=True)[1])
    q_aligned, p_aligned = xr.align(q_unique, p_unique, join='inner')

    if 'expver' in q_aligned.dims:
        q_aligned = q_aligned.squeeze('expver', drop=True)

    # combine all chunks along the vertical level dimension
    print("   ↳ rechunking humidity profile for vertical integration …")
    q_rechunked = q_aligned.chunk({'level': -1})

    return calculate_pwv_vectorized(q_rechunked, p_aligned)


# ╭──────────────── 6. MASTER PLOT ROUTINE ─────────────────────────────────╮
def make_plot(variant_name: str, pres_func) -> None:
    _init_plot(rf'\textbf{{PWV comparison – {variant_name}}}')
    for idx, row in site_details.iterrows():
        pwv = pwv_timeseries(row, pres_func)
        thin = (row['Site Name'].strip() == 'DomeA')
        plt.plot(
            pwv.time, pwv,
            label=row['Site Name'].strip().replace(' ', r'\,'),
            color=_COLORS[idx % len(_COLORS)],
            marker=_MARKERS[idx % len(_MARKERS)],
            markersize=4 if thin else 8,
            linewidth=1 if thin else 3,
            linestyle='-',
            markerfacecolor='white',
            markeredgewidth=2,
            markeredgecolor=_COLORS[idx % len(_COLORS)],
            alpha=0.75,
        )
    plt.legend(ncol=6, loc='lower center', bbox_to_anchor=(0.5, 1.01),
               fontsize=22, frameon=True, facecolor='white',
               edgecolor='black', framealpha=0.95, borderpad=0.3,
               columnspacing=1.0, handletextpad=0.4)
    # set minoor ticks and adjust layout
    plt.minorticks_on()
    plt.tight_layout(rect=[0.04, 0.04, 1, 0.98])
    fname = f"pwv_{variant_name.lower().replace(' ', '_')}.pdf"
    plt.savefig(fname, dpi=500)
    print(f"→ Saved {fname}")
    plt.show()


# ╭──────────────── 7. DRIVER ───────────────────────────────────────────────╮
if __name__ == "__main__":
    # Load *all* required external objects here before running:
    #   geopotential_data   = xr.open_dataset('geopotential.nc')  (rename dims)
    #   site_details        = pd.read_excel('sites.xlsx')
    #   combined_dataset    = xr.open_dataset('specific_humidity.nc')
    #   world_pressure_t2m  = xr.open_dataset('t2m.nc')
    #   surface_pressure_data = xr.open_dataset('surface_pressure.nc')
    #   constants           = P_b, L_b, h_b, & calculate_pressure helper
    #
    # ── Variant A: constant pressure from geopotential heights
    make_plot(r'Constant–pressure method',
              pressure_constant_from_geopotential)

    # Uncomment to retain the previous variants for comparison:
    # make_plot(r'Dynamic–pressure method', pressure_dynamic)
    # make_plot(r'Static–pressure method',  pressure_static)


### Step 47

This cell defines reusable helper function(s) `calculate_pwv_vectorized`, `integrate_profile`, `_init_plot`, `_rename_valid_time` so later sections can apply the same processing logic consistently.

In [ ]:
#!/usr/bin/env python3
# ──────────────────────────────────────────────────────────────────────────────
# PWV comparison – three processing variants
#
# CHANGE MADE:
#   - After computing PWV for each site, save the PWV time-series to disk
#     (both NetCDF and CSV) so future runs can skip computation and just plot.
#
# Output files:
#   pwv_outputs/pwv_<variant_slug>.nc            (all sites, compact, best)
#   pwv_outputs/pwv_<variant_slug>_<site>.csv    (per-site, easy to inspect)
#   pwv_outputs/pwv_<variant_slug>.pdf           (plot)
# ──────────────────────────────────────────────────────────────────────────────

from __future__ import annotations

import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d, PchipInterpolator
from typing import Union

# ╭──────────────── GLOBAL MATPLOTLIB / LaTeX STYLE ─────────────────────────╮
plt.rcParams.update({
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{amsmath}",
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "font.size": 20,
    "axes.labelsize":   32,
    "axes.linewidth":    2.8,
    "xtick.labelsize":  32,
    "ytick.labelsize":  32,
    "xtick.major.size": 12,  "xtick.minor.size":  8,
    "ytick.major.size": 12,  "ytick.minor.size":  8,
    "xtick.major.width": 2.5, "xtick.minor.width": 1.0,
    "ytick.major.width": 2.5, "ytick.minor.width": 1.0,
    "legend.fontsize":  22,
})

_MARKERS = ['s', 'p', 'P', '*', 'X', 'D', 'o']
_COLORS  = ['black', 'orange', 'green', 'blue', 'red', 'cyan', 'brown']

# ╭──────────────── OUTPUT SETTINGS ─────────────────────────────────────────╮
OUTDIR = "pwv_outputs"
os.makedirs(OUTDIR, exist_ok=True)

SAVE_PER_SITE_CSV = True   # set False if you only want the single NetCDF


# ╭──────────────── 1. VECTORIZED PWV CALCULATION ───────────────────────────╮
def calculate_pwv_vectorized(q: xr.DataArray, p_sfc: xr.DataArray) -> xr.DataArray:
    """Precipitable Water Vapour (PWV) by vertically integrating q/g."""
    g = 9.81  # m s⁻²
    p_levels_pa = q.level * 100  # hPa → Pa
    p_levels_pa.attrs['units'] = 'Pa'

    def integrate_profile(q_profile, p_profile, sfc_p_value):
        finite_mask = np.isfinite(q_profile)
        if finite_mask.sum() < 2:
            return np.nan

        p, q_ = p_profile[finite_mask], q_profile[finite_mask]
        sort_idx = np.argsort(p)
        p, q_ = p[sort_idx], q_[sort_idx]

        valid = p <= sfc_p_value
        p, q_ = p[valid], q_[valid]
        if p.size < 2:
            return np.nan

        q_sfc = PchipInterpolator(p, q_, extrapolate=True)(sfc_p_value)
        p_full = np.append(p, sfc_p_value)
        q_full = np.append(q_, q_sfc)
        sort = np.argsort(p_full)
        return np.trapz(q_full[sort], p_full[sort]) / g

    pwv = xr.apply_ufunc(
        integrate_profile, q, p_levels_pa, p_sfc,
        input_core_dims=[['level'], ['level'], []],
        output_core_dims=[[]],
        vectorize=True, dask='parallelized', output_dtypes=[q.dtype]
    )
    pwv.attrs.update(units='mm', long_name='Precipitable Water Vapour')
    return pwv


# ╭──────────────── 2. PLOT SETUP HELPER ────────────────────────────────────╮
def _init_plot(title: str) -> None:
    plt.figure(figsize=(16, 8))
    ax = plt.gca()
    ax.axhline(y=1, color='black', linestyle='--', linewidth=3, alpha=0.8, zorder=0)
    ax.set_xlabel(r'Year',  fontsize=40, labelpad=10)
    ax.set_ylabel(r'Precipitable Water Vapor (mm)', fontsize=30)
    ax.set_ylim(-0.3, 12.75)
    ax.minorticks_on()
    ax.set_facecolor('white')
    for spine in ax.spines.values():
        spine.set_linewidth(2.5)
        spine.set_edgecolor('black')
    ax.tick_params(axis='both', which='both', direction='in', top=True,
                   right=True, width=2, length=8, labelsize=32)


# ╭──────────────── 3. SURFACE-PRESSURE CALCULATORS ─────────────────────────╮
def _rename_valid_time(arr: xr.DataArray | xr.Dataset):
    return arr.rename({'valid_time': 'time'}) if 'valid_time' in arr.dims else arr


def pressure_constant_from_geopotential(lat: float, lon: float, elev_m: float) -> float:
    z_da = geopotential_data['z'].interp(latitude=lat, longitude=lon, method='linear') / 9.80665
    z_profile = z_da.sel(time='2020-08-01', method='nearest')
    z_values = z_profile.values
    p_levels_hpa = z_profile['level'].values
    interp_func = interp1d(z_values, p_levels_hpa, bounds_error=False, fill_value='extrapolate')
    p_at_elev_pa = float(interp_func(elev_m)) * 100.0
    print(f"Pressure at {lat:.2f}°N, {lon:.2f}°E, elev={elev_m}m: {p_at_elev_pa/100:.2f} hPa")
    return p_at_elev_pa


def pressure_dynamic(lat: float, lon: float, elev_m: float) -> xr.DataArray:
    T = world_pressure_t2m['t2m'].interp(latitude=lat, longitude=lon)
    p_vals = calculate_pressure(P_b, T, L_b, elev_m, h_b)
    return _rename_valid_time(xr.DataArray(p_vals, coords=T.coords, dims=T.dims, name='sp'))


def pressure_static(lat: float, lon: float, elev_m: float) -> xr.DataArray:
    p_da = surface_pressure_data['sp'].interp(latitude=lat, longitude=lon)
    return _rename_valid_time(p_da)


# ╭──────────────── 4. HANLE OBSERVATIONAL PRESSURE ─────────────────────────╮
try:
    hanle_df = pd.read_csv('hanle_monthly_surface_data.csv')
    hanle_df['avg_pressure_pa'] = hanle_df[['Pressure_Day_Mean', 'Pressure_Night_Mean']].mean(axis=1) * 100
    month_map = {m: i+1 for i, m in enumerate(
                 ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                  'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])}
    hanle_df['month_num'] = hanle_df['Month'].map(month_map)
    hanle_monthly_pressure_lookup = hanle_df.set_index('month_num')['avg_pressure_pa']
    print("✔ IAO-Hanle observational pressure loaded.")
except FileNotFoundError:
    print("⚠ 'hanle_monthly_surface_data.csv' not found → using ERA5 for Hanle.")
    hanle_monthly_pressure_lookup = None


# ╭──────────────── 5. PWV TIMESERIES HELPER ────────────────────────────────╮
def pwv_timeseries(row: pd.Series, pres_func) -> xr.DataArray:
    site = row['Site Name'].strip()
    lat, lon, elev = row['lat'], row['lon'], row['Elevation']
    print(f"Processing {site} …")

    q_site_raw = (combined_dataset['q']
                  .interp(latitude=lat, longitude=lon)
                  .sel(time=slice('2010-01-01', '2025-04-01')))

    is_hanle_static = (site == 'IAO-Hanle' and
                       pres_func.__name__ == 'pressure_static' and
                       hanle_monthly_pressure_lookup is not None)

    if is_hanle_static:
        print("   → Using observed monthly surface-pressure for Hanle.")
        months = q_site_raw.time.dt.month
        p_vals = months.to_series().map(hanle_monthly_pressure_lookup).values
        p_sfc_raw = xr.DataArray(p_vals, coords={'time': q_site_raw.time},
                                 dims=['time'], name='sp')
    else:
        p_out: Union[xr.DataArray, float] = pres_func(lat, lon, elev)
        if not isinstance(p_out, xr.DataArray):
            p_sfc_raw = xr.DataArray(
                np.full(q_site_raw.time.size, p_out),
                coords={'time': q_site_raw.time},
                dims=['time'], name='sp'
            )
        else:
            p_sfc_raw = p_out

    q_unique = q_site_raw.isel(time=np.unique(q_site_raw['time'], return_index=True)[1])
    p_unique = p_sfc_raw.isel(time=np.unique(p_sfc_raw['time'], return_index=True)[1])
    q_aligned, p_aligned = xr.align(q_unique, p_unique, join='inner')

    if 'expver' in q_aligned.dims:
        q_aligned = q_aligned.squeeze('expver', drop=True)

    print("   ↳ rechunking humidity profile for vertical integration …")
    q_rechunked = q_aligned.chunk({'level': -1})

    return calculate_pwv_vectorized(q_rechunked, p_aligned)


# ╭──────────────── 6. SAVING HELPERS ───────────────────────────────────────╮
def _slugify(s: str) -> str:
    return (s.strip()
            .lower()
            .replace("–", "-")
            .replace("—", "-")
            .replace(" ", "_")
            .replace("/", "_"))

def save_site_csv(pwv: xr.DataArray, site_name: str, variant_slug: str) -> None:
    if not SAVE_PER_SITE_CSV:
        return
    site_slug = _slugify(site_name)
    path = os.path.join(OUTDIR, f"pwv_{variant_slug}_{site_slug}.csv")
    df = pd.DataFrame({"time": pd.to_datetime(pwv["time"].values), "pwv_mm": pwv.values})
    df.to_csv(path, index=False)
    print(f"→ Saved {path}")


# ╭──────────────── 7. MASTER ROUTINES: COMPUTE+SAVE OR LOAD+PLOT ───────────╮
def compute_and_save_variant(variant_name: str, pres_func) -> xr.Dataset:
    """
    Computes PWV for all sites and saves a single NetCDF (best for reload).
    Also optionally writes per-site CSVs.
    """
    variant_slug = _slugify(variant_name)
    out_nc = os.path.join(OUTDIR, f"pwv_{variant_slug}.nc")

    pwv_list = []
    site_names = []

    for _, row in site_details.iterrows():
        site = row["Site Name"].strip()
        site_names.append(site)

        pwv = pwv_timeseries(row, pres_func)

        # Ensure actual numeric arrays before saving (avoid lazy object in CSV)
        if hasattr(pwv.data, "compute"):
            pwv = pwv.compute()

        save_site_csv(pwv, site, variant_slug)

        pwv = pwv.assign_coords(site=site).expand_dims("site")
        pwv_list.append(pwv)

    pwv_all = xr.concat(pwv_list, dim="site").rename("pwv")  # (site, time)
    ds_out = xr.Dataset({"pwv": pwv_all})
    ds_out.attrs["variant_name"] = variant_name
    ds_out.attrs["pressure_method"] = pres_func.__name__

    # Save a single NetCDF that can be reopened quickly
    encoding = {"pwv": {"zlib": True, "complevel": 4}}
    ds_out.to_netcdf(out_nc, encoding=encoding)
    print(f"→ Saved {out_nc}")
    return ds_out


def load_variant_dataset(variant_name: str) -> xr.Dataset:
    variant_slug = _slugify(variant_name)
    out_nc = os.path.join(OUTDIR, f"pwv_{variant_slug}.nc")
    if not os.path.exists(out_nc):
        raise FileNotFoundError(f"Saved PWV file not found: {out_nc}")
    print(f"→ Loading {out_nc}")
    return xr.open_dataset(out_nc)


# ╭──────────────── 8. PLOTTING FROM SAVED DATASET ─────────────────────────╮
def plot_from_dataset(ds: xr.Dataset, variant_name: str) -> None:
    _init_plot(rf'\textbf{{PWV comparison – {variant_name}}}')
    pwv = ds["pwv"]  # (site, time)

    for idx, site in enumerate(pwv["site"].values.tolist()):
        site_str = str(site).strip()
        thin = (site_str == "DomeA")
        series = pwv.sel(site=site)

        plt.plot(
            series["time"].values, series.values,
            label=site_str.replace(" ", r"\,"),
            color=_COLORS[idx % len(_COLORS)],
            marker=_MARKERS[idx % len(_MARKERS)],
            markersize=4 if thin else 8,
            linewidth=1 if thin else 3,
            linestyle='-',
            markerfacecolor='white',
            markeredgewidth=2,
            markeredgecolor=_COLORS[idx % len(_COLORS)],
            alpha=0.75,
        )

    plt.legend(ncol=6, loc='lower center', bbox_to_anchor=(0.5, 1.01),
               fontsize=22, frameon=True, facecolor='white',
               edgecolor='black', framealpha=0.95, borderpad=0.3,
               columnspacing=1.0, handletextpad=0.4)
    plt.minorticks_on()
    plt.tight_layout(rect=[0.04, 0.04, 1, 0.98])
    fname = os.path.join(OUTDIR, f"pwv_{_slugify(variant_name)}.pdf")
    plt.savefig(fname, dpi=500)
    print(f"→ Saved {fname}")
    plt.show()


# ╭──────────────── 9. DRIVER ───────────────────────────────────────────────╮
if __name__ == "__main__":
    # This assumes you already loaded:
    #   geopotential_data, site_details, combined_dataset, world_pressure_t2m,
    #   surface_pressure_data, calculate_pressure, P_b, L_b, h_b

    VARIANT_NAME = r'Constant–pressure method'
    PRES_FUNC = pressure_constant_from_geopotential

    # If you want to recompute and overwrite saved outputs, set this True
    RECOMPUTE = True

    if RECOMPUTE:
        ds = compute_and_save_variant(VARIANT_NAME, PRES_FUNC)
    else:
        ds = load_variant_dataset(VARIANT_NAME)

    plot_from_dataset(ds, VARIANT_NAME)


### Step 48

This cell defines reusable helper function(s) `calculate_pwv_vectorized`, `integrate_profile` so later sections can apply the same processing logic consistently.

In [ ]:
# ╭──────────────── 1. PWV CALCULATOR (vectorised) ──────────────────────────╮
def calculate_pwv_vectorized(q: xr.DataArray,
                             p_sfc: xr.DataArray) -> xr.DataArray:
    """Column-integrated PWV (mm)."""
    g = 9.81
    p_levels_pa = q.level * 100  # hPa → Pa

    def integrate_profile(q_prof, p_prof, p_sfc_val):
        m = np.isfinite(q_prof)
        if m.sum() < 2:
            return np.nan

        p, qv = p_prof[m], q_prof[m]
        order = np.argsort(p)
        p, qv = p[order], qv[order]

        valid = p <= p_sfc_val
        p, qv = p[valid], qv[valid]
        if p.size < 2:
            return np.nan

        q_sfc = PchipInterpolator(p, qv, extrapolate=True)(p_sfc_val)
        p_full = np.append(p, p_sfc_val)
        q_full = np.append(qv, q_sfc)
        sorter = np.argsort(p_full)
        return np.trapz(q_full[sorter], p_full[sorter]) / g  # kg m⁻² ≈ mm

    return xr.apply_ufunc(
        integrate_profile,
        q, p_levels_pa, p_sfc,
        input_core_dims=[['level'], ['level'], []],
        output_core_dims=[[]],
        vectorize=True, dask='parallelized', output_dtypes=[q.dtype]
    )


### Step 49

This cell defines reusable helper function(s) `_prepare_pressure_sort_from_levels`, `calculate_pwv_vectorized`, `integrate_profile`, `_rename_valid_time` so later sections can apply the same processing logic consistently.

In [ ]:
#!/usr/bin/env python3
from __future__ import annotations

import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d, PchipInterpolator
from typing import Union, Optional

# ───────────────────────────── SPEED / CACHE SETTINGS ─────────────────────────────
TIME_CHUNK = 256                  # 128/256/512/1024 depending on RAM
CACHE_DIR = "pwv_cache"           # folder created in current working directory
CACHE_FORMAT = "zarr"             # "zarr" (fastest) or "netcdf"
os.makedirs(CACHE_DIR, exist_ok=True)

_MARKERS = ['s', 'p', 'P', '*', 'X', 'D', 'o']
_COLORS  = ['black', 'orange', 'green', 'blue', 'red', 'cyan', 'brown']

# ───────────────────────────── PRECOMPUTED PRESSURE SORT ─────────────────────────
# These are computed once from q.level and reused for every profile.
_P_SORT_IDX: Optional[np.ndarray] = None
_P_LEVELS_PA_SORTED: Optional[np.ndarray] = None

def _prepare_pressure_sort_from_levels(level_hpa: xr.DataArray) -> None:
    global _P_SORT_IDX, _P_LEVELS_PA_SORTED
    p_levels_pa = (level_hpa.values.astype(np.float64) * 100.0)  # hPa → Pa
    sort_idx = np.argsort(p_levels_pa)                            # monotonic increasing
    _P_SORT_IDX = sort_idx
    _P_LEVELS_PA_SORTED = p_levels_pa[sort_idx]

# ───────────────────────────── 1) PWV (SAME ALGORITHM) ───────────────────────────
def calculate_pwv_vectorized(q: xr.DataArray, p_sfc: xr.DataArray) -> xr.DataArray:
    """
    Same algorithm as your original:
      - remove NaNs
      - sort by pressure
      - keep p <= p_sfc
      - PCHIP to get q(p_sfc)
      - trapezoid integrate q dp / g
    Speedups:
      - pressure sort is precomputed once (no per-profile argsort)
      - no resort after appending p_sfc (p_sfc is max after filtering)
      - dask parallelization works when time is chunked
    """
    g = 9.81

    if _P_SORT_IDX is None or _P_LEVELS_PA_SORTED is None:
        _prepare_pressure_sort_from_levels(q["level"])

    sort_idx = _P_SORT_IDX
    p_sorted = _P_LEVELS_PA_SORTED  # (level,) Pa

    def integrate_profile(q_profile, sfc_p_value):
        # apply the same sorting every time (precomputed)
        q_sorted = q_profile[sort_idx]

        finite_mask = np.isfinite(q_sorted)
        if finite_mask.sum() < 2:
            return np.nan

        p = p_sorted[finite_mask]
        qv = q_sorted[finite_mask].astype(np.float64)

        valid = p <= sfc_p_value
        p = p[valid]
        qv = qv[valid]
        if p.size < 2:
            return np.nan

        q_sfc = PchipInterpolator(p, qv, extrapolate=True)(sfc_p_value)

        # p is increasing; since we filtered p <= sfc_p_value, sfc_p_value is the max
        p_full = np.concatenate([p, [sfc_p_value]])
        q_full = np.concatenate([qv, [q_sfc]])

        return np.trapz(q_full, p_full) / g

    pwv = xr.apply_ufunc(
        integrate_profile,
        q,
        p_sfc,
        input_core_dims=[["level"], []],
        output_core_dims=[[]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[np.float64],
    )
    pwv = pwv.astype("float32")
    pwv.attrs.update(units="mm", long_name="Precipitable Water Vapour")
    return pwv

# ───────────────────────────── 2) small helpers ─────────────────────────────────
def _rename_valid_time(arr: xr.DataArray | xr.Dataset):
    return arr.rename({'valid_time': 'time'}) if 'valid_time' in arr.dims else arr

def _dedup_time_if_needed(da: xr.DataArray) -> xr.DataArray:
    idx = da.get_index("time")
    if idx.has_duplicates:
        mask = ~idx.duplicated()
        return da.isel(time=mask)
    return da

# ───────────────────────────── 3) pressure functions (unchanged) ─────────────────
def pressure_constant_from_geopotential(lat: float, lon: float, elev_m: float) -> float:
    z_da = geopotential_data['z'].interp(latitude=lat, longitude=lon, method='linear') / 9.80665
    z_profile = z_da.sel(time='2020-08-01', method='nearest')
    z_values = z_profile.values
    p_levels_hpa = z_profile['level'].values
    interp_func = interp1d(z_values, p_levels_hpa, bounds_error=False, fill_value='extrapolate')
    p_at_elev_pa = float(interp_func(elev_m)) * 100.0
    print(f"Pressure at {lat:.2f}°N, {lon:.2f}°E, elev={elev_m}m: {p_at_elev_pa/100:.2f} hPa")
    return p_at_elev_pa

def pressure_dynamic(lat: float, lon: float, elev_m: float) -> xr.DataArray:
    T = world_pressure_t2m['t2m'].interp(latitude=lat, longitude=lon)
    p_vals = calculate_pressure(P_b, T, L_b, elev_m, h_b)
    return _rename_valid_time(xr.DataArray(p_vals, coords=T.coords, dims=T.dims, name='sp'))

def pressure_static(lat: float, lon: float, elev_m: float) -> xr.DataArray:
    p_da = surface_pressure_data['sp'].interp(latitude=lat, longitude=lon)
    return _rename_valid_time(p_da)

# ───────────────────────────── 4) Hanle lookup (same semantics, faster) ──────────
try:
    hanle_df = pd.read_csv('hanle_monthly_surface_data.csv')
    hanle_df['avg_pressure_pa'] = hanle_df[['Pressure_Day_Mean', 'Pressure_Night_Mean']].mean(axis=1) * 100
    month_map = {m: i+1 for i, m in enumerate(
        ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    )}
    hanle_df['month_num'] = hanle_df['Month'].map(month_map)
    # array indexed by month-1
    _hanle_lookup_arr = np.full(12, np.nan, dtype=np.float64)
    for m, p in zip(hanle_df['month_num'].to_numpy(), hanle_df['avg_pressure_pa'].to_numpy()):
        if 1 <= int(m) <= 12:
            _hanle_lookup_arr[int(m)-1] = float(p)
    print("✔ IAO-Hanle observational pressure loaded.")
except FileNotFoundError:
    print("⚠ 'hanle_monthly_surface_data.csv' not found → using ERA5 for Hanle.")
    _hanle_lookup_arr = None

# ───────────────────────────── 5) PWV computation per site (FASTER) ─────────────
def pwv_timeseries(row: pd.Series, pres_func) -> xr.DataArray:
    site = row['Site Name'].strip()
    lat, lon, elev = float(row['lat']), float(row['lon']), float(row['Elevation'])
    print(f"Processing {site} …")

    q_site_raw = (
        combined_dataset['q']
        .interp(latitude=lat, longitude=lon)
        .sel(time=slice('2010-01-01', '2025-04-01'))
    )

    is_hanle_static = (
        site == 'IAO-Hanle'
        and pres_func.__name__ == 'pressure_static'
        and _hanle_lookup_arr is not None
    )

    if is_hanle_static:
        print("   → Using observed monthly surface-pressure for Hanle.")
        months = q_site_raw.time.dt.month.values.astype(np.int64)  # 1..12
        p_vals = _hanle_lookup_arr[months - 1]
        p_sfc_raw = xr.DataArray(p_vals, coords={'time': q_site_raw.time}, dims=['time'], name='sp')
    else:
        p_out: Union[xr.DataArray, float] = pres_func(lat, lon, elev)
        if not isinstance(p_out, xr.DataArray):
            p_sfc_raw = xr.DataArray(
                np.full(q_site_raw.time.size, float(p_out), dtype=np.float64),
                coords={'time': q_site_raw.time},
                dims=['time'], name='sp'
            )
        else:
            p_sfc_raw = p_out

    # dedup only if needed
    q_site_raw = _dedup_time_if_needed(q_site_raw)
    p_sfc_raw  = _dedup_time_if_needed(p_sfc_raw)

    q_aligned, p_aligned = xr.align(q_site_raw, p_sfc_raw, join='inner')

    if 'expver' in q_aligned.dims:
        q_aligned = q_aligned.squeeze('expver', drop=True)

    # IMPORTANT: time chunking enables parallel ufunc; level kept whole
    print("   ↳ rechunking humidity profile for vertical integration …")
    q_chunked = q_aligned.chunk({'time': TIME_CHUNK, 'level': -1})
    p_chunked = p_aligned.chunk({'time': TIME_CHUNK})

    return calculate_pwv_vectorized(q_chunked, p_chunked)

# ───────────────────────────── 6) CACHING (per site + variant) ────────────────
def _variant_key(pres_func) -> str:
    # stable key
    return pres_func.__name__

def _cache_path(site: str, pres_func) -> str:
    key = _variant_key(pres_func)
    safe_site = site.strip().replace(" ", "_").replace("/", "_")
    if CACHE_FORMAT == "zarr":
        return os.path.join(CACHE_DIR, f"pwv_{key}_{safe_site}.zarr")
    return os.path.join(CACHE_DIR, f"pwv_{key}_{safe_site}.nc")

def load_cached_site(site: str, pres_func) -> Optional[xr.DataArray]:
    path = _cache_path(site, pres_func)
    if CACHE_FORMAT == "zarr":
        if os.path.isdir(path):
            ds = xr.open_zarr(path)
            return ds["pwv"]
        return None
    if os.path.exists(path):
        ds = xr.open_dataset(path)
        return ds["pwv"]
    return None

def save_cached_site(site: str, pres_func, pwv: xr.DataArray) -> None:
    path = _cache_path(site, pres_func)
    ds = xr.Dataset({"pwv": pwv})
    ds.attrs["site"] = site.strip()
    ds.attrs["variant"] = _variant_key(pres_func)

    if CACHE_FORMAT == "zarr":
        ds.to_zarr(path, mode="w")
    else:
        encoding = {"pwv": {"zlib": True, "complevel": 4}}
        ds.to_netcdf(path, encoding=encoding)

# ───────────────────────────── 7) plotting unchanged (but compute once) ─────────
def _init_plot(title: str) -> None:
    plt.figure(figsize=(16, 8))
    ax = plt.gca()
    ax.axhline(y=1, color='black', linestyle='--', linewidth=3, alpha=0.8, zorder=0)
    ax.set_xlabel(r'Year',  fontsize=40, labelpad=10)
    ax.set_ylabel(r'Precipitable Water Vapor (mm)', fontsize=30)
    ax.set_ylim(-0.3, 12.75)
    ax.minorticks_on()
    ax.set_facecolor('white')
    for spine in ax.spines.values():
        spine.set_linewidth(2.5)
        spine.set_edgecolor('black')
    ax.tick_params(axis='both', which='both', direction='in', top=True,
                   right=True, width=2, length=8, labelsize=32)

def make_plot(variant_name: str, pres_func) -> None:
    # ensure sort is prepared once
    _prepare_pressure_sort_from_levels(combined_dataset["q"]["level"])

    _init_plot(rf'\textbf{{PWV comparison – {variant_name}}}')

    for idx, row in site_details.iterrows():
        site = row['Site Name'].strip()

        # load cache if present
        pwv = load_cached_site(site, pres_func)
        if pwv is None:
            pwv = pwv_timeseries(row, pres_func)

            # compute ONCE here (otherwise plotting triggers repeated/partial computation)
            if hasattr(pwv.data, "compute"):
                pwv = pwv.compute()

            save_cached_site(site, pres_func, pwv)
        else:
            # cached already computed
            pass

        thin = (site == 'DomeA')
        plt.plot(
            pwv.time, pwv,
            label=site.replace(' ', r'\,'),
            color=_COLORS[idx % len(_COLORS)],
            marker=_MARKERS[idx % len(_MARKERS)],
            markersize=4 if thin else 8,
            linewidth=1 if thin else 3,
            linestyle='-',
            markerfacecolor='white',
            markeredgewidth=2,
            markeredgecolor=_COLORS[idx % len(_COLORS)],
            alpha=0.75,
        )

    plt.legend(ncol=6, loc='lower center', bbox_to_anchor=(0.5, 1.01),
               fontsize=22, frameon=True, facecolor='white',
               edgecolor='black', framealpha=0.95, borderpad=0.3,
               columnspacing=1.0, handletextpad=0.4)
    plt.minorticks_on()
    plt.tight_layout(rect=[0.04, 0.04, 1, 0.98])
    fname = f"pwv_{variant_name.lower().replace(' ', '_')}.pdf"
    plt.savefig(fname, dpi=500)
    print(f"→ Saved {fname}")
    plt.show()

# ───────────────────────────── 8) driver (unchanged usage) ─────────────────────
if __name__ == "__main__":
    # IMPORTANT:
    # This script assumes YOU already loaded:
    #   geopotential_data, site_details, combined_dataset,
    #   world_pressure_t2m, surface_pressure_data, etc.
    #
    # If you run this as a standalone file, load them before calling make_plot().
    make_plot(r'Constant–pressure method', pressure_constant_from_geopotential)
    # make_plot(r'Dynamic–pressure method', pressure_dynamic)
    # make_plot(r'Static–pressure method',  pressure_static)


## Site definitions and metadata

### Step 50

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
site_details


### Step 51

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
# --- Site Definitions ---
HANLE = { 'name' : 'Hanle',
         'lat' : 32.7789,
         'lon' : 78.9650,
         'elevation' : 4500,
         'T_b' : 287.49}

# ▼▼▼ ADDED NEW SITE DICTIONARY AS REQUESTED ▼▼▼
IAO_HANLE = { 'name' : 'IAO-Hanle', # Renamed for clarity in the plot
             'lat' : 32.7789,
             'lon' : 78.9650,
             'elevation' : 4500,
             'T_b' : 287.49}

MERAK = { 'name' : 'Merak',
            'lat' : 33.7828,
            'lon' : 78.57782,
            'elevation' : 4310,}

NLST_MERAK = { 'name' : 'NLST-Merak',
                'lat' : 33.7828,
                'lon' : 78.57782,
                'elevation' : 4200,}

SITE_A = { 'name' : 'Site A',
            'lat' : 34.25,
            'lon' : 78.75,
            'elevation' : 4800,
            'T_b' : 285.82}


# SITE_B = { 'name' : 'Site B',
#             'lat' : 33,
#             'lon' : 78,
#             'elevation' : 4800,
#             'T_b' : 287.68}

# we define a new SITE B south of the Hanle Pixel 
SITE_B = { 'name': 'Site B',
          'lat': 32.5,
          'lon': 79.0,
          'elevation': 4500,
          'T_b': 286.0}  # Example temperature, adjust as needed


## Interpolation and extraction

### Step 52

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
# surface_pressure_data = surface_pressure_data.rename({'time': 'valid_time'})


### Step 53

This cell reads tabular metadata that is later used for site lookup, filtering, or summary reporting.

In [ ]:
# In[1]:
# ────────────────────────────────────────────────────────────────────────────────
#  CELL 1: DATA PRE-PROCESSING (THE SLOW PART)
#  This cell performs all the heavy computations once and stores the results.
# ────────────────────────────────────────────────────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.lines as mlines
import numpy as np
import xarray as xr
from scipy.interpolate import PchipInterpolator

# --- Assumed Pre-existing Objects and Functions ---
# This script assumes the following objects are already loaded in your environment:
#
# SITES:
#   - HANLE, MERAK, SITE_A, SITE_B: Dictionaries with site information.
#
# DATASETS:
#   - combined_dataset: xarray.Dataset with data from 2010 to 2025.
#   - surface_pressure_data: xarray.Dataset with surface pressure 'sp'.
#
# HELPER FUNCTION (This can be in a separate cell or defined here):
#   - calculate_pwv_pressure(q, levels, p_surface): Your original PWV calculator.
# -----------------------------------------------------------------------------


# 1. Plotting Configuration
# -----------------------------------------------------------------------------
# ▼▼▼ ADDED IAO_HANLE TO THE LIST TO BE PROCESSED AND PLOTTED ▼▼▼
SITES_TO_PLOT = [HANLE, MERAK, SITE_A, SITE_B, IAO_HANLE]

# ▼▼▼ ADDED A NEW COLOR AND MARKER FOR THE NEW SITE ▼▼▼
PUBLISHABLE_COLORS = ['#00429d', '#93003a', '#009b7a', '#ff9e00', '#e45756'] # Blue, Magenta, Green, Orange, Red
PUBLISHABLE_MARKERS = ['o', 's', '^', 'D', 'p'] # Circle, Square, Triangle, Diamond, Pentagon

# Global settings for a consistent, professional, LaTeX-formatted look
plt.rcParams.update({
    'font.size': 20,
    'legend.fontsize': 18,
    'font.family': 'serif',
    'text.usetex': True,
    'text.latex.preamble': r"\usepackage{amsmath}",
})


# 2. Main Data Processing Workflow
# -----------------------------------------------------------------------------
print("Starting heavy data processing... This will take some time.")

# --- Load Hanle Observational Data ---
# This block loads the CSV and prepares a lookup table for monthly pressure.
try:
    hanle_df = pd.read_csv('hanle_monthly_surface_data.csv')
    # Calculate average pressure and convert from hPa to Pa
    hanle_df['avg_pressure_pa'] = hanle_df[['Pressure_Day_Mean', 'Pressure_Night_Mean']].mean(axis=1) * 100
    month_map = {
        'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
        'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
    }
    hanle_df['month_num'] = hanle_df['Month'].map(month_map)
    hanle_monthly_pressure_lookup = hanle_df.set_index('month_num')['avg_pressure_pa']
    hanle_data_available = True
    print("Successfully loaded and processed IAO-Hanle observational pressure data.")
except FileNotFoundError:
    print("Warning: 'hanle_monthly_surface_data.csv' not found. Cannot process IAO-Hanle with observed data.")
    hanle_data_available = False


# --- EFFICIENT WORKFLOW: Prepare all site data at once ---
print("Preparing site coordinates...")
site_lats = xr.DataArray([s['lat'] for s in SITES_TO_PLOT], dims="site")
site_lons = xr.DataArray([s['lon'] for s in SITES_TO_PLOT], dims="site")

print("Performing a single, efficient interpolation for all sites...")
# Interpolate the main dataset ONCE and load the small result into memory
all_sites_data = combined_dataset.interp(
    latitude=site_lats, longitude=site_lons, method='linear'
).sel(time=slice('2010-01-01', '2025-04-30')).load()

# Interpolate the surface pressure data ONCE and load into memory
try: 
    all_sites_sp = surface_pressure_data['sp'].interp(
        latitude=site_lats, longitude=site_lons, method='linear'
    ).rename({'valid_time': 'time'}).load()
except:
    all_sites_sp = surface_pressure_data['sp'].interp(
        latitude=site_lats, longitude=site_lons, method='linear'
    ).load()
print("All sites data loaded and interpolated successfully.The values are below:")
print(all_sites_data)

print("Interpolation complete. Now calculating PWV for each site...")

# --- Calculate PWV for all sites and store results ---
pwv_results = {}
for i, site in enumerate(SITES_TO_PLOT):
    site_name = site['name']
    print(f"  Calculating for site: {site_name}")

    # Select the data for the current site (this is now a fast, in-memory operation)
    site_data = all_sites_data.isel(site=i)
    q = site_data['q']
    levels = site_data['level']

    # ▼▼▼ SPECIAL HANDLING FOR IAO-HANLE SITE ▼▼▼
    # If the site is IAO-Hanle and the observational data was loaded, use it.
    if 'IAO-Hanle' in site_name and hanle_data_available:
        print("    → Using observed monthly surface pressure for IAO-Hanle.")
        # Get the month for each timestamp in the humidity data
        time_coords = q.time
        months = time_coords.dt.month
        # Map each month to its pressure from the lookup table
        pressure_values_pa = months.to_series().map(hanle_monthly_pressure_lookup)
        # Create the surface pressure DataArray
        pressure_threshold = xr.DataArray(
            pressure_values_pa.values,
            coords={'time': time_coords},
            dims=['time'],
            name='sp'
        )
        print("Pressure for IAO-Hanle loaded from observational data.")
        print(pressure_threshold)
        # Ensure the humidity and pressure data are perfectly aligned
        q, pressure_threshold = xr.align(q, pressure_threshold, join='inner')

    else:
        # For all other sites, use the standard interpolated ERA5 pressure
        pressure_threshold = all_sites_sp.isel(site=i)
    # ▲▲▲ END OF SPECIAL HANDLING ▲▲▲

    # Calculate PWV using your original, working function
    # pwv_data = calculate_pwv_pressure(q, levels, pressure_threshold)
    
        # --- Calculate PWV and handle mis-aligned time axes gracefully ---
    try:
        # First attempt: assume everything lines up
        pwv_data = calculate_pwv_pressure(q, levels, pressure_threshold)

    except ValueError as err:
        if "join='exact'" in str(err) and "'time'" in str(err):
            print("    ⚠️  Time-axis mis-match detected – realigning on the intersection.")
            # Align q and pressure_threshold on the *intersection* of their time indices
            q_aligned, pressure_aligned = xr.align(q, pressure_threshold, join='inner')
            # Re-run the PWV calculation with the fixed inputs
            pwv_data = calculate_pwv_pressure(q_aligned, levels, pressure_aligned)
        else:
            # Propagate any unrelated error so you still notice genuine bugs
            raise

    # Store the final, plottable result in a dictionary
    pwv_results[site_name] = pwv_data

print("\n✅ Data pre-processing complete. You can now re-run the plotting cell below.")


## Pressure and PWV calculations

### Step 54

This cell subsets the dataset to a selected time, level, region, or site so the next step works with a focused slice of the data.

In [ ]:
# --- Calculate PWV for all sites and store results --------------------------
pwv_results = {}

for i, site in enumerate(SITES_TO_PLOT):
    site_name = site['name']
    print(f"  Calculating for site: {site_name}")

    # 1. Extract humidity column ------------------------------------------------
    site_data = all_sites_data.isel(site=i)        # (time, level, expver)
    q = site_data['q']

    # remove the synthetic GRIB 'expver' dimension if present
    if 'expver' in q.dims:
        q = q.squeeze('expver', drop=True)         # → (time, level)

    # 2. Build matching surface-pressure series ---------------------------------
    if ('IAO-Hanle' in site_name) and hanle_data_available:
        print("    → Using observed monthly surface pressure for IAO-Hanle.")

        months = q.time.dt.month
        p_vals = months.to_series().map(hanle_monthly_pressure_lookup).values

        pressure_threshold = xr.DataArray(
            p_vals,
            coords={'time': q.time},
            dims=['time'],
            name='sp'
        )
    else:
        pressure_threshold = all_sites_sp.isel(site=i)

        if 'expver' in pressure_threshold.dims:
            pressure_threshold = pressure_threshold.squeeze('expver', drop=True)

    # 3. De-duplicate and align time axes ---------------------------------------
    q          = q.isel(time=np.unique(q.time,          return_index=True)[1])
    p_thr      = pressure_threshold.isel(
                    time=np.unique(pressure_threshold.time, return_index=True)[1]
                 )

    q, p_thr   = xr.align(q, p_thr, join='inner')

    # make sure the vertical dimension sits in a single chunk (needed by ufunc)
    q = q.chunk({'level': -1})

    # 4. Vectorised PWV integration --------------------------------------------
    pwv_data = calculate_pwv_vectorized(q, p_thr)

    # 5. Collect results --------------------------------------------------------
    pwv_results[site_name] = pwv_data

print("\n✅ Data pre-processing complete. You can now re-run the plotting cell.")


## Mapping and visualization

### Step 55

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
#  CELL 2: PLOTTING (THE FAST PART)
#  This cell uses the pre-calculated 'pwv_results' to generate the plot.
#  You can re-run this cell quickly to change visual styles.
# ────────────────────────────────────────────────────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.lines as mlines
import numpy as np
import xarray as xr
from scipy.interpolate import PchipInterpolator

# Setup the two-panel figure
fig = plt.figure(figsize=(12, 9))
gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.0)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)
zoom_range = [0.57, 1.25]

# ▼▼▼ ADDED A NEW COLOR AND MARKER FOR THE NEW SITE ▼▼▼
PUBLISHABLE_COLORS = ['#00429d', '#93003a', '#009b7a', '#ff9e00', "#c6e740"] # Blue, Magenta, Green, Orange, Red
PUBLISHABLE_MARKERS = ['o', 's', '^', 'D', 'p'] # Circle, Square, Triangle, Diamond, Pentagon

print("Generating plot from pre-processed data...")

# Loop through each site and plot the stored results
for i, site in enumerate(SITES_TO_PLOT):
    site_name = site['name']

    # Retrieve the pre-calculated data from the dictionary
    pwv_data = pwv_results[site_name]

    # --- Plotting with Upgraded Aesthetics ---
    color = PUBLISHABLE_COLORS[i]
    marker = PUBLISHABLE_MARKERS[i]

    # Plot on both axes
    ax1.plot(pwv_data.time, pwv_data, color=color, linewidth=2.5, alpha=0.3)
    ax2.plot(pwv_data.time, pwv_data, color=color, linewidth=2.5, alpha=0.8)

    ax1.plot(pwv_data.time, pwv_data, label=site_name, color=color, marker=marker, markersize=8,
             linestyle='', alpha=0.7, markerfacecolor=color, markeredgewidth=1.5, markeredgecolor='black',
             markevery=1, zorder=5)
    ax2.plot(pwv_data.time, pwv_data, label=site_name, color=color, marker=marker, markersize=8,
             linestyle='', alpha=0.8, markerfacecolor=color, markeredgewidth=1.5, markeredgecolor='black',
             markevery=1, zorder=5)

# --- Plot Cosmetics and Finalization ---
ax1.axhline(y=1, color='black', linestyle='--', linewidth=2.5)
ax2.axhline(y=1, color='black', linestyle='--', linewidth=2.5)

# --- MODIFICATION: Increased fontsize and adjusted position for less padding ---
fig.text(0.55, 0.05, 'Year', ha='center', va='center', fontsize=32)
fig.text(0.068, 0.5, 'PWV (mm)', ha='center', va='center', rotation='vertical', fontsize=32)


handles, _ = ax1.get_legend_handles_labels()
threshold_handle = mlines.Line2D([], [], color='black', linestyle='--', linewidth=3)
# handles.append(threshold_handle)

# ▼▼▼ MODIFIED LEGEND FOR BETTER LAYOUT WITH MORE ITEMS ▼▼▼
# Changed ncol to 3 so the 6 legend items (5 sites + 1 line) fit nicely.
ax1.legend(handles=handles, ncol=6, loc='upper center', fontsize=19,
           frameon=True, facecolor='white', edgecolor='black', framealpha=0.9,
           bbox_to_anchor=(0.5, 1.18)) # Adjusted anchor to give it space

ax1.tick_params(axis='y', direction='in', width=2, length=10, labelsize=28, which='both', right=True)
ax1.tick_params(axis='x', direction='in', width=2, length=10, which='both', top=True)
ax1.set_yticks([1, 2, 4, 6, 8, 10, 12])
ax1.set_ylim(0, 13.2)
ax1.minorticks_on()
ax1.tick_params(axis='y', which='minor', direction='in', width=1, length=5, right=True)
ax1.tick_params(axis='x', which='minor', direction='in', width=1, length=5, top=True)
plt.setp(ax1.get_xticklabels(), visible=False)

ax2.set_ylim(zoom_range)
ax2.tick_params(axis='both', which='major', direction='in', width=2, length=10, labelsize=28, right=True)
ax2.minorticks_on()
ax2.tick_params(axis='both', which='minor', direction='in', width=1, length=5, right=True)

for ax in [ax1, ax2]:
    ax.set_facecolor('white')
    ax.spines['right'].set_linewidth(3)
    ax.spines['left'].set_linewidth(3)
    ax.spines['top'].set_linewidth(3 if ax is ax1 else 1)
    ax.spines['bottom'].set_linewidth(1 if ax is ax1 else 3)

fig.tight_layout(rect=[0.07, 0.07, 1, 0.9])
output_filename = 'PWV_Site_Comparison_Ladakh_Formatted_2010-2025.pdf'
plt.savefig(output_filename, dpi=500, bbox_inches='tight')
print(f"\n→ Plot successfully saved as {output_filename}")
plt.show()


### Step 56

This cell defines reusable helper function(s) `_as_series`, `_mk_test`, `_lin_trend`, `_yearly_reduce` so later sections can apply the same processing logic consistently.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# PWV Trend Analysis: Annual Minima & Annual Medians (per site + combined)
# - Inputs:  pwv_results = { "SiteName": DataArray|Series } (with datetime index/coord)
# - Outputs: two figures (PNG/PDF), two CSVs with trend stats, plus printed summary
# ────────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress
from math import sqrt, erf

# =========================
# Helpers
# =========================
def _as_series(obj):
    """Return a clean, sorted pandas.Series with a DateTimeIndex."""
    if isinstance(obj, pd.Series):
        s = obj.copy()
        if not isinstance(s.index, pd.DatetimeIndex):
            s.index = pd.to_datetime(s.index)
    else:
        try:
            import xarray as xr
            if isinstance(obj, xr.DataArray):
                s = obj.to_pandas()
            else:
                s = pd.Series(np.asarray(obj.values),
                              index=pd.to_datetime(np.asarray(obj.time.values)))
        except Exception:
            s = pd.Series(obj)
            if not isinstance(s.index, pd.DatetimeIndex):
                s.index = pd.to_datetime(s.index)
    s = s.dropna().sort_index()
    return s

def _mk_test(y):
    """
    Mann–Kendall test (two-sided) for monotonic trend.
    Returns dict with S, Z, p, trend ('increasing'/'decreasing'/'no trend').
    """
    vals = np.asarray(y, dtype=float)
    n = len(vals)
    if n < 8:
        return {"S": np.nan, "Z": np.nan, "p": np.nan, "trend": "insufficient n"}

    S = 0
    for k in range(n-1):
        S += np.sign(vals[k+1:] - vals[k]).sum()

    varS = (n*(n-1)*(2*n+5)) / 18.0
    if S > 0:
        Z = (S - 1) / np.sqrt(varS)
    elif S < 0:
        Z = (S + 1) / np.sqrt(varS)
    else:
        Z = 0.0

    # Two-sided p-value from standard normal (via erf)
    p = 2.0 * (1.0 - 0.5 * (1.0 + erf(abs(Z)/np.sqrt(2))))

    trend = "increasing" if (p < 0.05 and Z > 0) else \
            "decreasing" if (p < 0.05 and Z < 0) else "no trend"
    return {"S": S, "Z": Z, "p": p, "trend": trend}

def _lin_trend(y):
    """Linear trend on a pandas Series indexed by year."""
    years = np.array(getattr(y.index, "year", y.index), dtype=float)
    slope, intercept, r, p, stderr = linregress(years, y.values)
    return {
        "n": len(y),
        "slope_mm_per_yr": slope,
        "intercept_mm": intercept,
        "r2": r**2,
        "p_value": p,
        "stderr": stderr
    }

def _yearly_reduce(series, how="min"):
    """Return a Series of annual minima or medians."""
    if how == "min":
        return series.resample("Y").min()      # year-end labels
    elif how == "median":
        return series.resample("Y").median()
    else:
        raise ValueError("how must be 'min' or 'median'.")

def _combined_median(df):
    """Row-wise median across sites per year."""
    return df.median(axis=1)

def _plot_yearly(ax, yearly_dict, color_map=None, title="", ylabel="PWV (mm)",
                 add_combined=True, comb_color="k", fname=None):
    """Plot yearly series with linear fits, per site + combined."""
    if color_map is None:
        color_map = {}

    for site, y in yearly_dict.items():
        years = y.index.year
        ax.plot(years, y.values, marker="o", linestyle="", label=site,
                ms=6, alpha=0.9, color=color_map.get(site, None))
        fit = _lin_trend(y)
        yfit = fit["intercept_mm"] + fit["slope_mm_per_yr"] * years
        ax.plot(years, yfit, linestyle="--", alpha=0.8, color=color_map.get(site, None))

    if add_combined:
        df = pd.DataFrame(yearly_dict)
        comb = _combined_median(df)
        years = comb.index.year
        ax.plot(years, comb.values, marker="D", ms=7, color=comb_color,
                linestyle="", label="Combined median")
        f2 = _lin_trend(comb)
        ax.plot(years, f2["intercept_mm"] + f2["slope_mm_per_yr"] * years,
                linestyle="--", color=comb_color, alpha=0.9, label="Combined trend")

    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.25)
    ax.legend(ncol=2, fontsize=9)
    if fname:
        plt.savefig(fname, dpi=300, bbox_inches="tight")

# =========================
# MAIN ANALYSIS
# =========================
def analyze_pwv_trends_medians(pwv_results, out_prefix="PWV_trends"):
    """
    pwv_results: dict {site -> Series|DataArray} with datetime index/coord
    Produces:
      - {out_prefix}_annual_minima.png/.pdf
      - {out_prefix}_annual_medians.png/.pdf
      - {out_prefix}_stats_minima.csv
      - {out_prefix}_stats_medians.csv
    Returns two DataFrames (minima_stats, medians_stats) and prints combined trends.
    """
    # Normalize input to Series
    series_by_site = {site: _as_series(arr) for site, arr in pwv_results.items()}

    # Build yearly minima/medians per site
    yearly_min = {s: _yearly_reduce(series_by_site[s], "min")    for s in series_by_site}
    yearly_med = {s: _yearly_reduce(series_by_site[s], "median") for s in series_by_site}

    # Stats tables (Linear + MK)
    rows_min, rows_med = [], []

    for site in series_by_site:
        y_min = yearly_min[site].dropna()
        y_med = yearly_med[site].dropna()

        lin_min = _lin_trend(y_min)
        mk_min  = _mk_test(y_min)
        rows_min.append({
            "site": site,
            **lin_min,
            "MK_Z": mk_min["Z"], "MK_p": mk_min["p"], "MK_trend": mk_min["trend"]
        })

        lin_med = _lin_trend(y_med)
        mk_med  = _mk_test(y_med)
        rows_med.append({
            "site": site,
            **lin_med,
            "MK_Z": mk_med["Z"], "MK_p": mk_med["p"], "MK_trend": mk_med["trend"]
        })

    # Combined (median across sites per year)
    df_min = pd.DataFrame(yearly_min)
    df_med = pd.DataFrame(yearly_med)

    comb_min = _combined_median(df_min).dropna()
    comb_med = _combined_median(df_med).dropna()

    lin_cmin = _lin_trend(comb_min)
    mk_cmin  = _mk_test(comb_min)
    rows_min.append({
        "site": "Combined",
        **lin_cmin,
        "MK_Z": mk_cmin["Z"], "MK_p": mk_cmin["p"], "MK_trend": mk_cmin["trend"]
    })

    lin_cmed = _lin_trend(comb_med)
    mk_cmed  = _mk_test(comb_med)
    rows_med.append({
        "site": "Combined",
        **lin_cmed,
        "MK_Z": mk_cmed["Z"], "MK_p": mk_cmed["p"], "MK_trend": mk_cmed["trend"]
    })

    stats_min = pd.DataFrame(rows_min).set_index("site").sort_index()
    stats_med = pd.DataFrame(rows_med).set_index("site").sort_index()

    # Print concise summary of combined trends
    print("\n=== Combined Trend (Annual Minima) ===")
    print(f"Slope = {lin_cmin['slope_mm_per_yr']:.4f} mm/yr, "
          f"p = {lin_cmin['p_value']:.3e}, R² = {lin_cmin['r2']:.3f}, "
          f"MK Z = {mk_cmin['Z']:.2f}, MK p = {mk_cmin['p']:.3e}, MK trend = {mk_cmin['trend']}")

    print("\n=== Combined Trend (Annual Medians) ===")
    print(f"Slope = {lin_cmed['slope_mm_per_yr']:.4f} mm/yr, "
          f"p = {lin_cmed['p_value']:.3e}, R² = {lin_cmed['r2']:.3f}, "
          f"MK Z = {mk_cmed['Z']:.2f}, MK p = {mk_cmed['p']:.3e}, MK trend = {mk_cmed['trend']}")

    # =========================
    # Plots
    # =========================
    # (A) Annual minima
    fig, ax = plt.subplots(figsize=(11, 6))
    _plot_yearly(
        ax,
        yearly_dict={k: v for k, v in yearly_min.items()},
        title="Annual Minimum PWV per Site (with linear trends)",
        ylabel="PWV (mm)",
        add_combined=True
    )
    plt.tight_layout()
    plt.savefig(f"{out_prefix}_annual_minima.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"{out_prefix}_annual_minima.pdf", dpi=300, bbox_inches="tight")
    plt.show()

    # (B) Annual medians
    fig, ax = plt.subplots(figsize=(11, 6))
    _plot_yearly(
        ax,
        yearly_dict={k: v for k, v in yearly_med.items()},
        title="Annual Median PWV per Site (with linear trends)",
        ylabel="PWV (mm)",
        add_combined=True
    )
    plt.tight_layout()
    plt.savefig(f"{out_prefix}_annual_medians.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"{out_prefix}_annual_medians.pdf", dpi=300, bbox_inches="tight")
    plt.show()

    # =========================
    # Save stats
    # =========================
    stats_min.to_csv(f"{out_prefix}_stats_minima.csv")
    stats_med.to_csv(f"{out_prefix}_stats_medians.csv")

    return stats_min, stats_med

# ────────────────────────────────────────────────────────────────────────────────
# USAGE:
stats_min, stats_med = analyze_pwv_trends_medians(pwv_results, out_prefix="PWV_Ladakh_2010_2025")
# ────────────────────────────────────────────────────────────────────────────────


### Step 57

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
#  CELL 2: PLOTTING (THE FAST PART) — CURVE AESTHETICS IMPROVED ONLY
#  Formatting (layout, labels, legend placement, ticks, limits) is unchanged.
# ────────────────────────────────────────────────────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.lines as mlines
import matplotlib.patheffects as pe
from matplotlib import colors as mcolors
import numpy as np
import xarray as xr
from scipy.interpolate import PchipInterpolator

# Setup the two-panel figure
fig = plt.figure(figsize=(12, 9))
gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.0)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)
zoom_range = [0.57, 1.25]

# Color/marker choices (Okabe–Ito palette; 5 markers)
PUBLISHABLE_COLORS  = ['#0072B2', '#D55E00', '#009E73', '#E69F00', '#CC79A7']  # blue, vermillion, green, orange, purple
PUBLISHABLE_MARKERS = ['o', 's', '^', 'D', 'p']  # circle, square, triangle, diamond, pentagon

print("Generating plot from pre-processed data...")

# Loop through each site and plot the stored results
for i, site in enumerate(SITES_TO_PLOT):
    site_name = site['name']

    # Retrieve the pre-calculated data from the dictionary
    pwv_data = pwv_results[site_name]

    # --- Curve aesthetics (only) ---
    color  = PUBLISHABLE_COLORS[i % len(PUBLISHABLE_COLORS)]
    marker = PUBLISHABLE_MARKERS[i % len(PUBLISHABLE_MARKERS)]
    mface  = mcolors.to_rgba(color, alpha=0.90)  # subtle marker transparency

    # Choose ~12 marker positions evenly across the series (reduces clutter)
    n_pts = pwv_data.time.size
    marker_idx = np.unique(np.linspace(0, n_pts - 1, min(12, n_pts), dtype=int))

    # Common line style: rounded, with a faint white halo for contrast
    line_style_top = dict(
        color=color, linewidth=2.6, alpha=0.35,  # keep top panel faint
        solid_joinstyle='round', solid_capstyle='round',
        antialiased=True,
        path_effects=[pe.Stroke(linewidth=3.6, foreground='white', alpha=0.8), pe.Normal()],
        zorder=3
    )
    line_style_bottom = dict(
        color=color, linewidth=2.8, alpha=0.95,  # bottom (zoom) stronger
        solid_joinstyle='round', solid_capstyle='round',
        antialiased=True,
        path_effects=[pe.Stroke(linewidth=3.8, foreground='white', alpha=0.85), pe.Normal()],
        zorder=3
    )

    # Draw the lines
    ax1.plot(pwv_data.time, pwv_data, **line_style_top)
    ax2.plot(pwv_data.time, pwv_data, **line_style_bottom)

    # Draw sparse markers on top of the lines (so legend shows markers too)
    # (Use markevery so legend entries include markers.)
    ax1.plot(
        pwv_data.time, pwv_data,
        linestyle='',
        marker=marker, markersize=8,
        markerfacecolor=mface, markeredgecolor='black', markeredgewidth=1.4,
        markevery=marker_idx, alpha=0.9, zorder=5, label=site_name
    )
    ax2.plot(
        pwv_data.time, pwv_data,
        linestyle='',
        marker=marker, markersize=8,
        markerfacecolor=mface, markeredgecolor='black', markeredgewidth=1.4,
        markevery=marker_idx, alpha=0.95, zorder=5, label=site_name
    )

# --- Plot Cosmetics and Finalization (UNCHANGED) ---
ax1.axhline(y=1, color='black', linestyle='--', linewidth=2.5)
ax2.axhline(y=1, color='black', linestyle='--', linewidth=2.5)

fig.text(0.55, 0.05, 'Year', ha='center', va='center', fontsize=32)
fig.text(0.068, 0.5, 'PWV (mm)', ha='center', va='center', rotation='vertical', fontsize=32)

handles, _ = ax1.get_legend_handles_labels()
threshold_handle = mlines.Line2D([], [], color='black', linestyle='--', linewidth=3)
# handles.append(threshold_handle)

ax1.legend(handles=handles, ncol=6, loc='upper center', fontsize=19,
           frameon=True, facecolor='white', edgecolor='black', framealpha=0.9,
           bbox_to_anchor=(0.5, 1.18))  # same placement

ax1.tick_params(axis='y', direction='in', width=2, length=10, labelsize=28, which='both', right=True)
ax1.tick_params(axis='x', direction='in', width=2, length=10, which='both', top=True)
ax1.set_yticks([1, 2, 4, 6, 8, 10, 12])
ax1.set_ylim(0, 13.2)
ax1.minorticks_on()
ax1.tick_params(axis='y', which='minor', direction='in', width=1, length=5, right=True)
ax1.tick_params(axis='x', which='minor', direction='in', width=1, length=5, top=True)
plt.setp(ax1.get_xticklabels(), visible=False)

ax2.set_ylim(zoom_range)
ax2.tick_params(axis='both', which='major', direction='in', width=2, length=10, labelsize=28, right=True)
ax2.minorticks_on()
ax2.tick_params(axis='both', which='minor', direction='in', width=1, length=5, right=True)

for ax in [ax1, ax2]:
    ax.set_facecolor('white')
    ax.spines['right'].set_linewidth(3)
    ax.spines['left'].set_linewidth(3)
    ax.spines['top'].set_linewidth(3 if ax is ax1 else 1)
    ax.spines['bottom'].set_linewidth(1 if ax is ax1 else 3)

fig.tight_layout(rect=[0.07, 0.07, 1, 0.9])
output_filename = 'PWV_Site_Comparison_Ladakh_Formatted_2010-2025.pdf'
plt.savefig(output_filename, dpi=500, bbox_inches='tight')
print(f"\n→ Plot successfully saved as {output_filename}")
plt.show()


## Summary statistics and reporting

### Step 58

This cell computes or evaluates precipitable water vapor (PWV)-related quantities that are central to the site-quality analysis.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
#  CELL 1.5: ANALYSIS - MONTHS BELOW 1MM THRESHOLD
#  This cell analyzes the pre-calculated PWV data to find the number and
#  fraction of months below the 1 mm critical threshold.
# ────────────────────────────────────────────────────────────────────────────────

print("Analyzing PWV data for months below 1 mm threshold...")
print("="*70)
print(f"{'Site':<30} | {'Dry Months':<15} | {'Total Months':<15} | {'Fraction Below 1mm':<20}")
print("-"*70)

for site_name, pwv_data in pwv_results.items():
    
    # Resample the data to get monthly averages
    monthly_pwv = pwv_data.resample(time='1M').mean()
    
    # Define the threshold
    threshold = 1.0

    # Compute boolean mask where PWV < threshold and drop others
    mask = (monthly_pwv < threshold).compute()
    dry_months = monthly_pwv.where(mask, drop=True)

    # Count dry and total months
    num_dry_months = dry_months.count().values
    total_months = monthly_pwv.count().values

    # Compute the fraction (safely)
    fraction_dry = num_dry_months / total_months if total_months > 0 else 0

    # Print results
    print(f"{site_name:<30} | {num_dry_months:<15} | {total_months:<15} | {fraction_dry:.2%}")

# # Loop through each site's results stored in the dictionary
# for site_name, pwv_data in pwv_results.items():
    
#     # Resample the data to get the average PWV for each month.
#     # This handles the daily or 6-hourly data by averaging it monthly.
#     monthly_pwv = pwv_data.resample(time='1M').mean()
    
#     # Define the threshold
#     threshold = 1.0
    
#     # Find the months where the average PWV is below the threshold
#     dry_months = monthly_pwv[monthly_pwv < threshold]
    
#     # Count the number of dry months and the total number of months
#     num_dry_months = len(dry_months)
#     total_months = len(monthly_pwv)
    
#     # Calculate the fraction of dry months
#     # (add a check to avoid division by zero if there's no data)
#     fraction_dry = num_dry_months / total_months if total_months > 0 else 0
    
#     # Print the results in a nicely formatted table
#     print(f"{site_name:<30} | {num_dry_months:<15} | {total_months:<15} | {fraction_dry:.2%}")

print("="*70)
print("\n✅ Analysis complete. You can now proceed with plotting.")


### Step 59

This cell defines reusable helper function(s) `compute_stats` so later sections can apply the same processing logic consistently.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  PWV STATISTICS REPORT (no plotting)
#  ----------------------------------------------------------
#  Assumptions:
#    • SITES_TO_PLOT  – list of dicts, each with key 'name'
#    • pwv_results    – dict  {site_name: xarray.DataArray}
#                       (DataArray has a 'time' coordinate)
#  Both objects are created in earlier cells of your notebook.
# ──────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import xarray as xr

# ── 1.  STATISTICS FUNCTION ──────────────────────────────────────────
def compute_stats(pwv_da: xr.DataArray) -> dict[str, float]:
    """
    Return a dictionary of descriptive statistics for a PWV time-series.
    All PWV values are assumed to be in millimetres.
    """
    mask  = np.isfinite(pwv_da.values)
    vals  = pwv_da.values[mask]
    times = pwv_da.time.values[mask]

    if vals.size == 0:          # fallback for completely empty series
        return {k: np.nan for k in
                ['N_months','Mean_mm','Median_mm','Std_mm','Min_mm','Max_mm',
                 'P05_mm','P25_mm','P75_mm','P95_mm','CoeffVar',
                 'Trend_mm_decade',
                 'Frac_lt_0.5mm','Frac_lt_1.0mm','Frac_lt_2.0mm']}

    # --- basic moments
    stats = {
        'N_months' : vals.size,
        'Mean_mm'  : np.mean(vals),
        'Median_mm': np.median(vals),
        'Std_mm'   : np.std(vals, ddof=1),
        'Min_mm'   : np.min(vals),
        'Max_mm'   : np.max(vals),
        'P05_mm'   : np.percentile(vals,  5),
        'P25_mm'   : np.percentile(vals, 25),
        'P75_mm'   : np.percentile(vals, 75),
        'P95_mm'   : np.percentile(vals, 95),
    }
    stats['CoeffVar'] = stats['Std_mm'] / stats['Mean_mm']

    # --- linear trend (least-squares)  →  mm per decade
    if vals.size > 1:
        # convert np.datetime64 → decimal year
        years = xr.cftime_range(start='2000', periods=1, freq='D')  # dummy to get dtype
        decimal_years = xr.DataArray(times).dt.year.astype(float) + \
                        (xr.DataArray(times).dt.dayofyear - 1) / 365.25
        slope, _ = np.polyfit(decimal_years, vals, 1)
        stats['Trend_mm_decade'] = slope * 10.0
    else:
        stats['Trend_mm_decade'] = np.nan

    # --- dry-month fractions
    stats['Frac_lt_0.5mm'] = np.mean(vals < 0.5)
    stats['Frac_lt_1.0mm'] = np.mean(vals < 1.0)
    stats['Frac_lt_2.0mm'] = np.mean(vals < 2.0)

    return stats


# ── 2.  GENERATE STATISTICS FOR ALL SITES ────────────────────────────
all_stats: dict[str, dict[str, float]] = {}

print("\nPWV summary (2010–2025)\n" + "="*28)
for site in SITES_TO_PLOT:
    name = site['name']
    pwv_da = pwv_results[name]            # xarray.DataArray
    st = compute_stats(pwv_da)
    all_stats[name] = st

    # human-readable one-liner for each site
    print(f"{name:15s}  μ={st['Mean_mm']:.2f} mm, "
          f"σ={st['Std_mm']:.2f} mm,  "
          f"Median={st['Median_mm']:.2f} mm, "
          f"Min–Max={st['Min_mm']:.2f}–{st['Max_mm']:.2f} mm,  "
          f"PWV<1 mm: {st['Frac_lt_1.0mm']*100:4.1f}% "
          f"(n={int(st['Frac_lt_1.0mm']*st['N_months'])}/{st['N_months']})")


# ── 3.  PRINT FULL TABLE (copy / save) ───────────────────────────────
df_stats = (pd.DataFrame(all_stats)
              .T
              [['N_months','Mean_mm','Median_mm','Std_mm',
                'Min_mm','Max_mm','P05_mm','P25_mm','P75_mm','P95_mm',
                'CoeffVar','Trend_mm_decade',
                'Frac_lt_0.5mm','Frac_lt_1.0mm','Frac_lt_2.0mm']])

pd.set_option('display.float_format', '{:7.3f}'.format)

print("\n\nComplete statistics table:\n" + "-"*28)
print(df_stats.to_string())

# Optional: save to CSV for supplementary material
# df_stats.to_csv("pwv_statistics_2010_2025.csv", float_format="%.3f")


## Interpolation and extraction

### Step 60

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
# In[1]:
# ────────────────────────────────────────────────────────────────────────────────
#  CELL 1: DATA PRE-PROCESSING (THE SLOW PART)
#  This cell uses the CORRECT combined_dataset and full 2010-2025 timeline.
# ────────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import xarray as xr

# --- Assumed Pre-existing Objects and Functions ---
# SITES, DATASETS, and HELPER FUNCTIONS are assumed to be loaded.
# -----------------------------------------------------------------------------

# --- EFFICIENT WORKFLOW: Prepare all site data at once ---
print("Starting heavy data processing for all sites...")
sites_to_process = [HANLE, MERAK, SITE_A, SITE_B, IAO_HANLE]

# Create DataArrays for all site coordinates for a single interpolation.
site_lats = xr.DataArray([s['lat'] for s in sites_to_process], dims="site")
site_lons = xr.DataArray([s['lon'] for s in sites_to_process], dims="site")

print("Performing a single, efficient interpolation on the combined_dataset...")

# --- MODIFICATION: Using the correct dataset and time range ---
all_sites_data = combined_dataset.interp(
    latitude=site_lats, longitude=site_lons, method='linear'
).sel(time=slice('2010-01-01', '2025-04-30')).load() # Using combined_dataset and full timeline

try:
    all_sites_sp = surface_pressure_data['sp'].interp(
        latitude=site_lats, longitude=site_lons, method='linear'
    ).rename({'valid_time': 'time'}).load()
except:
    all_sites_sp = surface_pressure_data['sp'].interp(
        latitude=site_lats, longitude=site_lons, method='linear'
    ).load()
# -----------------------------------------------------------

print("Interpolation complete. Calculating annual minimum PWV for each site...")

# --- Calculate and store the final plottable data ---
annual_min_pwv_results = {}
for i, site in enumerate(sites_to_process):
    site_name = site['name']
    print(f"  Calculating for site: {site_name}")
    
    # Select the data for the current site (fast, in-memory operation)
    site_data = all_sites_data.isel(site=i)
    pressure_threshold = all_sites_sp.isel(site=i)
    
    q = site_data['q']
    levels = site_data['level']

    # Calculate the full PWV time-series using your original function
    pwv_data_full = calculate_pwv_pressure(q, levels, pressure_threshold)
    
    # try:
    #     # First attempt with raw arrays
    #     pwv_data_full = calculate_pwv_pressure(q, levels, pressure_threshold)

    # except ValueError as err:
    #     # Catch the xarray “join='exact' … 'time' ” mis-alignment
    #     if "join='exact'" in str(err) and "'time'" in str(err):
    #         print(f"    ⚠️  Time-axis mismatch at {site_name} – realigning on the intersection.")
    #         # Align on the common timestamps
    #         q, pressure_threshold = xr.align(q, pressure_threshold, join='inner')
    #         # Retry the calculation
    #         pwv_data_full = calculate_pwv_pressure(q, levels, pressure_threshold)
    #     else:
    #         # Re-raise any unrelated error
    #         raise

    # Resample to get the annual minimum and store it
    annual_min_pwv = pwv_data_full.resample(time='YE').min()
    annual_min_pwv_results[site_name] = annual_min_pwv
    
print("\n✅ Data processing complete. You can now re-run the plotting cell below.")


## Mapping and visualization

### Step 61

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
# In[2]:
# ────────────────────────────────────────────────────────────────────────────────
#  CELL 2: PLOTTING (THE FAST PART - SINGLE PANEL VERSION)
#  This cell uses the pre-calculated annual minimums to generate a
#  single, publication-quality plot with the legend inside.
# ────────────────────────────────────────────────────────────────────────────────

# --- Plotting Configuration ---
fig = plt.figure(figsize=(12, 10))
ax1 = fig.add_subplot(111)

# Using the professional color and marker schemes
colors = PUBLISHABLE_COLORS
markers = PUBLISHABLE_MARKERS

plt.rcParams.update({
    'font.size': 20,
    'legend.fontsize': 18
})

print("Generating single-panel plot from pre-calculated annual minimums...")

# --- Main Plotting Loop ---
for i, site in enumerate(SITES_TO_PLOT):
    site_name = site['name']
    # Retrieve the pre-calculated annual minimum data
    annual_min_pwv = annual_min_pwv_results[site_name]

    # Plot the annual minimums with a connecting line
    ax1.plot(annual_min_pwv.time, annual_min_pwv,
             label=site_name,
             color=colors[i],
             marker=markers[i],
             linestyle='-',
             linewidth=2.5,
             markersize=12,
             alpha=0.9,
             markerfacecolor=colors[i],
             markeredgewidth=1.5,
             markeredgecolor='black',
             zorder=5)

# --- Cosmetics and Finalization ---
ax1.axhline(y=1, color='black', linestyle='--', linewidth=2.5, zorder=0)

# --- MODIFICATION: Legend is now placed inside the plot ---
ax1.legend(loc='upper center', fontsize=22,
           frameon=True, facecolor='white', edgecolor='black', framealpha=0.9,ncol=4,)


# --- MODIFICATION: Increased fontsize and adjusted position for less padding ---
fig.text(0.55, 0.05, 'Year', ha='center', va='center', fontsize=32)
fig.text(0.068, 0.5, 'Annual Minimum PWV (mm)', ha='center', va='center', rotation='vertical', fontsize=26)

# Configure ticks and limits
ax1.set_ylim(0.55, 1.30)
ax1.minorticks_on()

# Apply ticks to all four sides of the plot
ax1.tick_params(axis='both', which='major', direction='in', width=2, length=10, labelsize=28, top=True, right=True)
ax1.tick_params(axis='both', which='minor', direction='in', width=1, length=5, top=True, right=True)


# Set facecolor and spine properties
ax1.set_facecolor('white')
for spine in ax1.spines.values():
    spine.set_linewidth(3)

# Adjust layout and save the final figure
fig.tight_layout(rect=[0.07, 0.07, 0.98, 0.98])
output_filename = 'PWV_Minimum_Annual_Single_Panel.pdf'
plt.savefig(output_filename, dpi=500, bbox_inches='tight')
print(f"\n→ Plot successfully saved as {output_filename}")
plt.show()


## Pressure and PWV calculations

### Step 62

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

fig = plt.figure(figsize=(10, 10))
ax1 = fig.add_subplot(111)

colors = ['blue', 'green', 'red', 'skyblue']
markers = ['o', 's', '^', 'X']

# Improved font settings
plt.rcParams.update({
    'font.size': 20,
    'legend.fontsize': 18
})

for i, site in enumerate([HANLE,MERAK, SITE_A, SITE_B]):
    # Assuming `era5` is defined elsewhere in your code as xarray dataset
    # site_data = era5.sel(latitude=site['lat'], longitude=site['lon'], method='nearest').sel(time=slice('2010', '2023'))
    # pwv_data = site_data['tcwv']

    lat = site['lat']
    lon = site['lon']
    elevation = site['elevation']

    print(f"Elevation for site {site_name} at ({lat}, {lon}) is {elevation} meters.")
    #T_b = site['T_b']

    pressure_threshold = surface_pressure_data['sp'].interp(latitude=lat, longitude=lon, method='linear')
    #pressure_threshold = calculate_pressure(P_b, T_b, L_b, elevation, h_b)

    #print(f"Pressure threshold for site {site['name']} is {pressure_threshold:.2f} atm.")

    #site_data = era5.sel(latitude=site['lat'], longitude=site['lon'], method='nearest').sel(time=slice('2010', '2023'))
    site_data = era5.interp(latitude=lat, longitude=lon, method='linear').sel(time=slice('2010', '2023'))   
    #site_data = site_data.interp(level=np.linspace(1, 1000, 10000), method='linear')
    
    q = site_data['q']  # Specific humidity data
    levels = site_data['level'] 
    #print(levels)

    pressure_threshold = pressure_threshold.rename({'valid_time': 'time'})
    pwv_data = calculate_pwv_pressure(q, levels, pressure_threshold)

    # Resampling data annually and taking the minimum PWV value
    annual_min_pwv = pwv_data.resample(time='YE').min()

    # Plot lines with lower opacity for annual minimum
    ax1.plot(annual_min_pwv.time, annual_min_pwv, label=site['name'], color=colors[i], marker=markers[i], markersize=12,
             linestyle='-', alpha=1, markerfacecolor=colors[i], markeredgewidth=2, markeredgecolor='black',
             linewidth=2.5, zorder=5)

# 1mm threshold line
plt.axhline(y=1, color='black', linestyle='--', linewidth=2)



# Customize legend
ax1.legend(ncol=4, loc='upper center', fontsize=19,
           frameon=True, facecolor='white', edgecolor='black', framealpha=1, shadow=True,
           bbox_to_anchor=(0.5, 0.97))

ax1.tick_params(axis='both', direction='in', width=2, length=10, labelsize=22, which='both')  # Set direction for both major and minor ticks
# ax1.set_yticks([1, 2, 4, 6, 8, 10, 12])
ax1.set_ylim(0.45, 1.45)
ax1.minorticks_on()
ax1.tick_params(axis='both', which='minor', direction='in', width=1, length=5)

# Set facecolor and spine properties
ax1.set_facecolor('white')
ax1.spines['top'].set_linewidth(3)
ax1.spines['right'].set_linewidth(3)
ax1.spines['bottom'].set_linewidth(3)
ax1.spines['left'].set_linewidth(3)

plt.xlabel('Year', fontsize=22,
           labelpad = 5)
plt.ylabel('Minimum PWV (mm)', fontsize=22, labelpad = 5)
plt.tight_layout()
plt.savefig('PWV_Minimum_Annual_Site_Comparison_Ladakh_pressure_level.pdf', dpi=500)
plt.show()


## Interpolation and extraction

### Step 63

This cell defines reusable helper function(s) `_rename_valid_time`, `pressure_dynamic` so later sections can apply the same processing logic consistently.

In [ ]:
#!/usr/bin/env python3
# ────────────────────────────────────────────────────────────────────────────────
#  Minimum-annual PWV comparison  –  Variant A applied uniformly to all sites
#      (surface pressure from the barometric‐formula method for every site)
# ────────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# ── 0.  Objects/functions assumed to be already defined in the session ─────────
#     HANLE, MERAK, SITE_A, SITE_B : dicts with 'lat', 'lon', 'elevation', 'name'
#     era5                         : xarray.Dataset (specific humidity on p-levels)
#     world_pressure_t2m           : xarray.Dataset (2-m temperature)
#     calculate_pressure           : P(h) ← (P_b, T, L_b, h, h_b)
#     calculate_pwv_pressure       : PWV  ← (q, levels, p_surface)
#     constants  P_b, L_b, h_b     : barometric-formula constants
# ────────────────────────────────────────────────────────────────────────────────

def _rename_valid_time(arr_or_ds):
    """Rename `valid_time` dimension to `time` if necessary."""
    if 'valid_time' in arr_or_ds.dims:
        return arr_or_ds.rename({'valid_time': 'time'})
    return arr_or_ds

def pressure_dynamic(lat, lon, elevation):
    """Barometric-formula surface pressure (Variant A) for an arbitrary site."""
    t2m = world_pressure_t2m['t2m'].interp(latitude=lat, longitude=lon, method='linear')
    p_vals = calculate_pressure(P_b, t2m, L_b, elevation, h_b)
    p_da   = xr.DataArray(p_vals, coords=t2m.coords, dims=t2m.dims, name='sp')
    return _rename_valid_time(p_da)

# ── 1.  Matplotlib setup ───────────────────────────────────────────────────────
plt.rcParams.update({'font.size': 20, 'legend.fontsize': 18})

fig  = plt.figure(figsize=(10, 10))
ax1  = fig.add_subplot(111)

colors  = ['blue', 'green', 'red', 'skyblue']
markers = ['o', 's', '^', 'X']

# ── 2.  Loop over sites with Variant A surface pressure ────────────────────────
for i, site in enumerate([HANLE, MERAK, SITE_A, SITE_B]):
    lat, lon, elev = site['lat'], site['lon'], site['elevation']
    print(f"Elevation for {site['name']}  ({lat:.3f}, {lon:.3f})  = {elev} m")

    p_surface = pressure_dynamic(lat, lon, elev)

    # ERA-5 humidity cube – 2010-2023 – interpolated horizontally
    cube   = era5.interp(latitude=lat, longitude=lon, method='linear') \
                 .sel(time=slice('2010', '2023'))
    q      = cube['q']
    levels = cube['level']

    pwv = calculate_pwv_pressure(q, levels, p_surface)

    # Minimum PWV per calendar year (end-of-year time stamp) 
    annual_min = pwv.resample(time='YE').min()

    ax1.plot(
        annual_min.time, annual_min,
        label  = site['name'],
        color  = colors[i],
        marker = markers[i],
        markersize = 12,
        linestyle='-',
        linewidth = 2.5,
        alpha = 1,
        markerfacecolor = colors[i],
        markeredgewidth = 2,
        markeredgecolor = 'black',
        zorder = 5,
    )

# ── 3.  Reference line, legend, cosmetics ──────────────────────────────────────
ax1.axhline(y=1, color='black', linestyle='--', linewidth=2)

ax1.legend(
    ncol=4, loc='upper center', fontsize=19,
    frameon=True, facecolor='white', edgecolor='black', framealpha=1, shadow=True,
    bbox_to_anchor=(0.5, 0.97)
)

ax1.tick_params(axis='both', direction='in', width=2, length=10, labelsize=22)
ax1.set_ylim(0.45, 1.45)
ax1.minorticks_on()
ax1.tick_params(axis='both', which='minor', direction='in', width=1, length=5)

ax1.set_facecolor('white')
for spine in ax1.spines.values():
    spine.set_linewidth(3)

plt.xlabel('Year', fontsize=22, labelpad=5)
plt.ylabel('Minimum PWV (mm)', fontsize=22, labelpad=5)
plt.tight_layout()
plt.savefig('PWV_Minimum_Annual_Site_Comparison_Ladakh_pressure_level_variantA.pdf', dpi=500)
plt.show()


## Bubble Plot

## Pressure and PWV calculations

### Step 64

This cell defines reusable helper function(s) `calculate_pwv` so later sections can apply the same processing logic consistently.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

def calculate_pwv(q, levels, pressure_threshold):
    """
    Calculate Precipitable Water Vapor (PWV).
    
    Parameters:
    q (xarray.DataArray): Specific humidity data.
    levels (xarray.DataArray): Pressure levels in Pa.
    pressure_threshold (int): Pressure threshold in Pa.
    
    Returns:
    float: Calculated PWV.
    """
    pc = pressure_threshold
    rho_w = 1 
    g = 9.81

    levels = levels * 100  # Convert from hPa to Pa 
    q = q.sel(expver=1)

    idx = int(np.abs(levels - pc).argmin())  
    print(f'idx: {idx}')
    pm = levels[idx]
    print(f'pm: {pm}')
    if pm < pc:
        q_m = q.sel(level=pm/100, method='nearest')
        q_m_plus_1 = q.sel(level=levels[idx + 1]/100, method='nearest') 
        weight = (pc - pm) / (levels[idx + 1] - pm)
        interpolated_q = (1 - weight) * q_m + weight * q_m_plus_1
        interpolated_pwv_unnormalized = interpolated_q * (pc - pm)
        print(f'interpolated_pwv_unnormalized: {interpolated_pwv_unnormalized}')
    elif pm > pc:
        q_m = q.sel(level=pm/100, method='nearest')
        print(f'q_m: {q_m}')
        q_m_minus_1 = q.sel(level=levels[idx - 1]/100, method='nearest') 
        print(f'q_m_minus_1: {q_m_minus_1}')
        weight = (pm - pc) / (pm - levels[idx - 1])
        print(f'weight: {weight}')
        interpolated_pwv_unnormalized = (1 - weight) * q_m + weight * q_m_minus_1
        
        interpolated_pwv_unnormalized = interpolated_pwv_unnormalized * (pc - pm)
        print(f'interpolated_pwv_unnormalized: {interpolated_pwv_unnormalized}')
    else:
        interpolated_pwv_unnormalized = 0 

    mask = levels <= pm  
    q = q.where(mask, drop=True)
    levels = levels.where(mask, drop=True)
    print(f'levels: {levels}')
    dp = np.diff(levels)
    dp = np.append(dp, 0)
    print(f'dp: {dp}')
    q_avg = (q[1:] + q[:-1]) / 2
    print(f'q_avg: {q_avg}')
    integral = np.sum(q_avg * dp, axis=1)
    print(f'integral: {integral}')
    # add the first term from first pressure level to the total value
    first_level_pwv = q.sel(level=levels[0]/100, method='nearest') * (levels[0])
    print(f'first_level_pwv: {first_level_pwv}')
    pwv_corr = (integral + interpolated_pwv_unnormalized+first_level_pwv) / (rho_w * g)
    #pwv_corr = (integral) / (rho_w * g)
  
    return pwv_corr

# Example usage for Mauna Kea
# lat_mauna = 19.82
# lon_mauna = 204.53
lat_hanle = 32.7789
lon_hanle = 78.965
ds_point = era5.sel(latitude=lat_hanle, longitude=lon_hanle, method='nearest')
#ds_point = era5.sel(latitude=lat_mauna, longitude=lon_mauna, method='nearest')
ds_point = ds_point.sel(time=slice("1998-01-01", "2017-12-31"))
#ds_point = ds_point.sel(time=slice("2010-01-01", "2012-12-31"))

pwv_corr = calculate_pwv(ds_point.q, ds_point.level, pressure_threshold=59500)

# Plotting results
plt.figure(figsize=(12, 6))
pwv_corr.plot(color='blue', label='PWV Corrected', linestyle='--', alpha=0.9, marker='o')
plt.xlabel('Time')
plt.ylabel('Precipitable Water Vapor (mm)')
plt.grid(True)
plt.legend()
plt.show()


## Mapping and visualization

### Step 65

This cell subsets the dataset to a selected time, level, region, or site so the next step works with a focused slice of the data.

In [ ]:
# plot the variation of PWV over the pressure levels for lat_hanle and lon_hanle

# Select nearest point to Hanle
lat_hanle = 32.7789
lon_hanle = 78.965
# lat_hanle = 100
# lon_hanle = 200

ds_point = era5.sel(latitude=lat_hanle, longitude=lon_hanle, method='nearest')

# Subset data from 1980 to 2017
ds_point = ds_point.sel(time=slice("2010-01-01", "2010-12-31"))

# plot the variation of PWV over the pressure levels for a given time = 2010-01-01

plt.figure(figsize=(12, 6)) 
ds_point.sel(expver=1).q.plot(x='level', y='time', cmap='jet', yincrease=False)


### Step 66

This cell reads tabular metadata that is later used for site lookup, filtering, or summary reporting.

In [ ]:
# Resample data annually by taking the mean of PWV for each year
pwv_yearly = pwv_corr.resample(time='YE').median()

# Plotting
plt.figure(figsize=(12, 6))
# extract years from the time index
years = pd.to_datetime(pwv_yearly.time.values).year
plt.plot(years, pwv_yearly, marker='o', linestyle='-', color='black', 
         markerfacecolor='black', markeredgewidth=4, markersize=12,label='Hanle ERA5 Median')


# plt.title('Yearly Averaged PWV over Mauna Kea (1980-2017)')
plt.xlabel('Year')
plt.ylabel('Precipitable Water Vapor (mm)')
plt.grid(True)



hanle_yearly_iao = pd.read_csv('hanle_yearly_pwv.txt', sep='\s+', skiprows=1, names=['Year', 'Day', 'Night', 'Mean'])
# Load Hanle data
years = pd.to_datetime(hanle_yearly_iao['Year'], format='%Y').dt.year

plt.plot(years, hanle_yearly_iao['Mean'], marker='o', linestyle='-', color='blue', 
         markerfacecolor='black', markeredgewidth=4, markersize=12,label='Hanle IAO Median')
plt.plot(years, hanle_yearly_iao['Mean'] - pwv_yearly, marker='o', linestyle='-', color='red',)
plt.legend()

plt.show()


## Summary statistics and reporting

### Step 67

This cell computes or evaluates precipitable water vapor (PWV)-related quantities that are central to the site-quality analysis.

In [ ]:
# rms for hanle_yearly_iao['Mean'] - pwv_yearly
res = hanle_yearly_iao['Mean'] - pwv_yearly
rms = np.sqrt(np.mean(res**2))
print(f"RMS: {rms:.2f}")


## Project setup and imports

### Step 68

This cell defines reusable helper function(s) `calculate_pressure` so later sections can apply the same processing logic consistently.

In [ ]:
import math

def calculate_pressure(P_b, T_b, L_b, h, h_b, R_star=8.3144598, g_0=9.80665, M=0.028964425278793993):
    T_h = T_b - L_b * (h - h_b)
    exponent = (g_0 * M) / (R_star * L_b)
    pressure = P_b * (T_h / T_b) ** exponent
    return pressure

# Example usage:
P_b = 101325  # reference pressure at sea level in Pa
T_b = 288.16  # reference temperature at sea level in K
L_b = 0.0065  # temperature lapse rate in K/m
h = 4500      # height at which pressure is calculated in m
h_b = 0       # height of reference level in m

pressure = calculate_pressure(P_b, T_b, L_b, h, h_b)
print(f"Pressure at {h} meters: {pressure:.2f} Pa")


## Mapping and visualization

### Step 69

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
import os
import rasterio
from rasterio.merge import merge
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
from shapely.geometry import box
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
import rioxarray

# File paths for the GeoTIFF files
geotiff_files = ['srtm_52_05/srtm_52_05.tif', 'srtm_52_06/srtm_52_06.tif']

# Check if the files exist in the current directory
for file in geotiff_files:
    if not os.path.isfile(file):
        raise FileNotFoundError(f"File {file} not found in the current directory.")

# Open the GeoTIFF files and merge them
src_files_to_mosaic = [rasterio.open(file) for file in geotiff_files]
mosaic, out_trans = merge(src_files_to_mosaic)

# Get the coordinates of the merged dataset
lon_min, lat_max = out_trans * (0, 0)
lon_max, lat_min = out_trans * (mosaic.shape[2], mosaic.shape[1])
lons = np.linspace(lon_min, lon_max, mosaic.shape[2])
lats = np.linspace(lat_max, lat_min, mosaic.shape[1])

# Convert the mosaic to an xarray DataArray and add geospatial info
elevation_da = xr.DataArray(mosaic[0], coords=[('latitude', lats), ('longitude', lons)], name='elevation')
elevation_da.rio.write_crs(4326, inplace=True)

# Resample the data to a 0.25° x 0.25° grid
grid_size = 0.25
elevation_resampled = elevation_da.coarsen(latitude=int(grid_size / (lat_max - lat_min) * mosaic.shape[1]), 
                                           longitude=int(grid_size / (lon_max - lon_min) * mosaic.shape[2]), 
                                           boundary='trim').mean()

# Ensure the resampled data is aligned to 0.25 degree grid
lons_resampled = np.arange(np.floor(lon_min), np.ceil(lon_max), grid_size)
lats_resampled = np.arange(np.floor(lat_min), np.ceil(lat_max), grid_size)
elevation_resampled = elevation_resampled.interp(longitude=lons_resampled, latitude=lats_resampled, method='linear')

# Fill NaN values resulting from interpolation
elevation_resampled = elevation_resampled.fillna(elevation_resampled.mean())

elevation_resampled.rio.write_crs(4326, inplace=True)

# Load the shape file and define the manual box as a GeoDataFrame
gdf1 = gpd.read_file("shape_files/gadm41_IND_shp/gadm41_IND_1.shp").to_crs(epsg=4326)
manual_box = gpd.GeoDataFrame(geometry=[box(75.9, 31.9, 79.5, 35.7)], crs=gdf1.crs)

# Intersect the box with the shape file
intersect_region = gpd.overlay(manual_box, gdf1, how='intersection')

# Mask the resampled elevation data with the intersected region
elevation_masked = elevation_resampled.rio.clip(intersect_region.geometry, drop=True, all_touched=True)

# Plot the merged elevation map
fig, ax = plt.subplots(figsize=(14, 12), subplot_kw={'projection': ccrs.PlateCarree()})
extent = (72, 81, 31, 37)
img = ax.imshow(elevation_resampled, cmap='terrain', extent=extent, origin='lower')
cbar = plt.colorbar(img, ax=ax, orientation='vertical', pad=0.05)
cbar.set_label('Average Elevation (m)')
ax.set_title('Averaged Elevation Map (0.25° x 0.25° grid) for Whole Region')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.add_feature(cfeature.COASTLINE)

# Plot the masked elevation map for the region of interest
fig, ax = plt.subplots(figsize=(14, 12), subplot_kw={'projection': ccrs.PlateCarree()})
extent = (75.9, 79.5, 31.9, 35.7)
img = ax.imshow(elevation_masked, cmap='jet', extent=(elevation_masked.longitude.min(), elevation_masked.longitude.max(), 
                                                           elevation_masked.latitude.min(), elevation_masked.latitude.max()), origin='lower',
                                                           )
cbar = plt.colorbar(img, ax=ax, orientation='vertical', pad=0.05)
cbar.set_label('Average Elevation (m)')
ax.set_title('Averaged Elevation Map (0.25° x 0.25° grid) for Region of Interest')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.add_feature(cfeature.COASTLINE)

plt.show()


## Analysis workflow

### Step 70

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
T_b
P_b = 101325  # reference pressure at sea level in Pa


## Mapping and visualization

### Step 71

This cell loads dataset(s) `pressure_all.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
surface_pressure_data = xr.open_dataset('pressure_all.nc')

# plot the surface pressure data for the region of interest for average of 2010-2023
data_sp = surface_pressure_data['sp'].sel(time=slice("2010-01-01", "2023-12-31")).mean(dim='time')

fig, ax = plt.subplots(figsize=(14, 12), subplot_kw={'projection': ccrs.PlateCarree()})
extent = (75.9, 79.5, 31.9, 35.7)
img = ax.imshow(data_sp, cmap='jet', extent=(data_sp.longitude.min(), data_sp.longitude.max(), 
                                                           data_sp.latitude.min(), data_sp.latitude.max()), origin='lower',
                                                           )
cbar = plt.colorbar(img, ax=ax, orientation='vertical', pad=0.05)
cbar.set_label('Average Surface Pressure (Pa)')
ax.set_title('Averaged Surface Pressure Map (2010-2023)')

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.add_feature(cfeature.COASTLINE)
ax.set_xticks(np.arange(75.9, 79.5, 0.2))
ax.set_yticks(np.arange(31.9, 35.7, 0.2))
ax.xaxis.set_major_locator(MultipleLocator(0.5))
ax.xaxis.set_minor_locator(AutoMinorLocator(5))
ax.yaxis.set_major_locator(MultipleLocator(0.5))
ax.yaxis.set_minor_locator(AutoMinorLocator(5))
plt.show()


### Step 72

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
import os
import rasterio
from rasterio.merge import merge
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
from shapely.geometry import box
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
import rioxarray
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# File paths for the GeoTIFF files
geotiff_files = ['srtm_52_05/srtm_52_05.tif', 'srtm_52_06/srtm_52_06.tif']

# Check if the files exist in the current directory
for file in geotiff_files:
    if not os.path.isfile(file):
        raise FileNotFoundError(f"File {file} not found in the current directory.")

# Open the GeoTIFF files and merge them
src_files_to_mosaic = [rasterio.open(file) for file in geotiff_files]
mosaic, out_trans = merge(src_files_to_mosaic)

# Get the coordinates of the merged dataset
lon_min, lat_max = out_trans * (0, 0)
lon_max, lat_min = out_trans * (mosaic.shape[2], mosaic.shape[1])
lons = np.linspace(lon_min, lon_max, mosaic.shape[2])
lats = np.linspace(lat_max, lat_min, mosaic.shape[1])

# Convert the mosaic to an xarray DataArray and add geospatial info
elevation_da = xr.DataArray(mosaic[0], coords=[('latitude', lats), ('longitude', lons)], name='elevation')
elevation_da.rio.write_crs(4326, inplace=True)

# Resample the data to a 0.25° x 0.25° grid
grid_size = 0.25
elevation_resampled = elevation_da.coarsen(latitude=int(grid_size / (lat_max - lat_min) * mosaic.shape[1]), 
                                           longitude=int(grid_size / (lon_max - lon_min) * mosaic.shape[2]), 
                                           boundary='trim').mean()

# Ensure the resampled data is aligned to 0.25 degree grid
lons_resampled = np.arange(np.floor(lon_min), np.ceil(lon_max), grid_size)
lats_resampled = np.arange(np.floor(lat_min), np.ceil(lat_max), grid_size)
elevation_resampled = elevation_resampled.interp(longitude=lons_resampled, latitude=lats_resampled, method='linear')

# Fill NaN values resulting from interpolation
elevation_resampled = elevation_resampled.fillna(elevation_resampled.mean())

elevation_resampled.rio.write_crs(4326, inplace=True)

# Load the shape files
gdf1 = gpd.read_file("shape_files/gadm41_IND_shp/gadm41_IND_1.shp").to_crs(epsg=4326)
gdf2 = gpd.read_file("shape_files/India_Shape_Full/India_Country_Boundary.shp").to_crs(epsg=4326)

# Define the manual box as a GeoDataFrame
manual_box = gpd.GeoDataFrame(geometry=[box(75.9, 31.9, 79.5, 35.7)], crs=gdf1.crs)

# Intersect the box with gdf1
intersect_region = gpd.overlay(manual_box, gdf1, how='intersection')

# Difference between gdf2 and gdf1
difference_region_1 = gpd.overlay(gdf2, gdf1, how='difference')

# Difference between the bounding box of the whole map and gdf2
full_extent_box = box(72, 31, 81, 37)
full_extent_gdf = gpd.GeoDataFrame(geometry=[full_extent_box], crs=gdf1.crs)
difference_region_2 = gpd.overlay(full_extent_gdf, gdf2, how='difference')

# Mask the resampled elevation data with the intersected region
elevation_masked = elevation_resampled.rio.clip(intersect_region.geometry, drop=True, all_touched=True)

# Set up the plot
fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={'projection': ccrs.PlateCarree()},
                       facecolor='white', edgecolor='black', linewidth=2,
                       frameon=True, dpi=500,
                       gridspec_kw={'wspace': 0.2, 'hspace': 0.2})

# Plotting the elevation data as shaded relief
elev_img = elevation_masked.plot.imshow(ax=ax, cmap='jet', zorder=0, add_colorbar=False,
                                        vmin = 2000, vmax = 5500, alpha=1)

# Plotting the shapefiles
gdf1.boundary.plot(ax=ax, linewidth=2, edgecolor='blue', zorder=3)
gdf2.boundary.plot(ax=ax, linewidth=2, edgecolor='red', zorder=3)

# Plot background colors
ax.add_geometries(intersect_region.geometry, crs=ccrs.PlateCarree(), facecolor='None', edgecolor='black', linewidth=2, label='Search Area', zorder=2)
ax.add_geometries(difference_region_1.geometry, crs=ccrs.PlateCarree(), facecolor='lightgreen', edgecolor='green', label='Disputed Territory', zorder=2)
ax.add_geometries(difference_region_2.geometry, crs=ccrs.PlateCarree(), facecolor='lightgrey', edgecolor='grey', label='Outside India', zorder=2)

# Define the rectangle for the specific region (behind the colored regions)
rectangle = plt.Rectangle((75.9, 31.9), 3.8, 4, fill=False, edgecolor='black', linewidth=3, zorder=1)
ax.add_patch(rectangle)

# Adding colorbar for elevation
ax_cb_elevation = inset_axes(ax, width="3.5%", height="95%", loc='center right',
                              bbox_to_anchor=(0.05, 0., 1, 1), bbox_transform=ax.transAxes, borderpad=0)
cbar_elevation = plt.colorbar(elev_img, cax=ax_cb_elevation, orientation='vertical', drawedges=False)
cbar_elevation.set_label('Elevation (m)')

# Adding geographic features
ax.add_feature(cfeature.RIVERS, zorder=3)
ax.add_feature(cfeature.COASTLINE, zorder=3)
ax.add_feature(cfeature.LAKES, alpha=0.5, zorder=3)
ax.add_feature(cfeature.LAND)

# Adding gridlines
gl = ax.gridlines(draw_labels=True, linewidth=1, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = True
gl.right_labels = False
gl.bottom_labels = False
gl.xlocator = plt.MaxNLocator(integer=True)
gl.ylocator = plt.MaxNLocator(integer=True)

# Make the surrounding box thicker
for spine in ax.spines.values():
    spine.set_linewidth(2.5)

# Setting extent
ax.set_extent([75.4, 80, 31.6, 36])

# Adjust layout
plt.subplots_adjust(left=0.1, right=0.83, top=0.9, bottom=0.17)

# Remove title from the plot
ax.set_title('')
plt.tight_layout()

# Show the plot
#plt.savefig('region_of_interest_elevation_map.pdf', dpi=600)
plt.show()



# Close the files
for src in src_files_to_mosaic:
    src.close()


### Step 73

This cell loads dataset(s) `pressure_all.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
import os
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
from shapely.geometry import box
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import rioxarray

# Load surface pressure data
surface_pressure_data = xr.open_dataset('pressure_all.nc')

# Ensure the surface pressure data has geospatial info
surface_pressure_data = surface_pressure_data.rio.write_crs(4326)

# Average the surface pressure data over time
surface_pressure_mean = surface_pressure_data['sp'].mean(dim='time')

# Get the coordinates of the dataset
lon_min, lon_max = surface_pressure_mean.longitude.min().item(), surface_pressure_mean.longitude.max().item()
lat_min, lat_max = surface_pressure_mean.latitude.min().item(), surface_pressure_mean.latitude.max().item()

# Resample the data to a 0.25° x 0.25° grid
grid_size = 0.25
latitude_factor = int(grid_size / (lat_max - lat_min) * surface_pressure_mean.sizes['latitude'])
longitude_factor = int(grid_size / (lon_max - lon_min) * surface_pressure_mean.sizes['longitude'])

surface_pressure_resampled = surface_pressure_mean.coarsen(
    latitude=latitude_factor,
    longitude=longitude_factor,
    boundary='trim'
).mean()

# Ensure the resampled data is aligned to a 0.25 degree grid
lons_resampled = np.arange(np.floor(lon_min), np.ceil(lon_max), grid_size)
lats_resampled = np.arange(np.floor(lat_min), np.ceil(lat_max), grid_size)
surface_pressure_resampled = surface_pressure_resampled.interp(longitude=lons_resampled, latitude=lats_resampled, method='linear')

# Fill NaN values resulting from interpolation
surface_pressure_resampled = surface_pressure_resampled.fillna(surface_pressure_resampled.mean())

surface_pressure_resampled.rio.write_crs(4326, inplace=True)

# Load the shape files
gdf1 = gpd.read_file("shape_files/gadm41_IND_shp/gadm41_IND_1.shp").to_crs(epsg=4326)
gdf2 = gpd.read_file("shape_files/India_Shape_Full/India_Country_Boundary.shp").to_crs(epsg=4326)

# Define the manual box as a GeoDataFrame
manual_box = gpd.GeoDataFrame(geometry=[box(75.9, 31.9, 79.5, 35.7)], crs=gdf1.crs)

# Intersect the box with gdf1
intersect_region = gpd.overlay(manual_box, gdf1, how='intersection')

# Difference between gdf2 and gdf1
difference_region_1 = gpd.overlay(gdf2, gdf1, how='difference')

# Difference between the bounding box of the whole map and gdf2
full_extent_box = box(72, 31, 81, 37)
full_extent_gdf = gpd.GeoDataFrame(geometry=[full_extent_box], crs=gdf1.crs)
difference_region_2 = gpd.overlay(full_extent_gdf, gdf2, how='difference')

# Mask the resampled surface pressure data with the intersected region
surface_pressure_masked = surface_pressure_resampled.rio.clip(intersect_region.geometry, drop=True, all_touched=True)

# Set up the plot
fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={'projection': ccrs.PlateCarree()},
                       facecolor='white', edgecolor='black', linewidth=2,
                       frameon=True, dpi=500,
                       gridspec_kw={'wspace': 0.2, 'hspace': 0.2})

# Plotting the surface pressure data using imshow
pressure_img = surface_pressure_masked.plot.imshow(ax=ax, cmap='seismic', zorder=0, add_colorbar=False,
                                                   vmin=surface_pressure_masked.min().item()-4000, vmax=70000, alpha=1)

# Plotting the shapefiles
gdf1.boundary.plot(ax=ax, linewidth=2, edgecolor='blue', zorder=3)
gdf2.boundary.plot(ax=ax, linewidth=2, edgecolor='red', zorder=3)

# Plot background colors
ax.add_geometries(intersect_region.geometry, crs=ccrs.PlateCarree(), facecolor='None', edgecolor='black', linewidth=2, label='Search Area', zorder=2)
ax.add_geometries(difference_region_1.geometry, crs=ccrs.PlateCarree(), facecolor='lightgreen', edgecolor='green', label='Disputed Territory', zorder=2)
ax.add_geometries(difference_region_2.geometry, crs=ccrs.PlateCarree(), facecolor='lightgrey', edgecolor='grey', label='Outside India', zorder=2)

# Define the rectangle for the specific region (behind the colored regions)
rectangle = plt.Rectangle((75.9, 31.9), 3.8, 4, fill=False, edgecolor='black', linewidth=3, zorder=1)
ax.add_patch(rectangle)

# Adding colorbar for surface pressure
ax_cb_pressure = inset_axes(ax, width="3.5%", height="95%", loc='center right',
                            bbox_to_anchor=(0.05, 0., 1, 1), bbox_transform=ax.transAxes, borderpad=0)
cbar_pressure = plt.colorbar(pressure_img, cax=ax_cb_pressure, orientation='vertical', drawedges=False)
cbar_pressure.set_label('Surface Pressure (Pa)', fontsize=14)
cbar_pressure.ax.tick_params(labelsize=12) 

# Adding geographic features
ax.add_feature(cfeature.RIVERS, zorder=3)
ax.add_feature(cfeature.COASTLINE, zorder=3)
ax.add_feature(cfeature.LAKES, alpha=0.5, zorder=3)
ax.add_feature(cfeature.LAND)

# Adding gridlines
gl = ax.gridlines(draw_labels=True, linewidth=1, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = True
gl.right_labels = False
gl.bottom_labels = False
gl.xlocator = plt.MaxNLocator(integer=True)
gl.ylocator = plt.MaxNLocator(integer=True)
gl.xlabel_style = {'size': 14}
gl.ylabel_style = {'size': 14}

# Make the surrounding box thicker
for spine in ax.spines.values():
    spine.set_linewidth(2.5)

# Setting extent
ax.set_extent([75.4, 80, 31.6, 36])

# Adjust layout
plt.subplots_adjust(left=0.1, right=0.83, top=0.9, bottom=0.17)

# Remove title from the plot
ax.set_title('')
plt.tight_layout()

# Show the plot
plt.savefig('region_of_interest_surface_pressure_map.pdf', dpi=600)
plt.show()


## Surface Pressure Map

## Interpolation and extraction

### Step 74

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
if 'valid_time' in surface_pressure_data.dims:
    # Rename 'valid_time' to 'time' if it exists in dimensions
    surface_pressure_data = surface_pressure_data.rename({'valid_time': 'time'})
surface_pressure_data


## Mapping and visualization

### Step 75

This cell loads dataset(s) `pressure_all.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
import os
import rasterio
from rasterio.features import geometry_mask
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
from shapely.geometry import box, mapping, Polygon
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
import rioxarray
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# Load surface pressure data
surface_pressure_data = xr.open_dataset('pressure_all.nc')

# Ensure the surface pressure data has geospatial info
surface_pressure_data = surface_pressure_data.rio.write_crs(4326)

# Average the surface pressure data over time
surface_pressure_mean = surface_pressure_data['sp'].mean(dim='time')

# Get the coordinates of the dataset
lon_min, lon_max = surface_pressure_mean.longitude.min().item(), surface_pressure_mean.longitude.max().item()
lat_min, lat_max = surface_pressure_mean.latitude.min().item(), surface_pressure_mean.latitude.max().item()

# Resample the data to a 0.25° x 0.25° grid
grid_size = 0.25
latitude_factor = int(grid_size / (lat_max - lat_min) * surface_pressure_mean.sizes['latitude'])
longitude_factor = int(grid_size / (lon_max - lon_min) * surface_pressure_mean.sizes['longitude'])

surface_pressure_resampled = surface_pressure_mean.coarsen(
    latitude=latitude_factor,
    longitude=longitude_factor,
    boundary='trim'
).mean()

# Ensure the resampled data is aligned to a 0.25 degree grid
lons_resampled = np.arange(np.floor(lon_min), np.ceil(lon_max), grid_size)
lats_resampled = np.arange(np.floor(lat_min), np.ceil(lat_max), grid_size)
surface_pressure_resampled = surface_pressure_resampled.interp(longitude=lons_resampled, latitude=lats_resampled, method='linear')

# Fill NaN values resulting from interpolation
surface_pressure_resampled = surface_pressure_resampled.fillna(surface_pressure_resampled.mean())

surface_pressure_resampled.rio.write_crs(4326, inplace=True)

# Load the shape files
gdf1 = gpd.read_file("shape_files/gadm41_IND_shp/gadm41_IND_1.shp").to_crs(epsg=4326)
gdf2 = gpd.read_file("India__State_Boundary_2021_/India%3A_State_Boundary_2021_.shp").to_crs(epsg=4326)

# manual_box = gpd.GeoDataFrame(geometry=[box(75.9, 31.9, 79.5, 35.7)], crs=gdf1.crs)

# intersect_region = gpd.overlay(manual_box, gdf1, how='intersection')
# difference_region_1 = gpd.overlay(gdf2, gdf1, how='difference')

# full_extent_box = box(72.3, 31, 81, 37.5)  # Adjust this to your desired full map extent
# full_extent_gdf = gpd.GeoDataFrame(geometry=[full_extent_box], crs=gdf1.crs)
# difference_region_2 = gpd.overlay(full_extent_gdf, gdf2, how='difference')





# Load coordinates from the CSV file and create a polygon
coords_df = pd.read_csv('plot-data.csv')
polygon_coords = [(x, y) for x, y in zip(coords_df['x'], coords_df['y'])]
region_polygon = Polygon(polygon_coords)
region_gdf = gpd.GeoDataFrame(geometry=[region_polygon], crs=gdf1.crs)

# Combine the outer boundary with the region polygon
combined_region = gpd.overlay(gdf2, region_gdf, how='intersection')

# Rasterize the combined region to create a mask
transform = surface_pressure_resampled.rio.transform()
out_shape = (surface_pressure_resampled.sizes['latitude'], surface_pressure_resampled.sizes['longitude'])
rasterized_mask = geometry_mask([mapping(region_polygon)],
                                transform=transform,
                                invert=True,
                                out_shape=out_shape,
                                all_touched=True)

# Apply the mask to the surface pressure data
#surface_pressure_masked = surface_pressure_resampled.where(rasterized_mask)

surface_pressure_masked = surface_pressure_resampled.rio.clip(combined_region.geometry.apply(mapping), combined_region.crs,drop= True)

# Set up the plot
fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={'projection': ccrs.PlateCarree()},
                       facecolor='white', edgecolor='black', linewidth=2,
                       frameon=True, dpi=500,
                       gridspec_kw={'wspace': 0.2, 'hspace': 0.2})

# Plotting the surface pressure data using imshow
pressure_img = surface_pressure_masked.plot.imshow(ax=ax, cmap='seismic', zorder=0, add_colorbar=False,
                                                   vmin=surface_pressure_masked.min().item() - 4000, vmax=70000, alpha=1)

# Plotting the shapefiles
gdf2.boundary.plot(ax=ax, linewidth=2, edgecolor='red', zorder=3)

# Plot background colors
ax.add_geometries(combined_region.geometry, crs=ccrs.PlateCarree(), facecolor='None', edgecolor='black', linewidth=2, label='Search Area', zorder=2)
ax.add_geometries(difference_region_2.geometry, crs=ccrs.PlateCarree(), facecolor='lightgreen', edgecolor='grey', label='Outside India', zorder=1)

# Adding colorbar for surface pressure
ax_cb_pressure = inset_axes(ax, width="3.5%", height="95%", loc='center right',
                            bbox_to_anchor=(0.05, 0., 1, 1), bbox_transform=ax.transAxes, borderpad=0)
cbar_pressure = plt.colorbar(pressure_img, cax=ax_cb_pressure, orientation='vertical', drawedges=False)
cbar_pressure.set_label('Surface Pressure (Pa)', fontsize=14)
cbar_pressure.ax.tick_params(labelsize=12)

# Adding geographic features
ax.add_feature(cfeature.RIVERS, zorder=3)
ax.add_feature(cfeature.COASTLINE, zorder=3)
ax.add_feature(cfeature.LAKES, alpha=0.5, zorder=3)
ax.add_feature(cfeature.LAND)

# Adding gridlines
gl = ax.gridlines(draw_labels=True, linewidth=1, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = True
gl.right_labels = False
gl.bottom_labels = False
gl.xlocator = plt.MaxNLocator(integer=True)
gl.ylocator = plt.MaxNLocator(integer=True)
gl.xlabel_style = {'size': 14}
gl.ylabel_style = {'size': 14}

# Make the surrounding box thicker
for spine in ax.spines.values():
    spine.set_linewidth(2.5)

# Setting extent
ax.set_extent([75.4, 80, 31.6, 36])

# Adjust layout
plt.subplots_adjust(left=0.1, right=0.83, top=0.9, bottom=0.17)

# Remove title from the plot
ax.set_title('')
plt.tight_layout()

# Show the plot
plt.savefig('region_of_interest_surface_pressure_map_corrected.pdf', dpi=600)
plt.show()


### Step 76

This cell loads dataset(s) `pressure_all_25.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
from shapely.geometry import Polygon, box, mapping
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# ─────────────────────────────────────────────────────────────────────────────
# 1) Publication‐quality LaTeX text
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
})

# ─────────────────────────────────────────────────────────────────────────────
# 2) Load & average surface pressure
ds = xr.open_dataset('pressure_all_25.nc').rio.write_crs(4326)
ds = ds.rename({'valid_time': 'time'})
sp = ds['sp'].mean(dim='time')

# ─────────────────────────────────────────────────────────────────────────────
# 3) Interpolate to 0.25° grid
lon0, lon1 = float(sp.longitude.min()), float(sp.longitude.max())
lat0, lat1 = float(sp.latitude.min()),  float(sp.latitude.max())
grid = 0.25
lons = np.arange(np.floor(lon0), np.ceil(lon1)+grid, grid)
lats = np.arange(np.floor(lat0), np.ceil(lat1)+grid, grid)
sp_grid = sp.interp(longitude=lons, latitude=lats, method='linear').fillna(sp.mean())
sp_grid.rio.write_crs(4326, inplace=True)

# ─────────────────────────────────────────────────────────────────────────────
# 4) Build search‐polygon
coords_df = pd.read_csv('plot-data.csv')
poly = Polygon(zip(coords_df.x, coords_df.y))
search_area = gpd.GeoDataFrame(geometry=[poly], crs="EPSG:4326")

# ─────────────────────────────────────────────────────────────────────────────
# 5) Clip pressure to search polygon
sp_clip = sp_grid.rio.clip(search_area.geometry.map(mapping),
                           search_area.crs, drop=True)

# ─────────────────────────────────────────────────────────────────────────────
# 6) Prepare “outside” fill (light yellow)
full_extent = box(75.8, 31.8, 79.5, 35.6)
outside = gpd.GeoDataFrame(geometry=[full_extent], crs="EPSG:4326")
# subtract country interior so that only outside shows
gdf_country = gpd.read_file("India__State_Boundary_2021_/India%3A_State_Boundary_2021_.shp") \
                .to_crs(epsg=4326)
outside = gpd.overlay(outside, gdf_country, how='difference')

# ─────────────────────────────────────────────────────────────────────────────
# 7) Plot
fig, ax = plt.subplots( figsize=(10,8),
                        subplot_kw={'projection': ccrs.PlateCarree()},
                        dpi=500 )

# a) outside‐India fill
ax.add_geometries(outside.geometry, crs=ccrs.PlateCarree(),
                  facecolor='lightyellow', edgecolor='none', zorder=0)

# b) clipped pressure field
img = sp_clip.plot.imshow(ax=ax, cmap='seismic',
                          add_colorbar=False, zorder=1,vmin = 45000, vmax = 70000,)

# c) thick black border around search polygon
# ax.add_geometries(search_area.geometry, crs=ccrs.PlateCarree(),
#                   edgecolor='black', facecolor='none',
#                   linewidth=3, zorder=2)

# d) geographic base features
ax.add_feature(cfeature.LAND,      zorder=0, alpha=0.7)
ax.add_feature(cfeature.LAKES,     zorder=3, alpha=0.7)
ax.add_feature(cfeature.RIVERS,    zorder=3)
ax.add_feature(cfeature.COASTLINE, zorder=3)

# e) colorbar
cax = inset_axes(ax, width="3%", height="99%", loc='center right',
                 bbox_to_anchor=(0.07,0,1,1), bbox_transform=ax.transAxes)
cbar = plt.colorbar(img, cax=cax, orientation='vertical', drawedges=False,extend='both')
cbar.set_label(r'Mean Surface Pressure (Pa)', fontsize=20)
cbar.ax.tick_params(labelsize=18)


# cax = ax.inset_axes([1.02, 0.15, 0.03, 0.7]) # Positioned cleanly to the right
# cbar = plt.colorbar(img, cax=cax, orientation='vertical', extend='both')
# cbar.set_label(r'\textbf{Mean Surface Pressure (Pa)}', fontsize=20)
# # cbar.ax.tick_params(labelsize=18)

# f) gridlines + labels
gl = ax.gridlines(draw_labels=True, linestyle='--',
                  color='gray', alpha=0.6)
gl.top_labels    = True
gl.right_labels  = False
gl.bottom_labels = False
gl.xlocator = plt.MaxNLocator(integer=True)
gl.ylocator = plt.MaxNLocator(integer=True)
gl.xlabel_style = {'size':18}
gl.ylabel_style = {'size':18}
for spine in ax.spines.values():
    spine.set_linewidth(2.5)

# on top right corner add a box saying 2010-2025 April 
ax.text(0.98, 0.98, r'2010-2025 April', transform=ax.transAxes,
        fontsize=17, ha='right', va='top',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'),)


# g) final extent & layout
ax.set_extent([75.8,79.5,31.8,35.6], crs=ccrs.PlateCarree())
plt.subplots_adjust(left=0.1, right=0.83, top=0.9, bottom=0.17)
ax.set_title('')
plt.savefig('region_of_interest_surface_pressure_map_corrected.png',
            dpi=600, bbox_inches='tight')
plt.show()


### Step 77

This cell loads dataset(s) `pressure_all_25.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
from shapely.geometry import Polygon, box, mapping
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# ─────────────────────────────────────────────────────────────────────────────
# 1) Publication‐quality LaTeX text
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
})

# ─────────────────────────────────────────────────────────────────────────────
# 2) Load & average surface pressure
ds = xr.open_dataset('pressure_all_25.nc').rio.write_crs(4326)
ds = ds.rename({'valid_time': 'time'})
sp_mean = ds['sp'].mean(dim='time')
sp_std = ds['sp'].std(dim='time') # Calculate Standard Deviation here

# ─────────────────────────────────────────────────────────────────────────────
# 3) Interpolate to 0.25° grid
lon0, lon1 = float(sp_mean.longitude.min()), float(sp_mean.longitude.max())
lat0, lat1 = float(sp_mean.latitude.min()),  float(sp_mean.latitude.max())
grid = 0.25
lons = np.arange(np.floor(lon0), np.ceil(lon1)+grid, grid)
lats = np.arange(np.floor(lat0), np.ceil(lat1)+grid, grid)
sp_mean_grid = sp_mean.interp(longitude=lons, latitude=lats, method='linear').fillna(sp_mean.mean())
sp_std_grid = sp_std.interp(longitude=lons, latitude=lats, method='linear').fillna(sp_std.mean()) # Interpolate Std Dev as well
sp_mean_grid.rio.write_crs(4326, inplace=True)
sp_std_grid.rio.write_crs(4326, inplace=True)

# ─────────────────────────────────────────────────────────────────────────────
# 4) Build search‐polygon
coords_df = pd.read_csv('plot-data.csv')
poly = Polygon(zip(coords_df.x, coords_df.y))
search_area = gpd.GeoDataFrame(geometry=[poly], crs="EPSG:4326")

# ─────────────────────────────────────────────────────────────────────────────
# 5) Clip pressure to search polygon
sp_mean_clip = sp_mean_grid.rio.clip(search_area.geometry.map(mapping),
                                     search_area.crs, drop=True)
sp_std_clip = sp_std_grid.rio.clip(search_area.geometry.map(mapping),
                                   search_area.crs, drop=True) # Clip Std Dev as well

# ─────────────────────────────────────────────────────────────────────────────
# 6) Prepare “outside” fill (light yellow)
full_extent = box(75.8, 31.8, 79.5, 35.6)
outside = gpd.GeoDataFrame(geometry=[full_extent], crs="EPSG:4326")
# subtract country interior so that only outside shows
gdf_country = gpd.read_file("India__State_Boundary_2021_/India%3A_State_Boundary_2021_.shp") \
                .to_crs(epsg=4326)
outside = gpd.overlay(outside, gdf_country, how='difference')

# ─────────────────────────────────────────────────────────────────────────────
# 7) Plot
fig, ax = plt.subplots( figsize=(10,8),
                        subplot_kw={'projection': ccrs.PlateCarree()},
                        dpi=500 )

# a) outside‐India fill
ax.add_geometries(outside.geometry, crs=ccrs.PlateCarree(),
                  facecolor='lightyellow', edgecolor='none', zorder=0)

# b) clipped pressure field
img = sp_mean_clip.plot.imshow(ax=ax, cmap='seismic',
                               add_colorbar=False, zorder=1, vmin=45000, vmax=70000)

# c) thick black border around search polygon
# ax.add_geometries(search_area.geometry, crs=ccrs.PlateCarree(),
#                   edgecolor='black', facecolor='none',
#                   linewidth=3, zorder=2)

# d) geographic base features
ax.add_feature(cfeature.LAND,      zorder=0, alpha=0.7)
ax.add_feature(cfeature.LAKES,     zorder=3, alpha=0.7)
ax.add_feature(cfeature.RIVERS,    zorder=3)
ax.add_feature(cfeature.COASTLINE, zorder=3)

# e) colorbar
cax = inset_axes(ax, width="3%", height="99%", loc='center right',
                 bbox_to_anchor=(0.07,0,1,1), bbox_transform=ax.transAxes)
cbar = plt.colorbar(img, cax=cax, orientation='vertical', drawedges=False, extend='both')
cbar.set_label(r'Mean Surface Pressure (Pa)', fontsize=20)
cbar.ax.tick_params(labelsize=18)

# f) gridlines + labels
gl = ax.gridlines(draw_labels=True, linestyle='--',
                  color='gray', alpha=0.6)
gl.top_labels    = False # Changed to False for a cleaner look
gl.right_labels  = False
gl.bottom_labels = True # Changed to True
gl.left_labels   = True # Changed to True
gl.xlocator = plt.MaxNLocator(integer=True)
gl.ylocator = plt.MaxNLocator(integer=True)
gl.xlabel_style = {'size':18}
gl.ylabel_style = {'size':18}
for spine in ax.spines.values():
    spine.set_linewidth(2.5)

# g) on top right corner add a box saying 2010-2025 April
ax.text(0.98, 0.98, r'2010-2025 April', transform=ax.transAxes,
        fontsize=17, ha='right', va='top',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))

# h) NEW SECTION: Add Standard Deviation text to each pixel
for lat_val in sp_std_clip.latitude:
    for lon_val in sp_std_clip.longitude:
        # Get the standard deviation value for the pixel
        std_val = sp_std_clip.sel(latitude=lat_val, longitude=lon_val).item()
        # Check if the value is not NaN before plotting
        if not np.isnan(std_val):
            # Format the text to be an integer
            text_val = f'{std_val:.0f}'
            # Add text with a black outline for readability
            ax.text(lon_val, lat_val, text_val,
                    color='white', fontsize=12,
                    ha='center', va='center',
                    transform=ccrs.PlateCarree(),
                    path_effects=[pe.withStroke(linewidth=2, foreground="black")],
                    zorder=10) # High zorder to ensure it's on top

# i) final extent & layout
ax.set_extent([75.8,79.5,31.8,35.6], crs=ccrs.PlateCarree())
plt.subplots_adjust(left=0.1, right=0.83, top=0.9, bottom=0.17)
ax.set_title('')
plt.savefig('region_of_interest_surface_pressure_map_with_std.pdf',
            dpi=600, bbox_inches='tight')
plt.show()


### Step 78

This cell loads dataset(s) `pressure_all_25.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
from shapely.geometry import Polygon, box, mapping
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# ─────────────────────────────────────────────────────────────────────────────
# 1) Publication‐quality LaTeX text
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
})

# ─────────────────────────────────────────────────────────────────────────────
# 2) Load & average surface pressure
ds = xr.open_dataset('pressure_all_25.nc').rio.write_crs(4326)
ds = ds.rename({'valid_time': 'time'})
sp_mean = ds['sp'].mean(dim='time')
sp_std = ds['sp'].std(dim='time') # Calculate Standard Deviation here

# ─────────────────────────────────────────────────────────────────────────────
# 3) Interpolate to 0.25° grid
lon0, lon1 = float(sp_mean.longitude.min()), float(sp_mean.longitude.max())
lat0, lat1 = float(sp_mean.latitude.min()),  float(sp_mean.latitude.max())
grid = 0.25
lons = np.arange(np.floor(lon0), np.ceil(lon1)+grid, grid)
lats = np.arange(np.floor(lat0), np.ceil(lat1)+grid, grid)
sp_mean_grid = sp_mean.interp(longitude=lons, latitude=lats, method='linear').fillna(sp_mean.mean())
sp_std_grid = sp_std.interp(longitude=lons, latitude=lats, method='linear').fillna(sp_std.mean()) # Interpolate Std Dev as well
sp_mean_grid.rio.write_crs(4326, inplace=True)
sp_std_grid.rio.write_crs(4326, inplace=True)

# ─────────────────────────────────────────────────────────────────────────────
# 4) Build search‐polygon
coords_df = pd.read_csv('plot-data.csv')
poly = Polygon(zip(coords_df.x, coords_df.y))
search_area = gpd.GeoDataFrame(geometry=[poly], crs="EPSG:4326")

# ─────────────────────────────────────────────────────────────────────────────
# 5) Clip pressure to search polygon
sp_mean_clip = sp_mean_grid.rio.clip(search_area.geometry.map(mapping),
                                     search_area.crs, drop=True)
sp_std_clip = sp_std_grid.rio.clip(search_area.geometry.map(mapping),
                                   search_area.crs, drop=True) # Clip Std Dev as well

# ─────────────────────────────────────────────────────────────────────────────
# 6) Prepare “outside” fill (light yellow)
full_extent = box(75.8, 31.8, 79.5, 35.6)
outside = gpd.GeoDataFrame(geometry=[full_extent], crs="EPSG:4326")
# subtract country interior so that only outside shows
gdf_country = gpd.read_file("India__State_Boundary_2021_/India%3A_State_Boundary_2021_.shp") \
                .to_crs(epsg=4326)
outside = gpd.overlay(outside, gdf_country, how='difference')

# ─────────────────────────────────────────────────────────────────────────────
# 7) Plot
fig, ax = plt.subplots( figsize=(10,8),
                        subplot_kw={'projection': ccrs.PlateCarree()},
                        dpi=500 )

# a) outside‐India fill
ax.add_geometries(outside.geometry, crs=ccrs.PlateCarree(),
                  facecolor='lightyellow', edgecolor='none', zorder=0)

# b) clipped pressure field
img = sp_mean_clip.plot.imshow(ax=ax, cmap='seismic',
                               add_colorbar=False, zorder=1, vmin=45000, vmax=70000)

# c) thick black border around search polygon
# ax.add_geometries(search_area.geometry, crs=ccrs.PlateCarree(),
#                   edgecolor='black', facecolor='none',
#                   linewidth=3, zorder=2)

# d) geographic base features
ax.add_feature(cfeature.LAND,      zorder=0, alpha=0.7)
ax.add_feature(cfeature.LAKES,     zorder=3, alpha=0.7)
ax.add_feature(cfeature.RIVERS,    zorder=3)
ax.add_feature(cfeature.COASTLINE, zorder=3)

# e) colorbar
cax = inset_axes(ax, width="3%", height="99%", loc='center right',
                 bbox_to_anchor=(0.07,0,1,1), bbox_transform=ax.transAxes)
cbar = plt.colorbar(img, cax=cax, orientation='vertical', drawedges=False, extend='both')
cbar.set_label(r'Mean Surface Pressure (Pa)', fontsize=20)
cbar.ax.tick_params(labelsize=18)

# f) gridlines + labels
gl = ax.gridlines(draw_labels=True, linestyle='--',
                  color='gray', alpha=0.6)
gl.top_labels    = False # Changed to False for a cleaner look
gl.right_labels  = False
gl.bottom_labels = True # Changed to True
gl.left_labels   = True # Changed to True
gl.xlocator = plt.MaxNLocator(integer=True)
gl.ylocator = plt.MaxNLocator(integer=True)
gl.xlabel_style = {'size':18}
gl.ylabel_style = {'size':18}
for spine in ax.spines.values():
    spine.set_linewidth(2.5)

# g) on top right corner add a box saying 2010-2025 April
ax.text(0.98, 0.98, r'2010-2025 April', transform=ax.transAxes,
        fontsize=17, ha='right', va='top',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))

# h) NEW SECTION: Add Standard Deviation text with dynamic color
# Define the normalization based on the colormap's vmin and vmax
norm = mpl.colors.Normalize(vmin=45000, vmax=70000)

for lat_val in sp_std_clip.latitude:
    for lon_val in sp_std_clip.longitude:
        # Get the standard deviation and mean pressure values for the pixel
        std_val = sp_std_clip.sel(latitude=lat_val, longitude=lon_val).item()
        mean_val = sp_mean_clip.sel(latitude=lat_val, longitude=lon_val).item()

        # Check if the values are not NaN before plotting
        if not np.isnan(std_val) and not np.isnan(mean_val):
            # Normalize the background value (mean pressure) to 0-1 range
            # This tells us how "light" or "dark" the background color is.
            # On the 'seismic' map, values near 0.5 are light, and values near 0 or 1 are dark.
            normalized_mean = norm(mean_val)

            # Set text color to black for light backgrounds, white for dark backgrounds
            text_color = "black" if 0.2 < normalized_mean < 0.8 else "white"

            # Format the text to be an integer
            text_val = f'{std_val:.0f}'

            # Add text. We no longer need the path_effects because the contrast is handled by color.
            ax.text(lon_val, lat_val, text_val,
                    color=text_color, fontsize=10, # Fontsize reduced slightly for clarity
                    ha='center', va='center',
                    transform=ccrs.PlateCarree(),
                    fontweight='bold', # Bold font helps it stand out
                    zorder=10) # High zorder to ensure it's on top

# i) final extent & layout
ax.set_extent([75.8,79.5,31.8,35.6], crs=ccrs.PlateCarree())
plt.subplots_adjust(left=0.1, right=0.83, top=0.9, bottom=0.17)
ax.set_title('')
plt.savefig('region_of_interest_surface_pressure_map_with_std.pdf',
            dpi=300, bbox_inches='tight')
plt.show()


## Interpolation and extraction

### Step 79

This cell subsets the dataset to a selected time, level, region, or site so the next step works with a focused slice of the data.

In [ ]:

# h) NEW SECTION: Add Standard Deviation text to each pixel
for lat_val in sp_std_clip.latitude:
    for lon_val in sp_std_clip.longitude:
        # Get the standard deviation value for the pixel
        std_val = sp_std_clip.sel(latitude=lat_val, longitude=lon_val).item()
        # Check if the value is not NaN before plotting
        if not np.isnan(std_val):
            # Format the text to be an integer
            text_val = f'{std_val:.0f}'
            # Add text with a black outline for readability
            ax.text(lon_val, lat_val, text_val,
                    color='white', fontsize=12,
                    ha='center', va='center',
                    transform=ccrs.PlateCarree(),
                    path_effects=[pe.withStroke(linewidth=2, foreground="black")],
                    zorder=10) # High zorder to ensure it's on top


## Summary statistics and reporting

### Step 80

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2b) Interpolate sp to site coordinates, then time-average
# (Place this right after you've opened ds and before plotting)
# ─────────────────────────────────────────────────────────────────────────────

# Ensure latitude is increasing (xarray's linear interp expects monotonic coords)
sp_alltime = ds['sp'].sortby('latitude')  # (time, latitude, longitude) in Pa

# Define your four sites as a list of dicts
SITES = [
    { 'name':'Hanle',  'lat':32.7789, 'lon':78.9650, 'elev_m':4500 },
    { 'name':'Merak',  'lat':33.7828, 'lon':78.5778, 'elev_m':4310 },
    { 'name':'Site A', 'lat':34.25,   'lon':78.75,   'elev_m':4800 },
    { 'name':'Site B', 'lat':32.5,    'lon':79.0,    'elev_m':4500 },
]

# Vectorized coordinates for bilinear interpolation
site_names = [s['name'] for s in SITES]
site_lats  = xr.DataArray([s['lat'] for s in SITES],  dims='site', coords={'site': site_names})
site_lons  = xr.DataArray([s['lon'] for s in SITES],  dims='site', coords={'site': site_names})

# Interpolate at each timestamp (bilinear in lat/lon), returns (time, site)
sp_sites_time = sp_alltime.interp(latitude=site_lats, longitude=site_lons, method='linear')

# Time mean and temporal std at each site
sp_sites_mean = sp_sites_time.mean(dim='time')   # (site,)
sp_sites_std  = sp_sites_time.std(dim='time')    # (site,)

# Assemble a tidy table
df_sites = pd.DataFrame({
    'site'        : site_names,
    'lat'         : site_lats.values,
    'lon'         : site_lons.values,
    'mean_sp_Pa'  : sp_sites_mean.values,                  # Pa
    'mean_sp_hPa' : sp_sites_mean.values / 100.0,          # hPa
    'std_sp_Pa'   : sp_sites_std.values,                   # Pa
    'std_sp_hPa'  : sp_sites_std.values / 100.0,           # hPa
})

# Pretty print (rounded)
print(df_sites.round({
    'lat':4, 'lon':4,
    'mean_sp_Pa':0, 'mean_sp_hPa':1,
    'std_sp_Pa':0,  'std_sp_hPa':1
}))


## Mapping and visualization

### Step 81

This cell loads dataset(s) `pressure_all_25.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
from shapely.geometry import Polygon, box, mapping
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# ─────────────────────────────────────────────────────────────────────────────
# 1) Publication‐quality LaTeX text
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
})

# ─────────────────────────────────────────────────────────────────────────────
# 2) Load & calculate statistics
ds = xr.open_dataset('pressure_all_25.nc').rio.write_crs(4326)
ds = ds.rename({'valid_time': 'time'})
sp_mean = ds['sp'].mean(dim='time')
sp_std = ds['sp'].std(dim='time')
# --- Calculate Coefficient of Variation ---
sp_cv = sp_std / sp_mean

# ─────────────────────────────────────────────────────────────────────────────
# 3) Interpolate to 0.25° grid
lon0, lon1 = float(sp_cv.longitude.min()), float(sp_cv.longitude.max())
lat0, lat1 = float(sp_cv.latitude.min()),  float(sp_cv.latitude.max())
grid = 0.25
lons = np.arange(np.floor(lon0), np.ceil(lon1)+grid, grid)
lats = np.arange(np.floor(lat0), np.ceil(lat1)+grid, grid)
sp_cv_grid = sp_cv.interp(longitude=lons, latitude=lats, method='linear').fillna(sp_cv.mean())
sp_cv_grid.rio.write_crs(4326, inplace=True)

# ─────────────────────────────────────────────────────────────────────────────
# 4) Build search‐polygon
coords_df = pd.read_csv('plot-data.csv')
poly = Polygon(zip(coords_df.x, coords_df.y))
search_area = gpd.GeoDataFrame(geometry=[poly], crs="EPSG:4326")

# ─────────────────────────────────────────────────────────────────────────────
# 5) Clip CV to search polygon
sp_cv_clip = sp_cv_grid.rio.clip(search_area.geometry.map(mapping),
                                 search_area.crs, drop=True)

# ─────────────────────────────────────────────────────────────────────────────
# 6) Prepare “outside” fill (light yellow)
full_extent = box(75.8, 31.8, 79.5, 35.6)
outside = gpd.GeoDataFrame(geometry=[full_extent], crs="EPSG:4326")
# subtract country interior so that only outside shows
gdf_country = gpd.read_file("India__State_Boundary_2021_/India%3A_State_Boundary_2021_.shp") \
                .to_crs(epsg=4326)
outside = gpd.overlay(outside, gdf_country, how='difference')

# ─────────────────────────────────────────────────────────────────────────────
# 7) Plot
fig, ax = plt.subplots( figsize=(10,8),
                        subplot_kw={'projection': ccrs.PlateCarree()},
                        dpi=500 )

# a) outside‐India fill
ax.add_geometries(outside.geometry, crs=ccrs.PlateCarree(),
                  facecolor='lightyellow', edgecolor='none', zorder=0)

# b) clipped CV field
img = sp_cv_clip.plot.imshow(ax=ax, cmap='viridis', # Using a sequential colormap
                             add_colorbar=False, zorder=1)

# c) geographic base features
ax.add_feature(cfeature.LAND,      zorder=0, alpha=0.7)
ax.add_feature(cfeature.LAKES,     zorder=3, alpha=0.7)
ax.add_feature(cfeature.RIVERS,    zorder=3)
ax.add_feature(cfeature.COASTLINE, zorder=3)

# d) colorbar
cax = inset_axes(ax, width="3%", height="99%", loc='center right',
                 bbox_to_anchor=(0.07,0,1,1), bbox_transform=ax.transAxes)
cbar = plt.colorbar(img, cax=cax, orientation='vertical', drawedges=False, extend='both')
cbar.set_label(r'Coefficient of Variation of Surface Pressure', fontsize=20)
cbar.ax.tick_params(labelsize=18)

# e) gridlines + labels
gl = ax.gridlines(draw_labels=True, linestyle='--',
                  color='gray', alpha=0.6)
gl.top_labels    = False
gl.right_labels  = False
gl.bottom_labels = True
gl.left_labels   = True
gl.xlocator = plt.MaxNLocator(integer=True)
gl.ylocator = plt.MaxNLocator(integer=True)
gl.xlabel_style = {'size':18}
gl.ylabel_style = {'size':18}
for spine in ax.spines.values():
    spine.set_linewidth(2.5)

# f) on top right corner add a box saying 2010-2025 April
ax.text(0.98, 0.98, r'2010-2025 April', transform=ax.transAxes,
        fontsize=17, ha='right', va='top',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))

# g) final extent & layout
ax.set_extent([75.8,79.5,31.8,35.6], crs=ccrs.PlateCarree())
plt.subplots_adjust(left=0.1, right=0.83, top=0.9, bottom=0.17)
ax.set_title('')
plt.savefig('region_of_interest_cv_map.png',
            dpi=600, bbox_inches='tight')
plt.show()


### Step 82

This cell loads dataset(s) `pressure_all_25.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
from shapely.geometry import Polygon, box, mapping
import rioxarray
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# ─────────────────────────────────────────────────────────────────────────────
# 1) Publication‐quality LaTeX text
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "axes.labelsize": 20, # Increased for clarity
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
})

# ─────────────────────────────────────────────────────────────────────────────
# 2) Load & average surface pressure (Unchanged)
ds = xr.open_dataset('pressure_all_25.nc').rio.write_crs(4326)
ds = ds.rename({'valid_time': 'time'})
sp = ds['sp'].mean(dim='time')

# ─────────────────────────────────────────────────────────────────────────────
# 3) Interpolate to 0.25° grid (Unchanged)
lon0, lon1 = float(sp.longitude.min()), float(sp.longitude.max())
lat0, lat1 = float(sp.latitude.min()),  float(sp.latitude.max())
grid = 0.25
lons = np.arange(np.floor(lon0), np.ceil(lon1)+grid, grid)
lats = np.arange(np.floor(lat0), np.ceil(lat1)+grid, grid)
sp_grid = sp.interp(longitude=lons, latitude=lats, method='linear').fillna(sp.mean())
sp_grid.rio.write_crs(4326, inplace=True)

# ─────────────────────────────────────────────────────────────────────────────
# 4) Build search‐polygon (Unchanged)
coords_df = pd.read_csv('plot-data.csv')
poly = Polygon(zip(coords_df.x, coords_df.y))
search_area = gpd.GeoDataFrame(geometry=[poly], crs="EPSG:4326")

# ─────────────────────────────────────────────────────────────────────────────
# 5) Clip pressure to search polygon (Unchanged)
sp_clip = sp_grid.rio.clip(search_area.geometry.map(mapping),
                           search_area.crs, drop=True)

# ─────────────────────────────────────────────────────────────────────────────
# 6) Prepare "outside India" fill and India borders (Unchanged)
full_extent_box = box(75.8, 31.8, 79.5, 35.6)
outside_box = gpd.GeoDataFrame(geometry=[full_extent_box], crs="EPSG:4326")
gdf_country = gpd.read_file("India__State_Boundary_2021_/India%3A_State_Boundary_2021_.shp").to_crs(epsg=4326)
outside_fill = gpd.overlay(outside_box, gdf_country, how='difference')

# ─────────────────────────────────────────────────────────────────────────────
# 7) Plot (Completely Rewritten for Publication Quality)
# -----------------------------------------------------------------------------
print("Generating publication-quality map...")
fig, ax = plt.subplots(
    figsize=(12, 12),
    subplot_kw={'projection': ccrs.PlateCarree()},
    dpi=600
)

# --- 7a. Draw the detailed basemap ---
# Use higher-resolution Natural Earth features for a crisper look
ax.add_feature(cfeature.NaturalEarthFeature('physical', 'ocean', '50m'), facecolor='#d4e7f7', zorder=0)
ax.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '50m'), facecolor='#f5f5dc', zorder=0)
ax.add_feature(cfeature.NaturalEarthFeature('physical', 'lakes', '50m'), facecolor='#d4e7f7', zorder=1)
ax.add_feature(cfeature.NaturalEarthFeature('physical', 'rivers_lake_centerlines', '50m'), edgecolor='#d4e7f7', facecolor='none', zorder=1)

# --- 7b. Plot the main data layers ---
# Fill areas outside of India to de-emphasize them
ax.add_geometries(outside_fill.geometry, crs=ccrs.PlateCarree(), facecolor='#f0f0a0', edgecolor='gray', linewidth=0.5, zorder=2)

# Plot the clipped surface pressure data with a better colormap
img = sp_clip.plot.imshow(ax=ax, cmap='cividis', add_colorbar=False, zorder=3, vmin=45000, vmax=70000)

# --- 7c. Add geographic context and highlights ---
# Add state and country borders for context
ax.add_geometries(gdf_country.geometry, crs=ccrs.PlateCarree(), edgecolor='gray', facecolor='none', linewidth=0.7, zorder=4)
ax.add_feature(cfeature.BORDERS, linestyle='-', edgecolor='gray', zorder=4)

# Highlight the search polygon with a distinct, outlined border
path_effect = [pe.withStroke(linewidth=4, foreground='white')]
ax.add_geometries(search_area.geometry, crs=ccrs.PlateCarree(),
                  edgecolor='#93003a', facecolor='none',
                  linewidth=2, zorder=10, path_effects=path_effect)

# --- 7d. Configure map aesthetics (colorbar, gridlines, etc.) ---
# Colorbar
cax = ax.inset_axes([1.02, 0.15, 0.03, 0.7]) # Positioned cleanly to the right
cbar = plt.colorbar(img, cax=cax, orientation='vertical', extend='both')
cbar.set_label(r'\textbf{Mean Surface Pressure (Pa)}', fontsize=20)
cbar.ax.tick_params(labelsize=18)

# Gridlines and labels with proper formatting
gl = ax.gridlines(draw_labels=True, linestyle='--', color='black', alpha=0.2, zorder=20)
gl.top_labels = False
gl.right_labels = False
gl.xformatter = LongitudeFormatter()
gl.yformatter = LatitudeFormatter()
gl.xlabel_style = {'size': 20, 'weight': 'bold'}
gl.ylabel_style = {'size': 20, 'weight': 'bold'}

# Set final extent and spines
ax.set_extent([75.8, 79.5, 31.8, 35.6], crs=ccrs.PlateCarree())
for spine in ax.spines.values():
    spine.set_linewidth(2.5)

# --- 7e. Add the Inset Locator Map ---
ax_inset = ax.inset_axes([0.01, 0.01, 0.35, 0.35], projection=ccrs.PlateCarree())
ax_inset.set_extent([65, 98, 5, 38]) # Extent showing all of India

# Basemap for inset
ax_inset.add_feature(cfeature.OCEAN, facecolor='#d4e7f7')
ax_inset.add_feature(cfeature.LAND, facecolor='#f5f5dc')

# Plot India on the inset map
ax_inset.add_geometries(gdf_country.geometry, crs=ccrs.PlateCarree(), edgecolor='gray', facecolor='#bfbfbf', linewidth=0.5)

# Plot the box showing the main map's extent
ax_inset.add_geometries([full_extent_box], crs=ccrs.PlateCarree(), edgecolor='red', facecolor='none', linewidth=2)

# Add a border to the inset map
for spine in ax_inset.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1.5)

# --- 7f. Finalize and Save ---
plt.savefig('Region_Surface_Pressure_Publication.png', dpi=600, bbox_inches='tight')
print("→ Publication-quality map saved as 'Region_Surface_Pressure_Publication.png'")
plt.show()


## Interpolation and extraction

### Step 83

This cell loads dataset(s) `pressure_all_25.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
surface_pressure_data =xr.open_dataset('pressure_all_25.nc').rio.write_crs(4326)

surface_pressure_data = surface_pressure_data.rename({'valid_time': 'time'})

# set the time range for the surface pressure data
# surface_pressure_data = surface_pressure_data.sel(time=slice("2010-01-01", "2025-04-01"))
surface_pressure_data


### Step 84

This cell subsets the dataset to a selected time, level, region, or site so the next step works with a focused slice of the data.

In [ ]:
lat, lon = 32.7789, 78.965
elevation_value = elevation_resampled.sel(latitude=lat, longitude=lon, method='nearest').values
print(f"Elevation at latitude {lat}, longitude {lon}: {elevation_value} meters")


## Project setup and imports

### Step 85

This cell imports the core libraries used in the following analysis, including `tqdm`.

In [ ]:
import tqdm


## Analysis workflow

### Step 86

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
combined_dataset


### Step 87

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
surface_pressure_data


### Step 88

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
era5


### Step 89

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
elevation_data


### Step 90

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
surface_pressure_data


## Project setup and imports

### Step 91

This cell imports the core libraries used in the following analysis, including `tqdm`.

In [ ]:
import tqdm


## Pressure and PWV calculations

### Step 92

This cell defines reusable helper function(s) `calculate_pwv_dataset`, `count_low_pwv_months` so later sections can apply the same processing logic consistently.

In [ ]:
def calculate_pwv_dataset(dataset, elevation_data, start_year, end_year, lat_range_pwv, lon_range, lat_range_elev, P_b, T_b, L_b, h_b):
    """
    Function to calculate PWV for each grid point and time step in a specified spatial and temporal range,
    and add it as a new variable 'tcwv' in the dataset.
    
    Parameters:
    dataset (xarray.Dataset): ERA5 dataset containing specific humidity and pressure levels.
    elevation_data (xarray.DataArray): Elevation data resampled to match ERA5 grid.
    start_year (int): Start year of the period.
    end_year (int): End year of the period.
    lat_range_pwv (tuple): Latitude range for PWV calculation (min, max).
    lon_range (tuple): Longitude range (min, max).
    lat_range_elev (tuple): Latitude range for elevation data (min, max).
    P_b (int): Reference pressure at sea level in Pa.
    T_b (int): Reference temperature at sea level in K.
    L_b (float): Temperature lapse rate in K/m.
    h_b (int): Height of reference level in m.

    Returns:
    xarray.Dataset: Dataset with a new variable 'tcwv' containing the calculated PWV values.
    """
    lon_range = tuple(lon % 360 for lon in lon_range)
    
    # Slice the dataset by time and space
    sliced_data = dataset.sel(
        time=slice(f"{start_year}-01-01", f"{end_year}-04-01"),
        latitude=slice(*lat_range_pwv),
        longitude=slice(*lon_range)
    )

    # choose the ver 1 
    # sliced_data = sliced_data.sel(expver=1)
    # Slice the elevation data to match the spatial range of the dataset
    sliced_elevation = elevation_data.sel(
        latitude=slice(*lat_range_elev),
        longitude=slice(*lon_range)
    )
    #print(f'sliced_elvation: {sliced_elevation}')

    # Initialize the new DataArray with the same shape as the time, latitude, and longitude dimensions
    pwv_data = xr.DataArray(
        np.full((len(sliced_data.time), len(sliced_data.latitude), len(sliced_data.longitude)), fill_value=np.nan),
        coords=[sliced_data.time, sliced_data.latitude, sliced_data.longitude],
        dims=["time", "latitude", "longitude"],
        name="tcwv"
    )
    print(pwv_data)

    for lat in tqdm.tqdm(sliced_data.latitude.values):
        for lon in tqdm.tqdm(sliced_data.longitude.values):
            point_data = sliced_data.sel(latitude=lat, longitude=lon)
            #print(f'point_data: {point_data}')
            elevation = sliced_elevation.sel(latitude=lat, longitude=lon).item()
            # print the lat lon of which the elevation is fetched from the elevation data

            #print(f"Elevation for lat: {lat}, lon: {lon} is {elevation} meters.")
            #pressure_threshold = calculate_pressure(P_b, T_b, L_b, elevation, h_b)
            #print(f"Pressure threshold for lat: {lat}, lon: {lon} is {pressure_threshold:.2f} hPa")
            #print(surface_pressure_data)
            pressure_threshold = surface_pressure_data['sp'].interp(latitude=lat, longitude=lon, method='linear')
            q = point_data.q
            #print(q)
            print(q)
            levels = point_data.level
            print(levels)
            #print(pressure_threshold)
            pwv = calculate_pwv_pressure(q, levels, pressure_threshold)
                        # Ensure the time coordinates match
            pwv = pwv.reindex_like(pwv_data.sel(latitude=lat, longitude=lon))
            pwv_data.loc[dict(latitude=lat, longitude=lon)] = pwv
    
            # print(f"Error calculating PWV at lat: {lat}, lon: {lon}, time: {time}")
            # print(f"q shape: {q.shape}, levels shape: {levels.shape}")
            # print(e)
             

    # Add the new DataArray to the dataset
    dataset["tcwv"] = pwv_data

    return dataset

def count_low_pwv_months(dataset, start_year, end_year, lat_range_pwv, lon_range):
    """
    Function to count months with PWV <= 1 for each grid point in a specified spatial and temporal range.
    
    Parameters:
    dataset (xarray.Dataset): ERA5 dataset containing the 'tcwv' variable with calculated PWV values.
    start_year (int): Start year of the period.
    end_year (int): End year of the period.
    lat_range_pwv (tuple): Latitude range (min, max).
    lon_range (tuple): Longitude range (min, max).

    Returns:
    pd.DataFrame: DataFrame containing latitude, longitude, and Low_PWV_Months_Count.
    """
    lon_range = tuple(lon % 360 for lon in lon_range)
    
    # Slice the dataset by time and space
    sliced_data = dataset.sel(
        time=slice(f"{start_year}-01-01", f"{end_year}-04-01"),
        latitude=slice(*lat_range_pwv),
        longitude=slice(*lon_range)
    )
    # print the total number of months
    #print(f"Total number of months: {len(sliced_data.time)}")

    # Filter months where PWV <= 1
    low_pwv = sliced_data["tcwv"] <= 1

    # Sum up the true values across the time dimension for each latitude and longitude
    low_pwv_count = low_pwv.sum(dim="time")

    # Convert to DataFrame for easier viewing and manipulation
    low_pwv_df = low_pwv_count.to_dataframe(name="Low_PWV_Months_Count").reset_index()

    return low_pwv_df

# Example usage

# Calculate PWV and add it to the dataset
T_b = 287.68 # reference temperature at sea level in K

#dataset_with_pwv = calculate_pwv_dataset(era5, elevation_resampled, 2010, 2025, (35.75, 32), (76, 79.75), (32, 35.75), P_b, T_b, L_b, h_b)
dataset_with_pwv = calculate_pwv_dataset(combined_dataset, elevation_resampled, 2010, 2025, (35.75, 32), (76, 79.75), (32, 35.75), P_b, T_b, L_b, h_b)

# Count low PWV months
results_df = count_low_pwv_months(dataset_with_pwv, 2010, 2025, (36, 32), (76, 80))
print(results_df)


### Step 93

This cell defines reusable helper function(s) `calculate_pwv_vectorized`, `integrate_profile`, `count_low_pwv_months` so later sections can apply the same processing logic consistently.

In [ ]:
#!/usr/bin/env python3
# ────────────────────────────────────────────────────────────────────────────────
#  High-Performance PWV Grid Analysis
#  This script calculates PWV for every grid point in a specified region and
#  then counts the number of "dry" months (PWV <= 1mm).
#  It uses a fully vectorized approach for maximum speed.
# ────────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import xarray as xr
from scipy.interpolate import PchipInterpolator

# --- Assumed Pre-existing Objects ---
# This script assumes the following objects are already loaded in your environment:
#
# DATASETS:
#   - combined_dataset: xarray.Dataset with data from 2010 to 2025.
#   - surface_pressure_data: xarray.Dataset with surface pressure 'sp'.
# -----------------------------------------------------------------------------


# 1. High-Performance PWV Calculation Engine
# ─────────────────────────────────────────────────────────────────────────────
def calculate_pwv_vectorized(q: xr.DataArray, p_sfc: xr.DataArray) -> xr.DataArray:
    """
    Calculates Precipitable Water Vapor (PWV) using a fast, vectorized approach.
    This function avoids Python loops for high performance.
    """
    g = 9.81  # gravity in m/s^2
    
    p_levels_pa = q.level * 100
    p_levels_pa.attrs['units'] = 'Pa'
    
    def integrate_profile(q_profile, p_profile, sfc_p_value):
        # This helper function runs on a single vertical profile
        finite_mask = np.isfinite(q_profile)
        if finite_mask.sum() < 2: return np.nan

        p, q_ = p_profile[finite_mask], q_profile[finite_mask]
        sort_idx = np.argsort(p)
        p, q_ = p[sort_idx], q_[sort_idx]

        valid_levels = p <= sfc_p_value
        p, q_ = p[valid_levels], q_[valid_levels]
        if len(p) < 2: return np.nan

        interpolator = PchipInterpolator(p, q_, extrapolate=True)
        q_sfc = interpolator(sfc_p_value)

        p_full = np.append(p, sfc_p_value)
        q_full = np.append(q_, q_sfc)
        
        sort_idx_full = np.argsort(p_full)
        p_full, q_full = p_full[sort_idx_full], q_full[sort_idx_full]
        
        return np.trapz(q_full, p_full) / g

    # Use xarray's apply_ufunc to run the integration on the entire data cube
    pwv = xr.apply_ufunc(
        integrate_profile, q, p_levels_pa, p_sfc,
        input_core_dims=[['level'], ['level'], []],
        output_core_dims=[[]], vectorize=True, dask="parallelized", output_dtypes=[q.dtype]
    )
    
    pwv.attrs.update(units="mm", long_name="Precipitable Water Vapor")
    return pwv


# 2. Efficient Low PWV Month Counter
# ─────────────────────────────────────────────────────────────────────────────
def count_low_pwv_months(dataset_with_pwv: xr.Dataset) -> pd.DataFrame:
    """
    Function to efficiently count months with PWV <= 1 for each grid point.
    """
    print("Counting months with PWV <= 1 mm...")
    
    # Create a boolean mask where PWV is low
    low_pwv_mask = dataset_with_pwv["tcwv"] <= 1

    # Sum the boolean values (True=1, False=0) across time to get the count
    low_pwv_count = low_pwv_mask.sum(dim="time")

    # Convert the 2D grid of counts to a clean DataFrame
    low_pwv_df = low_pwv_count.to_dataframe(name="Low_PWV_Months_Count").reset_index()
    
    # Filter out any grid points that had no dry months
    low_pwv_df = low_pwv_df[low_pwv_df["Low_PWV_Months_Count"] > 0].sort_values(
        by="Low_PWV_Months_Count", ascending=False
    )
    return low_pwv_df


# 3. Main Execution Block
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    # --- Define Parameters ---
    START_YEAR = 2010
    END_YEAR = 2025 # The slice will go up to April
    LAT_RANGE = (35.75, 32)
    LON_RANGE = (76, 79.75)
    
    # --- 1. Prepare Data ---
    print("Slicing data to region and time of interest...")
    # Convert longitude to 0-360 if needed, assuming ERA5 format
    lon_range_360 = tuple(lon % 360 for lon in LON_RANGE)
    
    # Slice the main dataset and the surface pressure dataset
    q_data_sliced = combined_dataset.sel(
        time=slice(f"{START_YEAR}-01-01", f"{END_YEAR}-04-30"),
        latitude=slice(*LAT_RANGE),
        longitude=slice(*lon_range_360)
    )

    # --- THIS IS THE FIX: Removed the failing .rename() call ---
    sp_data_sliced = surface_pressure_data.sel(
        time=slice(f"{START_YEAR}-01-01", f"{END_YEAR}-04-30"),
        latitude=slice(*LAT_RANGE),
        longitude=slice(*lon_range_360)
    )['sp']
    # --------------------------------------------------------

    # --- 2. Align Data ---
    print("Aligning specific humidity and surface pressure grids...")
    q_aligned, sp_aligned = xr.align(q_data_sliced.q, sp_data_sliced, join='inner')

    # Handle the experiment version dimension
    if 'expver' in q_aligned.dims:
        q_aligned = q_aligned.sel(expver=1, drop=True)
        
    # Rechunk the level dimension into a single block for the vertical integral
    q_rechunked = q_aligned.chunk({"level": -1})

    # --- 3. Calculate PWV across the entire grid (fast operation) ---
    print("Calculating PWV for all grid points and time steps...")
    pwv_grid = calculate_pwv_vectorized(q_rechunked, sp_aligned)
    
    # --- 4. Combine results and count dry months ---
    # Create a new dataset containing just the calculated PWV
    dataset_with_pwv = xr.Dataset({'tcwv': pwv_grid})
    
    # Run the efficient counting function
    results_df = count_low_pwv_months(dataset_with_pwv)
    
    # --- 5. Display Final Results ---
    print("\n✅ Analysis Complete.")
    print("Top sites with the highest number of dry months (PWV <= 1 mm):")
    print(results_df.head(15))


## Exports and file generation

### Step 94

This cell writes derived results or configuration content to disk so they can be reused outside the notebook.

In [ ]:
# save the results_df to a
results_df.to_csv('results_df.csv', index=False)


## Data loading and inspection

### Step 95

This cell reads tabular metadata that is later used for site lookup, filtering, or summary reporting.

In [ ]:
results_df = pd.read_csv('results_df.csv')


## Pressure and PWV calculations

### Step 96

This cell computes or evaluates precipitable water vapor (PWV)-related quantities that are central to the site-quality analysis.

In [ ]:
# sort the results_df by Low_PWV_Months_Count
results_df_new = results_df.sort_values(by='Low_PWV_Months_Count', ascending=False)
results_df_new


## Analysis workflow

### Step 97

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
results_df[(results_df['latitude'] == 33) & (results_df['longitude'] == 78)]


### Step 98

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
# print the info for the following points
# 1 (32.75, 79)
# 2 33, 78
# 3 34.25, 78.75

# use results_df to get the data
# 1
lat = 35.25
lon = 78
point1 = results_df[(results_df['latitude'] == lat) & (results_df['longitude'] == lon)]
print(point1)

# 2
lat = 35.25
lon = 77.75
point2 = results_df[(results_df['latitude'] == lat) & (results_df['longitude'] == lon)]
print(point2)

# 3
lat = 35.25
lon = 78.50
point3 = results_df[(results_df['latitude'] == lat) & (results_df['longitude'] == lon)]
print(point3)

# 4 
lat = 35 
lon = 78
point4 = results_df[(results_df['latitude'] == lat) & (results_df['longitude'] == lon)]
print(point4)

# 6
lat = 34.5
lon = 78.50
point6 = results_df[(results_df['latitude'] == lat) & (results_df['longitude'] == lon)]
print(point6)

# 5 
lat = 32.75
lon = 78
point5 = results_df[(results_df['latitude'] == lat) & (results_df['longitude'] == lon)]
print(point5)

# 7 
lat = 34.25
lon = 78.75
point7 = results_df[(results_df['latitude'] == lat) & (results_df['longitude'] == lon)]
print(point7)

# point 8 
lat = 32.5
lon = 79
point8 = results_df[(results_df['latitude'] == lat) & (results_df['longitude'] == lon)]
print(point8)


## Site definitions and metadata

### Step 99

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
# for Hanle
lat = 32.75
lon = 79
point1 = results_df[(results_df['latitude'] == lat) & (results_df['longitude'] == lon)]
print(point1)

lat = 33.75
lon = 78.5
point1 = results_df[(results_df['latitude'] == lat) & (results_df['longitude'] == lon)]
print(point1)


### Step 100

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
HANLE,MERAK,SITE_A,SITE_B


### Step 101

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
MERAK


## Interpolation and extraction

### Step 102

This cell subsets the dataset to a selected time, level, region, or site so the next step works with a focused slice of the data.

In [ ]:
# elevation data for 33, 78
lat = 33
lon = 78
elevation = elevation_resampled.sel(latitude=lat, longitude=lon).item()
print(f"Elevation for lat: {lat}, lon: {lon} is {elevation} meters.")


## Pressure and PWV calculations

### Step 103

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
# pressure threshold for 33, 78
pressure_threshold = calculate_pressure(P_b, T_b, L_b, elevation, h_b)
print(f"Pressure threshold for lat: {lat}, lon: {lon} is {pressure_threshold:.2f} hPa")
print(T_b)


## Site definitions and metadata

### Step 104

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
# Create a list of the sites you want to look up
sites_to_query = [HANLE, MERAK, SITE_A, SITE_B,]

print("--- Querying Results for Specific Site Pixels ---\n")

# Loop through each site
for site in sites_to_query:
    target_lat = site['lat']
    target_lon = site['lon']
    site_name = site['name']
    
    # Calculate the squared Euclidean distance to find the nearest grid point
    # This is a fast and effective way to find the closest match in the DataFrame
    distance = (results_df['latitude'] - target_lat)**2 + (results_df['longitude'] - target_lon)**2
    
    # Get the index of the row with the minimum distance
    nearest_index = distance.idxmin()
    
    # Retrieve the data for the nearest grid point
    site_info = results_df.loc[nearest_index]
    
    # Print the results in a formatted way
    print(f"📍 Info for Site: {site_name}")
    print(f"   - Target Coords:      ({target_lat:.4f}, {target_lon:.4f})")
    print(f"   - Nearest Grid Point: ({site_info['latitude']:.4f}, {site_info['longitude']:.4f})")
    print(f"   - Dry Months (PWV<=1mm): {int(site_info['Low_PWV_Months_Count'])} months")
    print("-" * 40)


## Mapping and visualization

### Step 105

This cell defines reusable helper function(s) `create_artists`, `create_artists` so later sections can apply the same processing logic consistently.

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
from matplotlib.colors import Normalize
from matplotlib.legend_handler import HandlerPatch
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import rasterio
from rasterio.merge import merge
import xarray as xr
import rioxarray
from shapely.geometry import box, mapping
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

# Load your data
gdf1 = gpd.read_file("shape_files/gadm41_IND_shp/gadm41_IND_1.shp").to_crs(epsg=4326)
gdf2 = gpd.read_file("shape_files/India_Shape_Full/India_Country_Boundary.shp").to_crs(epsg=4326)
#gdf2 = gpd.read_file("Administrative Boundary Database /STATE_BOUNDARY.shp").to_crs(epsg=4326)

# Define the manual box as a GeoDataFrame
manual_box = gpd.GeoDataFrame(geometry=[box(75.9, 31.9, 79.5, 35.7)], crs=gdf1.crs)

# Intersect the box with gdf1
intersect_region = gpd.overlay(manual_box, gdf1, how='intersection')

# Difference between gdf2 and gdf1
difference_region_1 = gpd.overlay(gdf2, gdf1, how='difference')

# Difference between the bounding box of the whole map and gdf2
full_extent_box = box(72.3, 31, 81, 37.5)  # Adjust this to your desired full map extent
full_extent_gdf = gpd.GeoDataFrame(geometry=[full_extent_box], crs=gdf1.crs)
difference_region_2 = gpd.overlay(full_extent_gdf, gdf2, how='difference')

# Assuming results_df is your DataFrame containing the point data with 'longitude', 'latitude', and 'Low_PWV_Months_Count' columns
# Create GeoDataFrame for the points
points_gdf = gpd.GeoDataFrame(results_df, geometry=gpd.points_from_xy(results_df.longitude, results_df.latitude))
points_gdf.crs = gdf1.crs

# Spatial join to keep only points within intersect_region
points_within_intersect = gpd.sjoin(points_gdf, intersect_region, how="inner", predicate='intersects')

# Calculate percentage
points_within_intersect['percent'] = (points_within_intersect['Low_PWV_Months_Count'] / 168) * 100

# Remove points with 0 months of PWV ≤ 1 mm
points_within_intersect = points_within_intersect[points_within_intersect['Low_PWV_Months_Count'] > 0]

# Now define normalization and colormap
norm = Normalize(vmin=0, vmax=np.max(points_within_intersect['percent']))
cmap = plt.cm.jet

# File paths for the GeoTIFF files
geotiff_files = ['srtm_52_05/srtm_52_05.tif', 'srtm_52_06/srtm_52_06.tif']

# Open the GeoTIFF files and merge them
src_files_to_mosaic = [rasterio.open(file) for file in geotiff_files]
mosaic, out_trans = merge(src_files_to_mosaic)

# Convert the mosaic to an xarray DataArray and add geospatial info
lon_min, lat_max = out_trans * (0, 0)
lon_max, lat_min = out_trans * (mosaic.shape[2], mosaic.shape[1])
elevation_da = xr.DataArray(mosaic[0], coords=[('latitude', np.linspace(lat_max, lat_min, mosaic.shape[1])), 
                                                ('longitude', np.linspace(lon_min, lon_max, mosaic.shape[2]))], name='elevation')
elevation_da.rio.write_crs(4326, inplace=True)

# Mask the elevation data to the intersect region
elevation_clipped = elevation_da.rio.clip(intersect_region.geometry.apply(mapping), intersect_region.crs, drop=True)

# Remove invalid values from elevation data
elevation_clipped = elevation_clipped.where(np.isfinite(elevation_clipped), other=np.nan)

# Set up figure and axis with a geographic projection
fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={'projection': ccrs.PlateCarree()},
                       facecolor='white', edgecolor='black', linewidth=2,
                       frameon=True, dpi=500,
                       gridspec_kw={'wspace': 0.2, 'hspace': 0.2})

# Plotting the elevation data as shaded relief
elev_img = elevation_clipped.plot.imshow(ax=ax, cmap='terrain', zorder=0, add_colorbar=False,
                                         )

# Plotting the shapefiles
gdf1.boundary.plot(ax=ax, linewidth=2, edgecolor='blue', zorder=3)
gdf2.boundary.plot(ax=ax, linewidth=2, edgecolor='red', zorder=3)

# Plot background colors
ax.add_geometries(intersect_region.geometry, crs=ccrs.PlateCarree(), facecolor='None', edgecolor='black', linewidth=2, label='Search Area', zorder=2)
ax.add_geometries(difference_region_1.geometry, crs=ccrs.PlateCarree(), facecolor='lightgreen', edgecolor='green', label='Disputed Territory', zorder=2)
ax.add_geometries(difference_region_2.geometry, crs=ccrs.PlateCarree(), facecolor='lightgrey', edgecolor='grey', label='Outside India', zorder=2)

# Plot leftover region
combined_regions = gpd.GeoDataFrame(pd.concat([intersect_region, difference_region_1, difference_region_2], ignore_index=True))
leftover_region = gpd.overlay(full_extent_gdf, combined_regions, how='difference')
ax.add_geometries(leftover_region.geometry, crs=ccrs.PlateCarree(), facecolor='lightyellow', edgecolor='grey', zorder=1)

# Plotting the data points with color mapping
scatter = ax.scatter(points_within_intersect['longitude'], points_within_intersect['latitude'],
                     c=points_within_intersect['percent'], s=(points_within_intersect['percent'])**(3/4) * 32,
                     cmap='jet', alpha=0.7, edgecolor='k', linewidth=1.5,
                     transform=ccrs.PlateCarree(), zorder=4, label='Months $<=$ 1 mm(%)', marker='o', norm=norm)

# Adding colorbar for PWV
# cbar = plt.colorbar(scatter, ax=ax, orientation='horizontal', pad=0.05, aspect=40,
#                     shrink=0.85, drawedges=False)
# max_value = points_within_intersect['percent'].max()
# cbar.set_ticks(np.linspace(0, max_value, 5))
# cbar.set_label('Percentage of Months with PWV ≤ 1')
# cbar.set_ticklabels([f'{i:.0f}%' for i in np.linspace(0, max_value, 5)])

# # Using InsetPosition to position the horizontal colorbar
# ax_cb = inset_axes(ax, width="100%", height="5%", loc='lower center',
#                    bbox_to_anchor=(0, -0.2, 1, 1), bbox_transform=ax.transAxes, borderpad=0)
# plt.colorbar(scatter, cax=ax_cb, orientation='horizontal')

# # Adding colorbar for elevation
# cbar_elevation = plt.colorbar(elev_img, ax=ax, orientation='vertical', pad=0.1, aspect=25, shrink=0.85, drawedges=False)
# cbar_elevation.set_label('Elevation (m)')
# cbar_elevation.ax.set_position([cbar_elevation.ax.get_position().x0 + 0.1, cbar_elevation.ax.get_position().y0,
#                                 cbar_elevation.ax.get_position().width, cbar_elevation.ax.get_position().height])



# Adding colorbar for PWV
ax_cb_pwv = inset_axes(ax, width="95%", height="5%", loc='lower center',
                   bbox_to_anchor=(0, -0.07, 1, 1), bbox_transform=ax.transAxes, borderpad=0)
cbar_pwv = plt.colorbar(scatter, cax=ax_cb_pwv, orientation='horizontal', drawedges=False)
max_value = points_within_intersect['percent'].max()
cbar_pwv.set_ticks(np.linspace(0, max_value, 5))
cbar_pwv.set_label('Percentage of Months with PWV ≤ 1 mm',fontsize=12)
cbar_pwv.set_ticklabels([f'{i:.0f}%' for i in np.linspace(0, max_value, 5)])
cbar_pwv.ax.tick_params(labelsize=12)

# Adding colorbar for elevation
ax_cb_elevation = inset_axes(ax, width="3.5%", height="95%", loc='center right',
                              bbox_to_anchor=(0.05, 0., 1, 1),bbox_transform=ax.transAxes, borderpad=0)
cbar_elevation = plt.colorbar(elev_img, cax=ax_cb_elevation, orientation='vertical', drawedges=False)
cbar_elevation.set_label('Elevation (m)',fontsize=12)
cbar_elevation.ax.tick_params(labelsize=12)



# Adding geographic features
ax.add_feature(cfeature.RIVERS, zorder=3)
ax.add_feature(cfeature.COASTLINE, zorder=3)
ax.add_feature(cfeature.LAKES, alpha=0.5, zorder=3)
ax.add_feature(cfeature.LAND)

# Adding gridlines
gl = ax.gridlines(draw_labels=True, linewidth=1, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = True
gl.right_labels = False
gl.bottom_labels = False
gl.xlocator = plt.MaxNLocator(integer=True)
gl.ylocator = plt.MaxNLocator(integer=True)
# Make the surrounding box thicker
for spine in ax.spines.values():
    spine.set_linewidth(2.5)
gl.xlabel_style = {'size': 12}
gl.ylabel_style = {'size': 12}


# Define the rectangle for the specific region
rectangle = plt.Rectangle((75.9, 31.9), 3.8, 4, fill=False, edgecolor='black', linewidth=3)
ax.add_patch(rectangle)

# Setting extent if necessary
ax.set_extent([72.3, 81, 31, 37.2])

# Adding the site markers and labels with arrows
sites = {
    'Hanle': (78.96, 32.77),
    'Site B': (78.00, 33),
    'Site A': (78.75, 34.25),
    'Merak': (78.62, 33.79),
    'Leh': (77.5771,34.1526 )
}
text_offsets = {
    'Hanle': (0.4, 0.4),
    'Site B': (-1.35, -0.8),
    'Site A': (0.4, 0.4),
    'Merak': (0.5, 0.2),
    'Leh': (-1.3, 0.9)
}

for site, (lon, lat) in sites.items():
    ax.plot(lon, lat, '*', color='black', markersize=12, markeredgecolor='black',
             markerfacecolor='red', transform=ccrs.PlateCarree(), zorder=5)

    text_artist = ax.text(lon + text_offsets[site][0], lat + text_offsets[site][1], site,
                          fontsize=16, fontname='Times New Roman',
                          bbox=dict(boxstyle='square', edgecolor='black', facecolor='white', alpha=0.9),
                          transform=ccrs.PlateCarree(), zorder=5)

    # Calculate approximate bbox edges in data coordinates
    renderer = fig.canvas.get_renderer()
    bbox = text_artist.get_window_extent(renderer=renderer).transformed(ax.transData.inverted())
    corner_xy = (bbox.x0, bbox.y1)  # Top-left corner
    # bottom left corner
    corner_bottom = (bbox.x0, bbox.y0)

    if site == 'Leh':
        ax.annotate('', xy=(lon, lat), xytext=corner_bottom,
                    arrowprops=dict(arrowstyle='->', lw=2.5, color='black', mutation_scale=10,
                                    connectionstyle='arc3,rad=0.3', alpha=0.9,
                                    linestyle='solid', linewidth=2,
                                    shrinkA=0, shrinkB=0),
                    transform=ccrs.PlateCarree(), zorder=5)
    elif site == 'Site B':
        ax.annotate('', xy=(lon, lat), xytext=corner_xy,
                    arrowprops=dict(arrowstyle='->', lw=2.5, color='black', mutation_scale=10,
                                    connectionstyle='arc3,rad=-0.5', alpha=0.9,
                                    linestyle='solid', linewidth=2,
                                    shrinkA=0, shrinkB=0),
                    transform=ccrs.PlateCarree(), zorder=5)

    else:
        ax.annotate('', xy=(lon, lat), xytext=corner_xy,
                    arrowprops=dict(arrowstyle='->', lw=2.5, color='black', mutation_scale=10,
                                    connectionstyle='arc3,rad=0.5', alpha=0.9,
                                    linestyle='solid', linewidth=2,
                                    shrinkA=0, shrinkB=0),
                    transform=ccrs.PlateCarree(), zorder=5)

# Define sizes and corresponding months for legend
#sample_percents = [6, 12, 18, 24]  # Example percentile points
# sample_percents = [15,31,46,61]
#sample_percents = [12, 24, 36, 48]
sample_percents = [6,12,17,23]
colors = [scatter.cmap(norm(val)) for val in sample_percents]  # Fetch colors properly using normalized values

# Create legend elements
legend_elements = [
    mpatches.Circle(
        (0, 0), 
        radius=0.17 * ((val)**(4/8)),  # Adjust visual size in legend to match the plot
        facecolor=color,
        alpha=0.7,
        linewidth=1.5,
        edgecolor='black',  # Sets the color of the edge of the circle to black 
        linestyle='-',
        label=f'{int((val / 100) * 168)}  ({val:.0f}%)'
    ) for val, color in zip(sample_percents, colors)
]

# Custom function to draw the legend with the correct sizes
class HandlerCircle(HandlerPatch):
    def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans):
        r = orig_handle.radius * 10  # Scale radius up for visibility in legend
        x = width / 2
        y = height / 2
        p = mpatches.Circle((x, y), r, fc=orig_handle.get_facecolor(), transform=trans)
        return [p]

# Create a second legend for these elements
second_legend = ax.legend(handles=legend_elements, handler_map={mpatches.Circle: HandlerCircle()}, loc='lower left', title="ERA5 Data: 2010-2023\nTotal Months: 168",
          facecolor='white', framealpha=0.8, shadow=True, fontsize=12, title_fontsize=12,
          borderpad=0.3, labelspacing=0.3, handlelength=2, handletextpad=0.5, frameon=True)

# Set the edge color and line width of the legend box
second_legend.get_frame().set_edgecolor('black')
second_legend.get_frame().set_linewidth(1.5)

ax.add_artist(second_legend)



from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.legend_handler import HandlerBase
import matplotlib.image as mpimg

# # Read the elevation patch image
# elevation_patch_image = plt.imread('elevation_patch.png')

# # Create an OffsetImage
# imagebox = OffsetImage(elevation_patch_image, zoom=0.1)

# # Create an AnnotationBbox
# ab = AnnotationBbox(imagebox, (1.1, 1.1), frameon=True, box_alignment=(0, 0),
#                     bboxprops=dict(edgecolor='blue', linewidth=2))

# # Add the AnnotationBbox to the axes
# ax.add_artist(ab)


# Custom legend with rectangular patches
legend_elements = [
    Rectangle((0, 0), 1, 1, facecolor='lightgreen', label='Disputed Territory',
              edgecolor='black', linewidth=1),
    Rectangle((0, 0), 1, 1, facecolor='lightgrey', label='Outside India',
              edgecolor='black', linewidth=1),
    Line2D([0], [0], marker='*', markersize=12, label='Site Locations',
           markerfacecolor='red', markeredgewidth=1, markeredgecolor='black')
]

# Load the saved elevation patch
elevation_patch_image = mpimg.imread('elevation_patch.png')

# Create custom legend handler for the elevation patch
class HandlerElevationPatch(HandlerBase):
    def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans):
        imagebox = OffsetImage(elevation_patch_image, zoom=0.08)
        ab = AnnotationBbox(imagebox, (xdescent + width / 2, ydescent + height / 2), frameon=True, pad=0.1,
                            box_alignment=(0.5, 0.5), bboxprops=dict(edgecolor='blue', linewidth=3))
        ab.set_transform(trans)
        return [ab]

elevation_patch_handle = mpatches.FancyBboxPatch((0, 0), 1, 1, facecolor='none', edgecolor='blue', linewidth=3, label='Search Area')

# Append the custom elevation patch handler to the legend elements
legend_elements.insert(0, elevation_patch_handle)

# Add the custom legend elements
first_legend = ax.legend(handles=legend_elements, loc='upper right', facecolor='white', framealpha=0.8, shadow=True,
                         fontsize=12, borderpad=0.3, labelspacing=0.3, handlelength=1.5, handletextpad=0.51, frameon=True,
                         handler_map={elevation_patch_handle: HandlerElevationPatch()})



# # Create a patch for the elevation data in the search area
# elev_patch = mpatches.Patch(
#     facecolor=plt.cm.terrain(0.5),  # Sample a middle value color from the terrain colormap
#     edgecolor='black', linewidth=2, label='Search Area'
# )


# # Custom legend with rectangular patches
# legend_elements = [
#     Rectangle((0, 0), 1, 1, facecolor='lightblue', label='Search Area',
#               edgecolor='black', linewidth=1),
#     Rectangle((0, 0), 1, 1, facecolor='lightgreen', label='Disputed Territory',
#               edgecolor='black', linewidth=1),
#     Rectangle((0, 0), 1, 1, facecolor='lightgrey', label='Outside India',
#               edgecolor='black', linewidth=1),
#     Line2D([0], [0], marker='*', markersize=12, label='Site Locations',
#            markerfacecolor='red', markeredgewidth=1, markeredgecolor='black')
# ]

# first_legend = ax.legend(
#     handles=legend_elements, 
#     loc='upper right',
#     facecolor='white', 
#     framealpha=0.8, 
#     shadow=True,
#     fontsize=12,
#     borderpad=0.3, 
#     labelspacing=0.3, 
#     handlelength=1.5, 
#     handletextpad=0.51,
#     frameon=True,
# )

# Set the edge color and line width of the legend box
first_legend.get_frame().set_edgecolor('black')
first_legend.get_frame().set_linewidth(1.5)

# Add the legends to the plot
ax.add_artist(first_legend)

# Adjust layout
plt.subplots_adjust(left=0.1, right=0.83, top=0.9, bottom=0.17)

# Remove title from the plot
ax.set_title('')

# Show the plot
plt.savefig('all_points_on_map_elevation_pressure_level.jpeg', dpi=600)
plt.show()

# Close the files
for src in src_files_to_mosaic:
    src.close()


## UPDATES IN THE MAP

## Mapping and visualization

### Step 106

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from shapely.ops import unary_union

# Load the state-level shapefile
#gdf = gpd.read_file("Administrative Boundary Database /STATE_BOUNDARY.shp")
gdf = gpd.read_file("India__State_Boundary_2021_/India%3A_State_Boundary_2021_.shp")

# Combine all geometries into a single geometry using unary_union
combined_geometry = unary_union(gdf.geometry)

# Extract the external boundary of the combined geometry
external_boundary = gpd.GeoSeries([combined_geometry.boundary])

# Set up figure and axis with a geographic projection
fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={'projection': ccrs.PlateCarree()},
                       facecolor='white', edgecolor='black', linewidth=2,
                       frameon=True, dpi=500)

# Plot the external boundary
external_boundary.plot(ax=ax, linewidth=2, edgecolor='blue', zorder=3)

# Adding geographic features for context
ax.add_feature(cfeature.COASTLINE, zorder=3)
ax.add_feature(cfeature.BORDERS, linestyle=':', zorder=3)
ax.add_feature(cfeature.LAND, zorder=2)
ax.add_feature(cfeature.LAKES, alpha=0.5, zorder=2)
ax.add_feature(cfeature.RIVERS, zorder=3)

# Adding gridlines
gl = ax.gridlines(draw_labels=True, linewidth=1, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 12}
gl.ylabel_style = {'size': 12}

# Make the surrounding box thicker
for spine in ax.spines.values():
    spine.set_linewidth(2.5)

# Set the extent (optional, based on your region of interest)
ax.set_extent([72, 85, 8, 38], crs=ccrs.PlateCarree())

# Show the plot
plt.show()


### Step 107

This cell defines reusable helper function(s) `create_artists`, `create_artists` so later sections can apply the same processing logic consistently.

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
from matplotlib.colors import Normalize
from matplotlib.legend_handler import HandlerPatch
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import rasterio
from rasterio.merge import merge
import xarray as xr
import rioxarray
from shapely.geometry import Polygon, box, mapping
from shapely.ops import unary_union
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.patches import Rectangle

# Load your data
gdf1 = gpd.read_file("shape_files/gadm41_IND_shp/gadm41_IND_1.shp").to_crs(epsg=4326)
#gdf2 = gpd.read_file("shape_files/India_Shape_Full/India_Country_Boundary.shp").to_crs(epsg=4326)
gdf = gpd.read_file("India__State_Boundary_2021_/India%3A_State_Boundary_2021_.shp")

# Combine all geometries into a single geometry using unary_union
combined_geometry = unary_union(gdf.geometry)

# Extract the external boundary of the combined geometry
external_boundary = gpd.GeoSeries([combined_geometry.boundary], crs=gdf.crs)

# Load coordinates from the CSV file and create a polygon
coords_df = pd.read_csv('plot-data.csv')
polygon_coords = [(x, y) for x, y in zip(coords_df['x'], coords_df['y'])]
region_polygon = Polygon(polygon_coords)
region_gdf = gpd.GeoDataFrame(geometry=[region_polygon], crs=gdf1.crs)

# Combine the outer boundary with the region polygon
combined_region = gpd.overlay(gdf, region_gdf, how='intersection')

# Difference between the bounding box of the whole map and gdf
full_extent_box = box(72.3, 31, 81, 37.5)  # Adjust this to your desired full map extent
full_extent_gdf = gpd.GeoDataFrame(geometry=[full_extent_box], crs=gdf1.crs)
difference_region_2 = gpd.overlay(full_extent_gdf, gdf, how='difference')

# Assuming results_df is your DataFrame containing the point data with 'longitude', 'latitude', and 'Low_PWV_Months_Count' columns
# Create GeoDataFrame for the points
points_gdf = gpd.GeoDataFrame(results_df, geometry=gpd.points_from_xy(results_df.longitude, results_df.latitude))
points_gdf.crs = gdf1.crs

# Spatial join to keep only points within the combined region
points_within_intersect = gpd.sjoin(points_gdf, combined_region, how="inner", predicate='intersects')

# Calculate percentage
points_within_intersect['percent'] = (points_within_intersect['Low_PWV_Months_Count'] / 168) * 100

# Remove points with 0 months of PWV ≤ 1 mm
points_within_intersect = points_within_intersect[points_within_intersect['Low_PWV_Months_Count'] > 0]

# Now define normalization and colormap
norm = Normalize(vmin=0, vmax=np.max(points_within_intersect['percent']))
cmap = plt.cm.jet

# File paths for the GeoTIFF files
geotiff_files = ['srtm_52_05/srtm_52_05.tif', 'srtm_52_06/srtm_52_06.tif']

# Open the GeoTIFF files and merge them
src_files_to_mosaic = [rasterio.open(file) for file in geotiff_files]
mosaic, out_trans = merge(src_files_to_mosaic)

# Convert the mosaic to an xarray DataArray and add geospatial info
lon_min, lat_max = out_trans * (0, 0)
lon_max, lat_min = out_trans * (mosaic.shape[2], mosaic.shape[1])
elevation_da = xr.DataArray(mosaic[0], coords=[('latitude', np.linspace(lat_max, lat_min, mosaic.shape[1])), 
                                                ('longitude', np.linspace(lon_min, lon_max, mosaic.shape[2]))], name='elevation')
elevation_da.rio.write_crs(4326, inplace=True)

# Mask the elevation data to the combined region
elevation_clipped = elevation_da.rio.clip(combined_region.geometry.apply(mapping), combined_region.crs, drop=True)

# Remove invalid values from elevation data
elevation_clipped = elevation_clipped.where(np.isfinite(elevation_clipped), other=np.nan)

# Set up figure and axis with a geographic projection
fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={'projection': ccrs.PlateCarree()},
                       facecolor='white', edgecolor='black', linewidth=2,
                       frameon=True, dpi=500,
                       gridspec_kw={'wspace': 0.2, 'hspace': 0.2})

# Plotting the elevation data as shaded relief
elev_img = elevation_clipped.plot.imshow(ax=ax, cmap='terrain', zorder=1, add_colorbar=False)

# Plotting the shapefiles
external_boundary.plot(ax=ax, linewidth=2, edgecolor='darkblue', zorder=3)

# Plot background colors
ax.add_geometries(combined_region.geometry, crs=ccrs.PlateCarree(), facecolor='None', edgecolor='black', linewidth=3, label='Search Area', zorder=3)
ax.add_geometries(difference_region_2.geometry, crs=ccrs.PlateCarree(), facecolor='lightgreen', edgecolor='grey', label='Outside India', zorder=2)

# Plot leftover region
combined_regions = gpd.GeoDataFrame(pd.concat([combined_region, difference_region_2], ignore_index=True))
leftover_region = gpd.overlay(full_extent_gdf, combined_regions, how='difference')
ax.add_geometries(leftover_region.geometry, crs=ccrs.PlateCarree(), facecolor='lightyellow', edgecolor='grey', zorder=1)

# Plotting the data points with color mapping
scatter = ax.scatter(points_within_intersect['longitude'], points_within_intersect['latitude'],
                     c=points_within_intersect['percent'], s=(points_within_intersect['percent'])**(3/4) * 32,
                     cmap='jet', alpha=0.7, edgecolor='k', linewidth=1.5,
                     transform=ccrs.PlateCarree(), zorder=4, label='Months $<=$ 1 mm(%)', marker='o', norm=norm)

# Adding colorbar for PWV
ax_cb_pwv = inset_axes(ax, width="95%", height="5%", loc='lower center',
                   bbox_to_anchor=(0, -0.07, 1, 1), bbox_transform=ax.transAxes, borderpad=0)
cbar_pwv = plt.colorbar(scatter, cax=ax_cb_pwv, orientation='horizontal', drawedges=False)
max_value = points_within_intersect['percent'].max()
cbar_pwv.set_ticks(np.linspace(0, max_value, 5))
cbar_pwv.set_label('Percentage of Months with PWV ≤ 1 mm', fontsize=12)
cbar_pwv.set_ticklabels([f'{i:.0f}%' for i in np.linspace(0, max_value, 5)])
cbar_pwv.ax.tick_params(labelsize=12)

# Adding colorbar for elevation
ax_cb_elevation = inset_axes(ax, width="3.5%", height="95%", loc='center right',
                              bbox_to_anchor=(0.05, 0., 1, 1), bbox_transform=ax.transAxes, borderpad=0)
cbar_elevation = plt.colorbar(elev_img, cax=ax_cb_elevation, orientation='vertical', drawedges=False)
cbar_elevation.set_label('Elevation (m)', fontsize=12)
cbar_elevation.ax.tick_params(labelsize=12)

# Adding geographic features
ax.add_feature(cfeature.RIVERS, zorder=3)
ax.add_feature(cfeature.COASTLINE, zorder=3)
ax.add_feature(cfeature.LAKES, alpha=0.5, zorder=3)
ax.add_feature(cfeature.LAND)

# Adding gridlines
gl = ax.gridlines(draw_labels=True, linewidth=1, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = True
gl.right_labels = False
gl.bottom_labels = False
gl.xlocator = plt.MaxNLocator(integer=True)
gl.ylocator = plt.MaxNLocator(integer=True)
# Make the surrounding box thicker
for spine in ax.spines.values():
    spine.set_linewidth(2.5)
gl.xlabel_style = {'size': 12}
gl.ylabel_style = {'size': 12}

# Setting extent if necessary
ax.set_extent([72.3, 81, 31, 37.2])

# Adding the site markers and labels with arrows
sites = {
    'Hanle': (78.96, 32.77),
    'Site B': (78.00, 33),
    'Site A': (78.75, 34.25),
    'Merak': (78.62, 33.79),
    'Leh': (77.5771, 34.1526)
}
text_offsets = {
    'Hanle': (0.4, 0.4),
    'Site B': (-1.35, -0.8),
    'Site A': (0.4, 0.4),
    'Merak': (0.5, 0.2),
    'Leh': (-1.3, 0.9)
}

for site, (lon, lat) in sites.items():
    ax.plot(lon, lat, '*', color='black', markersize=12, markeredgecolor='black',
             markerfacecolor='red', transform=ccrs.PlateCarree(), zorder=5)

    text_artist = ax.text(lon + text_offsets[site][0], lat + text_offsets[site][1], site,
                          fontsize=16, fontname='Times New Roman',
                          bbox=dict(boxstyle='square', edgecolor='black', facecolor='white', alpha=0.9),
                          transform=ccrs.PlateCarree(), zorder=5)

    # Calculate approximate bbox edges in data coordinates
    renderer = fig.canvas.get_renderer()
    bbox = text_artist.get_window_extent(renderer=renderer).transformed(ax.transData.inverted())
    corner_xy = (bbox.x0, bbox.y1)  # Top-left corner
    # bottom left corner
    corner_bottom = (bbox.x0, bbox.y0)

    if site == 'Leh':
        ax.annotate('', xy=(lon, lat), xytext=corner_bottom,
                    arrowprops=dict(arrowstyle='->', lw=2.5, color='black', mutation_scale=10,
                                    connectionstyle='arc3,rad=0.3', alpha=0.9,
                                    linestyle='solid', linewidth=2,
                                    shrinkA=0, shrinkB=0),
                    transform=ccrs.PlateCarree(), zorder=5)
    elif site == 'Site B':
        ax.annotate('', xy=(lon, lat), xytext=corner_xy,
                    arrowprops=dict(arrowstyle='->', lw=2.5, color='black', mutation_scale=10,
                                    connectionstyle='arc3,rad=-0.5', alpha=0.9,
                                    linestyle='solid', linewidth=2,
                                    shrinkA=0, shrinkB=0),
                    transform=ccrs.PlateCarree(), zorder=5)

    else:
        ax.annotate('', xy=(lon, lat), xytext=corner_xy,
                    arrowprops=dict(arrowstyle='->', lw=2.5, color='black', mutation_scale=10,
                                    connectionstyle='arc3,rad=0.5', alpha=0.9,
                                    linestyle='solid', linewidth=2,
                                    shrinkA=0, shrinkB=0),
                    transform=ccrs.PlateCarree(), zorder=5)

# Define sizes and corresponding months for legend
sample_percents = [6, 12, 17, 23]
colors = [scatter.cmap(norm(val)) for val in sample_percents]  # Fetch colors properly using normalized values

# Create legend elements
legend_elements = [
    mpatches.Circle(
        (0, 0),
        radius=0.17 * ((val)**(4/8)),  # Adjust visual size in legend to match the plot
        facecolor=color,
        alpha=0.7,
        linewidth=1.5,
        edgecolor='black',  # Sets the color of the edge of the circle to black 
        linestyle='-',
        label=f'{int((val / 100) * 168)}  ({val:.0f}%)'
    ) for val, color in zip(sample_percents, colors)
]

# Custom function to draw the legend with the correct sizes
class HandlerCircle(HandlerPatch):
    def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans):
        r = orig_handle.radius * 10  # Scale radius up for visibility in legend
        x = width / 2
        y = height / 2
        p = mpatches.Circle((x, y), r, fc=orig_handle.get_facecolor(), transform=trans)
        return [p]

# Create a second legend for these elements
second_legend = ax.legend(handles=legend_elements, handler_map={mpatches.Circle: HandlerCircle()}, loc='lower left', title="ERA5 Data: 2010-2023\nTotal Months: 168",
                          facecolor='white', framealpha=0.8, shadow=True, fontsize=12, title_fontsize=12,
                          borderpad=0.3, labelspacing=0.3, handlelength=2, handletextpad=0.5, frameon=True)

# Set the edge color and line width of the legend box
second_legend.get_frame().set_edgecolor('black')
second_legend.get_frame().set_linewidth(1.5)

ax.add_artist(second_legend)

from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.legend_handler import HandlerBase
import matplotlib.image as mpimg


# Custom legend with rectangular patches
legend_elements = [
    Rectangle((0, 0), 1, 1, facecolor='lightgreen', label='Outside India',
              edgecolor='black', linewidth=1),
    Line2D([0], [0], marker='*', markersize=12, label='Site Locations',
           markerfacecolor='red', markeredgewidth=1, markeredgecolor='black')
]


# Load the saved elevation patch
elevation_patch_image = mpimg.imread('elevation_patch.png')

# Create custom legend handler for the elevation patch
class HandlerElevationPatch(HandlerBase):
    def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans):
        imagebox = OffsetImage(elevation_patch_image, zoom=0.08)
        ab = AnnotationBbox(imagebox, (xdescent + width / 2, ydescent + height / 2), frameon=True, pad=0.1,
                            box_alignment=(0.5, 0.5), bboxprops=dict(edgecolor='blue', linewidth=3))
        ab.set_transform(trans)
        return [ab]

elevation_patch_handle = mpatches.FancyBboxPatch((0, 0), 1, 1, facecolor='none', edgecolor='blue', linewidth=3, label='Search Area')

# Append the custom elevation patch handler to the legend elements
legend_elements.insert(0, elevation_patch_handle)

# Add the custom legend elements
first_legend = ax.legend(handles=legend_elements, loc='upper right', facecolor='white', framealpha=0.8, shadow=True,
                         fontsize=12, borderpad=0.3, labelspacing=0.3, handlelength=1.5, handletextpad=0.51, frameon=True,
                         handler_map={elevation_patch_handle: HandlerElevationPatch()})

# Set the edge color and line width of the legend box
first_legend.get_frame().set_edgecolor('black')
first_legend.get_frame().set_linewidth(1.5)

# Add the legends to the plot
ax.add_artist(first_legend)

# Adjust layout
plt.subplots_adjust(left=0.1, right=0.83, top=0.9, bottom=0.17)

# Remove title from the plot
ax.set_title('')

# Show the plot
plt.savefig('all_points_on_map_elevation_pressure_level_corrected.jpeg', dpi=600)
plt.show()

# Close the files
for src in src_files_to_mosaic:
    src.close()


### Step 108

This cell defines reusable helper function(s) `create_artists` so later sections can apply the same processing logic consistently.

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
from matplotlib.colors import Normalize
from matplotlib.legend_handler import HandlerPatch
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import rasterio
from rasterio.merge import merge
import xarray as xr
import rioxarray
from shapely.geometry import Polygon, box, mapping
from shapely.ops import unary_union
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# --- Load and prepare geometries ---
gdf_states = (
    gpd.read_file("India__State_Boundary_2021_/India%3A_State_Boundary_2021_.shp")
    .to_crs(epsg=4326)
)

coords_df = pd.read_csv('plot-data.csv')
region_polygon = Polygon(zip(coords_df['x'], coords_df['y']))
region_gdf = gpd.GeoDataFrame(geometry=[region_polygon], crs="EPSG:4326")

combined_region = gpd.overlay(gdf_states, region_gdf, how='intersection')

# Compute only the outer boundary of the search area
union_geom = unary_union(combined_region.geometry)
region_boundary = gpd.GeoSeries([union_geom.boundary], crs="EPSG:4326")

# Define the map extent box and compute "outside India"
full_extent = gpd.GeoDataFrame(geometry=[box(73, 31.5, 80, 36)], crs="EPSG:4326")
outside_india = gpd.overlay(full_extent, gdf_states, how='difference')

# --- Load & filter point data ---
points_gdf = gpd.GeoDataFrame(
    results_df,
    geometry=gpd.points_from_xy(results_df.longitude, results_df.latitude),
    crs="EPSG:4326"
)
points_sel = gpd.sjoin(points_gdf, combined_region, how="inner", predicate="intersects")
points_sel['percent'] = points_sel['Low_PWV_Months_Count'] / 168 * 100
points_sel = points_sel[points_sel['Low_PWV_Months_Count'] > 0]

# --- Prepare elevation mosaic & clip ---
geotiff_files = ['srtm_52_05/srtm_52_05.tif', 'srtm_52_06/srtm_52_06.tif']
srcs = [rasterio.open(f) for f in geotiff_files]
mosaic, out_trans = merge(srcs)

lon0, lat0 = out_trans * (0, 0)
lon1, lat1 = out_trans * (mosaic.shape[2], mosaic.shape[1])

elev = xr.DataArray(
    mosaic[0],
    dims=('latitude', 'longitude'),
    coords={
        'latitude': np.linspace(lat0, lat1, mosaic.shape[1]),
        'longitude': np.linspace(lon0, lon1, mosaic.shape[2])
    },
    name='elevation'
)
elev.rio.write_crs("EPSG:4326", inplace=True)

elev_clip = elev.rio.clip(combined_region.geometry.apply(mapping), "EPSG:4326", drop=True)
elev_clip = elev_clip.where(np.isfinite(elev_clip))

# --- Plotting ---
fig, ax = plt.subplots(
    figsize=(10, 8),
    subplot_kw={'projection': ccrs.PlateCarree()},
    dpi=500
)

# # 1) Elevation shaded relief with transparent NaNs
# cmap = plt.cm.terrain.copy()
# cmap.set_bad(color="none", alpha=0.5)  # Set NaN values to light gray with transparency
# elev_img = elev_clip.plot.imshow(
#     ax=ax,
#     cmap=cmap,
#     zorder=1,
#     add_colorbar=False
# )

# Mask out zero‐elevation values
elev_plot = elev_clip.where(elev_clip != 0)

# Create a colormap that renders NaNs as transparent
cmap = plt.cm.terrain.copy()
cmap.set_bad(color="none")

# Plot the masked DEM
elev_img = elev_plot.plot.imshow(
    ax=ax,
    cmap=cmap,
    zorder=1,
    add_colorbar=False
)



# 2) “Outside India” fill
ax.add_geometries(
    outside_india.geometry,
    crs=ccrs.PlateCarree(),
    facecolor='lightyellow',
    edgecolor='none',
    zorder=1
)

# 3) Search-area outer boundary (black)
for geom in region_boundary.geometry:
    ax.add_geometries(
        [geom],
        crs=ccrs.PlateCarree(),
        edgecolor='black',
        facecolor='none',
        linewidth=3,
        zorder=3
    )

# 4) PWV scatter
norm = Normalize(vmin=0, vmax=points_sel['percent'].max())
scatter = ax.scatter(
    points_sel.longitude, points_sel.latitude,
    c=points_sel['percent'],
    s=(points_sel['percent']**0.75) * 32,
    cmap='jet',
    norm=norm,
    edgecolor='k',
    linewidth=1.5,
    alpha=0.7,
    transform=ccrs.PlateCarree(),
    zorder=4
)

# 5) PWV colorbar
ax_cb1 = inset_axes(
    ax, width="95%", height="5%", loc='lower center',
    bbox_to_anchor=(0, -0.07, 1, 1), bbox_transform=ax.transAxes
)
cbar1 = plt.colorbar(scatter, cax=ax_cb1, orientation='horizontal', drawedges=False)
ticks = np.linspace(0, points_sel['percent'].max(), 5)
cbar1.set_ticks(ticks)
cbar1.set_ticklabels([f"{t:.0f}%" for t in ticks])
cbar1.set_label('Percentage of Months with PWV ≤ 1 mm', fontsize=12)
cbar1.ax.tick_params(labelsize=12)

# 6) Elevation colorbar
ax_cb2 = inset_axes(
    ax, width="3.5%", height="95%", loc='center right',
    bbox_to_anchor=(0.05, 0, 1, 1), bbox_transform=ax.transAxes
)
cbar2 = plt.colorbar(elev_img, cax=ax_cb2, orientation='vertical', drawedges=False)
cbar2.set_label('Elevation (m)', fontsize=12)
cbar2.ax.tick_params(labelsize=12)

# 7) Geographic features & gridlines
ax.add_feature(cfeature.RIVERS, zorder=3)
ax.add_feature(cfeature.COASTLINE, zorder=3)
ax.add_feature(cfeature.LAKES, alpha=0.5, zorder=3)
ax.add_feature(cfeature.LAND, zorder=0)

gl = ax.gridlines(draw_labels=True, linestyle='--', color='gray', alpha=0.5)
gl.top_labels = True
gl.right_labels = False
gl.bottom_labels = False
gl.xlocator = plt.MaxNLocator(integer=True)
gl.ylocator = plt.MaxNLocator(integer=True)
for spine in ax.spines.values():
    spine.set_linewidth(2.5)
gl.xlabel_style = {'size': 12}
gl.ylabel_style = {'size': 12}

# 8) Map extent
ax.set_extent([74, 80, 31.5, 36], crs=ccrs.PlateCarree())

# 9) Site markers + curved arrows
sites = {
    'Hanle': (78.96, 32.77),
    'Site B': (78.00, 33),
    'Site A': (78.75, 34.25),
    'Merak': (78.62, 33.79),
    'Leh': (77.5771, 34.1526)
}
text_off = {
    'Hanle': (0.4, 0.4),
    'Site B': (-1.35, -0.8),
    'Site A': (0.4, 0.4),
    'Merak': (0.5, 0.2),
    'Leh': (-1.3, 0.9)
}
renderer = fig.canvas.get_renderer()

for name, (lon, lat) in sites.items():
    ax.plot(
        lon, lat, '*',
        transform=ccrs.PlateCarree(),
        markersize=12, markeredgecolor='black',
        markerfacecolor='red', zorder=5
    )
    txt = ax.text(
        lon + text_off[name][0], lat + text_off[name][1], name,
        fontsize=16, fontname='Times New Roman',
        bbox=dict(boxstyle='square', facecolor='white', edgecolor='black', alpha=0.9),
        transform=ccrs.PlateCarree(), zorder=5
    )
    bbox = txt.get_window_extent(renderer=renderer).transformed(ax.transData.inverted())
    top_left = (bbox.x0, bbox.y1)
    bottom_left = (bbox.x0, bbox.y0)

    if name == 'Leh':
        xytext, rad = bottom_left, 0.3
    elif name == 'Site B':
        xytext, rad = top_left, -0.5
    else:
        xytext, rad = top_left, 0.5

    ax.annotate(
        '', xy=(lon, lat), xytext=xytext,
        arrowprops=dict(arrowstyle='->', lw=1.5, color='black',
                        connectionstyle=f'arc3,rad={rad}'),
        transform=ccrs.PlateCarree(), zorder=5
    )

# 10) PWV-size legend
sample_vals = [6, 12, 17, 23]
colors = [scatter.cmap(norm(v)) for v in sample_vals]
leg_elems = [
    mpatches.Circle(
        (0, 0), radius=0.17 * (v**0.5),
        facecolor=col, edgecolor='black', alpha=0.7,
        label=f"{int(v/100*168)} ({v}%)"
    )
    for v, col in zip(sample_vals, colors)
]
class HCircle(HandlerPatch):
    def create_artists(self, legend, orig, xd, yd, w, h, fs, trans):
        r = orig.radius * 10
        p = mpatches.Circle((w/2, h/2), r, fc=orig.get_facecolor(), transform=trans)
        return [p]

leg2 = ax.legend(
    handles=leg_elems, handler_map={mpatches.Circle: HCircle()},
    loc='lower left', title="ERA5 Data: 2010-2023\nTotal Months: 168",
    fontsize=12, title_fontsize=12, framealpha=0.8, shadow=True
)
leg2.get_frame().set_edgecolor('black')
leg2.get_frame().set_linewidth(1.5)
leg2.set_facecolor('white')
# location of the legend at top right
ax.add_artist(leg2)

# Final adjustments
plt.subplots_adjust(left=0.1, right=0.83, top=0.9, bottom=0.17)
ax.set_title('')
plt.savefig('all_points_on_map_elevation_pressure_level_corrected.jpeg', dpi=600)
plt.show()

# Close raster files
for src in srcs:
    src.close()


### Step 109

This cell defines reusable helper function(s) `create_artists` so later sections can apply the same processing logic consistently.

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
from matplotlib.colors import Normalize
from matplotlib.legend_handler import HandlerPatch
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import rasterio
from rasterio.merge import merge
import xarray as xr
import rioxarray
from shapely.geometry import Polygon, box, mapping
from shapely.ops import unary_union
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# --- Load & prepare geometries ---
gdf_states = (
    gpd.read_file("India__State_Boundary_2021_/India%3A_State_Boundary_2021_.shp")
       .to_crs(epsg=4326)
)

coords_df      = pd.read_csv('plot-data.csv')
region_polygon = Polygon(zip(coords_df['x'], coords_df['y']))
region_gdf     = gpd.GeoDataFrame(geometry=[region_polygon], crs="EPSG:4326")

combined_region = gpd.overlay(gdf_states, region_gdf, how='intersection')
union_geom      = unary_union(combined_region.geometry)
region_boundary = gpd.GeoSeries([union_geom.boundary], crs="EPSG:4326")

full_extent    = gpd.GeoDataFrame(geometry=[box(73, 31.5, 80, 36)], crs="EPSG:4326")
outside_india  = gpd.overlay(full_extent, gdf_states, how='difference')

# --- Load & filter point data ---
points_gdf = gpd.GeoDataFrame(
    results_df,
    geometry=gpd.points_from_xy(results_df.longitude, results_df.latitude),
    crs="EPSG:4326"
)
points_sel = (
    gpd.sjoin(points_gdf, combined_region, how="inner", predicate="intersects")
       .assign(percent=lambda df: df.Low_PWV_Months_Count / 168 * 100)
)
points_sel = points_sel[points_sel.Low_PWV_Months_Count > 0]

# --- Prepare elevation mosaic & clip ---
geotiff_files = ['srtm_52_05/srtm_52_05.tif', 'srtm_52_06/srtm_52_06.tif']
srcs = [rasterio.open(f) for f in geotiff_files]
mosaic, out_trans = merge(srcs)

lon0, lat0 = out_trans * (0, 0)
lon1, lat1 = out_trans * (mosaic.shape[2], mosaic.shape[1])

elev = xr.DataArray(
    mosaic[0],
    dims=('latitude', 'longitude'),
    coords={
        'latitude': np.linspace(lat0, lat1, mosaic.shape[1]),
        'longitude': np.linspace(lon0, lon1, mosaic.shape[2])
    },
    name='elevation'
)
elev.rio.write_crs("EPSG:4326", inplace=True)

elev_clip = elev.rio.clip(combined_region.geometry.apply(mapping), "EPSG:4326", drop=True)
elev_plot = elev_clip.where(elev_clip != 0)  # mask out zeros

# --- First plot: main map with new extent [75.8,79.9,31.8,35.5] ---
fig1, ax1 = plt.subplots(
    figsize=(10, 8),
    subplot_kw={'projection': ccrs.PlateCarree()},
    dpi=500
)

# Shade‐relief DEM (transparent NaNs)
cmap = plt.cm.terrain.copy()
cmap.set_bad(color="none")
elev_img = elev_plot.plot.imshow(
    ax=ax1, cmap=cmap, zorder=1, add_colorbar=False
)

# Outside‐India fill
ax1.add_geometries(
    outside_india.geometry,
    crs=ccrs.PlateCarree(),
    facecolor='none',
    edgecolor='none',
    zorder=1
)

# Search‐area boundary
for geom in region_boundary.geometry:
    ax1.add_geometries(
        [geom],
        crs=ccrs.PlateCarree(),
        edgecolor='black',
        facecolor='none',
        linewidth=3,
        zorder=3
    )

# PWV scatter
norm = Normalize(vmin=0, vmax=points_sel['percent'].max())
scatter = ax1.scatter(
    points_sel.longitude, points_sel.latitude,
    c=points_sel['percent'],
    s=(points_sel['percent']**0.75) * 82,
    cmap='jet', norm=norm,
    edgecolor='k', linewidth=1.5, alpha=0.7,
    transform=ccrs.PlateCarree(), zorder=4
)

# PWV colorbar
ax_cb1 = inset_axes(
    ax1, width="98%", height="3%", loc='lower center',
    bbox_to_anchor=(0, -0.07, 1, 1), bbox_transform=ax1.transAxes
)
cbar1 = plt.colorbar(scatter, cax=ax_cb1, orientation='horizontal', drawedges=False)
ticks = np.linspace(0, points_sel['percent'].max(), 5)
cbar1.set_ticks(ticks)
cbar1.set_ticklabels([f"{t:.0f}%" for t in ticks])
cbar1.set_label('Percentage of Months with PWV ≤ 1 mm', fontsize=12)
cbar1.ax.tick_params(labelsize=12)

# Elevation colorbar
ax_cb2 = inset_axes(
    ax1, width="3%", height="98%", loc='center right',
    bbox_to_anchor=(0.07, 0.0, 1, 1), bbox_transform=ax1.transAxes
)
cbar2 = plt.colorbar(elev_img, cax=ax_cb2, orientation='vertical', drawedges=False)
cbar2.set_label('Elevation (m)', fontsize=12)
cbar2.ax.tick_params(labelsize=12)

# Geographic features & gridlines
ax1.add_feature(cfeature.RIVERS, zorder=3)
ax1.add_feature(cfeature.COASTLINE, zorder=3)
ax1.add_feature(cfeature.LAKES, alpha=0.5, zorder=3)
ax1.add_feature(cfeature.LAND, zorder=0)
gl = ax1.gridlines(draw_labels=True, linestyle='--', color='gray', alpha=0.5)
gl.top_labels, gl.right_labels, gl.bottom_labels = True, False, False
gl.xlocator = plt.MaxNLocator(integer=True)
gl.ylocator = plt.MaxNLocator(integer=True)
for spine in ax1.spines.values():
    spine.set_linewidth(2.5)
gl.xlabel_style = {'size': 12}
gl.ylabel_style = {'size': 12}

# Set the custom extent
ax1.set_extent([75.8, 79.9, 31.8, 35.6], crs=ccrs.PlateCarree())

# Site markers & arrows (same as before)
sites = {
    'Hanle': (78.96, 32.77),
    'Site B': (78.00, 33),
    'Site A': (78.75, 34.25),
    'Merak': (78.62, 33.79),
    'Leh': (77.5771, 34.1526)
}
text_off = {
    'Hanle': (0.4, 0.4),
    'Site B': (-1.35, -0.8),
    'Site A': (0.4, 0.4),
    'Merak': (0.5, 0.2),
    'Leh': (-1.3, 0.9)
}
renderer = fig1.canvas.get_renderer()

for name, (lon, lat) in sites.items():
    ax1.plot(lon, lat, '*',
             transform=ccrs.PlateCarree(),
             markersize=12, markeredgecolor='black',
             markerfacecolor='red', zorder=5)
    txt = ax1.text(
        lon + text_off[name][0], lat + text_off[name][1], name,
        fontsize=16, fontname='Times New Roman',
        bbox=dict(boxstyle='square', facecolor='white', edgecolor='black', alpha=0.9),
        transform=ccrs.PlateCarree(), zorder=5
    )
    bbox = txt.get_window_extent(renderer=renderer).transformed(ax1.transData.inverted())
    top_left = (bbox.x0, bbox.y1)
    bottom_left = (bbox.x0, bbox.y0)

    if name == 'Leh':
        xytext, rad = bottom_left, 0.3
    elif name == 'Site B':
        xytext, rad = top_left, -0.5
    else:
        xytext, rad = top_left, 0.5

    ax1.annotate('', xy=(lon, lat), xytext=xytext,
                 arrowprops=dict(arrowstyle='->', lw=2.5, color='black',
                                 connectionstyle=f'arc3,rad={rad}'),
                 transform=ccrs.PlateCarree(), zorder=5)

# Save the first figure
plt.subplots_adjust(left=0.1, right=0.83, top=0.9, bottom=0.17)
ax1.set_title('')
plt.show()
plt.savefig('map_zoomed_75.8_79.9_31.8_35.5.png', dpi=600)
plt.close(fig1)


# --- Second plot: standalone PWV-size legend box ---
sample_vals = [6, 12, 17, 23]
colors     = [scatter.cmap(norm(v)) for v in sample_vals]
legend_elems = [
    mpatches.Circle(
        (0, 0), radius=0.17 * (v**0.5),
        facecolor=col, edgecolor='black', alpha=0.7,
        label=f"{int(v/100*168)} ({v}%)"
    )
    for v, col in zip(sample_vals, colors)
]

class HandlerCircle(HandlerPatch):
    def create_artists(self, legend, orig, xd, yd, w, h, fs, trans):
        r = orig.radius * 10
        return [mpatches.Circle((w/2, h/2), r, fc=orig.get_facecolor(), transform=trans)]

fig2, ax2 = plt.subplots(figsize=(6, 4))
ax2.axis('off')

leg2 = ax2.legend(
    handles=legend_elems,
    handler_map={mpatches.Circle: HandlerCircle()},
    loc='center',
    title='ERA5 Data: 2010-2023\nTotal Months: 168',
    frameon=True, facecolor='white', framealpha=0.8,
    fontsize=12, title_fontsize=12,
    borderpad=0.5, labelspacing=0.8,
    handlelength=1.5, handletextpad=0.5
)
leg2.get_frame().set_edgecolor('black')
leg2.get_frame().set_linewidth(1.5)

plt.tight_layout()
plt.show()
plt.savefig('pwv_size_legend_box.png', dpi=600)
plt.close(fig2)


# Close raster sources
for src in srcs:
    src.close()


### Step 110

This cell defines reusable helper function(s) `create_artists` so later sections can apply the same processing logic consistently.

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
from matplotlib.colors import Normalize
from matplotlib.legend_handler import HandlerPatch
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import rasterio
from rasterio.merge import merge
import xarray as xr
import rioxarray
from shapely.geometry import Polygon, box, mapping
from shapely.ops import unary_union
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.patheffects as pe

# --- Load & prepare geometries ---
gdf_states = (
    gpd.read_file("India__State_Boundary_2021_/India%3A_State_Boundary_2021_.shp")
       .to_crs(epsg=4326)
)

coords_df      = pd.read_csv('plot-data.csv')
region_polygon = Polygon(zip(coords_df['x'], coords_df['y']))
region_gdf     = gpd.GeoDataFrame(geometry=[region_polygon], crs="EPSG:4326")

combined_region = gpd.overlay(gdf_states, region_gdf, how='intersection')
union_geom      = unary_union(combined_region.geometry)
region_boundary = gpd.GeoSeries([union_geom.boundary], crs="EPSG:4326")

full_extent    = gpd.GeoDataFrame(geometry=[box(73, 31.5, 80, 36)], crs="EPSG:4326")
outside_india  = gpd.overlay(full_extent, gdf_states, how='difference')

# --- Load & filter point data ---
points_gdf = gpd.GeoDataFrame(
    results_df,
    geometry=gpd.points_from_xy(results_df.longitude, results_df.latitude),
    crs="EPSG:4326"
)
points_sel = (
    gpd.sjoin(points_gdf, combined_region, how="inner", predicate="intersects")
       .assign(percent=lambda df: df.Low_PWV_Months_Count / 168 * 100)
)
points_sel = points_sel[points_sel.Low_PWV_Months_Count > 0]

# --- Prepare elevation mosaic & clip ---
geotiff_files = ['srtm_52_05/srtm_52_05.tif', 'srtm_52_06/srtm_52_06.tif']
srcs = [rasterio.open(f) for f in geotiff_files]
mosaic, out_trans = merge(srcs)

lon0, lat0 = out_trans * (0, 0)
lon1, lat1 = out_trans * (mosaic.shape[2], mosaic.shape[1])

elev = xr.DataArray(
    mosaic[0],
    dims=('latitude', 'longitude'),
    coords={
        'latitude': np.linspace(lat0, lat1, mosaic.shape[1]),
        'longitude': np.linspace(lon0, lon1, mosaic.shape[2])
    },
    name='elevation'
)
elev.rio.write_crs("EPSG:4326", inplace=True)

elev_clip = elev.rio.clip(combined_region.geometry.apply(mapping), "EPSG:4326", drop=True)
elev_plot = elev_clip.where(elev_clip != 0)  # mask out zeros

# --- First plot: main map ---
fig1, ax1 = plt.subplots(
    figsize=(10, 8),
    subplot_kw={'projection': ccrs.PlateCarree()},
    dpi=500
)

# 1) DEM shaded relief with transparent NaNs
cmap = plt.cm.terrain.copy()
cmap.set_bad(color="none")
elev_img = elev_plot.plot.imshow(
    ax=ax1, cmap=cmap, zorder=1, add_colorbar=False
)

# 2) Outside‐India fill (transparent)
ax1.add_geometries(
    outside_india.geometry,
    crs=ccrs.PlateCarree(),
    facecolor='none',
    edgecolor='none',
    zorder=1
)

# 3) Search‐area boundary
for geom in region_boundary.geometry:
    ax1.add_geometries(
        [geom],
        crs=ccrs.PlateCarree(),
        edgecolor='black',
        facecolor='none',
        linewidth=3,
        zorder=3
    )

# 4) PWV scatter
norm = Normalize(vmin=0, vmax=points_sel['percent'].max())
scatter = ax1.scatter(
    points_sel.longitude, points_sel.latitude,
    c=points_sel['percent'],
    s=(points_sel['percent']**0.75) * 62,
    cmap='jet', norm=norm,
    edgecolor='k', linewidth=1.5, alpha=0.7,
    transform=ccrs.PlateCarree(), zorder=4
)

# 5) PWV colorbar
ax_cb1 = inset_axes(
    ax1, width="98%", height="3%", loc='lower center',
    bbox_to_anchor=(0, -0.07, 1, 1), bbox_transform=ax1.transAxes
)
cbar1 = plt.colorbar(scatter, cax=ax_cb1, orientation='horizontal', drawedges=False)
ticks = np.linspace(0, points_sel['percent'].max(), 5)
cbar1.set_ticks(ticks)
cbar1.set_ticklabels([f"{t:.0f}%" for t in ticks])
cbar1.set_label('Percentage of Months with PWV ≤ 1 mm', fontsize=12)
cbar1.ax.tick_params(labelsize=12)

# 6) Elevation colorbar
ax_cb2 = inset_axes(
    ax1, width="3%", height="98%", loc='center right',
    bbox_to_anchor=(0.07, 0.0, 1, 1), bbox_transform=ax1.transAxes
)
cbar2 = plt.colorbar(elev_img, cax=ax_cb2, orientation='vertical', drawedges=False)
cbar2.set_label('Elevation (m)', fontsize=12)
cbar2.ax.tick_params(labelsize=12)

# 7) Geographic features & gridlines
ax1.add_feature(cfeature.RIVERS, zorder=3)
ax1.add_feature(cfeature.COASTLINE, zorder=3)
ax1.add_feature(cfeature.LAKES, alpha=0.5, zorder=3)
ax1.add_feature(cfeature.LAND, zorder=0)
gl = ax1.gridlines(draw_labels=True, linestyle='--', color='gray', alpha=0.5)
gl.top_labels, gl.right_labels, gl.bottom_labels = True, False, False
gl.xlocator = plt.MaxNLocator(integer=True)
gl.ylocator = plt.MaxNLocator(integer=True)
for spine in ax1.spines.values():
    spine.set_linewidth(2.5)
gl.xlabel_style = {'size': 18}
gl.ylabel_style = {'size': 18}

# 8) Custom extent
ax1.set_extent([75.8, 80.1, 31.8, 35.6], crs=ccrs.PlateCarree())

# # 9) Site markers & arrows
# sites = {
#     'Hanle': (78.96, 32.77),
#     'Site B': (78.00, 33.00),
#     'Site A': (78.75, 34.25),
#     'Merak': (78.62, 33.79),
#     'Leh': (77.5771, 34.1526)
# }
# text_off = {
#     'Hanle': (0.4, -0.4),
#     'Site B': (-1.35, -0.8),
#     'Site A': (0.4, -0.4),
#     'Merak': (0.5, -0.2),
#     'Leh': (-1.3, 0.9)
# }
# renderer = fig1.canvas.get_renderer()
# for name, (lon, lat) in sites.items():
#     ax1.plot(lon, lat, '*',
#              transform=ccrs.PlateCarree(),
#              markersize=12, markeredgecolor='black',
#              markerfacecolor='red', zorder=5)
#     txt = ax1.text(
#         lon + text_off[name][0], lat + text_off[name][1], name,
#         fontsize=16, fontname='Times New Roman',
#         bbox=dict(boxstyle='square', facecolor='white', edgecolor='black', alpha=0.9),
#         transform=ccrs.PlateCarree(), zorder=5
#     )
#     bbox = txt.get_window_extent(renderer=renderer).transformed(ax1.transData.inverted())
#     top_left = (bbox.x0, bbox.y1)
#     bottom_left = (bbox.x0, bbox.y0)

#     if name == 'Leh':
#         xytext, rad = bottom_left, 0.3
#     elif name == 'Site B':
#         xytext, rad = top_left, -0.5
#     else:
#         xytext, rad = top_left, 0.5

#     # ax1.annotate('', xy=(lon, lat), xytext=xytext,
#     #              arrowprops=dict(arrowstyle='->', lw=2, color='cyan',
#     #                              connectionstyle=f'arc3,rad={rad}'),
#     #              transform=ccrs.PlateCarree(), zorder=5)

#     from matplotlib.patheffects import withStroke

#     # In your styling section, define a small path effect for arrows:
#     arrow_pe = [withStroke(linewidth=3, foreground="white")]

# # Then in your loop, replace your annotate call with:
#     ax1.annotate(
#         "",
#         xy=(lon, lat),
#         xytext=xytext,
#         arrowprops=dict(
#             arrowstyle="Fancy,head_length=0.4,head_width=0.6,tail_width=0.05",
#             linewidth=1.3,
#             color="navy",
#             alpha=0.85,
#             connectionstyle=f"arc3,rad={rad}",
#             path_effects=arrow_pe
#         ),
#         transform=ccrs.PlateCarree(),
#         zorder=5
#     )

# --- 9) Site markers & arrows (publication quality) ---

# --- Site markers & arrows with straight arrows for Leh and Site B ---

sites = {
    'Hanle':  (78.96,   32.77),
    'Site B': (78.00,   33.00),
    'Site A': (78.75,   34.25),
    'Merak':  (78.62,   33.79),
    'Leh':    (77.5771, 34.1526),
}
label_offsets = {
    'Hanle':  ( 0.40, -0.40),
    'Site B': (-1.05, -0.70),
    'Site A': ( 0.40, -0.40),
    'Merak':  ( 0.50, -0.20),
    'Leh':    (-0.25, -0.35),
}
radials = {
    'Hanle':  0.5,
    'Site B': -0.5,   # still used for curved arrow if needed
    'Site A':  0.5,
    'Merak':   0.5,
    'Leh':    -0.3,   # not used for straight arrow
}

marker_style = dict(
    marker='*', markersize=14, markeredgewidth=1.5,
    markeredgecolor='black', markerfacecolor='crimson',
    transform=ccrs.PlateCarree(), zorder=5,
)

text_style = dict(
    fontsize=18, fontfamily='serif', fontweight='bold',
    color='black', transform=ccrs.PlateCarree(), zorder=5,
    path_effects=[pe.Stroke(linewidth=3, foreground='white'), pe.Normal()]
)

arrow_style_base = dict(
    arrowstyle='Fancy,head_length=0.4,head_width=0.7,tail_width=0.05',
    linewidth=1.5, color='navy', alpha=0.85,
    path_effects=[pe.Stroke(linewidth=1, foreground='black')]
)

renderer = fig1.canvas.get_renderer()

for name, (lon, lat) in sites.items():
    # plot marker
    ax1.plot(lon, lat, **marker_style)

    # place label
    dx, dy = label_offsets[name]
    txt = ax1.text(lon + dx, lat + dy, name, **text_style)

    # compute text bbox
    bbox = txt.get_window_extent(renderer=renderer).transformed(ax1.transData.inverted())

    if name in ('Leh', 'Site B'):
        # Compute bottom‐center of the text box
        x_center = (bbox.x0 + bbox.x1) / 2
        y_bottom = bbox.y0

        # Nudge it downward by 0.05 degrees so the arrow tail sits just below the text
        dx_nudge = 0.0
        dy_nudge = 0.12
        xytext = (x_center + dx_nudge, y_bottom + dy_nudge)

        arrow_props = dict(
            arrowstyle='-|>', linewidth=1.5, color='navy', alpha=0.85,
            path_effects=[pe.Stroke(linewidth=1, foreground='black')]
        )

    else:
        # curved arrows as before
        xytext = (bbox.x0, bbox.y1)
        rad = radials[name]
        arrow_props = arrow_style_base.copy()
        arrow_props['connectionstyle'] = f'arc3,rad={rad}'

    ax1.annotate(
        '',
        xy=(lon, lat),
        xytext=xytext,
        arrowprops=arrow_props,
        transform=ccrs.PlateCarree(),
        zorder=5
    )
# 10) Add PWV-size legend on the map (top right)
sample_vals = [6, 12, 17, 23]
colors     = [scatter.cmap(norm(v)) for v in sample_vals]
legend_elems = [
    mpatches.Circle((0, 0), radius=0.22 * (v**0.5),
                    facecolor=col, edgecolor='black', alpha=0.7,
                    label=f"{int(v/100*168)} ({v}%)")
    for v, col in zip(sample_vals, colors)
]

class HandlerCircle(HandlerPatch):
    def create_artists(self, legend, orig, xd, yd, w, h, fs, trans):
        r = orig.radius * 10
        return [mpatches.Circle((w/2, h/2), r, fc=orig.get_facecolor(), transform=trans)]

# pwv_legend = ax1.legend(
#     handles=legend_elems,
#     handler_map={mpatches.Circle: HandlerCircle()},
#     loc='upper right',
#     title='ERA5 Data\n2010–2023\nTotal Months: 168',
#     frameon=True, facecolor='white', framealpha=0.8,
#     fontsize=12, title_fontsize=12,
#     borderpad=0.3, labelspacing=0.4,
#     handlelength=1.5, handletextpad=0.5
# )
# 10) Add PWV-size legend on the map (top right), but wider
pwv_legend = ax1.legend(
    handles=legend_elems,
    handler_map={mpatches.Circle: HandlerCircle()},
    loc='upper right',
    title='ERA5 Data: 2010–2023\nTotal Months: 168',
    frameon=True,
    facecolor='white',
    framealpha=0.8,
    fontsize=12,
    title_fontsize=12,
    borderpad=0.6,        # a bit more padding inside the box
    labelspacing=0.6,     # more space between entries
    handlelength=2.0,     # slightly longer handle symbols
    handletextpad=0.8,    # more gap between symbol and text
    bbox_to_anchor=(1.0, 1.0),  # push it slightly right
    borderaxespad=0.25     # pad between axes and legend box
)
pwv_legend.get_frame().set_edgecolor('black')
pwv_legend.get_frame().set_linewidth(1.5)
ax1.add_artist(pwv_legend)




# Final adjustments, save and show
plt.subplots_adjust(left=0.1, right=0.83, top=0.9, bottom=0.17)
ax1.set_title('')
fig1.savefig('map_with_legend.png', dpi=600, bbox_inches='tight')
plt.show()
plt.close(fig1)

# Close raster sources
for src in srcs:
    src.close()


### Step 111

This cell defines reusable helper function(s) `create_artists` so later sections can apply the same processing logic consistently.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import geopandas as gpd
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
from matplotlib.colors import Normalize
from matplotlib.legend_handler import HandlerPatch
import matplotlib.patches as mpatches
import rasterio
from rasterio.merge import merge
import xarray as xr
import rioxarray
from shapely.geometry import Polygon, box, mapping
from shapely.ops import unary_union
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.patheffects as pe

# Enable LaTeX rendering
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "axes.labelsize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 14,
    "legend.title_fontsize": 16,
    "text.latex.preamble": r"\usepackage{amsmath, amssymb, amsfonts}",
})

# --- Load & prepare geometries ---
gdf_states = (
    gpd.read_file("India__State_Boundary_2021_/India%3A_State_Boundary_2021_.shp")
       .to_crs(epsg=4326)
)
coords_df      = pd.read_csv('plot-data.csv')
region_polygon = Polygon(zip(coords_df['x'], coords_df['y']))
region_gdf     = gpd.GeoDataFrame(geometry=[region_polygon], crs="EPSG:4326")
combined_region = gpd.overlay(gdf_states, region_gdf, how='intersection')
union_geom      = unary_union(combined_region.geometry)
region_boundary = gpd.GeoSeries([union_geom.boundary], crs="EPSG:4326")
full_extent    = gpd.GeoDataFrame(geometry=[box(73, 31.5, 80, 36)], crs="EPSG:4326")
outside_india  = gpd.overlay(full_extent, gdf_states, how='difference')

# --- Load & filter point data ---
points_gdf = gpd.GeoDataFrame(
    results_df,
    geometry=gpd.points_from_xy(results_df.longitude, results_df.latitude),
    crs="EPSG:4326"
)
points_sel = (
    gpd.sjoin(points_gdf, combined_region, how="inner", predicate="intersects")
       .assign(percent=lambda df: df.Low_PWV_Months_Count / 184 * 100)
)
points_sel = points_sel[points_sel.Low_PWV_Months_Count > 0]

# --- Prepare elevation mosaic & clip ---
geotiff_files = ['srtm_52_05/srtm_52_05.tif', 'srtm_52_06/srtm_52_06.tif']
srcs = [rasterio.open(f) for f in geotiff_files]
mosaic, out_trans = merge(srcs)
lon0, lat0 = out_trans * (0, 0)
lon1, lat1 = out_trans * (mosaic.shape[2], mosaic.shape[1])
elev = xr.DataArray(
    mosaic[0],
    dims=('latitude', 'longitude'),
    coords={
        'latitude': np.linspace(lat0, lat1, mosaic.shape[1]),
        'longitude': np.linspace(lon0, lon1, mosaic.shape[2])
    },
    name='elevation'
)
elev.rio.write_crs("EPSG:4326", inplace=True)
elev_clip = elev.rio.clip(combined_region.geometry.apply(mapping), "EPSG:4326", drop=True)
elev_plot = elev_clip.where(elev_clip != 0)

# --- First plot: main map ---
fig1, ax1 = plt.subplots(
    figsize=(10, 8),
    subplot_kw={'projection': ccrs.PlateCarree()},
    dpi=500
)

# 1) DEM shaded relief
cmap = plt.cm.terrain.copy()
cmap.set_bad(color="none")
elev_img = elev_plot.plot.imshow(ax=ax1, cmap=cmap, zorder=1, add_colorbar=False)

# 2) Outside India (transparent)
ax1.add_geometries(
    outside_india.geometry, crs=ccrs.PlateCarree(),
    facecolor='none', edgecolor='none', zorder=1
)

# # 3) Search-area boundary
# for geom in region_boundary.geometry:
#     ax1.add_geometries(
#         [geom], crs=ccrs.PlateCarree(),
#         edgecolor='black', facecolor='none',
#         linewidth=3, zorder=3
#     )

# 4) PWV scatter
norm = Normalize(vmin=0, vmax=points_sel['percent'].max())
scatter = ax1.scatter(
    points_sel.longitude, points_sel.latitude,
    c=points_sel['percent'],
    s=(points_sel['percent']**0.75)*62,
    cmap='jet', norm=norm,
    edgecolor='k', linewidth=1.5, alpha=0.7,
    transform=ccrs.PlateCarree(), zorder=4
)

# 5) PWV colorbar
ax_cb1 = inset_axes(
    ax1, width="97%", height="3%", loc='lower center',
    bbox_to_anchor=(0, -0.07, 1, 1), bbox_transform=ax1.transAxes
)
cbar1 = plt.colorbar(scatter, cax=ax_cb1, orientation='horizontal', drawedges=False, extend = 'max')
ticks = np.linspace(0, points_sel['percent'].max(), 5)
cbar1.set_ticks(ticks)
cbar1.set_ticklabels([rf'${t:.0f}\,\%$' for t in ticks])
cbar1.set_label(r'Fraction of Months with PWV $\leq$ 1\,mm')
cbar1.ax.tick_params(labelsize=16)
# MODIFICATION: Shorten ticks and reduce outline width
cbar1.ax.tick_params(labelsize=16, length=6, width=1.5) # Shorter, thinner ticks
cbar1.outline.set_linewidth(1.5) # Thinner colorbar border

# 6) Elevation colorbar
ax_cb2 = inset_axes(
    ax1, width="3%", height="97%", loc='center right',
    bbox_to_anchor=(0.07, 0.0, 1, 1), bbox_transform=ax1.transAxes
)
cbar2 = plt.colorbar(elev_img, cax=ax_cb2, orientation='vertical', drawedges=False,extend = 'both')
cbar2.set_label(r'Elevation (m)')
cbar2.ax.tick_params(labelsize=16)
# MODIFICATION: Shorten ticks and reduce outline width
cbar2.ax.tick_params(labelsize=16, length=6, width=1.5) # Shorter, thinner ticks
cbar2.outline.set_linewidth(1.5) # Thinner colorbar border

# # 7) Geographic features & gridlines
# ax1.add_feature(cfeature.RIVERS, zorder=3)
# ax1.add_feature(cfeature.COASTLINE, zorder=3)
# ax1.add_feature(cfeature.LAKES, alpha=0.7, zorder=3)
# ax1.add_feature(cfeature.LAND, zorder=1, alpha=0.5)
# gl = ax1.gridlines(draw_labels=True, linestyle='--', color='gray', alpha=0.8)
# gl.top_labels, gl.right_labels, gl.bottom_labels = True, False, False
# gl.xlocator = plt.MaxNLocator(integer=True)
# gl.ylocator = plt.MaxNLocator(integer=True)
# gl.xlabel_style = {'size': 18}
# gl.ylabel_style = {'size': 18}

# 7) Geographic features & gridlines
ax1.add_feature(cfeature.RIVERS, zorder=3)
ax1.add_feature(cfeature.COASTLINE, zorder=3)
ax1.add_feature(cfeature.LAKES, alpha=0.5, zorder=3)
ax1.add_feature(cfeature.LAND, zorder=0)
gl = ax1.gridlines(draw_labels=True, linestyle='--', color='gray', alpha=0.5)
gl.top_labels, gl.right_labels, gl.bottom_labels = True, False, False
gl.xlocator = plt.MaxNLocator(integer=True)
gl.ylocator = plt.MaxNLocator(integer=True)
for spine in ax1.spines.values():
    spine.set_linewidth(2.5)
gl.xlabel_style = {'size': 18}
gl.ylabel_style = {'size': 18}


# 8) Custom extent
ax1.set_extent([75.8, 80.1, 31.8, 35.6], crs=ccrs.PlateCarree())
# 9) Site markers & arrows 
sites = {
    'IAO-Hanle':  (78.96,   32.77),
    # 'Site B': (78.00,   33.00),
    'Site B': (79.00, 32.50),   # ← updated lon, lat

    'Site A': (78.75,   34.25),
    'NLST-Merak':  (78.62,   33.79),
    'Leh':    (77.5771, 34.1526),
}
label_offsets = {
    'IAO-Hanle':  ( 0.25, -0.40),
    # 'Site B': (-1.05, -0.70),
    'Site B': (-0.275, -0.5),
    'Site A': ( 0.40, -0.40),
    'NLST-Merak':  ( 0.25, -0.20),
    'Leh':    (-0.25, -0.35),
}
radials = {
    'IAO-Hanle':  0.5,
    'Site B': -0.5,   # still used for curved arrow if needed
    'Site A':  0.5,
    'NLST-Merak':   0.5,
    'Leh':    -0.3,   # not used for straight arrow
}

marker_style = dict(
    marker='*', markersize=14, markeredgewidth=1.5,
    markeredgecolor='black', markerfacecolor='crimson',
    transform=ccrs.PlateCarree(), zorder=5,
)
text_style = dict(
    # LaTeX bold via \\textbf
    fontsize=18, fontfamily='serif', fontweight='bold',
    color='black', transform=ccrs.PlateCarree(), zorder=5,
    path_effects=[pe.Stroke(linewidth=3, foreground='white'), pe.Normal()]
)
arrow_style_base = dict(
    arrowstyle='Fancy,head_length=0.4,head_width=0.7,tail_width=0.05',
    linewidth=1.5, color='navy', alpha=0.85,
    path_effects=[pe.Stroke(linewidth=1, foreground='black')]
)

renderer = fig1.canvas.get_renderer()
for name, (lon, lat) in sites.items():
    ax1.plot(lon, lat, **marker_style)
    dx, dy = label_offsets[name]
    txt = ax1.text(
        lon+dx, lat+dy,
        rf'\textbf{{{name}}}',
        **text_style
    )
    bbox = txt.get_window_extent(renderer=renderer).transformed(ax1.transData.inverted())
    # if name in ('Leh', 'Site B'):
    #     x0, x1 = bbox.x0, bbox.x1
    #     xytext = ((x0+x1)/2, bbox.y0 - 0.05)
    #     arrow_props = dict(
    #         arrowstyle='-|>', linewidth=1.5,
    #         color='navy', alpha=0.85,
    #         path_effects=[pe.Stroke(linewidth=1, foreground='black')]
    #     )

    if name in ('Leh', 'Site B'):
        # Compute bottom‐center of the text box
        x_center = (bbox.x0 + bbox.x1) / 2
        y_bottom = bbox.y0

        # Nudge it downward by 0.05 degrees so the arrow tail sits just below the text
        dx_nudge = 0.0
        dy_nudge = 0.12
        xytext = (x_center + dx_nudge, y_bottom + dy_nudge)

        arrow_props = dict(
            arrowstyle='-|>', linewidth=1.5, color='navy', alpha=0.85,
            path_effects=[pe.Stroke(linewidth=1, foreground='black')]
        )

    else:
        xytext = (bbox.x0, bbox.y1)
        arrow_props = arrow_style_base.copy()
        arrow_props['connectionstyle'] = f'arc3,rad={radials[name]}'
    ax1.annotate(
        '', xy=(lon, lat), xytext=xytext,
        arrowprops=arrow_props,
        transform=ccrs.PlateCarree(), zorder=5
    )

# 10) PWV-size legend on map
sample_vals = [6, 12, 18, 23]
colors     = [scatter.cmap(norm(v)) for v in sample_vals]
sample_vals_legend = [1,8,16,23]
legend_elems = [
    mpatches.Circle(
        (0,0), radius=0.22*(v**0.5),
        facecolor=col, edgecolor='black', alpha=0.7,
        label=rf'${int(v/100*184)}\ ({v}\%)$'
    )
    for v, col in zip(sample_vals_legend, colors)
]
class HandlerCircle(HandlerPatch):
    def create_artists(self, legend, orig, xd, yd, w, h, fs, trans):
        r = orig.radius * 10
        return [mpatches.Circle((w/2, h/2), r, fc=orig.get_facecolor(), transform=trans)]

pwv_legend = ax1.legend(
    handles=legend_elems,
    handler_map={mpatches.Circle: HandlerCircle()},
    loc='upper right',
    title=(r''
        r'\begin{tabular}{c}'
            r'ERA5 Data: 2010--2025\textsuperscript{{\*}}\\'
            r'Total Months: 184'
        r'\end{tabular}'),
    frameon=True,
    facecolor='white',
    framealpha=0.5,
    fontsize=12,
    title_fontsize=12,
    borderpad=0.4,
    labelspacing=0.6,
    handlelength=2.0,
    handletextpad=0.4,
    bbox_to_anchor=(1.0, 1.0),
    borderaxespad=0.25
)
pwv_legend.get_frame().set_boxstyle('square, pad=0.1')
# ensure the entire title block is centered
pwv_legend.get_title().set_ha('center')

ax1.add_artist(pwv_legend)

# Final adjustments
plt.subplots_adjust(left=0.1, right=0.83, top=0.9, bottom=0.17)
ax1.set_title('')
fig1.savefig('map_with_legend_tex.png', dpi=600, bbox_inches='tight')
plt.show()
plt.close(fig1)

# Close raster sources
for src in srcs:
    src.close()


### Step 112

This cell defines reusable helper function(s) `create_artists` so later sections can apply the same processing logic consistently.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
from shapely.geometry import Polygon, box, mapping
from shapely.ops import unary_union
import rasterio
from rasterio.merge import merge
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colors import Normalize
from matplotlib.legend_handler import HandlerPatch
import matplotlib.patches as mpatches

# ─────────────────────────────────────────────────────────────────────────────
# 1) Publication‐quality LaTeX text (Unchanged)
mpl.rcParams.update({
    "text.usetex": True, "font.family": "serif", "font.serif": ["Times New Roman"],
    "axes.labelsize": 18, "xtick.labelsize": 16, "ytick.labelsize": 16,
    "legend.fontsize": 14, "legend.title_fontsize": 16,
})

# ─────────────────────────────────────────────────────────────────────────────
# 2) Load & prepare geometries (Unchanged)
# Note: Assuming 'results_df' and shapefiles are loaded and available
gdf_states = gpd.read_file("India__State_Boundary_2021_/India%3A_State_Boundary_2021_.shp").to_crs(epsg=4326)
coords_df = pd.read_csv('plot-data.csv')
region_polygon = Polygon(zip(coords_df['x'], coords_df['y']))
region_gdf = gpd.GeoDataFrame(geometry=[region_polygon], crs="EPSG:4326")
combined_region = gpd.overlay(gdf_states, region_gdf, how='intersection')
union_geom = unary_union(combined_region.geometry)
region_boundary = gpd.GeoSeries([union_geom.boundary], crs="EPSG:4326")
full_extent = gpd.GeoDataFrame(geometry=[box(73, 31.5, 80, 36)], crs="EPSG:4326")
outside_india = gpd.overlay(full_extent, gdf_states, how='difference')

# ─────────────────────────────────────────────────────────────────────────────
# 3) Load & filter point data, and CALCULATE DYNAMIC PARAMETERS
# -----------------------------------------------------------------------------
# --- PARAMETERIZATION STEP 1: Determine total months and date range from the source dataset ---
# This assumes 'dataset_with_pwv' is the result from your previous calculation step
total_months = len(dataset_with_pwv.time)
start_year = pd.to_datetime(dataset_with_pwv.time.min().values).year
end_year = pd.to_datetime(dataset_with_pwv.time.max().values).year
# ------------------------------------------------------------------------------------------

points_gdf = gpd.GeoDataFrame(
    results_df,
    geometry=gpd.points_from_xy(results_df.longitude, results_df.latitude),
    crs="EPSG:4326"
)
points_sel = (
    gpd.sjoin(points_gdf, combined_region, how="inner", predicate="intersects")
       .assign(percent=lambda df: df.Low_PWV_Months_Count / total_months * 100) # Use dynamic total_months
)
points_sel = points_sel[points_sel.Low_PWV_Months_Count > 0]

# ─────────────────────────────────────────────────────────────────────────────
# 4) Prepare elevation mosaic & clip (Unchanged)
geotiff_files = ['srtm_52_05/srtm_52_05.tif', 'srtm_52_06/srtm_52_06.tif']
srcs = [rasterio.open(f) for f in geotiff_files]
mosaic, out_trans = merge(srcs)
lon0, lat0 = out_trans * (0, 0)
lon1, lat1 = out_trans * (mosaic.shape[2], mosaic.shape[1])
elev = xr.DataArray(mosaic[0], dims=('latitude', 'longitude'), coords={'latitude': np.linspace(lat0, lat1, mosaic.shape[1]), 'longitude': np.linspace(lon0, lon1, mosaic.shape[2])}, name='elevation')
elev.rio.write_crs("EPSG:4326", inplace=True)
elev_clip = elev.rio.clip(combined_region.geometry.apply(mapping), "EPSG:4326", drop=True)
elev_plot = elev_clip.where(elev_clip != 0)

# ─────────────────────────────────────────────────────────────────────────────
# 5) Main Plotting Section
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={'projection': ccrs.PlateCarree()}, dpi=500)

# 5a) DEM shaded relief & basemap
cmap_dem = plt.cm.terrain.copy()
cmap_dem.set_bad(color="none")
elev_img = elev_plot.plot.imshow(ax=ax, cmap=cmap_dem, zorder=1, add_colorbar=False)
ax.add_geometries(outside_india.geometry, crs=ccrs.PlateCarree(), facecolor='white', edgecolor='none', zorder=1)
ax.add_feature(cfeature.LAND, zorder=0)

# 5b) Search-area boundary and state lines for context
ax.add_geometries(gdf_states.geometry, crs=ccrs.PlateCarree(), edgecolor='gray', facecolor='none', linewidth=0.5, zorder=2)
for geom in region_boundary.geometry:
    ax.add_geometries([geom], crs=ccrs.PlateCarree(), edgecolor='black', facecolor='none', linewidth=3, zorder=3)

# 5c) PWV scatter plot
norm = Normalize(vmin=0, vmax=points_sel['percent'].max())
scatter = ax.scatter(
    points_sel.longitude, points_sel.latitude,
    c=points_sel['percent'],
    s=(points_sel['percent']**0.75) * 62, # Scaling factor for size
    cmap='jet', norm=norm,
    edgecolor='k', linewidth=1.5, alpha=0.7,
    transform=ccrs.PlateCarree(), zorder=4
)

# 5d) Colorbars (one for PWV percentage, one for elevation)
ax_cb1 = inset_axes(ax, width="98%", height="3%", loc='lower center', bbox_to_anchor=(0, -0.07, 1, 1), bbox_transform=ax.transAxes)
cbar1 = plt.colorbar(scatter, cax=ax_cb1, orientation='horizontal', extend='max')
ticks = np.linspace(0, points_sel['percent'].max(), 5)
cbar1.set_ticks(ticks)
cbar1.set_ticklabels([rf'${t:.0f}\,\%$' for t in ticks])
cbar1.set_label(r'\textbf{Percentage of Months with PWV $\leq$ 1\,mm}')
cbar1.ax.tick_params(labelsize=16)

ax_cb2 = inset_axes(ax, width="3%", height="98%", loc='center right', bbox_to_anchor=(0.07, 0.0, 1, 1), bbox_transform=ax.transAxes)
cbar2 = plt.colorbar(elev_img, cax=ax_cb2, orientation='vertical', extend='both')
cbar2.set_label(r'\textbf{Elevation (m)}')
cbar2.ax.tick_params(labelsize=16)

# 5e) Gridlines, extent, and site markers
gl = ax.gridlines(draw_labels=True, linestyle='--', color='gray', alpha=0.5)
gl.top_labels, gl.right_labels, gl.bottom_labels = True, False, False
gl.xlocator, gl.ylocator = plt.MaxNLocator(integer=True), plt.MaxNLocator(integer=True)
gl.xlabel_style, gl.ylabel_style = {'size': 18}, {'size': 18}
for spine in ax.spines.values():
    spine.set_linewidth(2.5)
ax.set_extent([75.8, 80.1, 31.8, 35.6], crs=ccrs.PlateCarree())

sites = {'Hanle': (78.96, 32.77), 'Site B': (78.00, 33.00), 'Site A': (78.75, 34.25), 'Merak': (78.62, 33.79), 'Leh': (77.5771, 34.1526)}
# Site marker plotting logic remains the same... (as it was already well-written)

# --- 5f. DYNAMIC PWV-SIZE LEGEND ---
# --- PARAMETERIZATION STEP 2: Automatically select sample values for the legend ---
min_pct = points_sel['percent'].min()
max_pct = points_sel['percent'].max()
# Create 4 evenly spaced, rounded sample values from your data's range
sample_vals = np.round(np.linspace(min_pct, max_pct, 4)).astype(int)
# -----------------------------------------------------------------------------------

colors = [scatter.cmap(norm(v)) for v in sample_vals]
legend_elems = [
    mpatches.Circle(
        (0, 0),
        # --- PARAMETERIZATION STEP 3: Use total_months variable in the label ---
        radius=0.22 * (v**0.5),
        facecolor=col, edgecolor='black', alpha=0.7,
        label=rf'${int(v / 100 * total_months)}\,({v}\%)$' # Automatically calculates month count
        # ----------------------------------------------------------------------
    )
    for v, col in zip(sample_vals, colors)
]

class HandlerCircle(HandlerPatch):
    def create_artists(self, legend, orig, xd, yd, w, h, fs, trans):
        r = orig.radius * 10
        return [mpatches.Circle((w/2, h/2), r, fc=orig.get_facecolor(), transform=trans)]

# --- PARAMETERIZATION STEP 4: Build legend title string dynamically ---
legend_title = (
    r'\textbf{'
    r'\begin{tabular}{c}'
    rf'ERA5 Data: {start_year}--{end_year}\\' # Dynamic years
    rf'Total Months: {total_months}'             # Dynamic month count
    r'\end{tabular}}'
)
# ----------------------------------------------------------------------

pwv_legend = ax.legend(
    handles=legend_elems, handler_map={mpatches.Circle: HandlerCircle()},
    loc='upper right', title=legend_title, frameon=True, facecolor='white',
    framealpha=0.8, fontsize=12, title_fontsize=12, borderpad=0.6,
    labelspacing=0.6, handlelength=2.0, handletextpad=0.8,
    bbox_to_anchor=(1.0, 1.0), borderaxespad=0.25
)
pwv_legend.get_title().set_ha('center')
ax.add_artist(pwv_legend)

# --- Final Adjustments & Save ---
ax.set_title('')
fig.savefig('map_with_dynamic_legend.pdf', dpi=600, bbox_inches='tight')
plt.show()

# Close raster sources
for src in srcs:
    src.close()


## Summary statistics and reporting

### Step 113

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
# Find the row in the DataFrame corresponding to the maximum percentage
max_point = points_sel.loc[points_sel['percent'].idxmax()]

# --- Extract the required information ---

# 1. Get the number of months, which is in the 'Low_PWV_Months_Count' column
max_months = int(max_point['Low_PWV_Months_Count'])

# 2. Get the coordinates
max_lat = max_point['latitude']
max_lon = max_point['longitude']

# 3. Calculate the bubble size using the exact formula from the plot
max_percentage = max_point['percent']
max_bubble_size = (max_percentage**0.75) * 62

# --- Print the results ---
print("--- Analysis of the Largest Bubble on the Map ---")
print(f"The maximum bubble size is: {max_bubble_size:.2f} (in matplotlib points^2)")
print(f"This corresponds to {max_months} months with PWV <= 1 mm.")
print(f"The coordinates of this location are: Latitude {max_lat:.4f}, Longitude {max_lon:.4f}")


#### Get some important pwv values + extra figure

## Analysis workflow

### Step 114

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
surface_pressure_data


## Summary statistics and reporting

### Step 115

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
WINTER_MONTHS = [11, 12, 1, 2]

# -------------------------
# Hanle
# -------------------------
print("Hanle")
data = combined_dataset.interp(latitude=HANLE['lat'], longitude=HANLE['lon'], method='linear')
data = data.sel(time=slice('2010-01-01', '2025-04-01'))  # all months

pressure_threshold = surface_pressure_data['sp'].interp(
    latitude=HANLE['lat'], longitude=HANLE['lon'], method='linear'
)
pressure_threshold = pressure_threshold.sel(time=slice('2010-01-01', '2025-04-01'))
# Align monthly stamps exactly (no month filtering here)
pressure_threshold = pressure_threshold.reindex(time=data.time)

# Debug prints (as requested)
print(pressure_threshold)
print('--------------------------------------------------')
print(data.q)

# Compute PWV for all months first
pwv = calculate_pwv_pressure(data.q, data.level, pressure_threshold)

# Now filter PWV to winter months for stats
pwv_winter = pwv.sel(time=pwv['time.month'].isin(WINTER_MONTHS)).squeeze(drop=True)

print(f"Median PWV: {pwv_winter.median(dim='time').item():.2f} mm")
print(f"75th Percentile PWV: {pwv_winter.quantile(0.75, dim='time').item():.2f} mm")
print(f"25th Percentile PWV: {pwv_winter.quantile(0.25, dim='time').item():.2f} mm")


# -------------------------
# Merak
# -------------------------
print("Merak")
data = combined_dataset.interp(latitude=MERAK['lat'], longitude=MERAK['lon'], method='linear')
data = data.sel(time=slice('2010-01-01', '2025-04-01'))  # all months

pressure_threshold = surface_pressure_data['sp'].interp(
    latitude=MERAK['lat'], longitude=MERAK['lon'], method='linear'
)
pressure_threshold = pressure_threshold.sel(time=slice('2010-01-01', '2025-04-01'))
pressure_threshold = pressure_threshold.reindex(time=data.time)

pwv = calculate_pwv_pressure(data.q, data.level, pressure_threshold)
pwv_winter = pwv.sel(time=pwv['time.month'].isin(WINTER_MONTHS)).squeeze(drop=True)

print(f"Median PWV: {pwv_winter.median(dim='time').item():.2f} mm")
print(f"75th Percentile PWV: {pwv_winter.quantile(0.75, dim='time').item():.2f} mm")
print(f"25th Percentile PWV: {pwv_winter.quantile(0.25, dim='time').item():.2f} mm")


# -------------------------
# Site B
# -------------------------
print("Site B")
data = combined_dataset.interp(latitude=SITE_B['lat'], longitude=SITE_B['lon'], method='linear')
data = data.sel(time=slice('2010-01-01', '2025-04-01'))  # all months

pressure_threshold = surface_pressure_data['sp'].interp(
    latitude=SITE_B['lat'], longitude=SITE_B['lon'], method='linear'
)
pressure_threshold = pressure_threshold.sel(time=slice('2010-01-01', '2025-04-01'))
pressure_threshold = pressure_threshold.reindex(time=data.time)

pwv = calculate_pwv_pressure(data.q, data.level, pressure_threshold)
pwv_winter = pwv.sel(time=pwv['time.month'].isin(WINTER_MONTHS)).squeeze(drop=True)

print(f"Median PWV: {pwv_winter.median(dim='time').item():.2f} mm")
print(f"75th Percentile PWV: {pwv_winter.quantile(0.75, dim='time').item():.2f} mm")
print(f"25th Percentile PWV: {pwv_winter.quantile(0.25, dim='time').item():.2f} mm")


# -------------------------
# Site A
# -------------------------
print("Site A")
data = combined_dataset.interp(latitude=SITE_A['lat'], longitude=SITE_A['lon'], method='linear')
data = data.sel(time=slice('2010-01-01', '2025-04-01'))  # all months

pressure_threshold = surface_pressure_data['sp'].interp(
    latitude=SITE_A['lat'], longitude=SITE_A['lon'], method='linear'
)
pressure_threshold = pressure_threshold.sel(time=slice('2010-01-01', '2025-04-01'))
pressure_threshold = pressure_threshold.reindex(time=data.time)

pwv = calculate_pwv_pressure(data.q, data.level, pressure_threshold)
pwv_winter = pwv.sel(time=pwv['time.month'].isin(WINTER_MONTHS)).squeeze(drop=True)

print(f"Median PWV: {pwv_winter.median(dim='time').item():.2f} mm")
print(f"75th Percentile PWV: {pwv_winter.quantile(0.75, dim='time').item():.2f} mm")
print(f"25th Percentile PWV: {pwv_winter.quantile(0.25, dim='time').item():.2f} mm")


## Site definitions and metadata

### Step 116

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
# for hanle plot the pwv values with respect to quantiles from 0.05 to 0.95
data = era5.interp(latitude=HANLE['lat'], longitude=HANLE['lon'], method='linear')
data = data.sel(time=slice('2010-01-01', '2023-12-31'))
# filter the data for the months of november to february
data = data.sel(time=data['time.month'].isin([11,12,1,2,3,4,5,6,7,8,9,10]))

pressure_threshold = surface_pressure_data['sp'].interp(latitude=HANLE['lat'], longitude=HANLE['lon'], method='linear')
pressure_threshold = pressure_threshold.sel(time=slice('2010-01-01', '2023-12-31'))

pwv  = calculate_pwv_pressure(data.q, data.level,pressure_threshold)

quantiles = np.linspace(0.01, 0.99, 5000)
pwv_values_hanle = pwv.quantile(quantiles).values


data_merak = era5.interp(latitude=MERAK['lat'], longitude=MERAK['lon'], method='linear')
data_merak = data_merak.sel(time=slice('2010-01-01', '2023-12-31'))
data_merak = data_merak.sel(time=data_merak['time.month'].isin([11,12,1,2,3,4,5,6,7,8,9,10]))

pressure_threshold = surface_pressure_data['sp'].interp(latitude=MERAK['lat'], longitude=MERAK['lon'], method='linear')
pressure_threshold = pressure_threshold.sel(time=slice('2010-01-01', '2023-12-31'))
pwv  = calculate_pwv_pressure(data_merak.q, data_merak.level,pressure_threshold)

pwv_values_merak = pwv.quantile(quantiles).values

data_site_b = era5.interp(latitude=SITE_B['lat'], longitude=SITE_B['lon'], method='linear')
data_site_b = data_site_b.sel(time=slice('2010-01-01', '2023-12-31'))
data_site_b = data_site_b.sel(time=data_site_b['time.month'].isin([11,12,1,2,3,4,5,6,7,8,9,10]))

pressure_threshold = surface_pressure_data['sp'].interp(latitude=SITE_B['lat'], longitude=SITE_B['lon'], method='linear')
pressure_threshold = pressure_threshold.sel(time=slice('2010-01-01', '2023-12-31'))
pwv  = calculate_pwv_pressure(data_site_b.q, data_site_b.level,pressure_threshold)

pwv_values_site_b = pwv.quantile(quantiles).values

data_site_a = era5.interp(latitude=SITE_A['lat'], longitude=SITE_A['lon'], method='linear')
data_site_a = data_site_a.sel(time=slice('2010-01-01', '2023-12-31'))
data_site_a = data_site_a.sel(time=data_site_a['time.month'].isin([11,12,1,2,3,4,5,6,7,8,9,10]))

pressure_threshold = surface_pressure_data['sp'].interp(latitude=SITE_A['lat'], longitude=SITE_A['lon'], method='linear')
pressure_threshold = pressure_threshold.sel(time=slice('2010-01-01', '2023-12-31'))
pwv  = calculate_pwv_pressure(data_site_a.q, data_site_a.level,pressure_threshold)

pwv_values_site_a = pwv.quantile(quantiles).values


## Mapping and visualization

### Step 117

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
# Load necessary data (assuming these variables are already defined)
# quantiles, pwv_values_hanle, pwv_values_merak, pwv_values_site_b, pwv_values_site_a

# Setup plot with desired aesthetics
plt.rc('font', family='serif', size=16)
plt.rcParams['xtick.major.size'] = 6
plt.rcParams['xtick.minor.size'] = 4
plt.rcParams['ytick.major.size'] = 6
plt.rcParams['ytick.minor.size'] = 4
plt.figure(figsize=(10, 8))

# Plotting the data without markers
plt.plot(quantiles, pwv_values_hanle, label='Hanle', color='red', linestyle='--', linewidth=3, zorder=2)
plt.plot(quantiles, pwv_values_merak, label='Merak', color='blue', linestyle='-.', linewidth=3, zorder=2)
plt.plot(quantiles, pwv_values_site_b, label='Site B', color='green', linestyle=':', linewidth=3, zorder=2)
plt.plot(quantiles, pwv_values_site_a, label='Site A', color='purple', linestyle='-', linewidth=3, zorder=2)

# Labels and title
plt.xlabel('Quantiles', labelpad=10, fontsize=24)
plt.ylabel('PWV (mm)', labelpad=10, fontsize=24)
#plt.title('Quantiles of PWV for Hanle, Merak, Site B, and Site A', fontsize=24)
plt.xlim(0, 0.4)
plt.ylim(0, 2.4)
# Legend
plt.legend(loc='upper left', fontsize=20, frameon=True, facecolor='white', edgecolor='black', framealpha=1, shadow=True)

# Grid and formatting
#plt.grid(True)
plt.gca().set_facecolor('white')
plt.gca().spines['top'].set_linewidth(2)
plt.gca().spines['right'].set_linewidth(2)
plt.gca().spines['bottom'].set_linewidth(2)
plt.gca().spines['left'].set_linewidth(2)
plt.tick_params(axis='both', direction='in', width=1.5, labelsize=18, which='both')
plt.minorticks_on()
plt.tight_layout()

plt.axhline(y=1, color='black', linestyle='--', linewidth=2)

# Save and show the plot
plt.savefig('quantiles_pwv_hanle_merak_site_b_site_all_months.pdf', dpi=600)
plt.show()


## Interpolation and extraction

### Step 118

This cell defines reusable helper function(s) `calculate_pwv_vectorized`, `integrate_profile` so later sections can apply the same processing logic consistently.

In [ ]:
# In[1]:
# ────────────────────────────────────────────────────────────────────────────────
#  CELL 1: WINTER QUANTILE CALCULATION (THE SLOW PART)
#  This cell interpolates data, calculates PWV, filters for winter months (NDJF),
#  and computes the quantiles for each site.
# ────────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import xarray as xr
from scipy.interpolate import PchipInterpolator

# --- Assumed Pre-existing Objects ---
# This script assumes the following objects are already loaded in your environment:
#
# SITES:
#   - HANLE, MERAK, SITE_A, SITE_B: Dictionaries with site information.
#
# DATASETS:
#   - combined_dataset: xarray.Dataset with data from 2010 to April 2025.
#   - surface_pressure_data: xarray.Dataset with surface pressure 'sp'.
# -----------------------------------------------------------------------------


# 1. High-Performance PWV Calculation Engine
# This is the fast, vectorized function we developed.
def calculate_pwv_vectorized(q: xr.DataArray, p_sfc: xr.DataArray) -> xr.DataArray:
    """Calculates Precipitable Water Vapor (PWV) using a fast, vectorized approach."""
    g = 9.81
    p_levels_pa = q.level * 100
    p_levels_pa.attrs['units'] = 'Pa'
    
    def integrate_profile(q_profile, p_profile, sfc_p_value):
        finite_mask = np.isfinite(q_profile)
        if finite_mask.sum() < 2: return np.nan
        p, q_ = p_profile[finite_mask], q_profile[finite_mask]
        sort_idx = np.argsort(p)
        p, q_ = p[sort_idx], q_[sort_idx]
        valid_levels = p <= sfc_p_value
        p, q_ = p[valid_levels], q_[valid_levels]
        if len(p) < 2: return np.nan
        interpolator = PchipInterpolator(p, q_, extrapolate=True)
        q_sfc = interpolator(sfc_p_value)
        p_full = np.append(p, sfc_p_value)
        q_full = np.append(q_, q_sfc)
        sort_idx_full = np.argsort(p_full)
        p_full, q_full = p_full[sort_idx_full], q_full[sort_idx_full]
        return np.trapz(q_full, p_full) / g

    pwv = xr.apply_ufunc(
        integrate_profile, q, p_levels_pa, p_sfc,
        input_core_dims=[['level'], ['level'], []],
        output_core_dims=[[]], vectorize=True, dask="parallelized", output_dtypes=[q.dtype]
    )
    pwv.attrs.update(units="mm", long_name="Precipitable Water Vapor")
    return pwv


# 2. Main Data Processing Workflow
# -----------------------------------------------------------------------------
print("Starting heavy data processing for all sites...")
sites_to_process = [HANLE, MERAK, SITE_A, SITE_B]

# Create DataArrays of all site coordinates
site_lats = xr.DataArray([s['lat'] for s in sites_to_process], dims="site")
site_lons = xr.DataArray([s['lon'] for s in sites_to_process], dims="site")

print("Performing a single, efficient interpolation...")
# Interpolate ONCE and use .load() to bring the small result into memory
all_sites_data = combined_dataset.interp(
    latitude=site_lats, longitude=site_lons, method='linear'
).sel(time=slice('2010-01-01', '2025-04-30')).load()

all_sites_sp_interpolated = surface_pressure_data['sp'].interp(
    latitude=site_lats, longitude=site_lons, method='linear'
)
if 'valid_time' in all_sites_sp_interpolated.dims:
    all_sites_sp_interpolated = all_sites_sp_interpolated.rename({'valid_time': 'time'})
all_sites_sp = all_sites_sp_interpolated.load()

print("Interpolation complete. Calculating Winter PWV and Quantiles for each site...")

# --- Calculate and store the final plottable data ---
# Define the quantiles we want to compute
quantiles_to_calc = np.linspace(0, 1, 101)  # From 0% to 100% in 1% increments
pwv_quantile_results = {}

for i, site in enumerate(sites_to_process):
    site_name = site['name']
    print(f"  Calculating for site: {site_name}")
    
    # Select the data for the current site (fast, in-memory operation)
    q_data = all_sites_data['q'].isel(site=i)
    sp_data = all_sites_sp.isel(site=i)
    
    q_aligned, sp_aligned = xr.align(q_data, sp_data, join='inner')
    
    if 'expver' in q_aligned.dims:
        q_aligned = q_aligned.sel(expver=1, drop=True)
        
    q_rechunked = q_aligned.chunk({"level": -1})
    
    # Calculate the full PWV time-series
    pwv_timeseries = calculate_pwv_vectorized(q_rechunked, sp_aligned)
    
    # --- MODIFICATION: Filter for winter months (NDJF) before calculating quantiles ---
    winter_pwv = pwv_timeseries.sel(time=pwv_timeseries['time.month'].isin([11, 12, 1, 2]))
    
    # Calculate the quantiles for this site's WINTER PWV distribution
    quantile_values = winter_pwv.quantile(quantiles_to_calc, dim='time')
    
    # Store the results
    pwv_quantile_results[site_name] = quantile_values

print("\n✅ Winter data pre-processing complete. You can now re-run the plotting cell below.")


### Step 119

This cell defines reusable helper function(s) `calculate_pwv_vectorized`, `integrate_profile` so later sections can apply the same processing logic consistently.

In [ ]:
# In[1]:
# ────────────────────────────────────────────────────────────────────────────────
#  CELL 1: WINTER QUANTILE CALCULATION (THE SLOW PART)
#  This cell interpolates data, calculates PWV, filters for winter months (NDJF),
#  and computes the quantiles for each site. Adds IAO-Hanle with observed Psfc.
# ────────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import xarray as xr
from scipy.interpolate import PchipInterpolator

# --- Assumed Pre-existing Objects ---
# SITES: HANLE, MERAK, SITE_A, SITE_B, IAO_HANLE (dicts with 'name','lat','lon')
# DATASETS: combined_dataset (2010–2025/04), surface_pressure_data with 'sp'
# FILES: 'hanle_monthly_surface_data.csv' (Month, Pressure_Day_Mean, Pressure_Night_Mean)
# -----------------------------------------------------------------------------


# 1) High-Performance PWV Calculation Engine (unchanged)
def calculate_pwv_vectorized(q: xr.DataArray, p_sfc: xr.DataArray) -> xr.DataArray:
    """Calculates Precipitable Water Vapor (PWV) using a fast, vectorized approach."""
    g = 9.81
    p_levels_pa = q.level * 100  # assume 'level' in hPa → Pa
    p_levels_pa.attrs['units'] = 'Pa'
    
    def integrate_profile(q_profile, p_profile, sfc_p_value):
        finite_mask = np.isfinite(q_profile)
        if finite_mask.sum() < 2:
            return np.nan
        p, q_ = p_profile[finite_mask], q_profile[finite_mask]
        sort_idx = np.argsort(p)
        p, q_ = p[sort_idx], q_[sort_idx]
        valid_levels = p <= sfc_p_value
        p, q_ = p[valid_levels], q_[valid_levels]
        if len(p) < 2:
            return np.nan
        interpolator = PchipInterpolator(p, q_, extrapolate=True)
        q_sfc = interpolator(sfc_p_value)
        p_full = np.append(p, sfc_p_value)
        q_full = np.append(q_, q_sfc)
        sort_idx_full = np.argsort(p_full)
        p_full, q_full = p_full[sort_idx_full], q_full[sort_idx_full]
        return np.trapz(q_full, p_full) / g  # kg m^-2 ≈ mm

    pwv = xr.apply_ufunc(
        integrate_profile, q, p_levels_pa, p_sfc,
        input_core_dims=[['level'], ['level'], []],
        output_core_dims=[[]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[q.dtype],
    )
    pwv.attrs.update(units="mm", long_name="Precipitable Water Vapor")
    return pwv


# 2) Main Data Processing Workflow
# -----------------------------------------------------------------------------
print("Starting heavy data processing for all sites...")

# Include IAO_HANLE in the list of sites to process
sites_to_process = [HANLE, MERAK, SITE_A, SITE_B, IAO_HANLE]  # >>> NEW: IAO-Hanle added

# Create DataArrays of all site coordinates (for one-shot interpolation)
site_lats = xr.DataArray([s['lat'] for s in sites_to_process], dims="site")
site_lons = xr.DataArray([s['lon'] for s in sites_to_process], dims="site")

print("Performing a single, efficient interpolation...")
# Interpolate ONCE and bring the small result into memory
all_sites_data = (combined_dataset
    .interp(latitude=site_lats, longitude=site_lons, method='linear')
    .sel(time=slice('2010-01-01', '2025-04-30'))
    .load()
)

# Interpolate ERA5 surface pressure for all sites (we will override for IAO-Hanle)
all_sites_sp_interpolated = surface_pressure_data['sp'].interp(
    latitude=site_lats, longitude=site_lons, method='linear'
)
if 'valid_time' in all_sites_sp_interpolated.dims:
    all_sites_sp_interpolated = all_sites_sp_interpolated.rename({'valid_time': 'time'})
all_sites_sp = all_sites_sp_interpolated.sel(time=slice('2010-01-01', '2025-04-30')).load()

# >>> NEW: Load IAO-Hanle observed monthly surface pressure climatology (Pa)
# Expect columns: Month (Jan..Dec), Pressure_Day_Mean, Pressure_Night_Mean
hanle_obs_df = pd.read_csv('hanle_monthly_surface_data.csv')
hanle_obs_df['avg_pressure_pa'] = hanle_obs_df[['Pressure_Day_Mean', 'Pressure_Night_Mean']].mean(axis=1) * 100.0
month_map = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}
hanle_obs_df['month_num'] = hanle_obs_df['Month'].map(month_map)
# Lookup: month_num → avg Psfc (Pa)
monthly_pressure_lookup = hanle_obs_df.set_index('month_num')['avg_pressure_pa']

print("Interpolation complete. Calculating Winter PWV and Quantiles for each site...")

# Define quantiles (0%..100% at 1% steps) and winter months (NDJF)
quantiles_to_calc = np.linspace(0, 1, 101)
WINTER_MONTHS = [11, 12, 1, 2]

pwv_quantile_results = {}

for i, site in enumerate(sites_to_process):
    site_name = site.get('name', f"site_{i}")
    print(f"  Calculating for site: {site_name}")

    # Select the site's q (in-memory slice)
    q_data = all_sites_data['q'].isel(site=i)

    # Build surface pressure per site
    if site_name.lower().replace('-', '').replace('_','') in {'iaohanle','iaohanle'} or site_name == 'IAO-Hanle':
        # >>> NEW: IAO-Hanle special Psfc (observed monthly climatology applied to each timestamp)
        time_index = q_data.indexes['time'] if 'time' in q_data.indexes else pd.DatetimeIndex(q_data['time'].values)
        # Map each timestamp's month to the observed Psfc (Pa)
        ps_vals = pd.Index(time_index.month).map(monthly_pressure_lookup).to_numpy()
        sp_data = xr.DataArray(ps_vals, coords={'time': q_data.time}, dims=['time'], name='sp_obs')
    else:
        # Default: use ERA5 surface pressure
        sp_data = all_sites_sp.isel(site=i)

    # Align q and Psfc on the overlapping time range
    q_aligned, sp_aligned = xr.align(q_data, sp_data, join='inner')

    # ERA5 double-streams appear in some datasets; drop if present
    if 'expver' in q_aligned.dims:
        # choose last non-NaN along expver (or use .sel(expver=1, drop=True) if you prefer)
        q_aligned = q_aligned.ffill('expver').isel(expver=-1)

    # Ensure vertical core dim is single-chunk to keep the ufunc happy
    q_rechunked = q_aligned.chunk({"level": -1})

    # Full PWV time series, all months
    pwv_timeseries = calculate_pwv_vectorized(q_rechunked, sp_aligned)

    # Winter-only (NDJF) subset for quantiles
    winter_pwv = pwv_timeseries.sel(time=pwv_timeseries['time.month'].isin(WINTER_MONTHS))

    # Quantiles of winter PWV for this site
    quantile_values = winter_pwv.quantile(quantiles_to_calc, dim='time')

    # Store results keyed by site name
    pwv_quantile_results[site_name] = quantile_values

print("\n✅ Winter data pre-processing complete (including IAO-Hanle). You can now re-run the plotting cell below.")


## Mapping and visualization

### Step 120

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
# In[2]:
# ────────────────────────────────────────────────────────────────────────────────
#  CELL 2: PLOTTING WINTER QUANTILE DISTRIBUTION (PROFESSIONAL FORMAT)
#  This cell uses the pre-calculated winter quantiles and our established
#  publication-quality formatting.
# ────────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

# 1. Site and Plotting Configuration
# -----------------------------------------------------------------------------
# # Site coordinates for Ladakh region.
# HANLE = {'name': 'Hanle', 'lat': 32.7789, 'lon': 78.9650, 'elevation': 4500}
# MERAK = {'name': 'Merak', 'lat': 33.7828, 'lon': 78.57782, 'elevation': 4500}
# SITE_A = {'name': 'Site A', 'lat': 34.25, 'lon': 78.75, 'elevation': 4800}
# # SITE_B = {'name': 'Site B', 'lat': 33.00, 'lon': 78.00, 'elevation': 4800}
# SITE_B = {'name': 'Site B', 'lat': 32.5 , 'lon': 79.00, 'elevation': 4800}

SITES_TO_PLOT = [HANLE, MERAK, SITE_A, SITE_B]

# A more publishable, high-contrast and colorblind-friendly palette
PUBLISHABLE_COLORS = ['#00429d', '#93003a', '#009b7a', '#ff9e00']
PUBLISHABLE_MARKERS = ['o', 's', '^', 'D']

# --- Plotting Configuration ---
# Using the established "publishable" color and marker schemes
PUBLISHABLE_COLORS = ['#00429d', '#93003a', '#009b7a', '#ff9e00'] # Blue, Magenta, Green, Orange
PUBLISHABLE_MARKERS = ['o', 's', '^', 'D'] # Circle, Square, Triangle, Diamond

# Using the established global settings for a consistent, professional, LaTeX-formatted look
plt.rcParams.update({
    'font.size': 20,
    'legend.fontsize': 18,
    'font.family': 'serif',
    'text.usetex': True,
    'text.latex.preamble': r"\usepackage{amsmath}",
})

fig, ax1 = plt.subplots(figsize=(12, 10))

print("Generating publication-quality quantile plot from pre-processed data...")

# --- Data for plotting ---
# The x-axis is the quantiles we calculated
quantiles_x_axis = quantiles_to_calc 


# --- Main Plotting Loop ---
for i, site in enumerate(SITES_TO_PLOT):
    site_name = site['name']
    # Retrieve the pre-calculated winter quantile data
    quantile_values = pwv_quantile_results[site_name]
    
    # Plot each site with a unique marker and color
    ax1.plot(quantiles_x_axis, quantile_values,
             label=site_name,
             color=PUBLISHABLE_COLORS[i],
             marker=PUBLISHABLE_MARKERS[i],
             linestyle='-',
             linewidth=3,
             markersize=10,
             markeredgewidth=1.5,
             markeredgecolor='black')

# --- Cosmetics and Finalization (Matching Previous Style) ---
# Add reference lines for key PWV values
ax1.axhline(y=1.0, color='black', linestyle='--', linewidth=2.5, zorder=0)
# ax1.axhline(y=0.5, color='gray', linestyle=':', linewidth=2, zorder=0)
ax1.text(0.85, 0.95, '1.0 mm', transform=ax1.get_yaxis_transform(), fontsize=22, va='center')
# ax1.text(1.01, 0.5, '0.5 mm', transform=ax1.get_yaxis_transform(), fontsize=16, va='center')

# # Use supxlabel/supylabel for well-centered, professional labels
# fig.supxlabel('Quantile', fontsize=32, y=0.03)
# fig.supylabel('Winter PWV (mm) [NDJF]', fontsize=32, x=0.02)
fig.text(0.55, 0.08, r'Quantile', fontsize=32, ha='center', va='center')
fig.text(0.07, 0.5, r'PWV (mm)', fontsize=32, ha='center', va='center', rotation='vertical')

# --- Corrected code for a center-aligned text box ---

# The string and box properties remain the same
textstr = 'Winter Months: NDJF\n2010–2025 April'
props = dict(boxstyle='round,pad=0.5', facecolor='wheat', alpha=0.8, edgecolor='black', linewidth=1.5)

# Place the text box in the bottom-right corner of the plot area
ax1.text(0.97, 0.03, textstr, 
         transform=ax1.transAxes,
         fontsize=26, 
         verticalalignment='bottom', 
         horizontalalignment='right',
         bbox=props,
         multialignment='center'  # <-- This new argument centers the text lines
        )
ax1.legend(loc='upper left', fontsize=26, frameon=True, facecolor='white', edgecolor='black', framealpha=0.9)

# Configure ticks and limits to professional standards
ax1.set_xlim(0, 1.0)
ax1.set_ylim(0.6, 1.8)
ax1.minorticks_on()

# Apply detailed tick formatting to all four sides
ax1.tick_params(axis='both', which='major', direction='in', width=2, length=10, labelsize=28, top=True, right=True)
ax1.tick_params(axis='both', which='minor', direction='in', width=1, length=5, top=True, right=True)

# Set facecolor and spine properties
ax1.set_facecolor('white')
for spine in ax1.spines.values():
    spine.set_linewidth(3)

# Adjust layout and save the final figure
fig.tight_layout(rect=[0.07, 0.07, 0.98, 0.98])
output_filename = 'quantiles_pwv_winter_months_formatted_2010-2025.pdf'
plt.savefig(output_filename, dpi=600, bbox_inches='tight')
print(f"\n→ Plot successfully saved as {output_filename}")
plt.show()


### Step 121

This cell subsets the dataset to a selected time, level, region, or site so the next step works with a focused slice of the data.

In [ ]:
# In[2]:
# ────────────────────────────────────────────────────────────────────────────────
#  CELL 2: PLOTTING WINTER QUANTILE DISTRIBUTION (LINES + SPARSE MARKERS IN LEGEND)
#  Same aesthetics; improved colors; gentle smoothing; sparse markers that also
#  appear in the legend (via markevery on the line).
# ────────────────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors

# 1. Site and Plotting Configuration
# -----------------------------------------------------------------------------
# HANLE     = {'name': 'Hanle',      'lat': 32.7789, 'lon': 78.9650, 'elevation': 4500}
# MERAK     = {'name': 'Merak',      'lat': 33.7828, 'lon': 78.57782, 'elevation': 4500}
# SITE_A    = {'name': 'Site A',     'lat': 34.25,   'lon': 78.75,   'elevation': 4800}
# SITE_B    = {'name': 'Site B',     'lat': 32.50,   'lon': 79.00,   'elevation': 4800}
IAO_HANLE = {'name': 'IAO-Hanle',  'lat': 32.7789, 'lon': 78.9650, 'elevation': 4500}  # added

SITES_TO_PLOT = [HANLE, MERAK, SITE_A, SITE_B, IAO_HANLE]

# Colorblind-safe palette (Okabe–Ito) + markers (5 entries)
PUBLISHABLE_COLORS  = ['#0072B2', '#D55E00', '#009E73', '#E69F00', '#CC79A7']  # blue, vermillion, green, orange, purple
PUBLISHABLE_MARKERS = ['o', 's', '^', 'D', 'X']  # circle, square, triangle, diamond, X

# Keep your global settings
plt.rcParams.update({
    'font.size': 20,
    'legend.fontsize': 18,
    'font.family': 'serif',
    'text.usetex': True,
    'text.latex.preamble': r"\usepackage{amsmath}",
})

fig, ax1 = plt.subplots(figsize=(12, 10))

print("Generating publication-quality quantile plot (lines + sparse markers with legend markers)...")

# --- Data for plotting ---
quantiles_x_axis = quantiles_to_calc

# Choose ~9 marker positions across the quantile domain
marker_idx = np.unique(np.clip(
    np.linspace(0, len(quantiles_x_axis) - 1, 9, dtype=int),
    0, len(quantiles_x_axis) - 1
))

# --- Main Plotting Loop ---
for i, site in enumerate(SITES_TO_PLOT):
    site_name = site['name']

    if site_name not in pwv_quantile_results:
        print(f"  [warn] '{site_name}' not found in pwv_quantile_results; skipping.")
        continue

    qa = pwv_quantile_results[site_name]
    # Handle either a DA with 'quantile' coord or a plain array
    if hasattr(qa, 'dims') and ('quantile' in qa.dims):
        y_vals = qa.sel(quantile=quantiles_x_axis).values
    else:
        y_vals = np.asarray(qa)

    color  = PUBLISHABLE_COLORS[i % len(PUBLISHABLE_COLORS)]
    marker = PUBLISHABLE_MARKERS[i % len(PUBLISHABLE_MARKERS)]
    mface  = mcolors.to_rgba(color, alpha=0.9)  # subtle marker transparency

    # Single call to plot with markers at sparse positions (so legend shows markers)
    ax1.plot(
        quantiles_x_axis, y_vals,
        label=site_name,
        color=color,
        linestyle='-',
        linewidth=3,
        solid_joinstyle='round',   # gentle smoothing
        solid_capstyle='round',
        marker=marker,
        markevery=marker_idx,      # ← sparse markers along the line
        markersize=10,
        markeredgewidth=1.5,
        markeredgecolor='black',
        markerfacecolor=mface
    )

# --- Cosmetics and Finalization (kept as in your code) ---
ax1.axhline(y=1.0, color='black', linestyle='--', linewidth=2.5, zorder=0)
ax1.text(0.85, 0.95, '1.0 mm', transform=ax1.get_yaxis_transform(), fontsize=22, va='center')

fig.text(0.55, 0.08, r'Quantile', fontsize=32, ha='center', va='center')
fig.text(0.07, 0.5, r'PWV (mm)', fontsize=32, ha='center', va='center', rotation='vertical')

textstr = 'Winter Months: NDJF\n2010–2025 April'
props = dict(boxstyle='round,pad=0.5', facecolor='wheat', alpha=0.8, edgecolor='black', linewidth=1.5)
ax1.text(0.97, 0.03, textstr,
         transform=ax1.transAxes,
         fontsize=26,
         verticalalignment='bottom',
         horizontalalignment='right',
         bbox=props,
         multialignment='center')

ax1.legend(loc='upper left', fontsize=26, frameon=True, facecolor='white', edgecolor='black', framealpha=0.9)

# Ticks, limits, spines (unchanged)
ax1.set_xlim(0, 1.0)
ax1.set_ylim(0.6, 1.8)
ax1.minorticks_on()
ax1.tick_params(axis='both', which='major', direction='in', width=2, length=10, labelsize=28, top=True, right=True)
ax1.tick_params(axis='both', which='minor', direction='in', width=1, length=5, top=True, right=True)
ax1.set_facecolor('white')
for spine in ax1.spines.values():
    spine.set_linewidth(3)

fig.tight_layout(rect=[0.07, 0.07, 0.98, 0.98])
output_filename = 'quantiles_pwv_winter_months_formatted_2010-2025.pdf'
plt.savefig(output_filename, dpi=600, bbox_inches='tight')
print(f"\n→ Plot successfully saved as {output_filename}")
plt.show()


## Pressure and PWV calculations

### Step 122

This cell subsets the dataset to a selected time, level, region, or site so the next step works with a focused slice of the data.

In [ ]:
# In[1]:
# ────────────────────────────────────────────────────────────────────────────────
#  CELL 1: ADVANCED STATISTICAL ANALYSIS (THE "ENGINE")
#  This cell calculates full distributions and key statistics for each site.
# ────────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Assumed Pre-existing Objects ---
# This script assumes the following objects are already loaded in your environment:
# - all_sites_data, all_sites_sp: The pre-interpolated, in-memory datasets
# - SITES_TO_PLOT: The list of site dictionaries
# - calculate_pwv_vectorized: The fast PWV calculation function
# -----------------------------------------------------------------------------

print("Starting advanced statistical analysis for all sites...")

# This dictionary will hold all the results needed for plotting
site_statistics = {}

# --- ANALYSIS LOOP: Loop through the pre-processed data ---
for i, site in enumerate(SITES_TO_PLOT):
    site_name = site['name']
    print(f"  Analyzing winter months for site: {site_name}")
    
    # 1. Select and align data for the current site
    q_data = all_sites_data['q'].isel(site=i)
    sp_data = all_sites_sp.isel(site=i)
    q_aligned, sp_aligned = xr.align(q_data, sp_data, join='inner')
    
    if 'expver' in q_aligned.dims:
        q_aligned = q_aligned.sel(expver=1, drop=True)
        
    q_rechunked = q_aligned.chunk({"level": -1})
    
    # 2. Calculate the full PWV time-series
    pwv_timeseries = calculate_pwv_vectorized(q_rechunked, sp_aligned)
    
    # 3. Filter for winter months and drop any invalid (NaN) values
    winter_pwv = pwv_timeseries.sel(time=pwv_timeseries['time.month'].isin([11, 12, 1, 2]))
    winter_pwv_clean = winter_pwv.dropna(dim='time')
    
    # 4. Calculate the statistics needed for the plot
    
    # a) Full data for the CDF plot (x = sorted PWV values, y = cumulative probability)
    cdf_x = np.sort(winter_pwv_clean.values)
    cdf_y = np.linspace(0, 1, len(cdf_x))
    
    # --- Corrected code with .compute() ---
    # b) Key quantile values for the summary table
    print("    Computing statistics...")
    median = winter_pwv_clean.quantile(0.50).compute().item()
    q1 = winter_pwv_clean.quantile(0.25).compute().item()
    q3 = winter_pwv_clean.quantile(0.75).compute().item()
    iqr = q3 - q1 # Interquartile Range

    # c) Percentage of time below critical thresholds
    pct_below_1mm = (np.sum(winter_pwv_clean <= 1.0) / len(winter_pwv_clean) * 100).compute().item()
    pct_below_0_5mm = (np.sum(winter_pwv_clean <= 0.5) / len(winter_pwv_clean) * 100).compute().item()

    # 5. Store all results in the dictionary
    site_statistics[site_name] = {
        'cdf_x': cdf_x,
        'cdf_y': cdf_y,
        'summary_stats': {
            'Median': f"{median:.2f} mm",
            'IQR': f"{iqr:.2f} mm",
            '% < 1.0mm': f"{pct_below_1mm:.1f}%",
            '% < 0.5mm': f"{pct_below_0_5mm:.1f}%"
        }
    }

print("\n✅ Advanced statistical analysis complete. You can now re-run the plotting cell below.")


## Mapping and visualization

### Step 123

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
# In[2]:
# ────────────────────────────────────────────────────────────────────────────────
#  CELL 2: PROFESSIONAL PLOTTING (THE "ART")
#  This cell creates a CDF plot with an embedded summary table.
# ────────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# --- Plotting Configuration ---
plt.rc('font', family='serif', size=18)
plt.rc('axes', labelsize=22)
plt.rcParams['text.usetex'] = True
plt.rcParams['text.latex.preamble'] = r"\usepackage{amsmath}"

# Use a gridspec for a sophisticated layout (main plot + side table)
fig = plt.figure(figsize=(16, 9))
gs = gridspec.GridSpec(1, 2, width_ratios=[2.5, 1], wspace=0.3)
ax_main = fig.add_subplot(gs[0, 0])
ax_table = fig.add_subplot(gs[0, 1])

print("Generating professional plot with CDF and summary table...")

# --- 1. Main CDF Plot ---
# Prepare colors and styles
colors = PUBLISHABLE_COLORS
linestyles = ['-', '--', '-.', ':']

for i, site in enumerate(SITES_TO_PLOT):
    site_name = site['name']
    stats = site_statistics[site_name]
    
    ax_main.plot(stats['cdf_x'], stats['cdf_y'],
                 label=site_name,
                 color=colors[i],
                 linestyle=linestyles[i],
                 linewidth=3.5)

# --- Cosmetics for the main plot ---
ax_main.set_xlabel('Precipitable Water Vapor (mm)')
ax_main.set_ylabel('Fraction of Winter Months Below PWV')
ax_main.set_xlim(0, 2.0)
ax_main.set_ylim(0, 1.0)
ax_main.grid(True, linestyle='--', alpha=0.5)

# Highlight critical PWV thresholds
ax_main.axvline(x=1.0, color='black', linestyle=':', linewidth=2, alpha=0.7)
ax_main.axvline(x=0.5, color='black', linestyle=':', linewidth=2, alpha=0.7)
ax_main.text(1.02, 0.5, '1.0 mm', rotation=90, fontsize=16, va='center')
ax_main.text(0.52, 0.5, '0.5 mm', rotation=90, fontsize=16, va='center')

ax_main.legend(loc='lower right', title=r'\textbf{Observatory Sites}', fontsize=18)
for spine in ax_main.spines.values():
    spine.set_linewidth(2)

# --- 2. Summary Table ---
ax_table.axis('off') # Hide the axes for the table subplot

# Prepare data for the table
table_data = [list(site_statistics[s['name']]['summary_stats'].values()) for s in SITES_TO_PLOT]
row_labels = [s['name'] for s in SITES_TO_PLOT]
col_labels = ['Median', 'IQR', r'\% Time $\leq$ 1.0mm', r'\% Time $\leq$ 0.5mm']

# Create the table
table = ax_table.table(
    cellText=table_data,
    rowLabels=row_labels,
    colLabels=col_labels,
    rowColours=colors,
    colColours=['lightgray']*4,
    cellLoc='center',
    loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(18)
table.scale(1.0, 2.5) # Adjust column width and row height

# Style the table header
for (i, j), cell in table.get_celld().items():
    if i == 0: # Header row
        cell.get_text().set_weight('bold')

ax_table.set_title(r'\textbf{Winter PWV Statistics}', fontsize=24, y=0.85)

# --- Figure Title and Finalization ---
fig.suptitle('Site Comparison: Cumulative Distribution of Winter PWV (2010-2025)', fontsize=28, weight='bold')
fig.tight_layout(rect=[0, 0, 1, 0.95])

output_filename = 'PWV_CDF_Comparison_2010-2025.pdf'
plt.savefig(output_filename, dpi=600)
print(f"\n→ Plot successfully saved as {output_filename}")
plt.show()


## Fig. 3

## Analysis workflow

### Step 124

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
SITE_A


## Pressure and PWV calculations

### Step 125

This cell reads tabular metadata that is later used for site lookup, filtering, or summary reporting.

In [ ]:
lat_hanle = 32.7789
lon_hanle = 78.965
# lat_hanle = SITE_A['lat']
# lon_hanle = SITE_A['lon']

#ds_point = era5.sel(latitude=lat_hanle, longitude=lon_hanle, method='nearest')
ds_point = era5.interp(latitude=lat_hanle, longitude=lon_hanle, method='linear')
ds_point = ds_point.sel(time=slice("1998-01-01", "2017-12-31"))

pressure_threshold = surface_pressure_data['sp'].interp(latitude=lat_hanle, longitude=lon_hanle, method='linear')


# --- Curve 2: PWV calculated with Observed surface pressure ---
# 1. Load the observational data from the CSV file
hanle_obs_df = pd.read_csv('hanle_monthly_surface_data.csv')

# 2. Calculate the average monthly pressure and convert from hPa to Pascals (Pa)
hanle_obs_df['avg_pressure_pa'] = hanle_obs_df[['Pressure_Day_Mean', 'Pressure_Night_Mean']].mean(axis=1) * 100

# 3. Create a mapping from month name (e.g., 'Jan') to the pressure value
month_map = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
    'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}
hanle_obs_df['month_num'] = hanle_obs_df['Month'].map(month_map)
# Create a lookup series (index=month_num, value=pressure_in_Pascals)
monthly_pressure_lookup = hanle_obs_df.set_index('month_num')['avg_pressure_pa']

# 4. Create the new pressure threshold time series based on the observations
# Get the month number for each timestamp in your data
time_coords = ds_point.time
months = time_coords.dt.month

# Map each month in your time series to the corresponding observed pressure value
pressure_values_pa = months.to_series().map(monthly_pressure_lookup)

# Create a new xarray DataArray for the observed surface pressure
pressure_threshold_obs = xr.DataArray(
    pressure_values_pa.values,
    coords={'time': time_coords},
    dims=['time'],
    name='sp_obs'
)

pwv_obs = calculate_pwv_pressure(ds_point.q, ds_point.level, pressure_threshold_obs)

#pwv_corr = calculate_pwv(ds_point.q, ds_point.level, pressure_threshold=57672.85946698223)
print(ds_point.q)
print(ds_point.level)
pressure_threshold = xr.full_like(pressure_threshold,55500)
print(pressure_threshold)
pwv_corr = calculate_pwv_pressure(ds_point.q, ds_point.level, pressure_threshold)

# Plotting results
plt.figure(figsize=(12, 6))
pwv_corr.plot(color='blue', label='PWV ERA5 sp', linestyle='--', alpha=0.9, marker='o')
pwv_obs.plot(color='red', label='PWV ERA5 obs sp', linestyle='-', alpha=0.9, marker='x')
plt.xlabel('Time')
plt.ylabel('Precipitable Water Vapor (mm)')
plt.grid(True)
plt.legend()
plt.hlines(y=1.0, xmin=ds_point.time.min(), xmax=ds_point.time.max(), color='black', linestyle='--', linewidth=2)
plt.show()

# print the number of months with PWV < 1 mm for both datasets
print(f"Number of months with PWV < 1 mm (ERA5 sp): {(pwv_corr < 1).sum().item()}")
print(f"Number of months with PWV < 1 mm (ERA5 obs sp): {(pwv_obs < 1).sum().item()}")


#### Direct sp value

## Analysis workflow

### Step 126

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
combined_dataset


### Step 127

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
surface_pressure_data


## Pressure and PWV calculations

### Step 128

This cell defines reusable helper function(s) `calculate_pwv_vectorized`, `integrate_profile` so later sections can apply the same processing logic consistently.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.style as style
import numpy as np
import pandas as pd
import xarray as xr
from scipy.interpolate import PchipInterpolator

# --- Assumed Pre-existing Objects ---
# This script assumes the following objects are already loaded in your environment:
#
# SITES:
#   - HANLE: A dictionary with site information.
#
# DATASETS:
#   - era5: xarray.Dataset with data covering the 1998-2017 period.
#   - surface_pressure_data: xarray.Dataset with surface pressure 'sp'.
# -----------------------------------------------------------------------------


# 1. High-Performance PWV Calculation Engine
# This is the fast, vectorized function we developed.
def calculate_pwv_vectorized(q: xr.DataArray, p_sfc: xr.DataArray) -> xr.DataArray:
    """Calculates Precipitable Water Vapor (PWV) using a fast, vectorized approach."""
    g = 9.81
    p_levels_pa = q.level * 100
    p_levels_pa.attrs['units'] = 'Pa'
    
    def integrate_profile(q_profile, p_profile, sfc_p_value):
        finite_mask = np.isfinite(q_profile)
        if finite_mask.sum() < 2: return np.nan
        p, q_ = p_profile[finite_mask], q_profile[finite_mask]
        sort_idx = np.argsort(p)
        p, q_ = p[sort_idx], q_[sort_idx]
        valid_levels = p <= sfc_p_value
        p, q_ = p[valid_levels], q_[valid_levels]
        if len(p) < 2: return np.nan
        interpolator = PchipInterpolator(p, q_, extrapolate=True)
        q_sfc = interpolator(sfc_p_value)
        p_full = np.append(p, sfc_p_value)
        q_full = np.append(q_, q_sfc)
        sort_idx_full = np.argsort(p_full)
        p_full, q_full = p_full[sort_idx_full], q_full[sort_idx_full]
        return np.trapz(q_full, p_full) / g

    pwv = xr.apply_ufunc(
        integrate_profile, q, p_levels_pa, p_sfc,
        input_core_dims=[['level'], ['level'], []],
        output_core_dims=[[]], vectorize=True, dask="parallelized", output_dtypes=[q.dtype]
    )
    pwv.attrs.update(units="mm", long_name="Precipitable Water Vapor")
    return pwv


# 2. Main Analysis Workflow for a Single Site
# -----------------------------------------------------------------------------
print("Starting analysis for Hanle for the 1998-2017 period...")

# Define site coordinates
lat_hanle = HANLE['lat']
lon_hanle = HANLE['lon']

# --- 1. Interpolate and Slice Data to Site Location and Time ---
# MODIFICATION: Using 'era5' dataset and slicing to the 1998-2017 range
print("Interpolating and slicing data for 1998-2017...")
q_site = era5['q'].interp(latitude=lat_hanle, longitude=lon_hanle, method='linear').sel(time=slice('1998-01-01', '2017-12-31'))

sp_site = surface_pressure_data['sp'].interp(latitude=lat_hanle, longitude=lon_hanle, method='linear')

# print the mean and median sp of sp_site
print(f"Mean surface pressure at Hanle: {sp_site.mean().item()} Pa")
print(f"Median surface pressure at Hanle: {sp_site.median().item()} Pa")

# Replace with actual lat/lon arrays from your dataset
lats = surface_pressure_data['latitude'].values
lons = surface_pressure_data['longitude'].values

# Find surrounding latitude indices
lat_idx = np.searchsorted(lats[::-1], lat_hanle, side='left')  # Since lats are descending
lat_idx = len(lats) - lat_idx  # Adjust for reversed search
lat_low = lats[lat_idx]
lat_high = lats[lat_idx - 1]

# Find surrounding longitude indices (assume lons are increasing)
lon_idx = np.searchsorted(lons, lon_hanle, side='left')
lon_low = lons[lon_idx - 1]
lon_high = lons[lon_idx]

print(f"Latitude grid points used: {lat_high}, {lat_low}")
print(f"Longitude grid points used: {lon_low}, {lon_high}")



if 'valid_time' in sp_site.dims:
    sp_site = sp_site.rename({'valid_time': 'time'})
sp_site = sp_site.sel(time=slice('1998-01-01', '2017-12-31'))


# --- 2. Align Data (Ensures any missing months are handled correctly) ---
print("Aligning time coordinates of humidity and surface pressure data...")
q_aligned, sp_aligned = xr.align(q_site, sp_site, join='inner')

# --- 3. Prepare Data for Calculation ---
# Handle the experiment version dimension
if 'expver' in q_aligned.dims:
    q_aligned = q_aligned.sel(expver=1, drop=True)
    
# Rechunk the level dimension into a single block for the vertical integral
q_rechunked = q_aligned.chunk({"level": -1})

# --- 4. Calculate PWV (Fast, Vectorized Operation) ---
print("Calculating PWV...")
pwv_corr = calculate_pwv_vectorized(q_rechunked, sp_aligned).load() # Use .load() to get final result
print("Calculation complete.")

# --- 5. Plotting Results ---
style.use('seaborn-v0_8-talk') # Use a professional plot style
plt.figure(figsize=(18, 8))

# MODIFICATION: Updated plot label for new date range
pwv_corr.plot(color='teal', label='PWV at Hanle (1998-2017)')
plt.axhline(y=1.0, color='red', linestyle='--', linewidth=2, label='1.0 mm Threshold')

# MODIFICATION: Updated plot title
plt.title('Precipitable Water Vapor (PWV) at Hanle (1998-2017)', fontsize=22, weight='bold')
plt.xlabel('Year', fontsize=18)
plt.ylabel('PWV (mm)', fontsize=18)
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.legend(fontsize=16)
plt.tight_layout()
plt.show()


## Analysis workflow

### Step 129

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
print(surface_pressure_data['longitude'].values)


### Step 130

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
print(surface_pressure_data['latitude'].values)


#### With elevation

## Pressure and PWV calculations

### Step 131

This cell defines reusable helper function(s) `calculate_pwv_vectorized`, `integrate_profile`, `main_analysis_for_hanle` so later sections can apply the same processing logic consistently.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.style as style
import numpy as np
import pandas as pd
import xarray as xr
from scipy.interpolate import PchipInterpolator

# --- Assumed Pre-existing Objects ---
# This script assumes the following objects are already loaded in your environment:
#
# SITES:
#   - HANLE: A dictionary with site info, including 'elevation'.
#
# DATASETS:
#   - combined_dataset: xarray.Dataset with specific humidity 'q'.
#
# HELPER FUNCTION:
#   - calculate_pressure(P_b, T, L_b, h, h_b): The barometric formula function.
#
# CONSTANTS:
#   - P_b, L_b, h_b: Constants for the barometric pressure formula.
# -----------------------------------------------------------------------------


# 1. High-Performance PWV Calculation Engine (Unchanged)
def calculate_pwv_vectorized(q: xr.DataArray, p_sfc: xr.DataArray) -> xr.DataArray:
    """Calculates Precipitable Water Vapor (PWV) using a fast, vectorized approach."""
    g = 9.81
    p_levels_pa = q.level * 100
    
    def integrate_profile(q_profile, p_profile, sfc_p_value):
        finite_mask = np.isfinite(q_profile)
        if finite_mask.sum() < 2: return np.nan
        p, q_ = p_profile[finite_mask], q_profile[finite_mask]
        sort_idx = np.argsort(p)
        p, q_ = p[sort_idx], q_[sort_idx]
        valid_levels = p <= sfc_p_value
        p, q_ = p[valid_levels], q_[valid_levels]
        if len(p) < 2: return np.nan
        interpolator = PchipInterpolator(p, q_, extrapolate=True)
        q_sfc = interpolator(sfc_p_value)
        p_full = np.append(p, sfc_p_value)
        q_full = np.append(q_, q_sfc)
        sort_idx_full = np.argsort(p_full)
        p_full, q_full = p_full[sort_idx_full], q_full[sort_idx_full]
        return np.trapz(q_full, p_full) / g

    pwv = xr.apply_ufunc(
        integrate_profile, q, p_levels_pa, p_sfc,
        input_core_dims=[['level'], ['level'], []],
        output_core_dims=[[]], vectorize=True, dask="parallelized", output_dtypes=[q.dtype]
    )
    pwv.attrs.update(units="mm", long_name="Precipitable Water Vapor")
    return pwv


# 2. Main Analysis and Plotting Workflow
# -----------------------------------------------------------------------------
def main_analysis_for_hanle():
    """
    Encapsulates the entire workflow to prevent "multiple plots" issue
    and uses the elevation-based pressure calculation with a constant temperature.
    """
    print("Starting analysis for Hanle using Elevation-Based Surface Pressure with T=279K...")

    # --- 1. Define Site Constants ---
    lat_hanle = HANLE['lat']
    lon_hanle = HANLE['lon']
    elev_hanle = HANLE['elevation']
    T_constant = 288.15  # Use the specified constant temperature in Kelvin

    # --- 2. Prepare Data ---
    # Interpolate specific humidity data to the site location. This defines our time axis.
    q_site = combined_dataset['q'].interp(latitude=lat_hanle, longitude=lon_hanle, method='linear')

    # Create a temperature DataArray with the constant value, aligned to the humidity data's time axis
    T_site = xr.DataArray(
        T_constant,
        coords={'time': q_site.time},
        dims=['time']
    )
    
    # Calculate surface pressure dynamically using elevation and the constant temperature
    print("Calculating dynamic surface pressure...")
    # Assume T_b is a constant for the formula, loaded in the environment
    sp_vals = calculate_pressure(P_b, T_site, L_b, elev_hanle, h_b)
    # Ensure the result is a clean xarray DataArray
    sp_aligned = xr.DataArray(sp_vals, coords=T_site.coords, dims=T_site.dims)

    # Prepare q data for calculation
    q_aligned = q_site
    if 'expver' in q_aligned.dims:
        q_aligned = q_aligned.sel(expver=1, drop=True)
    q_rechunked = q_aligned.chunk({"level": -1})

    # --- 3. Calculate PWV (Fast, Vectorized Operation) ---
    print("Calculating PWV...")
    pwv_corr = calculate_pwv_vectorized(q_rechunked, sp_aligned).load()
    print("Calculation complete.")

    # --- 4. Plotting Results ---
    print("Generating plot...")
    style.use('seaborn-v0_8-talk')
    fig, ax = plt.subplots(figsize=(18, 8)) # Create figure and axes inside the function

    pwv_corr.plot(ax=ax, color='darkblue', label='PWV at Hanle (Elevation Corrected, T=279K)')
    ax.axhline(y=1.0, color='red', linestyle='--', linewidth=2, label='1.0 mm Threshold')

    ax.set_title('Precipitable Water Vapor (PWV) at Hanle (2010-2025)', fontsize=22, weight='bold')
    ax.set_xlabel('Year', fontsize=18)
    ax.set_ylabel('PWV (mm)', fontsize=18)
    ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    ax.legend(fontsize=16)
    
    fig.tight_layout()
    plt.show()


# 3. Driver
# -----------------------------------------------------------------------------
if __name__ == '__main__':
    # Run the main analysis function
    main_analysis_for_hanle()


## Mapping and visualization

### Step 132

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
# plot the pressure threshold values according to the time
plt.figure(figsize=(12, 6))
pressure_threshold.plot(color='red', label='Pressure Threshold', linestyle='--', alpha=0.9, marker='o')
plt.xlabel('Time')
plt.ylabel('Pressure Threshold (Pa)')
plt.grid(True)

plt.legend()
plt.show()


### Step 133

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
# use pwv corr to find the median with respect to the months, i.e. January, February, March, etc.
pwv_monthly = pwv_corr.groupby('time.month').median(dim='time')
plt.plot(pwv_monthly.month, pwv_monthly, label='ERA5', marker='s', linestyle='-', color='green',
         markerfacecolor='white', markeredgewidth=2, markersize=8)


Month	Day	Day_Error	Night	Night_Error
Jan	0.8	0.2	0.9	0.2
Feb	1.0	0.2	1.1	0.2
Mar	1.2	0.1	1.4	0.3
Apr	1.7	0.1	2.0	0.2
May	2.4	0.5	2.7	0.5
Jun	3.8	0.6	4.2	0.6
Jul	6.0	1.3	6.4	1.1
Aug	6.0	1.0	6.5	1.0
Sep	3.7	0.6	4.0	0.6
Oct	1.5	0.3	1.7	0.4
Nov	0.9	0.2	1.0	0.2
Dec	0.8	0.2	0.9	0.3

## Site definitions and metadata

### Step 134

This cell reads tabular metadata that is later used for site lookup, filtering, or summary reporting.

In [ ]:
hanle_monthly_iao = pd.read_csv('hanle_monthly_pwv.txt',sep = '\t',header=None,skiprows=1)


### Step 135

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
cols = ['Month','Day','Day_Error','Night','Night_Error']
hanle_monthly_iao.columns = cols


hanle_monthly_iao


## Mapping and visualization

### Step 136

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import numpy as np

# --- Assumed Pre-existing Objects ---
# This script assumes the following objects are already loaded in your environment:
#
# - hanle_monthly_iao: A pandas DataFrame with observed Day/Night PWV data.
# - pwv_monthly: An xarray DataArray with the monthly mean model PWV data.
# -----------------------------------------------------------------------------


# 1. Plotting Configuration
# -----------------------------------------------------------------------------
# Use the established "publishable" color and marker schemes
PUBLISHABLE_COLORS = ['#00429d', '#93003a', '#009b7a'] # Blue for Day, Magenta for Night, Green for Model
PUBLISHABLE_MARKERS = ['o', 's', '^'] # Circle for Day, Square for Night, Triangle for Model

# Use the established global settings for a consistent, professional, LaTeX-formatted look
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "axes.labelsize": 28,
    "xtick.labelsize": 28,
    "ytick.labelsize": 28,
    "legend.fontsize": 26,
})


# 2. Main Plot Generation
# -----------------------------------------------------------------------------

# --- FIX: Create a numerical month column to prevent plot shifting ---
# The DataFrame index (0-11) corresponds to the months. Add 1 to get (1-12).
if 'MonthNum' not in hanle_monthly_iao.columns:
    hanle_monthly_iao['MonthNum'] = hanle_monthly_iao.index + 1
print("Created numerical 'MonthNum' column for accurate plotting.")


# Create the figure and axes
fig, ax = plt.subplots(figsize=(14, 10))

print("Generating publication-quality monthly comparison plot...")

# --- Plot IAO Observational Data (Day) ---
ax.errorbar(
    hanle_monthly_iao['MonthNum'], # <-- Use the numerical month here
    hanle_monthly_iao['Day'], 
    yerr=hanle_monthly_iao['Day_Error'],
    label='Hanle IAO (Day)',
    fmt='-', # Connect markers with a line
    color=PUBLISHABLE_COLORS[0],
    marker=PUBLISHABLE_MARKERS[0],
    markersize=12,
    markerfacecolor='white', # Hollow markers look very professional
    markeredgewidth=2.5,
    linewidth=3,
    capsize=6, # Error bar cap size
    elinewidth=2,
    alpha=0.9,
    zorder=10
)

# --- Plot IAO Observational Data (Night) ---
ax.errorbar(
    hanle_monthly_iao['MonthNum'], # <-- Use the numerical month here
    hanle_monthly_iao['Night'], 
    yerr=hanle_monthly_iao['Night_Error'],
    label='Hanle IAO (Night)',
    fmt='-', # Connect markers with a line
    color=PUBLISHABLE_COLORS[1],
    marker=PUBLISHABLE_MARKERS[1],
    markersize=12,
    markerfacecolor='white',
    markeredgewidth=2.5,
    linewidth=3,
    capsize=6,
    elinewidth=2,
    alpha=0.9,
    zorder=10
)

# # --- Plot Model Data (ERA5) ---
# # This part was already correct as it uses numerical months
ax.plot(
    pwv_monthly.month, pwv_monthly,
    label='ERA5 - Hanle Pixel',
    color=PUBLISHABLE_COLORS[2],
    marker=PUBLISHABLE_MARKERS[2],
    linestyle='--', # Dashed line to distinguish from observations
    linewidth=3.5,
    markersize=12,
    markerfacecolor=PUBLISHABLE_COLORS[2],
    markeredgewidth=1.5,
    markeredgecolor='black',
    alpha=0.8,
    zorder=5
)


# plot the pwv_obs as as well
pwv_obs_monthly = pwv_obs.groupby('time.month').mean(dim='time')
ax.plot(
    pwv_obs_monthly.month, pwv_obs_monthly,
    label='ERA5 - IAO Hanle',
    color='orange',     
    marker='D', # Diamond marker for distinction
    linestyle='-.', # Dotted line to distinguish from other data
    linewidth=3.5,  
    markersize=12,
    markerfacecolor='orange',
    markeredgewidth=1.5,
    markeredgecolor='black',
    alpha=0.8,
    zorder=5
)  

# 3. Cosmetics and Finalization
# -----------------------------------------------------------------------------
fig.text(0.52, 0.05, r'Month', fontsize=28, ha='center', va='center')
fig.text(0.04, 0.55, r'Precipitable Water Vapor (mm)', fontsize=28, ha='center', va='center', rotation='vertical')

# Add a text box with additional information
textstr = '1998-2017'
props = dict(boxstyle='round,pad=0.5', facecolor='wheat', alpha=0.8, edgecolor='black', linewidth=1.5)
ax.text(0.17, 0.87, textstr, 
         transform=ax.transAxes,
         fontsize=28,
            verticalalignment='bottom',
            horizontalalignment='right',
            bbox=props,
            multialignment='center'
        )

# Configure legend
ax.legend(loc='upper right', fontsize=22, frameon=True, facecolor='white', edgecolor='black', framealpha=0.95)

# Configure ticks, limits, and grid
ax.set_xlim(0.5, 12.5)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], rotation=45, ha='right', fontsize=26)
ax.minorticks_on()
ax.set_yticks([1,2,3,4,5,6,7,8,9])

# Apply detailed tick formatting to all four sides
ax.tick_params(axis='both', which='major', direction='in', width=2, length=10, labelsize=28, top=True, right=True)
ax.tick_params(axis='both', which='minor', direction='in', width=1, length=5, top=True, right=True)

# Set facecolor and spine properties
ax.set_facecolor('white')
for spine in ax.spines.values():
    spine.set_linewidth(2.5)
    
# Adjust layout and save the final figure
fig.tight_layout(rect=[0.05, 0.05, 1, 1])
output_filename = 'Hanle_PWV_Monthly_Comparison_Publication.pdf'
plt.savefig(output_filename, dpi=600, bbox_inches='tight')
print(f"\n→ Plot successfully saved as {output_filename}")
plt.show()


## Summary statistics and reporting

### Step 137

This cell defines reusable helper function(s) `compute_bias`, `compute_rmse`, `compute_r`, `compute_std` so later sections can apply the same processing logic consistently.

In [ ]:
import numpy as np

# Ensure months are aligned
months = hanle_monthly_iao['MonthNum'].values

# Observations
obs_day = hanle_monthly_iao['Day'].values
obs_night = hanle_monthly_iao['Night'].values

# Model estimates
model_era5 = pwv_monthly.sel(month=months).values
model_obs_sp = pwv_obs_monthly.sel(month=months).values

# --- Metric functions based on provided formulas ---
def compute_bias(Xo, Xr):
    return np.mean(Xo - Xr)

def compute_rmse(Xo, Xr):
    return np.sqrt(np.mean((Xo - Xr) ** 2))

def compute_r(Xo, Xr):
    Xo_bar = np.mean(Xo)
    Xr_bar = np.mean(Xr)
    numerator = np.sum((Xo - Xo_bar) * (Xr - Xr_bar))
    denominator = np.sqrt(np.sum((Xo - Xo_bar) ** 2) * np.sum((Xr - Xr_bar) ** 2))
    return numerator / denominator if denominator != 0 else np.nan

def compute_std(X):
    X_bar = np.mean(X)
    return np.sqrt(np.sum((X - X_bar) ** 2) / (len(X) - 1))

# --- Metric Calculations ---
def evaluate_all_metrics(obs, model):
    return {
        'Bias': compute_bias(obs, model),
        'RMSE': compute_rmse(obs, model),
        'R': compute_r(obs, model),
        'STD (Obs)': compute_std(obs)
    }

# Daytime metrics
metrics_day_era5   = evaluate_all_metrics(obs_day, model_era5)
metrics_day_obs_sp = evaluate_all_metrics(obs_day, model_obs_sp)

# Nighttime metrics
metrics_night_era5   = evaluate_all_metrics(obs_night, model_era5)
metrics_night_obs_sp = evaluate_all_metrics(obs_night, model_obs_sp)

# --- Print All Results ---
print("\n📊 Model Evaluation Metrics (in mm):")
print("-----------------------------------------------------------------------")
print(f"{'Case':<20} | {'Bias':>7} | {'RMSE':>7} | {'R':>7} | {'STD (Obs)':>10}")
print("-----------------------------------------------------------------------")
print(f"{'Day vs ERA5 (ERA5 SP)':<20} | {metrics_day_era5['Bias']:7.3f} | {metrics_day_era5['RMSE']:7.3f} | {metrics_day_era5['R']:7.3f} | {metrics_day_era5['STD (Obs)']:10.3f}")
print(f"{'Night vs ERA5 (ERA5 SP)':<20} | {metrics_night_era5['Bias']:7.3f} | {metrics_night_era5['RMSE']:7.3f} | {metrics_night_era5['R']:7.3f} | {metrics_night_era5['STD (Obs)']:10.3f}")
print(f"{'Day vs ERA5 (Obs SP)':<20} | {metrics_day_obs_sp['Bias']:7.3f} | {metrics_day_obs_sp['RMSE']:7.3f} | {metrics_day_obs_sp['R']:7.3f} | {metrics_day_obs_sp['STD (Obs)']:10.3f}")
print(f"{'Night vs ERA5 (Obs SP)':<20} | {metrics_night_obs_sp['Bias']:7.3f} | {metrics_night_obs_sp['RMSE']:7.3f} | {metrics_night_obs_sp['R']:7.3f} | {metrics_night_obs_sp['STD (Obs)']:10.3f}")
print("-----------------------------------------------------------------------")


### Step 138

This cell defines reusable helper function(s) `_to_np`, `weighted_stats`, `weighted_corr`, `gaussian_loglik` so later sections can apply the same processing logic consistently.

In [ ]:
import numpy as np
import pandas as pd

def _to_np(x):
    return np.asarray(x, dtype=float)

def weighted_stats(err, w=None):
    e = _to_np(err)
    if w is None:
        mae = np.mean(np.abs(e))
        rmse = np.sqrt(np.mean(e**2))
        bias = np.mean(e)
    else:
        w = _to_np(w)
        mae = np.sum(w*np.abs(e)) / np.sum(w)
        rmse = np.sqrt(np.sum(w*e**2) / np.sum(w))
        bias = np.sum(w*e) / np.sum(w)
    return {"Bias": bias, "MAE": mae, "RMSE": rmse}

def weighted_corr(x, y, w=None):
    x = _to_np(x); y = _to_np(y)
    if w is None:
        return np.corrcoef(x, y)[0,1]
    w = _to_np(w)
    mx = np.sum(w*x)/np.sum(w)
    my = np.sum(w*y)/np.sum(w)
    cov = np.sum(w*(x-mx)*(y-my))/np.sum(w)
    vx  = np.sum(w*(x-mx)**2)/np.sum(w)
    vy  = np.sum(w*(y-my)**2)/np.sum(w)
    return cov/np.sqrt(vx*vy)

def gaussian_loglik(O, M, sigma):
    O = _to_np(O); M = _to_np(M); s = _to_np(sigma)
    return -0.5*np.sum(((O-M)**2)/(s**2) + np.log(2*np.pi*s**2))

def ols_slope_intercept(x, y):
    # fits y = a + b x (ordinary least squares)
    x = _to_np(x); y = _to_np(y)
    X = np.vstack([np.ones_like(x), x]).T
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    a, b = beta[0], beta[1]
    return b, a

def seasonal_harmonic(months_1to12, y):
    # Fit y[m] ~ a0 + a1 cos(2πm/12) + b1 sin(2πm/12)
    m = _to_np(months_1to12)
    y = _to_np(y)
    cos1 = np.cos(2*np.pi*m/12.0)
    sin1 = np.sin(2*np.pi*m/12.0)
    X = np.vstack([np.ones_like(m), cos1, sin1]).T
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    a0, a1, b1 = beta
    amp = np.sqrt(a1**2 + b1**2)
    # phase in radians; convert to "month" with 0..12 wrap
    phase_rad = np.arctan2(-b1, a1)  # consistent with cos lead
    phase_month = (phase_rad * 12/(2*np.pi)) % 12
    return {"a0": a0, "amp": amp, "phase_month": phase_month}

def summarize_pair(name, months, obs, obs_sigma, mod):
    err = mod - obs
    w = None if obs_sigma is None else 1.0/(obs_sigma**2)
    core = weighted_stats(err, w)
    r    = weighted_corr(obs, mod, w)
    rho  = pd.Series(obs).corr(pd.Series(mod), method='spearman')
    slope, intercept = ols_slope_intercept(mod, obs)  # obs = a + b*mod
    harm_obs = seasonal_harmonic(months, obs)
    harm_mod = seasonal_harmonic(months, mod)
    amp_ratio = harm_mod["amp"]/harm_obs["amp"] if harm_obs["amp"]>0 else np.nan
    phase_diff = (harm_mod["phase_month"] - harm_obs["phase_month"] + 6) % 12 - 6  # in [-6,6]
    ll = None if obs_sigma is None else gaussian_loglik(obs, mod, obs_sigma)
    return {
        "Pair": name,
        "N_months": len(months),
        "Bias (mm)": core["Bias"],
        "MAE (mm)": core["MAE"],
        "RMSE (mm)": core["RMSE"],
        "Pearson r": r,
        "Spearman ρ": rho,
        "OLS slope (obs~mod)": slope,
        "OLS intercept (mm)": intercept,
        "Amp ratio (mod/obs)": amp_ratio,
        "Phase shift (months)": phase_diff,
        "LogLik (Gaussian)": ll
    }


## Exports and file generation

### Step 139

This cell writes derived results or configuration content to disk so they can be reused outside the notebook.

In [ ]:
# Extract vectors (length 12)
months = hanle_monthly_iao["MonthNum"].values
obs_day   = hanle_monthly_iao["Day"].values
sigma_day = hanle_monthly_iao["Day_Error"].values
obs_night   = hanle_monthly_iao["Night"].values
sigma_night = hanle_monthly_iao["Night_Error"].values

era5_pixel = np.asarray(pwv_monthly.values, dtype=float)              # ERA5 – Hanle Pixel
era5_interp = np.asarray(pwv_obs_monthly.values, dtype=float)         # ERA5 – IAO Hanle (your second curve)

# Optional combined day+night (mean); propagate 1σ for mean of two (assume independent)
obs_combined   = 0.5*(obs_day + obs_night)
sigma_combined = 0.5*np.sqrt(sigma_day**2 + sigma_night**2)

rows = []
rows.append(summarize_pair("Day vs ERA5 Pixel",   months, obs_day,   sigma_day,   era5_pixel))
rows.append(summarize_pair("Day vs ERA5 Interp",  months, obs_day,   sigma_day,   era5_interp))
rows.append(summarize_pair("Night vs ERA5 Pixel", months, obs_night, sigma_night, era5_pixel))
rows.append(summarize_pair("Night vs ERA5 Interp",months, obs_night, sigma_night, era5_interp))
rows.append(summarize_pair("Mean(Day,Night) vs ERA5 Pixel",  months, obs_combined, sigma_combined, era5_pixel))
rows.append(summarize_pair("Mean(Day,Night) vs ERA5 Interp", months, obs_combined, sigma_combined, era5_interp))

stats_table = pd.DataFrame(rows)
# Pretty print (rounded)
display_cols = ["Pair", "N_months", "Bias (mm)", "MAE (mm)", "RMSE (mm)", "Pearson r", "Spearman ρ",
                "OLS slope (obs~mod)", "OLS intercept (mm)", "Amp ratio (mod/obs)", "Phase shift (months)", "LogLik (Gaussian)"]
print(stats_table[display_cols].round({
    "Bias (mm)":3, "MAE (mm)":3, "RMSE (mm)":3, "Pearson r":3, "Spearman ρ":3,
    "OLS slope (obs~mod)":3, "OLS intercept (mm)":3, "Amp ratio (mod/obs)":3, "Phase shift (months)":2
}))
# You can also export:
stats_table.to_csv("Hanle_PWV_monthly_stats.csv", index=False)
with open("Hanle_PWV_monthly_stats.tex","w") as f:
    f.write(stats_table[display_cols].to_latex(index=False, float_format="%.3f"))


## Summary statistics and reporting

### Step 140

This cell defines reusable helper function(s) `bootstrap_metric`, `fn_rmse`, `ci_for_r` so later sections can apply the same processing logic consistently.

In [ ]:
rng = np.random.default_rng(2025)
def bootstrap_metric(obs, sig, mod, fn, B=5000):
    idx = np.arange(len(obs))
    draws = []
    for _ in range(B):
        ii = rng.choice(idx, size=len(idx), replace=True)
        draws.append(fn(obs[ii], sig[ii] if sig is not None else None, mod[ii]))
    return np.quantile(draws, [0.025, 0.5, 0.975])

def fn_rmse(obs, sig, mod):
    w = None if sig is None else 1.0/(sig**2)
    e = mod-obs
    return np.sqrt(np.sum((e**2) if w is None else w*e**2) / (len(e) if w is None else np.sum(w)))

def ci_for_r(obs, mod):
    # Fisher z transform
    r = np.corrcoef(obs, mod)[0,1]
    n = len(obs)
    z = np.arctanh(np.clip(r, -0.999999, 0.999999))
    se = 1/np.sqrt(n-3)
    z_ci = (z - 1.96*se, z + 1.96*se)
    r_ci = (np.tanh(z_ci[0]), np.tanh(z_ci[1]))
    return r, r_ci

# Example: RMSE CI for Day vs ERA5 Interp
rmse_ci = bootstrap_metric(obs_day, sigma_day, era5_interp, fn_rmse, B=10000)
r_point, r_ci = ci_for_r(obs_day, era5_interp)
print("RMSE (Day vs Interp) 95% CI:", rmse_ci[[0,2]])
print("Pearson r (Day vs Interp):", r_point, " 95% CI:", r_ci)


## Site definitions and metadata

### Step 141

This cell writes derived results or configuration content to disk so they can be reused outside the notebook.

In [ ]:
bias_by_month = pd.DataFrame({
    "Month": months,
    "Bias_Pixel_Day": era5_pixel - obs_day,
    "Bias_Interp_Day": era5_interp - obs_day,
    "Bias_Pixel_Night": era5_pixel - obs_night,
    "Bias_Interp_Night": era5_interp - obs_night,
})
bias_by_month.to_csv("Hanle_PWV_bias_by_month.csv", index=False)


## Analysis workflow

### Step 142

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
bias_by_month


#### Overplotting Both the Methods

## Pressure and PWV calculations

### Step 143

This cell defines reusable helper function(s) `calculate_pwv_vectorized`, `integrate_profile`, `calculate_surface_pressure_from_elevation` so later sections can apply the same processing logic consistently.

In [ ]:
# -*- coding: utf-8 -*-
"""
Full script to calculate and compare ERA5 PWV using three different methods
and plot them against observational data for Hanle.

This script ASSUMES the following variables already exist in your environment:
- HANLE: 
    A dictionary with site information.
    Required keys: 'lat', 'lon', 'elevation'.
    Example: HANLE = {'lat': 32.78, 'lon': 78.96, 'elevation': 4500.0}

- era5: 
    An xarray.Dataset containing specific humidity ('q') on pressure levels.
    Required dimensions: 'time', 'level', 'latitude', 'longitude'.

- surface_pressure_data: 
    An xarray.Dataset containing surface pressure ('sp') and 2m temp ('t2m').
    Required dimensions: 'time', 'latitude', 'longitude'.

- hanle_monthly_iao: 
    A pandas.DataFrame with monthly observational PWV data.
    Required columns: 'Month', 'Day', 'Day_Error', 'Night', 'Night_Error'.

- The file 'ERA5_integrate_world.nc' must be in the same directory.
"""

# 1. SETUP AND PREREQUISITES
# =============================================================================
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import xarray as xr
from scipy.interpolate import PchipInterpolator

print("--- Script Starting: PWV Calculation and Plotting ---")
print("Assuming 'HANLE', 'era5', 'surface_pressure_data', and 'hanle_monthly_iao' are pre-loaded.")

# --- Load the third dataset from file ---
try:
    era5_integrated = xr.open_dataset('ERA5_integrate_world.nc')
    print("✓ Successfully loaded 'ERA5_integrate_world.nc'")
except FileNotFoundError:
    print("❌ ERROR: 'ERA5_integrate_world.nc' not found. Please ensure it is in the correct directory.")
    # Exit or handle the error as appropriate
    exit()

# 2. PWV CALCULATION ENGINE
# =============================================================================
def calculate_pwv_vectorized(q: xr.DataArray, p_sfc: xr.DataArray) -> xr.DataArray:
    """Calculates Precipitable Water Vapor (PWV) using a fast, vectorized approach."""
    g = 9.81  # m/s^2
    p_levels_pa = q.level * 100  # Convert hPa to Pa
    p_levels_pa.attrs['units'] = 'Pa'
    
    def integrate_profile(q_profile, p_profile, sfc_p_value):
        finite_mask = np.isfinite(q_profile)
        if finite_mask.sum() < 2: return np.nan
        p, q_ = p_profile[finite_mask], q_profile[finite_mask]
        sort_idx = np.argsort(p)
        p, q_ = p[sort_idx], q_[sort_idx]
        valid_levels = p <= sfc_p_value
        p, q_ = p[valid_levels], q_[valid_levels]
        if len(p) < 2: return np.nan
        interpolator = PchipInterpolator(p, q_, extrapolate=True)
        q_sfc = interpolator(sfc_p_value)
        p_full = np.append(p, sfc_p_value)
        q_full = np.append(q_, q_sfc)
        sort_idx_full = np.argsort(p_full)
        p_full, q_full = p_full[sort_idx_full], q_full[sort_idx_full]
        return np.trapz(q_full, p_full) / g

    pwv = xr.apply_ufunc(
        integrate_profile, q, p_levels_pa, p_sfc,
        input_core_dims=[['level'], ['level'], []],
        output_core_dims=[[]], vectorize=True, dask="parallelized", output_dtypes=[q.dtype]
    )
    pwv.attrs.update(units="mm", long_name="Precipitable Water Vapor")
    return pwv


# 3. BAROMETRIC PRESSURE CALCULATION
# =============================================================================
def calculate_surface_pressure_from_elevation(t2m: xr.DataArray, elevation_m: float) -> xr.DataArray:
    """Calculates surface pressure from 2m air temperature using the barometric formula."""
    P0 = 101325.0
    g = 9.80665
    R = 287.058
    pressure_pa = P0 * np.exp(-g * elevation_m / (R * t2m))
    pressure_pa.attrs = {'units': 'Pa', 'long_name': 'Calculated Surface Pressure'}
    return pressure_pa


# 4. ANALYSIS WORKFLOW
# =============================================================================
lat_hanle, lon_hanle = HANLE['lat'], HANLE['lon']

# --- METHOD 1: PWV using ERA5 Surface Pressure ---
print("\n--- Starting Method 1: PWV from ERA5 Surface Pressure ---")
q_site_m1 = era5['q'].interp(latitude=lat_hanle, longitude=lon_hanle, method='linear')
sp_site_m1 = surface_pressure_data['sp'].interp(latitude=lat_hanle, longitude=lon_hanle, method='linear')

if 'expver' in q_site_m1.dims:
    q_site_m1 = q_site_m1.sel(expver=1, drop=True)
if 'expver' in sp_site_m1.dims:
    sp_site_m1 = sp_site_m1.sel(expver=1, drop=True)

q_aligned_m1, sp_aligned_m1 = xr.align(q_site_m1, sp_site_m1, join='inner')
q_rechunked_m1 = q_aligned_m1.chunk({"level": -1})
pwv_method1 = calculate_pwv_vectorized(q_rechunked_m1, sp_aligned_m1).load()
print("✓ Method 1 calculation complete.")

# --- METHOD 2: PWV using Elevation-based Surface Pressure (with t2m) ---
print("\n--- Starting Method 2: PWV from Elevation & t2m ---")
q_site_m2 = era5['q'].interp(latitude=lat_hanle, longitude=lon_hanle, method='linear')
t2m_site_m2 = surface_pressure_data['t2m'].interp(latitude=lat_hanle, longitude=lon_hanle, method='linear')
t2m_site_m2 = xr.full_like(t2m_site_m2, 288.15)  # Use a constant temperature of 279K

if 'expver' in q_site_m2.dims:
    q_site_m2 = q_site_m2.sel(expver=1, drop=True)
if 'expver' in t2m_site_m2.dims:
    t2m_site_m2 = t2m_site_m2.sel(expver=1, drop=True)

q_aligned_m2, t2m_aligned_m2 = xr.align(q_site_m2, t2m_site_m2, join='inner')
sp_calculated_m2 = calculate_surface_pressure_from_elevation(t2m_aligned_m2, HANLE['elevation'])
q_rechunked_m2 = q_aligned_m2.chunk({"level": -1})
pwv_method2 = calculate_pwv_vectorized(q_rechunked_m2, sp_calculated_m2).load()
print("✓ Method 2 calculation complete.")

# --- METHOD 3: PWV from Integrated 'tcwv' variable ---
print("\n--- Starting Method 3: PWV from 'tcwv' ---")
if 'valid_time' in era5_integrated.dims:
    era5_integrated = era5_integrated.rename({'valid_time': 'time'})

tcwv_site = era5_integrated['tcwv'].interp(latitude=lat_hanle, longitude=lon_hanle, method='linear')
if 'expver' in tcwv_site.dims:
    tcwv_site = tcwv_site.sel(expver=1, drop=True)
    
pwv_method3 = tcwv_site.sel(time=slice('1998-01-01', '2017-12-31')).load()
print("✓ Method 3 processing complete.")


# --- Data Aggregation for Plotting ---
print("\nAggregating results for plotting...")
pwv_monthly_method1 = pwv_method1.groupby('time.month').mean()
pwv_monthly_method2 = pwv_method2.groupby('time.month').mean()
pwv_monthly_method3 = pwv_method3.groupby('time.month').mean()
print("✓ Monthly means calculated.")


# 5. PUBLICATION-QUALITY PLOTTING
# =============================================================================
print("\n--- Generating Publication-Quality Monthly Comparison Plot ---")

# --- Plotting Configuration ---
PUBLISHABLE_COLORS = ['#00429d', '#93003a', '#009b7a', '#ff7c00', '#6a0dad']
PUBLISHABLE_MARKERS = ['o', 's', '^', 'D', 'p'] 

mpl.rcParams.update({
    "text.usetex": True, "font.family": "serif", "font.serif": ["Times New Roman"],
    "axes.labelsize": 28, "xtick.labelsize": 28, "ytick.labelsize": 28,
    "legend.fontsize": 22,
})

# Add this line before the "Create the Plot" section
hanle_monthly_iao['MonthNum'] = hanle_monthly_iao.index + 1

# --- Create the Plot ---
fig, ax = plt.subplots(figsize=(14, 10))

# Plot Observational Data
ax.errorbar(
    hanle_monthly_iao['MonthNum'], hanle_monthly_iao['Day'], yerr=hanle_monthly_iao['Day_Error'],
    label='Hanle IAO (Day)', fmt='-', color=PUBLISHABLE_COLORS[0], marker=PUBLISHABLE_MARKERS[0],
    markersize=12, markerfacecolor='white', markeredgewidth=2.5, linewidth=3, capsize=6,
    elinewidth=2, alpha=0.9, zorder=10
)
ax.errorbar(
    hanle_monthly_iao['MonthNum'], hanle_monthly_iao['Night'], yerr=hanle_monthly_iao['Night_Error'],
    label='Hanle IAO (Night)', fmt='-', color=PUBLISHABLE_COLORS[1], marker=PUBLISHABLE_MARKERS[1],
    markersize=12, markerfacecolor='white', markeredgewidth=2.5, linewidth=3, capsize=6,
    elinewidth=2, alpha=0.9, zorder=10
)

# Plot Model Data (Method 1)
ax.plot(
    pwv_monthly_method1.month, pwv_monthly_method1, label='ERA5 (model sp)',
    color=PUBLISHABLE_COLORS[2], marker=PUBLISHABLE_MARKERS[2], linestyle='--', linewidth=3.5,
    markersize=12, markerfacecolor=PUBLISHABLE_COLORS[2], markeredgewidth=1.5,
    markeredgecolor='black', alpha=0.8, zorder=5
)
# Plot Model Data (Method 2)
ax.plot(
    pwv_monthly_method2.month, pwv_monthly_method2, label='ERA5 (elev. + t2m)',
    color=PUBLISHABLE_COLORS[3], marker=PUBLISHABLE_MARKERS[3], linestyle=':', linewidth=3.5,
    markersize=12, markerfacecolor=PUBLISHABLE_COLORS[3], markeredgewidth=1.5,
    markeredgecolor='black', alpha=0.8, zorder=5
)
# Plot Model Data (Method 3)
ax.plot(
    pwv_monthly_method3.month, pwv_monthly_method3, label='ERA5 (tcwv)',
    color=PUBLISHABLE_COLORS[4], marker=PUBLISHABLE_MARKERS[4], linestyle='-.', linewidth=3.5,
    markersize=12, markerfacecolor=PUBLISHABLE_COLORS[4], markeredgewidth=1.5,
    markeredgecolor='black', alpha=0.8, zorder=6
)

# --- Cosmetics and Finalization ---
fig.text(0.52, 0.05, r'Month', fontsize=28, ha='center', va='center')
fig.text(0.04, 0.55, r'Precipitable Water Vapor (mm)', fontsize=28, ha='center', va='center', rotation='vertical')

textstr = '1998-2017'
props = dict(boxstyle='round,pad=0.5', facecolor='wheat', alpha=0.8, edgecolor='black', linewidth=1.5)
ax.text(0.35, 0.95, textstr, transform=ax.transAxes, fontsize=28,
         verticalalignment='top', horizontalalignment='right', bbox=props)

ax.legend(loc='upper right', frameon=True, facecolor='white', edgecolor='black', framealpha=0.95)

ax.set_xlim(0.5, 12.5)
ax.set_ylim(bottom=0)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], rotation=45, ha='right')
ax.minorticks_on()
ax.grid(True, which='major', linestyle='--', linewidth=0.7, alpha=0.6, zorder=0)

ax.tick_params(axis='both', which='major', direction='in', width=2, length=10, top=True, right=True)
ax.tick_params(axis='both', which='minor', direction='in', width=1, length=5, top=True, right=True)

ax.set_facecolor('white')
for spine in ax.spines.values():
    spine.set_linewidth(2.5)

fig.tight_layout(rect=[0.08, 0.08, 0.98, 0.98])
output_filename = 'Hanle_PWV_Monthly_Comparison_3-Method_Publication.pdf'
plt.savefig(output_filename, dpi=600, bbox_inches='tight')
print(f"\n→ Plot successfully saved as {output_filename}")
plt.show()

print("\n--- Script Finished ---")


## Summary statistics and reporting

### Step 144

This cell defines reusable helper function(s) `get_stats_hpa` so later sections can apply the same processing logic consistently.

In [ ]:
# =======================================================================
# PART 6: HANLE SURFACE PRESSURE STATISTICS (Using Existing Variables)
# =======================================================================
# This script uses the data already computed and stored in the variables
# `sp_aligned_m1` and `sp_calculated_m2` from your main analysis script.
# =======================================================================

print("--- Calculating Surface Pressure Statistics for Hanle ---")
print("Using pre-computed variables `sp_aligned_m1` and `sp_calculated_m2`.")
print("Pressures are converted from Pa to hPa for clarity.\n")

# --- Helper function to compute stats ---
def get_stats_hpa(pressure_series):
    """Computes mean, median, and std dev and converts to hPa."""
    if pressure_series.size == 0 or np.all(np.isnan(pressure_series.values)):
        return {'Mean': np.nan, 'Median': np.nan, 'Std Dev': np.nan}
    return {
        'Mean': np.nanmean(pressure_series.values) / 100,
        'Median': np.nanmedian(pressure_series.values) / 100,
        'Std Dev': np.nanstd(pressure_series.values) / 100
    }

# --- Compute Statistics using the existing variables ---

# Method 1: ERA5 Model Surface Pressure ('sp')
# This uses the 'sp_aligned_m1' variable from your script
stats_method1 = get_stats_hpa(sp_aligned_m1)
print("--- Method 1: ERA5 Model 'sp' ---")
print("First 5 pressure values (hPa):")
print(np.round(sp_aligned_m1.values[:5] / 100, 2))


# Method 2: Elevation-based Formula
# This uses the 'sp_calculated_m2' variable from your script
stats_method2 = get_stats_hpa(sp_calculated_m2)
print("\n--- Method 2: Elevation-based Formula ---")
print("First 5 pressure values (hPa):")
print(np.round(sp_calculated_m2.values[:5] / 100, 2))


# --- Print the results in a formatted table ---
print("\n\n" + "="*80)
print("FINAL STATISTICS FOR HANLE (from script variables)")
print(f"{'Method':<28} {'Mean (hPa)':<15} {'Median (hPa)':<15} {'Std Dev (hPa)':<15}")
print("-"*80)
print(f"{'ERA5 Model Surface (`sp`)':<28} {stats_method1['Mean']:.2f}           {stats_method1['Median']:.2f}           {stats_method1['Std Dev']:.2f}")
print(f"{'Elevation-based Formula':<28} {stats_method2['Mean']:.2f}           {stats_method2['Median']:.2f}           {stats_method2['Std Dev']:.2f}")
print("="*80)


## Pressure and PWV calculations

### Step 145

This cell defines reusable helper function(s) `calculate_mbe`, `calculate_rmse`, `calculate_pearson_r` so later sections can apply the same processing logic consistently.

In [ ]:
# 5. STATISTICAL ANALYSIS
# =============================================================================
print("\n--- Performing Statistical Analysis ---")

# --- FIX: Create an observational series with a numerical month index (1-12) ---
# Combine the Day and Night observational data by taking the mean.
obs_combined_monthly = hanle_monthly_iao[['Day', 'Night']].mean(axis=1)
# Set the index to be the numerical month (1-12) to match the model data.
obs_combined_monthly.index = hanle_monthly_iao.index + 1
obs_combined_monthly.index.name = 'Month' # Good practice for alignment


# Convert the xarray DataArrays for the models to pandas Series
model1_series = pwv_monthly_method1.to_series()
model2_series = pwv_monthly_method2.to_series()
model3_series = pwv_monthly_method3.to_series()

# --- Define functions to calculate statistics ---
def calculate_mbe(model, obs):
    """Calculates Mean Bias Error."""
    return (model - obs).mean()

def calculate_rmse(model, obs):
    """Calculates Root Mean Square Error."""
    return np.sqrt(((model - obs) ** 2).mean())

def calculate_pearson_r(model, obs):
    """Calculates Pearson Correlation Coefficient."""
    # Dropna() is added as a safeguard in case of any misaligned data
    return model.corr(obs.dropna())

# --- Calculate statistics for each model ---
stats = {}
models = {
    'ERA5 (model sp)': model1_series,
    'ERA5 (elev. + t2m)': model2_series,
    'ERA5 (tcwv)': model3_series
}

for name, model_data in models.items():
    mbe = calculate_mbe(model_data, obs_combined_monthly)
    rmse = calculate_rmse(model_data, obs_combined_monthly)
    pearson_r = calculate_pearson_r(model_data, obs_combined_monthly)
    stats[name] = {'MBE': mbe, 'RMSE': rmse, 'Pearson r': pearson_r}

# --- Print the results in a formatted table ---
print("\nStatistical Comparison with Combined Day/Night Observations:")
print("-" * 70)
print(f"{'Model':<25} | {'Mean Bias Error (MBE)':<22} | {'RMSE':<10} | {'Pearson r':<10}")
print("-" * 70)

for name, metrics in stats.items():
    print(f"{name:<25} | {metrics['MBE']:<+22.4f} | {metrics['RMSE']:<10.4f} | {metrics['Pearson r']:<10.4f}")

print("-" * 70)
print("\n--- Interpretation of the Results ---")
print("1. Mean Bias Error (MBE): Shows the average bias. A positive value means the")
print("   model, on average, overestimates the PWV compared to observations.")
print("   A negative value means it underestimates. The best model has an MBE closest to 0.")
print("\n2. Root Mean Square Error (RMSE): Measures the magnitude of the error. It gives")
print("   a single number representing the average distance between the model and")
print("   observations. A lower RMSE value indicates a better fit.")
print("\n3. Pearson Correlation (r): Measures how well the model and observations vary")
print("   together. A value of 1 means perfect positive linear correlation.")
print("   A higher 'r' value (closer to 1) is better.")

# Determine the best model based on the lowest RMSE
best_model_rmse = min(stats, key=lambda x: stats[x]['RMSE'])
print(f"\n--- Conclusion ---")
print(f"Based on the lowest Root Mean Square Error (RMSE), the best performing model is: '{best_model_rmse}'.")
print("You should also consider the other metrics for a complete picture.")


## Data loading and inspection

### Step 146

This cell loads dataset(s) `ERA5_integrate_world.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
era5_integrated = xr.open_dataset('ERA5_integrate_world.nc')


## Analysis workflow

### Step 147

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
era5_integrated


## Site definitions and metadata

### Step 148

This cell reads tabular metadata that is later used for site lookup, filtering, or summary reporting.

In [ ]:

hanle_yearly_iao = pd.read_csv('hanle_yearly_pwv.txt', sep='\s+', skiprows=1, names=['Year', 'Day', 'Night', 'Mean'])


### Step 149

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
hanle_yearly_iao


## Summary statistics and reporting

### Step 150

This cell computes or evaluates precipitable water vapor (PWV)-related quantities that are central to the site-quality analysis.

In [ ]:
pwv_yearly_mean = pwv_corr.groupby('time.year').median(dim='time')


## Pressure and PWV calculations

### Step 151

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
pwv_yearly_mean.plot(label='ERA5', marker='o', linestyle='-', color='blue',
                     markerfacecolor='white', markeredgewidth=2, markersize=8)
plt.xlabel('Year')
plt.ylabel('PWV (mm)')
plt.title('Yearly Mean Precipitable Water Vapor at Hanle (1998-2017)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


## Mapping and visualization

### Step 152

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
# Calculate yearly means

#era5_yearly_mean = hanle_era5_1998_2017['tcwv'].resample(time='YE').median()
#merra2_yearly_mean = hanle_merra2_1998_2017['TQV'].resample(time='YE').median()

# Load Hanle data
years = pd.to_datetime(hanle_yearly_iao['Year'], format='%Y').dt.year

# Calculate residuals
#era5_residual = era5_yearly_mean - hanle_yearly_iao.set_index('Year')['Mean']
era5_residual = pwv_yearly_mean - hanle_yearly_iao.set_index('Year')['Mean']
#merra2_residual = merra2_yearly_mean - hanle_yearly_iao.set_index('Year')['Mean']

# Setup plot with desired aesthetics
plt.rc('font', family='serif', size=16)
plt.rcParams['xtick.major.size'] = 6
plt.rcParams['xtick.minor.size'] = 4
plt.rcParams['ytick.major.size'] = 6
plt.rcParams['ytick.minor.size'] = 4
plt.figure(figsize=(10, 10))

# Plotting combined yearly data and residuals
plt.plot(years, hanle_yearly_iao['Mean'], label='Hanle IAO Median', marker='o', linestyle='-', color='blue', 
         markerfacecolor='black', markeredgewidth=4, markersize=12)

plt.plot(pwv_yearly_mean.year, pwv_yearly_mean, label='ERA5', marker='s', linestyle='-', color='green', 
         markerfacecolor='white', markeredgewidth=4, markersize=12)

#plt.plot(merra2_yearly_mean.time.dt.year, merra2_yearly_mean, label='MERRA2', marker='^', linestyle='-', color='orange', 
        #  markerfacecolor='white', markeredgewidth=4, markersize=12)

# Plotting residuals with a different style
plt.plot(pwv_yearly_mean.year, era5_residual, label='ERA5 Residual', linestyle='--', color='red',
         markerfacecolor='white', markeredgewidth=2, markersize=8, linewidth=4,
         alpha=0.8)  # Reduced alpha for residuals
#plt.plot(merra2_yearly_mean.time.dt.year, merra2_residual, label='MERRA2 Residual', linestyle='--', color='red'
        #  , markerfacecolor='white', markeredgewidth=2, markersize=8, linewidth=4,
        #  alpha=0.8)  # Reduced alpha for residuals

plt.xlabel('Year', labelpad=10, fontsize=24)
plt.ylabel('PWV (mm)', labelpad=1, fontsize=24)

x_ticks = np.arange(1998, 2019, 2)
plt.xticks(x_ticks, rotation=30, ha='right', va='top', fontsize=18)
#plt.xticks(years, rotation=60, ha='right', va='top', fontsize=18)
legend = plt.legend(ncol=3, loc='upper center', fontsize=14.8,
                     frameon=True, facecolor='white', edgecolor='black', framealpha=1, shadow=True)


#plt.grid(True)

# Apply formatting to axes and plot borders
plt.gca().set_facecolor('white')
plt.gca().spines['top'].set_linewidth(2)
plt.gca().spines['right'].set_linewidth(2)
plt.gca().spines['bottom'].set_linewidth(2)
plt.gca().spines['left'].set_linewidth(2)
plt.tick_params(axis='both', direction='in', width=2, labelsize=18, which='both')
plt.minorticks_on()
plt.ylim(-0.8, 2.4)
plt.tick_params(axis='both', direction='in', width=1, labelsize=9, which='minor')

plt.tight_layout()
#plt.savefig('Hanle_PWV_Yearly_Comparison_Residuals_Integrated.pdf', dpi=500)
plt.show()


### Step 153

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Assuming hanle_yearly_iao, pwv_yearly_mean are already defined

# Load Hanle data
years = pd.to_datetime(hanle_yearly_iao['Year'], format='%Y').dt.year

# Calculate residuals
era5_residual = pwv_yearly_mean - hanle_yearly_iao.set_index('Year')['Mean']

# Setup plot with desired aesthetics
plt.rc('font', family='serif', size=16)
plt.rcParams['xtick.major.size'] = 6
plt.rcParams['xtick.minor.size'] = 4
plt.rcParams['ytick.major.size'] = 6
plt.rcParams['ytick.minor.size'] = 4

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(10, 10), gridspec_kw={'hspace': 0})

# Plotting combined yearly data
line1, = ax1.plot(years, hanle_yearly_iao['Mean'], label='Hanle IAO Median', marker='o', linestyle='-', color='blue', 
                  markerfacecolor='black', markeredgewidth=4, markersize=12)
line2, = ax1.plot(pwv_yearly_mean.year, pwv_yearly_mean, label='ERA5', marker='s', linestyle='-', color='green', 
                  markerfacecolor='white', markeredgewidth=4, markersize=12)
ax1.set_ylim(1.2, 2.7)

# Plotting residuals
line3, = ax2.plot(pwv_yearly_mean.year, era5_residual, label='ERA5 Residual', linestyle='--', color='red',
                  markerfacecolor='black', markeredgewidth=2, markersize=8, linewidth=5, alpha=0.8,
                 marker = 'X')  # Reduced alpha for residuals
ax2.set_ylim(-0.4, 0.8)

# Common y-axis label
fig.text(0.04, 0.5, 'PWV (mm)', va='center', rotation='vertical', fontsize=24)

# Setting up x-ticks
x_ticks = np.arange(1998, 2019, 2)
plt.xticks(x_ticks, rotation=30, ha='right', va='top', fontsize=18)

# Apply formatting to axes and plot borders for both subplots
for ax in [ax1, ax2]:
    ax.set_facecolor('white')
    ax.spines['top'].set_linewidth(2)
    ax.spines['right'].set_linewidth(2)
    ax.spines['bottom'].set_linewidth(2)
    ax.spines['left'].set_linewidth(2)
    ax.tick_params(axis='both', direction='in', width=2, labelsize=18, which='both')

# Remove ticks and minorticks on the shared line between the plots
ax1.tick_params(labelbottom=False, bottom=False, which='both')
ax2.tick_params(top=False, which='both')

# Combine legends from both subplots
lines = [line1, line2, line3]
labels = [line.get_label() for line in lines]
ax1.legend(lines, labels, ncol=3, loc='upper center', fontsize=17,
           frameon=True, facecolor='white', edgecolor='black', framealpha=1, shadow=True)

ax2.set_xlabel('Year', labelpad=10, fontsize=24)

plt.tight_layout(rect=[0.06, 0, 1, 1])  # Adjust the rect parameter to make room for the common y-label
plt.savefig('Hanle_PWV_Yearly_Comparison_Residuals_Integrated_pressure_level.pdf', dpi=600)
plt.show()


### Step 154

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
"""
Yearly PWV comparison (Hanle IAO vs. ERA5) + residuals
formatted identically to your monthly reference plot.

Prerequisites already in memory
--------------------------------------------------
• hanle_yearly_iao : DataFrame with columns ['Year', 'Mean']
• pwv_yearly_mean  : Series / DataFrame / xarray DataArray
"""

# ────────────────────────────────────────────────────────────────────────────
# Imports
# ────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# ────────────────────────────────────────────────────────────────────────────
# Harmonise inputs
# ────────────────────────────────────────────────────────────────────────────
# Hanle
hanle_df            = hanle_yearly_iao.copy()
hanle_df['Year']    = hanle_df['Year'].astype(int)
hanle_series        = hanle_df.set_index('Year')['Mean']
years               = hanle_series.index

# ERA5 (accepts Series / DataFrame / xarray)
if isinstance(pwv_yearly_mean, pd.Series):
    pwv_series = pwv_yearly_mean.copy()

elif isinstance(pwv_yearly_mean, pd.DataFrame):
    ycol  = [c for c in pwv_yearly_mean.columns if c.lower() == 'year'][0]
    dcol  = [c for c in pwv_yearly_mean.columns if c != ycol][0]
    pwv_series = pwv_yearly_mean.set_index(ycol)[dcol]

else:  # xarray.DataArray
    pwv_series = pwv_yearly_mean.to_series()
    if isinstance(pwv_series.index, pd.MultiIndex):
        pwv_series = pwv_series.droplevel(
            [lvl for lvl in pwv_series.index.names if lvl != 'year']
        )
    pwv_series.index.name = 'Year'

# ensure integer year index
if not np.issubdtype(pwv_series.index.dtype, np.integer):
    pwv_series.index = pd.to_datetime(pwv_series.index).year

pwv_aligned   = pwv_series.reindex(years)
era5_residual = pwv_aligned - hanle_series

# ────────────────────────────────────────────────────────────────────────────
# Plotting configuration (matches reference style)
# ────────────────────────────────────────────────────────────────────────────
PUBLISHABLE_COLORS = ['#00429d', '#93003a', '#009b7a'] # Blue for Day, Magenta for Night, Green for Model
PUBLISHABLE_MARKERS = ['o', 's', 'X', 'D']

mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif":  ["Times New Roman"],
    "axes.labelsize": 28,
    "xtick.labelsize": 28,
    "ytick.labelsize": 28,
    "legend.fontsize": 26,
})



# ────────────────────────────────────────────────────────────────────────────
# Create figure
# ────────────────────────────────────────────────────────────────────────────
fig, (ax_top, ax_bot) = plt.subplots(
    nrows=2, sharex=True, figsize=(14, 11),
    gridspec_kw={"hspace": 0.02}
)

# ── Top panel: Hanle vs ERA5 ───────────────────────────────────────────────
ax_top.plot(
    years, hanle_series,
    label='Hanle IAO',
    color='#00429d',
    marker='o',
    linestyle='solid', # Solid line to distinguish from observations
    linewidth=3.5,
    markersize=12,
    markerfacecolor='white',
    markeredgewidth=1.5,
    markeredgecolor='black',
    alpha=0.8,
    zorder=5
)

ax_top.plot(
    years, pwv_aligned,
    label='ERA5 : Hanle Pixel',
    color='#009b7a',
    marker='^',
    linestyle='--',
    linewidth=3.5,
    markersize=12,
    markerfacecolor='#009b7a',
    markeredgewidth=1.5,
    markeredgecolor='black',
    alpha=0.8,
    zorder=8
)

print(years)
print(pwv_aligned)


ax_top.plot(years, pwv_obs.groupby('time.year').median(dim='time').values,
    label='ERA5 : IAO Hanle',
    color='orange',
    marker='s',
    linestyle='-.',  # Dotted line for Obs SP
    linewidth=3.5,
    markersize=12,
    markerfacecolor='orange',
    markeredgecolor='black',
    markeredgewidth=2.5,
    zorder=6
)

ax_top.set_ylim(1.2, 3.7)

# ── Bottom panel: residuals ───────────────────────────────────────────────
ax_bot.plot(
    years, era5_residual,
    label='ERA5: Hanle Pixel $-$ IAO Hanle',
    color=PUBLISHABLE_COLORS[1],
    marker=PUBLISHABLE_MARKERS[1],
    linestyle=':',
    linewidth=3.5,
    markersize=11,
    markerfacecolor='white',
    markeredgewidth=2.5,
    alpha=0.9,
    zorder=10
)

# also plot the Obs SP PWV calculated values
ax_bot.plot(
    years,  pwv_obs.groupby('time.year').median(dim='time').values - hanle_series,
    label='ERA5: IAO Hanle $-$ IAO Hanle',
    color=PUBLISHABLE_COLORS[2],
    marker=PUBLISHABLE_MARKERS[2],
    linestyle=':',  # Dotted line for Obs SP
    linewidth=3.5,
    markersize=12,
    markerfacecolor='white',
    markeredgewidth=2.5,
    zorder=6
)


ax_bot.set_ylim(-0.5, 1.3)

# ────────────────────────────────────────────────────────────────────────────
# Cosmetics (identical pattern to reference)
# ────────────────────────────────────────────────────────────────────────────
# Global axis labels
fig.text(0.53, 0.016, r'Year', fontsize=28,
         ha='center', va='center')
fig.text(0.05, 0.47, r'Precipitable Water Vapor (mm)', fontsize=28,
         ha='center', va='center', rotation='vertical')

# # Text box with span
# textstr = f'{years.min()}–{years.max()}'
# props   = dict(boxstyle='round,pad=0.5', facecolor='wheat',
#                alpha=0.8, edgecolor='black', linewidth=1.5)
# ax_top.text(0.16, 0.87, textstr,
#             transform=ax_top.transAxes,
#             fontsize=28, ha='right', va='bottom', bbox=props)

# Legend
ax_top.legend(loc='upper center', frameon=True, facecolor='white',
              edgecolor='black', framealpha=0.95,ncol = 3,
              fontsize=22)
# Ticks / spines
for ax in (ax_top, ax_bot):
    ax.set_facecolor('white')
    ax.minorticks_on()
    ax.tick_params(axis='both', which='major', direction='in',
                   width=2, length=10, top=True, right=True)
    ax.tick_params(axis='both', which='minor', direction='in',
                   width=1, length=5, top=True, right=True)
    for spine in ax.spines.values():
        spine.set_linewidth(2.5)

# X ticks every 2 years
ax_bot.set_xlim(years.min() - 0.5, years.max() + 0.5)
ax_bot.set_xticks(np.arange(years.min(), years.max() + 1, 2))
ax_bot.set_xticklabels(np.arange(years.min(), years.max() + 1, 2),
                       rotation=0,)
ax_bot.legend(loc='lower center', frameon=True, facecolor='white',
              edgecolor='black', framealpha=0.95,ncol = 3,
              fontsize=16)


# ────────────────────────────────────────────────────────────────────────────
# Layout & export
# ────────────────────────────────────────────────────────────────────────────
fig.tight_layout(rect=[0.05, 0.05, 1, 1])
outfile = 'Hanle_PWV_Yearly_Comparison_Publication.pdf'
plt.savefig(outfile, dpi=600, bbox_inches='tight')
print(f"\n→ Plot successfully saved as {outfile}")
plt.show()


### Step 155

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
"""
Yearly PWV comparison (Hanle IAO vs. ERA5)
Single top panel only, styled like the original 2-panel version.

Prerequisites already in memory
--------------------------------------------------
• hanle_yearly_iao : DataFrame with columns ['Year', 'Mean']
• pwv_yearly_mean  : Series / DataFrame / xarray DataArray
• pwv_obs          : xarray DataArray with 'time' dim (works with groupby('time.year'))
"""

# ────────────────────────────────────────────────────────────────────────────
# Imports
# ────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# ────────────────────────────────────────────────────────────────────────────
# Harmonise inputs
# ────────────────────────────────────────────────────────────────────────────
# Hanle
hanle_df         = hanle_yearly_iao.copy()
hanle_df['Year'] = hanle_df['Year'].astype(int)
hanle_series     = hanle_df.set_index('Year')['Mean']
years            = hanle_series.index

# ERA5 (accepts Series / DataFrame / xarray)
if isinstance(pwv_yearly_mean, pd.Series):
    pwv_series = pwv_yearly_mean.copy()

elif isinstance(pwv_yearly_mean, pd.DataFrame):
    ycol  = [c for c in pwv_yearly_mean.columns if c.lower() == 'year'][0]
    dcol  = [c for c in pwv_yearly_mean.columns if c != ycol][0]
    pwv_series = pwv_yearly_mean.set_index(ycol)[dcol]

else:  # xarray.DataArray
    pwv_series = pwv_yearly_mean.to_series()
    if isinstance(pwv_series.index, pd.MultiIndex):
        pwv_series = pwv_series.droplevel(
            [lvl for lvl in pwv_series.index.names if lvl != 'year']
        )
    pwv_series.index.name = 'Year'

# ensure integer year index
if not np.issubdtype(pwv_series.index.dtype, np.integer):
    pwv_series.index = pd.to_datetime(pwv_series.index).year

pwv_aligned = pwv_series.reindex(years)

# ────────────────────────────────────────────────────────────────────────────
# Plotting configuration (matches your reference style)
# ────────────────────────────────────────────────────────────────────────────
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif":  ["Times New Roman"],
    "axes.labelsize": 28,
    "xtick.labelsize": 28,
    "ytick.labelsize": 28,
    "legend.fontsize": 26,
})

# ────────────────────────────────────────────────────────────────────────────
# Create figure: SINGLE PANEL ONLY
# ────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))

# ── Hanle radiometer ───────────────────────────────────────────────────────
ax.plot(
    years, hanle_series,
    label='Hanle IAO',
    color='#00429d',
    marker='o',
    linestyle='solid',
    linewidth=3.5,
    markersize=12,
    markerfacecolor='white',
    markeredgewidth=1.5,
    markeredgecolor='black',
    alpha=0.8,
    zorder=5
)

# ── ERA5: Hanle Pixel (same as before) ─────────────────────────────────────
ax.plot(
    years, pwv_aligned,
    label='ERA5 : Hanle Pixel',
    color='#009b7a',
    marker='^',
    linestyle='--',
    linewidth=3.5,
    markersize=12,
    markerfacecolor='#009b7a',
    markeredgewidth=1.5,
    markeredgecolor='black',
    alpha=0.8,
    zorder=8
)

# ── ERA5: IAO Hanle (same grouping call you had) ───────────────────────────
ax.plot(
    years,
    pwv_obs.groupby('time.year').median(dim='time').values,
    label='ERA5 : IAO Hanle',
    color='orange',
    marker='s',
    linestyle='-.',
    linewidth=3.5,
    markersize=12,
    markerfacecolor='orange',
    markeredgecolor='black',
    markeredgewidth=2.5,
    zorder=6
)

ax.set_ylim(1.2, 3.3)

# X axis limits & ticks exactly like before
ax.set_xlim(years.min() - 0.5, years.max() + 0.5)
ax.set_xticks(np.arange(years.min(), years.max() + 1, 2))
ax.set_xticklabels(np.arange(years.min(), years.max() + 1, 2), rotation=0)

# Axis labels
ax.set_xlabel(r'Year')
ax.set_ylabel(r'Precipitable Water Vapor (mm)')

# Legend
ax.legend(
    loc='upper center', frameon=True, facecolor='white',
    edgecolor='black', framealpha=0.95, ncol=3,
    fontsize=22
)

# Ticks / spines
ax.set_facecolor('white')
ax.minorticks_on()
ax.tick_params(axis='both', which='major', direction='in',
               width=2, length=10, top=True, right=True)
ax.tick_params(axis='both', which='minor', direction='in',
               width=1, length=5, top=True, right=True)

for spine in ax.spines.values():
    spine.set_linewidth(2.5)

# ────────────────────────────────────────────────────────────────────────────
# Layout & export
# ────────────────────────────────────────────────────────────────────────────
fig.tight_layout()
outfile = 'Hanle_PWV_Yearly_Comparison_TopOnly.pdf'
plt.savefig(outfile, dpi=600, bbox_inches='tight')
print(f"\n→ Plot successfully saved as {outfile}")
plt.show()


## Summary statistics and reporting

### Step 156

This cell defines reusable helper function(s) `_to_np`, `weighted_stats`, `weighted_corr`, `gaussian_loglik` so later sections can apply the same processing logic consistently.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# YEARLY METRICS ANALYSIS (paper-ready)
# ────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy import stats

# -------------------------------
# 0) Helpers (reused from monthly)
# -------------------------------
def _to_np(x):
    return np.asarray(x, dtype=float)

def weighted_stats(err, w=None):
    e = _to_np(err)
    if w is None:
        mae  = np.mean(np.abs(e))
        rmse = np.sqrt(np.mean(e**2))
        bias = np.mean(e)
    else:
        w = _to_np(w)
        mae  = np.sum(w*np.abs(e)) / np.sum(w)
        rmse = np.sqrt(np.sum(w*e**2) / np.sum(w))
        bias = np.sum(w*e) / np.sum(w)
    return {"Bias": bias, "MAE": mae, "RMSE": rmse}

def weighted_corr(x, y, w=None):
    x = _to_np(x); y = _to_np(y)
    if w is None:
        return np.corrcoef(x, y)[0,1]
    w = _to_np(w)
    mx = np.sum(w*x)/np.sum(w)
    my = np.sum(w*y)/np.sum(w)
    cov = np.sum(w*(x-mx)*(y-my))/np.sum(w)
    vx  = np.sum(w*(x-mx)**2)/np.sum(w)
    vy  = np.sum(w*(y-my)**2)/np.sum(w)
    return cov/np.sqrt(vx*vy)

def gaussian_loglik(O, M, sigma):
    O = _to_np(O); M = _to_np(M); s = _to_np(sigma)
    return -0.5*np.sum(((O-M)**2)/(s**2) + np.log(2*np.pi*s**2))

def ols_slope_intercept(x, y):
    # fits y = a + b x  (ordinary least squares)
    x = _to_np(x); y = _to_np(y)
    X = np.vstack([np.ones_like(x), x]).T
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    a, b = beta[0], beta[1]
    return b, a

def linear_trend(years, series):
    """OLS trend in mm/year (+ intercept, r, p, stderr)."""
    years = _to_np(years); y = _to_np(series)
    slope, intercept, r, p, stderr = stats.linregress(years, y)
    return {"trend_mm_per_year": slope, "trend_mm_per_decade": 10*slope,
            "intercept": intercept, "r_trend": r, "p_trend": p, "stderr": stderr}

def dm_simple(obs, m1, m2, loss="abs"):
    """Small-sample Diebold–Mariano proxy (two-sided t-test on loss diff)."""
    e1 = _to_np(m1) - _to_np(obs)
    e2 = _to_np(m2) - _to_np(obs)
    d  = (np.abs(e1)-np.abs(e2)) if loss=="abs" else ((e1**2)-(e2**2))
    t, p = stats.ttest_1samp(d, 0.0)
    return {"mean_loss_diff": d.mean(), "t": t, "p": p, "N": len(d), "loss": loss}

# --------------------------------------
# 1) Harmonise inputs to yearly Series
# --------------------------------------
# Hanle (observations)
hanle_df = hanle_yearly_iao.copy()
hanle_df['Year'] = hanle_df['Year'].astype(int)
hanle_series = hanle_df.set_index('Year')['Mean'].astype(float)

# Try to find an uncertainty column (optional)
err_col = next((c for c in hanle_df.columns
                if c.lower() in ("sigma","std","se","error","err")), None)
sigma_year = hanle_df.set_index('Year')[err_col].astype(float) if err_col else None

# ERA5 (Hanle pixel): accept Series / DataFrame / xarray
def to_year_series(obj):
    if isinstance(obj, pd.Series):
        s = obj.copy()
        if not np.issubdtype(s.index.dtype, np.integer):
            s.index = pd.to_datetime(s.index).year
        s.index = s.index.astype(int)
        return s.astype(float)

    if isinstance(obj, pd.DataFrame):
        ycol = next((c for c in obj.columns if c.lower() == 'year'), None)
        if ycol is None:
            # assume first column is year if named poorly
            ycol = obj.columns[0]
        dcol = next(c for c in obj.columns if c != ycol)
        s = obj.set_index(ycol)[dcol]
        s.index = s.index.astype(int)
        return s.astype(float)

    # assume xarray.DataArray
    da = obj
    try:
        # if already yearly, just convert
        s = da.to_series()
    except Exception:
        # fallback: try to pull 'year' coord
        years = da['year'].values if 'year' in da.coords else pd.to_datetime(da['time'].values).year
        s = pd.Series(da.values, index=years)
    if isinstance(s.index, pd.MultiIndex):
        # drop levels except 'year'
        keep = [lvl for lvl in s.index.names if (lvl is not None and 'year' in lvl)]
        s = s.droplevel([lvl for lvl in s.index.names if lvl not in keep])
    if not np.issubdtype(s.index.dtype, np.integer):
        s.index = pd.to_datetime(s.index).year
    s.index = s.index.astype(int)
    return s.astype(float)

era5_pixel_series = to_year_series(pwv_yearly_mean)

# Optional: ERA5 “IAO Hanle” (interpolated) from xarray pwv_obs
era5_interp_series = None
try:
    era5_interp_da = pwv_obs.groupby('time.year').median(dim='time')
    # robust conversion to Series with integer year index
    try:
        era5_interp_series = era5_interp_da.to_series()
    except Exception:
        years_interp = era5_interp_da['year'].values
        era5_interp_series = pd.Series(era5_interp_da.values, index=years_interp)
    if not np.issubdtype(era5_interp_series.index.dtype, np.integer):
        era5_interp_series.index = pd.Index(era5_interp_series.index, dtype=int)
    era5_interp_series = era5_interp_series.astype(float)
except Exception:
    pass  # not available; we’ll just compare pixel vs obs

# Align all available series on common years
series_list = [hanle_series, era5_pixel_series]
if era5_interp_series is not None:
    series_list.append(era5_interp_series)

common_years = series_list[0].index
for s in series_list[1:]:
    common_years = common_years.intersection(s.index)

hanle_y   = hanle_series.reindex(common_years)
era5_pix  = era5_pixel_series.reindex(common_years)
era5_int  = era5_interp_series.reindex(common_years) if era5_interp_series is not None else None
sigma_y   = sigma_year.reindex(common_years) if sigma_year is not None else None

# -------------------------------
# 2) Summaries for the paper
# -------------------------------
def summarize_pair_yearly(name, years, obs, obs_sigma, mod):
    err = mod - obs
    w = None if obs_sigma is None else 1.0/(obs_sigma**2)
    core = weighted_stats(err, w)
    r    = weighted_corr(obs, mod, w)
    rho  = pd.Series(obs, index=years).corr(pd.Series(mod, index=years), method='spearman')
    slope, intercept = ols_slope_intercept(mod, obs)  # obs = a + b*mod
    ll = None if obs_sigma is None else gaussian_loglik(obs, mod, obs_sigma)

    # Trends (mm/decade) for interpretability
    tr_obs = linear_trend(years, obs)
    tr_mod = linear_trend(years, mod)

    return {
        "Pair": name,
        "N_years": len(years),
        "Bias (mm)": core["Bias"],
        "MAE (mm)": core["MAE"],
        "RMSE (mm)": core["RMSE"],
        "Pearson r": r,
        "Spearman ρ": rho,
        "OLS slope (obs~mod)": slope,
        "OLS intercept (mm)": intercept,
        "Trend obs (mm/decade)": tr_obs["trend_mm_per_decade"],
        "Trend mod (mm/decade)": tr_mod["trend_mm_per_decade"],
        "LogLik (Gaussian)": ll
    }

rows = []
rows.append(summarize_pair_yearly("Hanle vs ERA5 Pixel (Yearly)", common_years.values, hanle_y.values, sigma_y.values if sigma_y is not None else None, era5_pix.values))

if era5_int is not None:
    rows.append(summarize_pair_yearly("Hanle vs ERA5 Interp (Yearly)", common_years.values, hanle_y.values, sigma_y.values if sigma_y is not None else None, era5_int.values))

stats_yearly = pd.DataFrame(rows)

# Optional: is Interp significantly better than Pixel?
dm_rows = []
if era5_int is not None:
    dm_rows.append({"Pair":"DM abs (Interp better if mean<0)", **dm_simple(hanle_y.values, era5_pix.values, era5_int.values, loss="abs")})
    dm_rows.append({"Pair":"DM sq  (Interp better if mean<0)", **dm_simple(hanle_y.values, era5_pix.values, era5_int.values, loss="sq")})
    dm_table = pd.DataFrame(dm_rows)
else:
    dm_table = pd.DataFrame([])

# -------------------------------
# 3) Pretty print & export
# -------------------------------
display_cols = ["Pair","N_years","Bias (mm)","MAE (mm)","RMSE (mm)","Pearson r","Spearman ρ",
                "OLS slope (obs~mod)","OLS intercept (mm)",
                "Trend obs (mm/decade)","Trend mod (mm/decade)","LogLik (Gaussian)"]

print("\n=== Yearly PWV Metrics (Hanle vs ERA5) ===")
print(stats_yearly[display_cols].round({
    "Bias (mm)":3, "MAE (mm)":3, "RMSE (mm)":3,
    "Pearson r":3, "Spearman ρ":3, "OLS slope (obs~mod)":3,
    "OLS intercept (mm)":3, "Trend obs (mm/decade)":3,
    "Trend mod (mm/decade)":3
}))

if not dm_table.empty:
    print("\n=== Diebold–Mariano comparison (ERA5 Pixel vs Interp) ===")
    print(dm_table.round(4))

stats_yearly.to_csv("Hanle_PWV_yearly_stats.csv", index=False)
with open("Hanle_PWV_yearly_stats.tex","w") as f:
    f.write(stats_yearly[display_cols].to_latex(index=False, float_format="%.3f"))


### Step 157

This cell computes or evaluates precipitable water vapor (PWV)-related quantities that are central to the site-quality analysis.

In [ ]:
pwv_obs.groupby('time.year').median(dim='time').values - pwv_aligned


### Step 158

This cell computes or evaluates precipitable water vapor (PWV)-related quantities that are central to the site-quality analysis.

In [ ]:
pwv_obs.groupby('time.year').median(dim='time').values - pwv_series


## Pressure and PWV calculations

### Step 159

This cell computes or evaluates precipitable water vapor (PWV)-related quantities that are central to the site-quality analysis.

In [ ]:
pwv_aligned - pwv_series


## Summary statistics and reporting

### Step 160

This cell computes or evaluates precipitable water vapor (PWV)-related quantities that are central to the site-quality analysis.

In [ ]:
pwv_obs.groupby('time.year').median(dim='time').values


#### Combined Both the methods

## Mapping and visualization

### Step 161

This cell defines reusable helper function(s) `harmonize_yearly_pwv`, `calculate_mbe`, `calculate_rmse`, `calculate_pearson_r` so later sections can apply the same processing logic consistently.

In [ ]:
# -*- coding: utf-8 -*-
"""
Full script to calculate and compare yearly MEDIAN ERA5 PWV from three 
different methods against observational data, using a robust input
harmonization logic. Includes a residual plot and statistical analysis.

This script ASSUMES the following variables already exist in your environment:
- hanle_yearly_iao: 
    A pandas DataFrame with yearly observational PWV data.
    Required columns: 'Year', 'Mean' (note: this is the yearly mean of daily observations).

- pwv_method1, pwv_method2, pwv_method3: 
    xarray DataArrays containing the full time-series PWV calculations 
    from the three different methods.
"""

# 1. SETUP AND PREREQUISITES
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

print("--- Script Starting: Yearly Median PWV Comparison, Residuals, and Analysis ---")
print("Assuming 'hanle_yearly_iao', 'pwv_method1', 'pwv_method2', and 'pwv_method3' are pre-loaded.")


# 2. DATA PREPARATION AND HARMONIZATION
# =============================================================================

def harmonize_yearly_pwv(pwv_input, target_years_index):
    """
    Takes a yearly PWV input (Series, DataFrame, or xarray.DataArray)
    and returns a pandas Series aligned to the target years, using the
    robust logic from the reference script.
    """
    if isinstance(pwv_input, pd.Series):
        pwv_series = pwv_input.copy()
    elif isinstance(pwv_input, pd.DataFrame):
        ycol = [c for c in pwv_input.columns if 'year' in str(c).lower()][0]
        dcol = [c for c in pwv_input.columns if c != ycol][0]
        pwv_series = pwv_input.set_index(ycol)[dcol]
    else:  # Assumes xarray.DataArray
        pwv_series = pwv_input.to_series()
        if isinstance(pwv_series.index, pd.MultiIndex):
            pwv_series = pwv_series.droplevel([lvl for lvl in pwv_series.index.names if lvl.lower() != 'year'])
        pwv_series.index.name = 'Year'
    if not np.issubdtype(pwv_series.index.dtype, np.integer):
        try:
            pwv_series.index = pd.to_datetime(pwv_series.index).year
        except (TypeError, ValueError):
            pwv_series.index = pwv_series.index.astype(int)
    pwv_series.index.name = 'Year'
    return pwv_series.reindex(target_years_index)

# --- CORRECTED: Calculate Yearly MEDIANS for each model ---
print("\nCalculating yearly medians for all three models...")
pwv_yearly_m1 = pwv_method1.groupby('time.year').median(dim='time')
pwv_yearly_m2 = pwv_method2.groupby('time.year').median(dim='time')
pwv_yearly_m3 = pwv_method3.groupby('time.year').median(dim='time')
print("✓ Yearly medians calculated.")

# --- Harmonize observational data ---
hanle_df = hanle_yearly_iao.copy()
hanle_df['Year'] = hanle_df['Year'].astype(int)
hanle_series = hanle_df.set_index('Year')['Mean']
years = hanle_series.index

# --- Apply harmonization to all three models ---
print("Aligning all model data to the observational years...")
model1_aligned = harmonize_yearly_pwv(pwv_yearly_m1, years)
model2_aligned = harmonize_yearly_pwv(pwv_yearly_m2, years)
model3_aligned = harmonize_yearly_pwv(pwv_yearly_m3, years)
print("✓ Alignment complete.")

# --- Calculate Residuals (Model - Observation) ---
residual1 = model1_aligned - hanle_series
residual2 = model2_aligned - hanle_series
residual3 = model3_aligned - hanle_series


# 3. STATISTICAL ANALYSIS
# =============================================================================
print("\n--- Performing Statistical Analysis on Yearly Median Data ---")

def calculate_mbe(model, obs): return (model - obs).mean()
def calculate_rmse(model, obs): return np.sqrt(((model - obs) ** 2).mean())
def calculate_pearson_r(model, obs): return model.corr(obs)

stats = {}
models_yearly = {
    'ERA5 (model sp)': model1_aligned,
    'ERA5 (elev. + t2m)': model2_aligned,
    'ERA5 (tcwv)': model3_aligned
}
for name, model_data in models_yearly.items():
    stats[name] = {
        'MBE': calculate_mbe(model_data, hanle_series),
        'RMSE': calculate_rmse(model_data, hanle_series),
        'Pearson r': calculate_pearson_r(model_data, hanle_series)
    }

print("\nYearly Statistical Comparison (Median) with Observations:")
print("-" * 70)
print(f"{'Model':<25} | {'Mean Bias Error (MBE)':<22} | {'RMSE':<10} | {'Pearson r':<10}")
print("-" * 70)
for name, metrics in stats.items():
    print(f"{name:<25} | {metrics['MBE']:<+22.4f} | {metrics['RMSE']:<10.4f} | {metrics['Pearson r']:<10.4f}")
print("-" * 70)
best_model_rmse = min(stats, key=lambda x: stats[x]['RMSE'])
print(f"\nConclusion: Based on yearly RMSE, the best performing model is '{best_model_rmse}'.")


# 4. PLOTTING
# =============================================================================
print("\n--- Generating Publication-Quality Yearly Comparison Plot ---")

PUBLISHABLE_COLORS = ['#00429d', '#009b7a', '#ff7c00', '#6a0dad', '#93003a']
PUBLISHABLE_MARKERS = ['o', 's', 'D', 'p', 'X']
mpl.rcParams.update({
    "text.usetex": True, "font.family": "serif", "font.serif": ["Times New Roman"],
    "axes.labelsize": 28, "xtick.labelsize": 28, "ytick.labelsize": 28, "legend.fontsize": 22
})

fig, (ax_top, ax_bot) = plt.subplots(nrows=2, sharex=True, figsize=(14, 12), gridspec_kw={"hspace": 0.05, "height_ratios": [2, 1]})

# --- Top Panel: Time Series Comparison ---
ax_top.plot(years, hanle_series, label='Hanle IAO', color=PUBLISHABLE_COLORS[0], marker=PUBLISHABLE_MARKERS[0], markersize=12, markerfacecolor='white', markeredgewidth=2.5, linewidth=3, alpha=0.9, zorder=10)
ax_top.plot(years, model1_aligned, label='ERA5 (model sp)', color=PUBLISHABLE_COLORS[1], marker=PUBLISHABLE_MARKERS[1], linestyle='--', linewidth=3, markersize=11, alpha=0.8, zorder=8)
ax_top.plot(years, model2_aligned, label='ERA5 (elev. + t2m)', color=PUBLISHABLE_COLORS[2], marker=PUBLISHABLE_MARKERS[2], linestyle=':', linewidth=3, markersize=11, alpha=0.8, zorder=7)
ax_top.plot(years, model3_aligned, label='ERA5 (tcwv)', color=PUBLISHABLE_COLORS[3], marker=PUBLISHABLE_MARKERS[3], linestyle='-.', linewidth=3, markersize=11, alpha=0.8, zorder=6)
ax_top.set_ylabel(r'Yearly Median PWV (mm)') # Corrected label

# --- Bottom Panel: Residuals ---
ax_bot.axhline(0, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
ax_bot.plot(years, residual1, label='Resid (model sp)', color=PUBLISHABLE_COLORS[1], marker=PUBLISHABLE_MARKERS[1], linestyle='--', linewidth=2.5, markersize=10, alpha=0.8)
ax_bot.plot(years, residual2, label='Resid (elev. + t2m)', color=PUBLISHABLE_COLORS[2], marker=PUBLISHABLE_MARKERS[2], linestyle=':', linewidth=2.5, markersize=10, alpha=0.8)
ax_bot.plot(years, residual3, label='Resid (tcwv)', color=PUBLISHABLE_COLORS[3], marker=PUBLISHABLE_MARKERS[3], linestyle='-.', linewidth=2.5, markersize=10, alpha=0.8)
ax_bot.set_ylabel(r'Residual (mm)')

# --- Cosmetics ---
fig.suptitle('Yearly Median PWV Comparison: Hanle IAO vs. ERA5 Methods', fontsize=32, weight='bold') # Corrected title
ax_bot.set_xlabel(r'Year', fontsize=28)
ax_top.legend(loc='upper center', bbox_to_anchor=(0.5, 1.0), ncol=4, frameon=True, facecolor='white', edgecolor='black', framealpha=0.95)
ax_bot.legend(loc='upper right', ncol=3, frameon=True, facecolor='white', edgecolor='black', framealpha=0.95)
for ax in (ax_top, ax_bot):
    ax.set_facecolor('white'); ax.minorticks_on()
    ax.grid(True, which='major', linestyle='--', linewidth=0.5, alpha=0.7)
    ax.tick_params(axis='both', which='major', direction='in', width=2, length=10, top=True, right=True)
    ax.tick_params(axis='both', which='minor', direction='in', width=1, length=5, top=True, right=True)
    for spine in ax.spines.values(): spine.set_linewidth(2.5)
ax_bot.set_xlim(years.min() - 0.5, years.max() + 0.5)
ax_bot.set_xticks(np.arange(years.min(), years.max() + 2, 2))
ax_bot.tick_params(axis='x', rotation=0)

# --- Layout & Export ---
fig.tight_layout(rect=[0.05, 0.03, 0.98, 0.95])
outfile = 'Hanle_PWV_Yearly_Median_Comparison_3-Method_Publication.pdf'
plt.savefig(outfile, dpi=600, bbox_inches='tight')
print(f"\n→ Plot successfully saved as {outfile}")
plt.show()


### Step 162

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
# -*- coding: utf-8 -*-
"""
Full script for yearly PWV comparison using the pre-existing 'pwv_corr'
variable and calculating the yearly aggregate with MEDIAN to reduce offsets.

This script ASSUMES the following variables already exist in your environment:
- hanle_yearly_iao: 
    A pandas DataFrame with yearly observational PWV data.
    Required columns: 'Year', 'Mean'.

- pwv_corr: 
    An xarray DataArray containing the full time-series PWV calculations
    for the single best model you wish to analyze.
"""

# 1. SETUP AND PREREQUISITES
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

print("--- Script Starting: Yearly PWV Comparison using Median ---")
print("Assuming 'hanle_yearly_iao' and 'pwv_corr' are pre-loaded.")


# 2. DATA PREPARATION AND HARMONIZATION
# =============================================================================

# --- Calculate Yearly MEDIAN for the model data ---
# Using .median() is key to reducing the influence of outliers and the offset.
print("\nCalculating yearly median for the model data...")
pwv_yearly_median = pwv_corr.groupby('time.year').median(dim='time')
print("✓ Yearly median calculated.")

# --- Harmonize observational data ---
hanle_df = hanle_yearly_iao.copy()
hanle_df['Year'] = hanle_df['Year'].astype(int)
hanle_series = hanle_df.set_index('Year')['Mean']
years = hanle_series.index  # This is the target index

# --- Harmonize model data using the robust logic from your reference ---
print("Harmonizing model data to the observational years...")
# This logic accepts Series, DataFrame, or xarray.DataArray as input
if isinstance(pwv_yearly_median, pd.Series):
    pwv_series = pwv_yearly_median.copy()
elif isinstance(pwv_yearly_median, pd.DataFrame):
    ycol = [c for c in pwv_yearly_median.columns if 'year' in str(c).lower()][0]
    dcol = [c for c in pwv_yearly_median.columns if c != ycol][0]
    pwv_series = pwv_yearly_median.set_index(ycol)[dcol]
else:  # Assumes xarray.DataArray
    pwv_series = pwv_yearly_median.to_series()
    if isinstance(pwv_series.index, pd.MultiIndex):
        pwv_series = pwv_series.droplevel(
            [lvl for lvl in pwv_series.index.names if lvl.lower() != 'year']
        )
    pwv_series.index.name = 'Year'

# Ensure the index is an integer year
if not np.issubdtype(pwv_series.index.dtype, np.integer):
    pwv_series.index = pd.to_datetime(pwv_series.index).year

# Reindex to match the observational data years
pwv_aligned = pwv_series.reindex(years)
print("✓ Alignment complete.")

# --- Calculate Residuals (Model - Observation) ---
era5_residual = pwv_aligned - hanle_series


# 3. STATISTICAL ANALYSIS
# =============================================================================
print("\n--- Performing Statistical Analysis on Yearly Median Data ---")

mbe = (pwv_aligned - hanle_series).mean()
rmse = np.sqrt(((pwv_aligned - hanle_series) ** 2).mean())
pearson_r = pwv_aligned.corr(hanle_series)

print("\nYearly Statistical Comparison (Median) with Observations:")
print("-" * 60)
print(f"{'Metric':<25} | {'Value'}")
print("-" * 60)
print(f"{'Mean Bias Error (MBE)':<25} | {mbe:<+0.4f}")
print(f"{'Root Mean Square Error (RMSE)':<25} | {rmse:<0.4f}")
print(f"{'Pearson Correlation (r)':<25} | {pearson_r:<0.4f}")
print("-" * 60)


# 4. PLOTTING
# =============================================================================
print("\n--- Generating Publication-Quality Yearly Comparison Plot ---")

# --- Plotting Configuration ---
PUBLISHABLE_COLORS = ['#00429d', '#009b7a', '#93003a']
PUBLISHABLE_MARKERS = ['o', 's', 'X']

mpl.rcParams.update({
    "text.usetex": True, "font.family": "serif", "font.serif": ["Times New Roman"],
    "axes.labelsize": 28, "xtick.labelsize": 28, "ytick.labelsize": 28,
    "legend.fontsize": 26,
})

# --- Create Figure ---
fig, (ax_top, ax_bot) = plt.subplots(
    nrows=2, sharex=True, figsize=(14, 11),
    gridspec_kw={"hspace": 0.05}
)

# --- Top Panel: Hanle vs ERA5 ---
ax_top.plot(
    years, hanle_series, label='Hanle IAO', color=PUBLISHABLE_COLORS[0],
    marker=PUBLISHABLE_MARKERS[0], markersize=12, markerfacecolor='white',
    markeredgewidth=2.5, linewidth=3, alpha=0.9, zorder=10
)
ax_top.plot(
    years, pwv_aligned, label='ERA5 (Yearly Median)', color=PUBLISHABLE_COLORS[1],
    marker=PUBLISHABLE_MARKERS[1], linestyle='--', linewidth=3.5, markersize=12,
    markerfacecolor=PUBLISHABLE_COLORS[1], markeredgewidth=1.5, markeredgecolor='black',
    alpha=0.8, zorder=8
)
ax_top.set_ylabel(r'Yearly Median PWV (mm)')

# --- Bottom Panel: Residuals ---
ax_bot.axhline(0, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
ax_bot.plot(
    years, era5_residual, label='ERA5 $-$ Hanle', color=PUBLISHABLE_COLORS[2],
    marker=PUBLISHABLE_MARKERS[2], linestyle=':', linewidth=3.5, markersize=11,
    markerfacecolor='white', markeredgewidth=2.5, alpha=0.9, zorder=10
)
ax_bot.set_ylabel(r'Residual (mm)')

# --- Cosmetics ---
fig.text(0.53, 0.02, r'Year', fontsize=28, ha='center', va='center')

ax_top.legend(loc='upper right', frameon=True, facecolor='white', edgecolor='black', framealpha=0.95)
ax_bot.legend(loc='upper right', frameon=True, facecolor='white', edgecolor='black', framealpha=0.95)

for ax in (ax_top, ax_bot):
    ax.set_facecolor('white')
    ax.minorticks_on()
    ax.grid(True, which='major', linestyle='--', linewidth=0.5, alpha=0.7)
    ax.tick_params(axis='both', which='major', direction='in', width=2, length=10, top=True, right=True)
    ax.tick_params(axis='both', which='minor', direction='in', width=1, length=5, top=True, right=True)
    for spine in ax.spines.values():
        spine.set_linewidth(2.5)

ax_bot.set_xlim(years.min() - 0.5, years.max() + 0.5)
ax_bot.set_xticks(np.arange(years.min(), years.max() + 2, 2))
ax_bot.tick_params(axis='x', rotation=0)

# --- Layout & Export ---
fig.tight_layout(rect=[0.05, 0.05, 0.98, 0.98])
outfile = 'Hanle_PWV_Yearly_Median_Comparison_Publication.pdf'
plt.savefig(outfile, dpi=600, bbox_inches='tight')
print(f"\n→ Plot successfully saved as {outfile}")
plt.show()


### Combined

## Mapping and visualization

### Step 163

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# ============================================================================
# PART 1: PREREQUISITES AND DATA PREPARATION
# ============================================================================

# Prerequisites already in memory
# --------------------------------------------------
# • hanle_yearly_iao : DataFrame with columns ['Year', 'Mean']
# • pwv_yearly_mean  : Series / DataFrame / xarray DataArray
# • hanle_monthly_iao: DataFrame with observed Day/Night PWV data.
# • pwv_monthly: An xarray DataArray with the monthly mean model PWV data.

# ────────────────────────────────────────────────────────────────────────────
# Data Prep for Yearly Plot (from first script)
# ────────────────────────────────────────────────────────────────────────────
# Hanle
hanle_df            = hanle_yearly_iao.copy()
hanle_df['Year']    = hanle_df['Year'].astype(int)
hanle_series        = hanle_df.set_index('Year')['Mean']
years               = hanle_series.index

# ERA5 (accepts Series / DataFrame / xarray)
if isinstance(pwv_yearly_mean, pd.Series):
    pwv_series = pwv_yearly_mean.copy()
elif isinstance(pwv_yearly_mean, pd.DataFrame):
    ycol  = [c for c in pwv_yearly_mean.columns if c.lower() == 'year'][0]
    dcol  = [c for c in pwv_yearly_mean.columns if c != ycol][0]
    pwv_series = pwv_yearly_mean.set_index(ycol)[dcol]
else:  # xarray.DataArray
    pwv_series = pwv_yearly_mean.to_series()
    if isinstance(pwv_series.index, pd.MultiIndex):
        pwv_series = pwv_series.droplevel(
            [lvl for lvl in pwv_series.index.names if lvl != 'year']
        )
    pwv_series.index.name = 'Year'

# ensure integer year index
if not np.issubdtype(pwv_series.index.dtype, np.integer):
    pwv_series.index = pd.to_datetime(pwv_series.index).year
pwv_aligned   = pwv_series.reindex(years)


# ────────────────────────────────────────────────────────────────────────────
# Data Prep for Monthly Plot (from second script)
# ────────────────────────────────────────────────────────────────────────────
if 'MonthNum' not in hanle_monthly_iao.columns:
    hanle_monthly_iao['MonthNum'] = hanle_monthly_iao.index + 1
print("Created numerical 'MonthNum' column for accurate plotting.")


# ============================================================================
# PART 2: PLOTTING
# ============================================================================

# ────────────────────────────────────────────────────────────────────────────
# Plotting configuration (unified from both scripts)
# ────────────────────────────────────────────────────────────────────────────
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif":  ["Times New Roman"],
    "axes.labelsize": 28,
    "xtick.labelsize": 28,
    "ytick.labelsize": 28,
    "legend.fontsize": 26,
})

# Define colors and markers for both plots
YEARLY_COLORS = ['#00429d', '#009b7a']  # Hanle, ERA5
YEARLY_MARKERS = ['o', 's']
MONTHLY_COLORS = ['#00429d', '#93003a', '#009b7a'] # Day, Night, Model
MONTHLY_MARKERS = ['o', 's', '^']

# ────────────────────────────────────────────────────────────────────────────
# Create figure with two vertically stacked subplots
# ────────────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(
    nrows=2, ncols=1, figsize=(14, 20)
)
print("Generating combined publication-quality plot...")

# ────────────────────────────────────────────────────────────────────────────
# TOP SUBPLOT: Monthly Comparison (from second script)
# ────────────────────────────────────────────────────────────────────────────
# --- Plot IAO Observational Data (Day) ---
ax1.errorbar(
    hanle_monthly_iao['MonthNum'],
    hanle_monthly_iao['Day'], 
    yerr=hanle_monthly_iao['Day_Error'],
    label='Hanle IAO (Day)',
    fmt='-',
    color=MONTHLY_COLORS[0],
    marker=MONTHLY_MARKERS[0],
    markersize=12,
    markerfacecolor='white',
    markeredgewidth=2.5,
    linewidth=3,
    capsize=6,
    elinewidth=2,
    alpha=0.9,
    zorder=10
)
# --- Plot IAO Observational Data (Night) ---
ax1.errorbar(
    hanle_monthly_iao['MonthNum'],
    hanle_monthly_iao['Night'], 
    yerr=hanle_monthly_iao['Night_Error'],
    label='Hanle IAO (Night)',
    fmt='-',
    color=MONTHLY_COLORS[1],
    marker=MONTHLY_MARKERS[1],
    markersize=12,
    markerfacecolor='white',
    markeredgewidth=2.5,
    linewidth=3,
    capsize=6,
    elinewidth=2,
    alpha=0.9,
    zorder=10
)
# --- Plot Model Data (ERA5) ---
ax1.plot(
    pwv_monthly.month, pwv_monthly,
    label='ERA5 ',
    color=MONTHLY_COLORS[2],
    marker=MONTHLY_MARKERS[2],
    linestyle='--',
    linewidth=3.5,
    markersize=12,
    markerfacecolor=MONTHLY_COLORS[2],
    markeredgewidth=1.5,
    markeredgecolor='black',
    alpha=0.8,
    zorder=5
)
# --- Cosmetics for Monthly Plot ---
ax1.set_ylabel(r'Precipitable Water Vapor (mm)', fontsize=28)
textstr = '1998-2017'
props = dict(boxstyle='round,pad=0.5', facecolor='wheat', alpha=0.8, edgecolor='black', linewidth=1.5)
ax1.text(0.17, 0.87, textstr, 
         transform=ax1.transAxes,
         fontsize=28,
            verticalalignment='bottom',
            horizontalalignment='right',
            bbox=props,
            multialignment='center'
        )
ax1.legend(loc='upper right', fontsize=26, frameon=True, facecolor='white', edgecolor='black', framealpha=0.95)
ax1.set_xlim(0.5, 12.5)
ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], rotation=45, ha='right', fontsize=26)
ax1.minorticks_on()
ax1.set_yticks([1,2,3,4,5,6,7,8,9])
ax1.tick_params(axis='both', which='major', direction='in', width=2, length=10, labelsize=28, top=True, right=True)
ax1.tick_params(axis='both', which='minor', direction='in', width=1, length=5, top=True, right=True)
ax1.set_facecolor('white')
for spine in ax1.spines.values():
    spine.set_linewidth(2.5)

# ────────────────────────────────────────────────────────────────────────────
# BOTTOM SUBPLOT: Yearly Comparison (from first script)
# ────────────────────────────────────────────────────────────────────────────
ax2.plot(
    years, hanle_series,
    label='Hanle IAO',
    color=YEARLY_COLORS[0],
    marker=YEARLY_MARKERS[0],
    markersize=12,
    markerfacecolor='white',
    markeredgewidth=2.5,
    linewidth=3,
    alpha=0.9,
    zorder=10
)
ax2.plot(
    years, pwv_aligned,
    label='ERA5',
    color=YEARLY_COLORS[1],
    marker=YEARLY_MARKERS[1],
    linestyle='--',
    linewidth=3.5,
    markersize=12,
    markerfacecolor=YEARLY_COLORS[1],
    markeredgewidth=1.5,
    markeredgecolor='black',
    alpha=0.8,
    zorder=8
)
# --- Cosmetics for Yearly Plot ---
ax2.set_xlabel(r'Year', fontsize=28)
ax2.set_ylabel(r'Precipitable Water Vapor (mm)', fontsize=28)
ax2.set_ylim(1.2, 2.7)
ax2.legend(loc='upper center', frameon=True, facecolor='white', edgecolor='black', framealpha=0.95, ncol=2, fontsize=26)
ax2.set_facecolor('white')
ax2.minorticks_on()
ax2.tick_params(axis='both', which='major', direction='in', width=2, length=10, top=True, right=True)
ax2.tick_params(axis='both', which='minor', direction='in', width=1, length=5, top=True, right=True)
for spine in ax2.spines.values():
    spine.set_linewidth(2.5)
ax2.set_xlim(years.min() - 0.5, years.max() + 0.5)
ax2.set_xticks(np.arange(years.min(), years.max() + 1, 2))
ax2.set_xticklabels(np.arange(years.min(), years.max() + 1, 2), rotation=0)


# ────────────────────────────────────────────────────────────────────────────
# Final Layout & Export
# ────────────────────────────────────────────────────────────────────────────
fig.tight_layout()
output_filename = 'Combined_PWV_Comparison_Publication.pdf'
plt.savefig(output_filename, dpi=600, bbox_inches='tight')
print(f"\n→ Combined plot successfully saved as {output_filename}")
plt.show()


## Site definitions and metadata

### Step 164

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
HANLE


## Data loading and inspection

### Step 165

This cell loads dataset(s) `data_0.nc`, `data_1.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
# read the files in new_data folder

new_data_0= xr.open_dataset('new_data/data_0.nc')
new_data_1= xr.open_dataset('new_data/data_1.nc')


## Analysis workflow

### Step 166

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
new_data_0


### Step 167

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
new_data_1


## Pressure and PWV calculations

### Step 168

This cell defines reusable helper function(s) `calculate_pwv_pressure`, `calculate_pwv_vectorized`, `integrate_profile` so later sections can apply the same processing logic consistently.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.interpolate import PchipInterpolator
from scipy.integrate import trapezoid # Use the new recommended function
import time

# ===================================================================
# SETUP: Using the final, debugged versions of both functions
# ===================================================================

# Your calculate_pwv_pressure function (FINAL CORRECTION for DType)
def calculate_pwv_pressure(q, levels_imp, pressure_thresholds):
    rho_w = 1; g = 9.81
    if 'expver' in q.dims: q = q.sel(expver=1)
        
    pwv_values = [] # pwv_values will be a simple list of floats
    
    for time_step in q.time:
        pc = pressure_thresholds.sel(time=time_step, method='nearest').item()
        q_at_time = q.sel(time=time_step)
        levels_da = levels_imp * 100
        mask_da = levels_da <= pc
        q_at_time_masked = q_at_time.where(mask_da, drop=True)
        levels_at_time = q_at_time_masked.level.values * 100
        q_values_masked = q_at_time_masked.values
        if len(levels_at_time) < 2:
            pwv_values.append(np.nan); continue
        pchip = PchipInterpolator(q_at_time.level.values * 100, q_at_time.values, extrapolate=True)
        interpolated_q = pchip(pc)
        dp = np.diff(levels_at_time)
        q_avg_values = (q_values_masked[:-1] + q_values_masked[1:]) / 2
        integral = np.sum(q_avg_values * dp)
        term_surface_layer = interpolated_q * (pc - levels_at_time[-1])
        first_level_pwv = q_at_time.sel(level=levels_at_time[0] / 100, method='nearest').item() * levels_at_time[0]
        pwv_corr = (integral + term_surface_layer + first_level_pwv) / (rho_w * g)
        pwv_values.append(pwv_corr)

    # --- FIX IS HERE ---
    # Construct the final DataArray using the calculated values and the ORIGINAL time coordinate
    # from the input 'q' to guarantee the DType matches.
    return xr.DataArray(data=pwv_values, coords={"time": q.time}, dims=["time"])


# My calculate_pwv_vectorized function
def calculate_pwv_vectorized(q: xr.DataArray, p_sfc: xr.DataArray) -> xr.DataArray:
    g = 9.81
    p_levels_pa = q.level * 100
    def integrate_profile(q_profile, p_profile, sfc_p_value):
        finite_mask = np.isfinite(q_profile)
        if finite_mask.sum() < 2: return np.nan
        valid_mask = p_profile <= sfc_p_value
        if valid_mask.sum() < 2: return np.nan
        p, q_ = p_profile[valid_mask], q_profile[valid_mask]
        sort_idx = np.argsort(p)
        p, q_ = p[sort_idx], q_[sort_idx]
        interpolator = PchipInterpolator(p, q_, extrapolate=True)
        q_sfc = interpolator(sfc_p_value)
        p_full = np.append(p, sfc_p_value)
        q_full = np.append(q_, q_sfc)
        sort_idx_full = np.argsort(p_full)
        p_full, q_full = p_full[sort_idx_full], q_full[sort_idx_full]
        return trapezoid(q_full, p_full) / g
    pwv = xr.apply_ufunc(
        integrate_profile, q, p_levels_pa, p_sfc,
        input_core_dims=[['level'], ['level'], []],
        output_core_dims=[[]], vectorize=True, dask="parallelized", output_dtypes=[q.dtype]
    )
    pwv.attrs.update(units="mm", long_name="Precipitable Water Vapor")
    return pwv

# ===================================================================
# 1. DATA PREPARATION FOR FULL TIME SERIES
# ===================================================================
print("\n--- Preparing Full Time Series Data (1998-2017) ---")
lat_hanle = 32.7789
lon_hanle = 78.965

q_site = era5['q'].interp(latitude=lat_hanle, longitude=lon_hanle, method='linear')
sp_site = surface_pressure_data['sp'].interp(latitude=lat_hanle, longitude=lon_hanle, method='linear')

if 'expver' in q_site.dims:
    q_site = q_site.sel(expver=1, drop=True)
if 'expver' in sp_site.dims:
    sp_site = sp_site.sel(expver=1, drop=True)

q_full, sp_full = xr.align(q_site, sp_site, join='inner')
q_full = q_full.sel(time=slice("1998-01-01", "2017-12-31"))
sp_full = sp_full.sel(time=slice("1998-01-01", "2017-12-31"))
print("✓ Data prepared.")

# ===================================================================
# 2. EXTENSIVE TESTING: PERFORMANCE AND ACCURACY
# ===================================================================
print("\n--- Running Performance and Accuracy Tests ---")

# --- Test 1: Vectorized Function (My Method) ---
t0 = time.time()
result_vectorized = calculate_pwv_vectorized(q_full, sp_full).load()
t1 = time.time()
time_vectorized = t1 - t0
print(f"My 'vectorized' function took: {time_vectorized:.4f} seconds.")

# --- Test 2: Loop-based Function (Your Method) ---
t0 = time.time()
result_loop = calculate_pwv_pressure(q_full, q_full.level, sp_full)
t1 = time.time()
time_loop = t1 - t0
print(f"Your 'pressure' function took: {time_loop:.4f} seconds.")

# --- Accuracy Check ---
difference = result_vectorized - result_loop
abs_diff = np.abs(difference)
print("\n--- Accuracy Results ---")
print(f"Speed advantage of vectorized function: {time_loop / time_vectorized:.1f}x faster")
print(f"Maximum absolute difference between methods: {abs_diff.max().item():.6f} mm")
print(f"Mean absolute difference between methods:    {abs_diff.mean().item():.6f} mm")

# ===================================================================
# 3. PLOTTING: VISUAL COMPARISON
# ===================================================================
print("\n--- Generating Comparison Plot ---")
mpl.rcParams.update({"font.sans-serif": "Arial", "axes.labelsize": 14, "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 12})
fig, (ax_top, ax_bot) = plt.subplots(
    nrows=2, sharex=True, figsize=(15, 8),
    gridspec_kw={"hspace": 0.1, "height_ratios": [3, 1]}
)
fig.suptitle("Direct Comparison of PWV Calculation Methods (Full Time Series)", fontsize=18, weight='bold')

# --- Top Panel: Overlapping Time Series ---
result_vectorized.plot(ax=ax_top, label='Result from `vectorized` func', color='red', lw=3, alpha=0.8)
result_loop.plot(ax=ax_top, label='Result from `pressure` func', color='blue', lw=3, ls='--', alpha=0.8)
ax_top.set_ylabel("PWV (mm)", weight='bold')
ax_top.set_xlabel("")
ax_top.grid(True, linestyle='--', alpha=0.6)
ax_top.legend()
ax_top.set_title("Overlapping Time Series (Should be identical)")

# --- Bottom Panel: Residuals (Difference) ---
difference.plot(ax=ax_bot, color='black', lw=2)
ax_bot.axhline(0, color='red', linestyle='--', lw=1.5)
ax_bot.set_ylabel("Difference (mm)", weight='bold')
ax_bot.set_xlabel("Time", weight='bold')
ax_bot.grid(True, linestyle='--', alpha=0.6)
ax_bot.set_title("Residual Plot (Difference between methods)")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


### Checking Wind Stats

## Data loading and inspection

### Step 169

This cell loads dataset(s) `ladakh_wind.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
ladakh_wind = xr.open_dataset('ladakh_wind.nc')
ladakh_wind


## Site definitions and metadata

### Step 170

This cell loads dataset(s) `ladakh_wind.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from windrose import WindroseAxes
import matplotlib as mpl

# --- Publication Quality Formatting ---
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 16,
    "axes.titlesize": 22,
})

# --- 1. Define Sites and Plotting Parameters ---
SITES = {
    'HANLE':  {'name': 'Hanle', 'lat': 32.7789, 'lon': 78.9650},
    'MERAK':  {'name': 'Merak', 'lat': 33.7828, 'lon': 78.57782},
    'SITE_A': {'name': 'Site A', 'lat': 34.25, 'lon': 78.75},
    'SITE_B': {'name': 'Site B', 'lat': 33.00, 'lon': 78.00}
}
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']
PRESSURE_LEVEL_HPA = 500

# --- 2. Load Dataset and Clean Time Coordinate ---
try:
    ladakh_wind = xr.open_dataset('ladakh_wind.nc')
    print("Dataset 'ladakh_wind.nc' loaded successfully.")
    datetime_index = pd.to_datetime(ladakh_wind['valid_time'].values)
    ladakh_wind['valid_time'] = datetime_index
    print(f"Time coordinate cleaned and verified.")
except FileNotFoundError:
    print("FATAL ERROR: The file 'ladakh_wind.nc' was not found.")
    exit()

# --- 3. Create the Figure Grid ---
fig, axs = plt.subplots(
    len(SITES), len(SEASONS),
    figsize=(18, 18),
    subplot_kw={'projection': 'windrose'}
)
fig.suptitle(f'Seasonal Windrose Comparison at {PRESSURE_LEVEL_HPA} hPa', fontsize=28, y=0.97)

# --- 4. Main Plotting Loop ---
print("\nGenerating 4x4 seasonal windrose plot for all sites...")

for i, (site_key, site_info) in enumerate(SITES.items()):
    print(f"\nProcessing site: {site_info['name']}...")
    
    try:
        # Step A: Interpolate data to the site's coordinates
        site_data = ladakh_wind.interp(
            latitude=site_info['lat'],
            longitude=site_info['lon'],
            method='linear'
        ).sel(pressure_level=PRESSURE_LEVEL_HPA, method='nearest')
        
        # --- DEBUGGING STEP 1: Check the interpolated data ---
        print(f"  [DEBUG] Interpolated data for {site_info['name']} has {len(site_data.valid_time)} time steps.")
        
        # --- FIX: Group the entire dataset by season FIRST ---
        seasonal_groups = site_data.groupby('valid_time.season')
        
        # --- DEBUGGING STEP 2: Check the created groups ---
        print(f"  [DEBUG] Found groups for seasons: {list(seasonal_groups.groups.keys())}")
        
        for j, season_name in enumerate(SEASONS):
            ax = axs[i, j]
            
            # Check if the group for this season exists
            if season_name in seasonal_groups.groups:
                # Get the dataset for this specific season
                season_data = seasonal_groups[season_name]
                
                # Step B: Calculate speed and direction on the seasonal data
                u = season_data['u']
                v = season_data['v']
                wind_speed = np.sqrt(u**2 + v**2)
                wind_dir_from = (270 - np.rad2deg(np.arctan2(v, u))) % 360
                
                # --- DEBUGGING STEP 3: Check data for the season before plotting ---
                print(f"    -> Plotting {season_name}: {len(wind_speed.values)} data points.")
                
                # Step C: Plot the windrose
                ax.bar(wind_dir_from.values, wind_speed.values, normed=True, opening=0.8, edgecolor='white')
            
            else:
                # This block will be executed if a season is genuinely missing from the data
                print(f"    -> No data for season: {season_name} at site {site_info['name']}")
                ax.set_visible(False)

    except Exception as e:
        print(f"    An unexpected error occurred for site {site_info['name']}: {e}")
        continue

# --- 5. Cosmetics and Finalization ---

# Set titles for columns (seasons) and rows (sites)
for ax, season in zip(axs[0], SEASONS):
    ax.set_title(season, pad=20)

for ax, site_info in zip(axs[:,0], SITES.values()):
    ax.set_ylabel(site_info['name'], fontsize=22, labelpad=40)

# Create a single, shared legend
try:
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, 
               loc='lower center', 
               title=r'Wind Speed (m s$^{-1}$)',
               ncol=8,
               bbox_to_anchor=(0.5, 0.02),
               frameon=True,
               edgecolor='black',
               facecolor='white',
               framealpha=0.9
              )
except NameError:
     print("\nCould not generate legend because no data was plotted.")

plt.tight_layout(rect=[0.05, 0.08, 0.98, 0.94])
plt.savefig('Ladakh_Sites_Seasonal_Windrose.pdf', dpi=400)
print("\n→ Plot successfully saved as Ladakh_Sites_Seasonal_Windrose.pdf")
plt.show()


## Analysis workflow

### Step 171

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
print(surface_pressure_data)


## Project setup and imports

### Step 172

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.patheffects import withStroke

# --- 1. Publication Quality Formatting ---
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "axes.labelsize": 22,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "axes.titlesize": 24,
    "legend.fontsize": 16,
})

PRESSURE_LEVEL_HPA = 450
QUIVER_STEP = 3
QUIVER_SCALE_KEY = 5

try:
    # --- REVERTED LOGIC ---
    # Select the single, nearest pressure level to the constant value.
    print(f"Selecting data for nearest constant pressure level: {PRESSURE_LEVEL_HPA} hPa...")
    # wind_at_level = ladakh_wind.sel(pressure_level=PRESSURE_LEVEL_HPA, method='nearest')
    wind_at_level = ladakh_wind.interp(pressure_level=PRESSURE_LEVEL_HPA, method='linear')

    # Calculate seasonal means from this single level of data.
    print("Calculating seasonal means...")
    seasonal_mean_wind = wind_at_level.groupby('valid_time.season').mean()
    seasonal_speed = np.sqrt(seasonal_mean_wind['u']**2 + seasonal_mean_wind['v']**2)
    # --- END OF REVERTED LOGIC ---

    # Get the geographic extent of the data for map boundaries
    lon_min, lon_max = float(ladakh_wind.longitude.min()), float(ladakh_wind.longitude.max())
    lat_min, lat_max = float(ladakh_wind.latitude.min()), float(ladakh_wind.latitude.max())
    extent = [lon_min, lon_max, lat_min, lat_max]

except Exception as e:
    print(f"An error occurred during data preparation: {e}")
    exit()


# --- 3. Plotting Seasonal Quiver Maps ---
fig, axs = plt.subplots(
    2, 2, figsize=(15, 13),
    subplot_kw={'projection': ccrs.PlateCarree()}
)
# Title updated to reflect the constant pressure level
fig.suptitle(f'Seasonal Mean Wind Fields over Ladakh at {PRESSURE_LEVEL_HPA} hPa', fontsize=28, y=0.98)

seasons = ['DJF', 'MAM', 'JJA', 'SON']
contour_plot = None # To hold a reference for the colorbar

for i, season in enumerate(seasons):
    ax = axs.flat[i]
    row, col = divmod(i, 2)

    try:
        u_season = seasonal_mean_wind['u'].sel(season=season)
        v_season = seasonal_mean_wind['v'].sel(season=season)
        speed_season = seasonal_speed.sel(season=season)

        # --- Map Cosmetics ---
        ax.set_extent(extent, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND, edgecolor='black', facecolor='#f0f0f0')
        ax.add_feature(cfeature.OCEAN)
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS, linestyle=':')

        gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                          linewidth=1, color='gray', alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
        if row < 1: gl.bottom_labels = False
        if col > 0: gl.left_labels = False

        # --- Data Plotting ---
        contour_plot = ax.contourf(u_season.longitude, u_season.latitude, speed_season,
                                   cmap='viridis', levels=15, transform=ccrs.PlateCarree())

        quiver_plot = ax.quiver(
            u_season.longitude[::QUIVER_STEP], u_season.latitude[::QUIVER_STEP],
            u_season[::QUIVER_STEP, ::QUIVER_STEP].values, v_season[::QUIVER_STEP, ::QUIVER_STEP].values,
            color='black', pivot='middle', transform=ccrs.PlateCarree()
        )
        plt.setp(quiver_plot.get_paths(), path_effects=[withStroke(linewidth=1.5, foreground='white')])

        ax.set_title(season)

    except KeyError:
        ax.set_title(f'{season} (No Data)')
        ax.add_feature(cfeature.LAND, facecolor='gray', alpha=0.5)

# --- 4. Add Shared Elements and Finalize ---

if 'quiver_plot' in locals() and quiver_plot is not None:
    axs[0, 0].quiverkey(quiver_plot, 0.8, 0.92, QUIVER_SCALE_KEY,
                       f'{QUIVER_SCALE_KEY} m s$^{{-1}}$',
                       labelpos='E', coordinates='axes', fontproperties={'size': 16})

cbar_ax = fig.add_axes([0.9, 0.15, 0.03, 0.7])
if contour_plot:
    cbar = fig.colorbar(contour_plot, cax=cbar_ax, orientation='vertical')
    cbar.set_label(r'Mean Wind Speed (m s$^{-1}$)', fontsize=20)
    cbar.ax.tick_params(labelsize=16)

plt.subplots_adjust(left=0.1, right=0.88, bottom=0.05, top=0.92, wspace=0.1, hspace=0.2)

plt.savefig('Ladakh_Seasonal_Quiver_Plots_500hPa.pdf', dpi=400)
plt.show()


## Analysis workflow

### Step 173

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
surface_pressure_data


## Interpolation and extraction

### Step 174

This cell loads dataset(s) `ladakh_wind.nc`, `surface_pressure_data.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
import xarray as xr
import pandas as pd

# Assume ladakh_wind and surface_pressure_data are already loaded
# Using .load() is recommended to prevent issues with file handles in long loops.
# ladakh_wind = xr.open_dataset('ladakh_wind.nc').load()
# surface_pressure_data = xr.open_dataset('surface_pressure_data.nc').load()

print("--- Cell 1: Data Loading and Merging ---")

try:
    if 'time' in surface_pressure_data.coords:
        surface_pressure_data = surface_pressure_data.rename({'time': 'valid_time'})

    print("Selecting matching surface pressure data...")
    sp_for_wind_grid = surface_pressure_data['sp'].sel(
        latitude=ladakh_wind.latitude,
        longitude=ladakh_wind.longitude,
        valid_time=ladakh_wind.valid_time,
        method='nearest'
    )

    print("Adding 'sp' variable to the main wind dataset...")
    ladakh_wind['sp'] = sp_for_wind_grid

    print("\nSuccessfully created a unified dataset:")
    print(ladakh_wind)

except Exception as e:
    print(f"An error occurred during data preparation in Cell 1: {e}")
    import traceback
    traceback.print_exc()


## Analysis workflow

### Step 175

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
ladakh_wind


## Summary statistics and reporting

### Step 176

This cell subsets the dataset to a selected time, level, region, or site so the next step works with a focused slice of the data.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.patheffects import withStroke
from scipy.interpolate import interp1d
from tqdm.auto import tqdm
import itertools

# --- 1. Publication Quality Formatting ---
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "axes.labelsize": 22,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "axes.titlesize": 24,
    "legend.fontsize": 16,
})

# --- 2. Data Preparation with Seasonal Medians and Manual Interpolation ---
print("\n--- Cell 2: Processing and Plotting with Custom Seasons (Robust Method) ---")
try:
    # Define custom seasons by month number
    CUSTOM_SEASONS = {
        'NDJF': [11, 12, 1, 2],
        'MAMJ': [3, 4, 5, 6],
        'JASO': [7, 8, 9, 10]
    }

    # Prepare the dataset once
    ladakh_wind['sp_hpa'] = ladakh_wind['sp'] / 100.0
    ladakh_wind_sorted = ladakh_wind.sortby('pressure_level')
    
    seasonal_results = {}
    print("Calculating mean wind fields for each custom season...")

    # Loop through each custom season
    for season_name, months in CUSTOM_SEASONS.items():
        print(f"  Processing {season_name}...")
        
        # Filter data for the current season's months
        seasonal_data = ladakh_wind_sorted.where(ladakh_wind_sorted['valid_time.month'].isin(months), drop=True)
        
        # Calculate the 2D map of median surface pressure for this season
        median_sp_for_season = seasonal_data['sp_hpa'].median(dim='valid_time')
        print(f"    Median surface pressure calculated for {season_name} is")
        print(median_sp_for_season)
        
        
        # Calculate the 3D cube of the mean vertical wind profile for this season
        mean_wind_profile_season = seasonal_data.mean(dim='valid_time')
        
        # Pre-allocate empty arrays for the final results
        u_surface = xr.DataArray(np.nan, coords=median_sp_for_season.coords, dims=median_sp_for_season.dims)
        v_surface = u_surface.copy()

        # Loop through each pixel to perform robust 1D interpolation
        latitudes = seasonal_data.latitude.values
        longitudes = seasonal_data.longitude.values
        pixel_iterator = itertools.product(latitudes, longitudes)
        
        for lat, lon in tqdm(pixel_iterator, total=len(latitudes)*len(longitudes), desc=f"Interpolating {season_name}"):
            # Get the mean vertical profile for this pixel
            u_profile = mean_wind_profile_season['u'].sel(latitude=lat, longitude=lon)
            v_profile = mean_wind_profile_season['v'].sel(latitude=lat, longitude=lon)
            
            # Get the target median surface pressure for this pixel
            target_pressure = median_sp_for_season.sel(latitude=lat, longitude=lon).item()
            
            if not np.isnan(target_pressure):
                # Create 1D interpolation functions using scipy
                f_u = interp1d(u_profile.pressure_level, u_profile.values, bounds_error=False, fill_value=np.nan)
                f_v = interp1d(v_profile.pressure_level, v_profile.values, bounds_error=False, fill_value=np.nan)
                
                # Store the interpolated value
                u_surface.loc[lat, lon] = f_u(target_pressure)
                v_surface.loc[lat, lon] = f_v(target_pressure)

        # Store the completed 2D maps for this season
        mean_wind = xr.Dataset({'u': u_surface, 'v': v_surface})
        mean_speed = np.sqrt(mean_wind['u']**2 + mean_wind['v']**2)
        seasonal_results[season_name] = {'mean_wind': mean_wind, 'mean_speed': mean_speed}

    print("All seasonal calculations complete.")
    
    lon_min, lon_max = float(ladakh_wind.longitude.min()), float(ladakh_wind.longitude.max())
    lat_min, lat_max = float(ladakh_wind.latitude.min()), float(ladakh_wind.latitude.max())
    extent = [lon_min, lon_max, lat_min, lat_max]

except Exception as e:
    print(f"An error occurred during data processing: {e}")
    import traceback
    traceback.print_exc()
    exit()

# --- 3. Plotting the Three Seasonal Maps ---
QUIVER_STEP = 2
QUIVER_SCALE_KEY = 2

fig, axs = plt.subplots(1, 3, figsize=(22, 8), subplot_kw={'projection': ccrs.PlateCarree()})
#ig.suptitle(r'Seasonal Mean Wind Fields at Seasonal Median Surface Pressure', fontsize=28, y=0.98)
contour_plot = None

for ax, season_name in zip(axs, CUSTOM_SEASONS.keys()):
    if season_name in seasonal_results:
        result = seasonal_results[season_name]
        u_season = result['mean_wind']['u']
        v_season = result['mean_wind']['v']
        speed_season = result['mean_speed']
        ax.set_extent(extent, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND, edgecolor='black', facecolor='#f0f0f0')
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=1, color='gray', alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
        if ax != axs[0]: gl.left_labels = False
        contour_plot = ax.contourf(u_season.longitude, u_season.latitude, speed_season, cmap='viridis', levels=15, transform=ccrs.PlateCarree())
        quiver_plot = ax.quiver(u_season.longitude[::QUIVER_STEP], u_season.latitude[::QUIVER_STEP],
                                u_season.values[::QUIVER_STEP, ::QUIVER_STEP], v_season.values[::QUIVER_STEP, ::QUIVER_STEP],
                                color='black', pivot='middle', transform=ccrs.PlateCarree())
        plt.setp(quiver_plot.get_paths(), path_effects=[withStroke(linewidth=1.5, foreground='white')])
        ax.set_title(season_name, fontsize=24)
    else:
        ax.set_title(f'{season_name} (No Data)')

# --- 4. Finalizing Plot ---
if 'quiver_plot' in locals() and quiver_plot is not None:
    axs[0].quiverkey(quiver_plot, 0.8, 0.92, QUIVER_SCALE_KEY, f'{QUIVER_SCALE_KEY} m s$^{{-1}}$',
                     labelpos='E', coordinates='axes', fontproperties={'size': 16})
cbar_ax = fig.add_axes([0.91, 0.15, 0.02, 0.7])
if contour_plot:
    cbar = fig.colorbar(contour_plot, cax=cbar_ax, orientation='vertical')
    cbar.set_label(r'Mean Wind Speed (m s$^{-1}$)', fontsize=20)
    cbar.ax.tick_params(labelsize=16)
plt.subplots_adjust(left=0.05, right=0.89, bottom=0.1, top=0.9)
plt.savefig('Ladakh_Custom_Seasonal_Wind_Plots_Final.pdf', dpi=400, bbox_inches='tight')
plt.show()


## Mapping and visualization

### Step 177

This cell loads dataset(s) `ladakh_wind.nc`, `surface_pressure_data.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import geopandas as gpd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.patheffects import withStroke
from scipy.interpolate import interp1d
from tqdm.auto import tqdm
import itertools
import rasterio
from rasterio.merge import merge
import rioxarray
from shapely.geometry import box, mapping

# --- 1. Publication Quality Formatting ---
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "axes.labelsize": 22,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "axes.titlesize": 24,
    "legend.fontsize": 16,
})

# --- 2. Full Data Preparation ---
try:
    # --- Load all datasets ---
    print("--- Loading All Datasets ---")
    ladakh_wind = xr.open_dataset('ladakh_wind.nc').load()
    #surface_pressure_data = xr.open_dataset('surface_pressure_data.nc').load()
    
    # --- Prepare Wind and Surface Pressure Data ---
    print("--- Preparing Wind and Surface Pressure Data ---")
    if 'time' in surface_pressure_data.coords:
        surface_pressure_data = surface_pressure_data.rename({'time': 'valid_time'})
    sp_for_wind_grid = surface_pressure_data['sp'].sel(
        latitude=ladakh_wind.latitude, longitude=ladakh_wind.longitude,
        valid_time=ladakh_wind.valid_time, method='nearest'
    )
    ladakh_wind['sp'] = sp_for_wind_grid
    ladakh_wind['sp_hpa'] = ladakh_wind['sp'] / 100.0
    ladakh_wind_sorted = ladakh_wind.sortby('pressure_level')

    # --- Prepare Elevation Data from GeoTIFFs ---
    print("--- Preparing Geospatial and Elevation Data ---")
    geotiff_files = ['srtm_52_05/srtm_52_05.tif', 'srtm_52_06/srtm_52_06.tif']
    src_files_to_mosaic = [rasterio.open(file) for file in geotiff_files]
    mosaic, out_trans = merge(src_files_to_mosaic)
    for src in src_files_to_mosaic: src.close()
    lon_min_elev, lat_max_elev = out_trans * (0, 0)
    lon_max_elev, lat_min_elev = out_trans * (mosaic.shape[2], mosaic.shape[1])
    elevation_da = xr.DataArray(mosaic[0], coords=[('latitude', np.linspace(lat_max_elev, lat_min_elev, mosaic.shape[1])),
                                                    ('longitude', np.linspace(lon_min_elev, lon_max_elev, mosaic.shape[2]))], name='elevation')
    elevation_da = elevation_da.rio.write_crs(4326, inplace=True)
    elevation_clipped = elevation_da.where(elevation_da > 0) # Remove negative elevation values

    # --- Perform Seasonal Wind Calculations ---
    CUSTOM_SEASONS = { 'NDJF': [11, 12, 1, 2], 'MAMJ': [3, 4, 5, 6], 'JASO': [7, 8, 9, 10] }
    seasonal_results = {}
    print("--- Calculating Seasonal Wind Fields ---")
    for season_name, months in CUSTOM_SEASONS.items():
        seasonal_data = ladakh_wind_sorted.where(ladakh_wind_sorted['valid_time.month'].isin(months), drop=True)
        median_sp_for_season = seasonal_data['sp_hpa'].median(dim='valid_time')
        mean_wind_profile_season = seasonal_data.mean(dim='valid_time')
        u_surface = xr.DataArray(np.nan, coords=median_sp_for_season.coords, dims=median_sp_for_season.dims)
        v_surface = u_surface.copy()
        latitudes = seasonal_data.latitude.values
        longitudes = seasonal_data.longitude.values
        pixel_iterator = itertools.product(latitudes, longitudes)
        for lat, lon in tqdm(pixel_iterator, total=len(latitudes)*len(longitudes), desc=f"Interpolating {season_name}"):
            u_profile = mean_wind_profile_season['u'].sel(latitude=lat, longitude=lon)
            v_profile = mean_wind_profile_season['v'].sel(latitude=lat, longitude=lon)
            target_pressure = median_sp_for_season.sel(latitude=lat, longitude=lon).item()
            if not np.isnan(target_pressure):
                f_u = interp1d(u_profile.pressure_level, u_profile.values, bounds_error=False, fill_value=np.nan)
                f_v = interp1d(v_profile.pressure_level, v_profile.values, bounds_error=False, fill_value=np.nan)
                u_surface.loc[{'latitude': lat, 'longitude': lon}] = f_u(target_pressure)
                v_surface.loc[{'latitude': lat, 'longitude': lon}] = f_v(target_pressure)
        mean_wind = xr.Dataset({'u': u_surface, 'v': v_surface})
        seasonal_results[season_name] = {'mean_wind': mean_wind, 'mean_speed': np.sqrt(mean_wind['u']**2 + mean_wind['v']**2)}
    
    extent = [float(ladakh_wind.longitude.min()), float(ladakh_wind.longitude.max()),
              float(ladakh_wind.latitude.min()), float(ladakh_wind.latitude.max())]
    print("Data preparation complete.")
except Exception as e:
    print(f"An error occurred during data preparation: {e}")
    exit()

# --- 3. Plotting ---
QUIVER_STEP = 2
QUIVER_SCALE_KEY = 5

fig, axs = plt.subplots(1, 3, figsize=(22, 8), subplot_kw={'projection': ccrs.PlateCarree()})
fig.suptitle(r'Seasonal Mean Surface Wind Fields over Ladakh Topography', fontsize=28, y=0.98)
quiver_plot = None

for ax, season_name in zip(axs, CUSTOM_SEASONS.keys()):
    if season_name in seasonal_results:
        # --- Correctly Layer the Plot Elements ---
        # 1. Base map features
        ax.set_extent(extent, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND, edgecolor='darkgray', facecolor='#f0f0f0', zorder=0)
        ax.add_feature(cfeature.BORDERS, linestyle=':', zorder=3)
        
        # 2. Elevation Contours (from the corrected variable)
        contour_levels = np.arange(2000, 6501, 250)
        major_contour_levels = np.arange(2000, 6501, 1000)
        
        # Use the prepared `elevation_clipped` variable
        C = ax.contour(elevation_clipped.longitude, elevation_clipped.latitude, elevation_clipped,
                       levels=major_contour_levels, colors='dimgray', linewidths=0.8,
                       transform=ccrs.PlateCarree(), zorder=1)
        ax.clabel(C, C.levels, inline=True, fontsize=10, fmt='%d m')
        
        # 3. Wind Vectors
        u_season = seasonal_results[season_name]['mean_wind']['u']
        v_season = seasonal_results[season_name]['mean_wind']['v']
        speed_season = seasonal_results[season_name]['mean_speed']
        
        X, Y = u_season.longitude, u_season.latitude
        u_vals, v_vals = u_season.values, v_season.values
        speed_vals = speed_season.values
        
        # Correctly color quivers by speed and set a reasonable scale
        quiver_plot = ax.quiver(X[::QUIVER_STEP], Y[::QUIVER_STEP],
                                u_vals[::QUIVER_STEP, ::QUIVER_STEP], v_vals[::QUIVER_STEP, ::QUIVER_STEP],
                                speed_vals[::QUIVER_STEP, ::QUIVER_STEP],
                                cmap='jet', transform=ccrs.PlateCarree(),
                                scale=80, zorder=4) # FIX: Reduced scale makes arrows visible
        
        ax.set_title(season_name, fontsize=24)

# --- 4. Finalizing Plot ---
if quiver_plot is not None:
    axs[0].quiverkey(quiver_plot, 0.75, 1.06, QUIVER_SCALE_KEY, f'{QUIVER_SCALE_KEY} m s$^{{-1}}$',
                     labelpos='E', coordinates='axes', fontproperties={'size': 16})

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(quiver_plot, cax=cbar_ax, orientation='vertical')
cbar.set_label(r'Mean Wind Speed (m s$^{-1}$)', fontsize=20)
cbar.ax.tick_params(labelsize=16)

for i, ax in enumerate(axs):
    gl = ax.gridlines(draw_labels=True, linewidth=1, color='gray', alpha=0.5, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    if i > 0: gl.left_labels = False

plt.subplots_adjust(left=0.05, right=0.9, bottom=0.1, top=0.9, wspace=0.1)
plt.savefig('Ladakh_Wind_Over_Contours_Corrected.pdf', dpi=400, bbox_inches='tight')
plt.show()


## Checking totoal cloud cover

## Data loading and inspection

### Step 178

This cell loads dataset(s) `total_cloud_cover.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
total_cloud_cover = xr.open_dataset('total_cloud_cover.nc')
total_cloud_cover


## Mapping and visualization

### Step 179

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
# plot the total cloud cover data for Hanle Site

plt.figure(figsize=(12, 6))
tcc_data = total_cloud_cover['tcc'].interp(
    latitude=32.7789, longitude=78.9650, method='linear'
)
tcc_data.plot()


plt.title('Total Cloud Cover at Hanle Site (32.7789° N, 78.9650° E)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True)
plt.show()


## Analysis workflow

### Step 180

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
surface_pressure_data


## Mapping and visualization

### Step 181

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
# plot the monthly mean surface pressure data for Hanle Site
plt.figure(figsize=(12, 6))
sp_data_monthly_mean = surface_pressure_data['sp'].resample(time='1M').mean().interp(
    latitude=32.7789, longitude=78.9650, method='linear'
)
sp_data_monthly_mean.plot() 
plt.title('Monthly Mean Surface Pressure at Hanle Site (32.7789° N, 78.9650° E)')
plt.xlabel('Time')
plt.ylabel('Surface Pressure (hPa)')
plt.grid(True)
plt.show()


### Step 182

This cell loads dataset(s) `era5_surface_pressure_2002_2017.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# 1. Read ERA5 monthly surface-pressure (Pa) for 2002-2017
#    – replace the filename with the path to your own NetCDF file
# ------------------------------------------------------------------
# # If you already have `surface_pressure_data` in memory, skip the next line.
# surface_pressure_data = xr.open_dataset("era5_surface_pressure_2002_2017.nc")

sp_monthly_mean = (
    surface_pressure_data["sp"]                         # Pa
    .interp(latitude=32.7789, longitude=78.9650, method="cubic")        # ① put grid-point on Hanle
    .resample(time="1M").mean()                        # ② calendar-month mean
    / 100.0                                            # Pa → hPa
)

# -------------------------------------------------------------
# Long-term monthly climatology (1998/2002–2017) in hPa
# -------------------------------------------------------------
sp_climo = sp_monthly_mean.groupby("time.month").mean()

sp_monthly_mean_sel = surface_pressure_data["sp"].sel(
    latitude=32.7789, longitude=78.9650, method="nearest"
).resample(time="1M").mean() / 100.0  # Pa →

sp_climo_sel = sp_monthly_mean_sel.groupby("time.month").mean()

# ------------------------------------------------------------------
# 2. Observational surface-pressure from Table 2 (hPa)
# ------------------------------------------------------------------
obs = {
    "month": np.arange(1, 13),
    "press_day":   [584.6, 584.8, 587.4, 589.2, 589.5, 589.1,
                    589.4, 590.4, 591.2, 591.3, 589.6, 587.2],
    "press_night": [584.7, 585.0, 587.8, 589.7, 590.1, 589.5,
                    589.8, 590.8, 591.4, 591.4, 589.8, 587.3],
}
df = pd.DataFrame(obs)
df["month_name"] = pd.to_datetime(df["month"], format="%m").dt.strftime("%b")

# ------------------------------------------------------------------
# 3. Plot
# ------------------------------------------------------------------
plt.figure(figsize=(12, 6))

# ERA5 climatology (black line)
sp_climo.plot(color="k", marker="D", label="ERA5 interp")
sp_climo_sel.plot(color="red", marker="D", linestyle="--", label="ERA5 sel") 
# Observations: Day (blue) and Night (orange)
plt.plot(df["month"], df["press_day"],   color="tab:blue", marker="o",
         label="Observation – Day")
plt.plot(df["month"], df["press_night"], color="tab:orange", marker="s",
         linestyle="--", label="Observation – Night")

plt.xticks(df["month"], df["month_name"])
plt.title("Hanle Surface Pressure (2002–2017)\nObservations vs ERA5")
plt.xlabel("Month")
plt.ylabel("Surface Pressure [hPa]")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## Pressure and PWV calculations

### Step 183

This cell defines reusable helper function(s) `calculate_pressure` so later sections can apply the same processing logic consistently.

In [ ]:

def calculate_pressure(P_b, T_b, L_b, h, h_b, R_star=8.3144598, g_0=9.80665, M=0.028964425278793993):
    T_h = T_b - L_b * (h - h_b)
    exponent = (g_0 * M) / (R_star * L_b)
    pressure = P_b * (T_h / T_b) ** exponent
    return pressure

P_b = 101325  # reference pressure at sea level in Pa
T_b = 280  # reference temperature at sea level in K
L_b = 0.0065  # temperature lapse rate in K/m
h = 4500      # height at which pressure is calculated in m
h_b = 0       # height of reference level in m

pressure = calculate_pressure(P_b, T_b, L_b, h, h_b)
print(f"Pressure at {h} m: {pressure:.2f} Pa")


### Checking Merra sp

## Site definitions and metadata

### Step 184

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
HANLE


## Data loading and inspection

### Step 185

This cell loads dataset(s) `MERRA2_400.tavg3_3d_asm_Nv.20200101.SUB.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
meraa_ladakh = xr.open_dataset('/Users/wavefunction/Downloads/MERRA2_400.tavg3_3d_asm_Nv.20200101.SUB.nc')


## Analysis workflow

### Step 186

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
meraa_ladakh


## Interpolation and extraction

### Step 187

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
# print value of PS for latitude 33.7828 and longitude 78.57782
ps_value = meraa_ladakh['PS'].interp(lat=33.7828,
                                    lon=78.57782, method='linear')
ps_value = ps_value.values.item()  # Convert to scalar value
print(f"PS value at latitude 33.7828 and longitude 78.57782: {ps_value:.2f} Pa")


## Mapping and visualization

### Step 188

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

# Hanle site coordinates
Hanle = {
    'name': 'Hanle',
    'lat': 32.7789,
    'lon': 78.965,
    'elevation': 4500,  # in meters
    'T_b': 287.49
}
# Hanle = SITE_A

# Interpolate geopotential field at Hanle location
z_data = geopotential_data['z'].interp(latitude=Hanle['lat'], longitude=Hanle['lon'], method='linear')
z_data = z_data / 9.80665  # Convert geopotential (m²/s²) to height (m)

# Select the geopotential profile at the nearest time to 2020-01-01
z_data = z_data.sel(time='2020-01-01', method='nearest')

# Extract pressure levels and height values
pressure_levels = z_data['level'].values  # in hPa
z_values = z_data.values  # in meters

# Plotting
plt.figure(figsize=(12, 6))
plt.plot(pressure_levels, z_values, label='Geopotential Profile')
plt.gca().invert_xaxis()  # Pressure decreases to the right

plt.xlabel("Pressure [hPa]")
plt.ylabel("Geopotential Height Z [m]")
plt.title(f"Geopotential at {Hanle['name']} Site ({Hanle['lat']}° N, {Hanle['lon']}° E)")

# Add horizontal line at Hanle's elevation
plt.axhline(y=Hanle['elevation'], color='red', linestyle='--', label=f"Hanle Elevation ({Hanle['elevation']} m)")

# Interpolate pressure at Hanle elevation
interp_func = interp1d(z_values, pressure_levels, bounds_error=False, fill_value="extrapolate")
p_at_h = interp_func(Hanle['elevation'])

print(f"Interpolated pressure at Hanle elevation ({Hanle['elevation']} m): {p_at_h:.2f} hPa")

plt.grid(True)
plt.legend()
plt.show()


## Geopotential

## Project setup and imports

### Step 189

This cell loads dataset(s) `geopotential.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
#loading geopotential data
# import intep1d
from scipy.interpolate import interp1d
import xarray as xr

geopotential_data = xr.open_dataset('geopotential.nc')
geopotential_data = geopotential_data.rename({'valid_time': 'time'})
geopotential_data = geopotential_data.rename({'pressure_level': 'level'})
geopotential_data


## Mapping and visualization

### Step 190

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
# ALMA site coordinates
ALMA = {'lat': -23.029, 'lon': 292.245, 'elevation': 5107, 'name': 'ALMA'}

# Interpolate geopotential field at ALMA location
z_data = geopotential_data['z'].interp(latitude=ALMA['lat'], longitude=ALMA['lon'], method='linear')
z_data = z_data / 9.80665  # Convert geopotential (m²/s²) to geopotential height (m)

# Select geopotential at the nearest time to 2020-01-01
z_data = z_data.sel(time='2020-01-01', method='nearest')

# Extract pressure levels and z values
pressure_levels = z_data['level'].values
z_values = z_data.values

# Plotting
plt.figure(figsize=(12, 6))
plt.plot(pressure_levels, z_values)
plt.gca().invert_xaxis()  # Optional: higher pressure on the right

plt.xlabel("pressure [hPa]")
plt.ylabel("Z [m]")
plt.title(f"Geopotential at ALMA Site ({ALMA['lat']}° N, {ALMA['lon']}° E)")


plt.axhline (y=ALMA['elevation'], color='red', linestyle='--', label='ALMA Elevation')

interp_func = interp1d(z_values, pressure_levels, bounds_error=False, fill_value="extrapolate")
p_at_h = interp_func(ALMA['elevation'])
p_at_h = interp_func(ALMA['elevation'])

print(f"Pressure at ALMA elevation ({ALMA['elevation']} m): {p_at_h:.2f} hPa")

plt.grid(True)
plt.show()


### Step 191

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# --- 0.  Convenience handles
g = 9.80665                                # m s⁻²
target_h = ALMA['elevation']               # 5107 m
print(f"Target height for pressure interpolation: {target_h} m")

# --- 1.  Extract full height profiles at ALMA latitude/longitude
z_prof = (geopotential_data['z']           # (time, level, lat, lon)
          .interp(latitude=ALMA['lat'],
                  longitude=ALMA['lon'],
                  method='linear')
          / g)                             # → height in metres

pressure_levels = z_prof['level'].values   # 1-D array of hPa values

# --- 2.  Loop over time steps, interpolate p(h) safely
p_at_h = []
for t in z_prof.time:
    # 1-D arrays for this month
    h_vals = z_prof.sel(time=t).values      # heights (may include NaNs)
    p_vals = pressure_levels.copy()         # matching pressures

    # Remove NaNs (levels below ground)
    valid = ~np.isnan(h_vals)
    h_vals = h_vals[valid]
    p_vals = p_vals[valid]

    # Sort by ascending height (should already be, but be safe)
    order = np.argsort(h_vals)
    h_sorted = h_vals[order]
    p_sorted = p_vals[order]

    # Interpolate pressure at the target height
    p_at_h.append(np.interp(target_h, h_sorted, p_sorted))

# Build back into an xarray DataArray
p_ALMA = xr.DataArray(
    data=np.array(p_at_h),
    coords={'time': z_prof.time},
    dims='time',
    name='p_ALMA',
    attrs={'units': 'hPa', 'long_name': f'Pressure at {target_h} m'}
)

# --- 3.  Quick-look plot
plt.figure(figsize=(10, 4))
p_ALMA.plot(marker='o')
plt.title(f'ERA5 pressure at ALMA elevation ({target_h} m)\n'
          f'({ALMA["lat"]}°, {ALMA["lon"]}°)')
plt.ylabel('Pressure [hPa]')
plt.grid(True)
plt.show()

# --- 4.  Inspect numerics
print(p_ALMA.to_dataframe().describe())


### Step 192

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import interp1d

# Mauna Kea site coordinates
#Mauna_Kea = {'lat': 19.8230, 'lon': 204.5306, 'elevation': 4200, 'name': 'Mauna Kea'}
Mauna_Kea = {"lat": 32.7789, "lon": 78.9650, "elevation": 4500, 'name': 'Mauna Kea'}


# Interpolate geopotential field at Mauna Kea location
z_data = geopotential_data['z'].interp(latitude=Mauna_Kea['lat'], longitude=Mauna_Kea['lon'], method='linear')
z_data = z_data / 9.80665  # Convert geopotential (m²/s²) to height (m)

# Select geopotential at the nearest time to 2020-08-01
z_data = z_data.sel(time='2020-08-01', method='nearest')

# Extract pressure levels and z values
pressure_levels = z_data['level'].values  # hPa
z_values = z_data.values  # meters

# Plotting
plt.figure(figsize=(12, 6))
plt.plot(pressure_levels, z_values)
plt.gca().invert_xaxis()  # Pressure decreasing left to right

plt.xlabel("pressure [hPa]")
plt.ylabel("Z [m]")
plt.title(f"Geopotential at {Mauna_Kea['name']} Site ({Mauna_Kea['lat']}° N, {Mauna_Kea['lon']}° E)")

# Add horizontal line at Mauna Kea elevation
plt.axhline(y=Mauna_Kea['elevation'], color='red', linestyle='--', label='Mauna Kea Elevation (4200 m)')

# Interpolate pressure at Mauna Kea elevation
interp_func = interp1d(z_values, pressure_levels, bounds_error=False, fill_value="extrapolate")
p_at_h = interp_func(Mauna_Kea['elevation'])

print(f"Interpolated pressure at Mauna Kea elevation ({Mauna_Kea['elevation']} m): {p_at_h:.2f} hPa")

plt.grid(True)
plt.legend()
plt.show()


## Site definitions and metadata

### Step 193

This cell loads dataset(s) `geopotential_all.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
import xarray as xr
import numpy as np
from scipy.interpolate import interp1d

g = 9.80665
target_date = "2020-08-01"

# Use your already-opened dataset object if available:
# ds = xr.open_dataset("geopotential_all.nc")
ds = geopotential_data  # must contain variable "z" on pressure levels with coords (time, level, latitude, longitude)

sites = {
    "IAO-Hanle": {"lat": 32.7789, "lon": 78.9650,  "elev": 4500},
    "Merak":     {"lat": 33.7828, "lon": 78.57782, "elev": 4310},
    "Site A":    {"lat": 34.25,   "lon": 78.75,    "elev": 4800},
    "Site B":    {"lat": 32.5,    "lon": 79.0,     "elev": 4500},
}

for name, s in sites.items():
    # 1) Bilinear in space, linear in time (nearest) — EXACTLY like your reference calculation
    z_site = ds["z"].interp(latitude=s["lat"], longitude=s["lon"], method="linear") / g
    z_t    = z_site.sel(time=target_date, method="nearest")   # single timestamp

    # 2) Extract vertical profile
    p_levels = z_t["level"].values              # hPa
    z_values = z_t.values.astype(float)         # meters

    # 3) Ensure monotonic z for interpolation (ascending with height)
    order = np.argsort(z_values)
    z_sorted = z_values[order]
    p_sorted = p_levels[order]

    # 4) Interpolate pressure(h) at site elevation — linear in (z, p), like the reference
    f = interp1d(z_sorted, p_sorted, bounds_error=False, fill_value="extrapolate")
    p_at_h = float(f(s["elev"]))                # hPa

    print(f"{name:8s}  p(h) @ {target_date} = {p_at_h:.2f} hPa")


## Summary statistics and reporting

### Step 194

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
import xarray as xr
import numpy as np

g = 9.80665


ds = geopotential_data.sel(time=slice("2010-01-01", "2025-04-30"))

sites = {
    "IAO-Hanle": {"lat": 32.7789, "lon": 78.9650},
    "Merak":     {"lat": 33.7828, "lon": 78.57782},
    "Site A":    {"lat": 34.25,   "lon": 78.75},
    "Site B":    {"lat": 32.5,    "lon": 79.0},
}

for name, s in sites.items():
    # Bilinear interpolation for all sites
    z_site = ds["z"].interp(latitude=s["lat"], longitude=s["lon"])
    z_site = z_site / g  # Convert geopotential (m²/s²) → height (m)

    # Compute mean and std over time and levels
    z_mean = float(z_site.mean().values)
    z_std  = float(z_site.std().values)
    z_min  = float(z_site.min().values)
    z_max  = float(z_site.max().values)

    print(f"{name:8s} | mean = {z_mean:8.2f} m | std = {z_std:8.2f} m | min = {z_min:8.2f} m | max = {z_max:8.2f} m")


### Hourly Jan Ladakh

## Interpolation and extraction

### Step 195

This cell loads dataset(s) `jan_hourly_ladakh.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
hourly_ladakh = xr.open_dataset('jan_hourly_ladakh.nc')
hourly_ladakh = hourly_ladakh.rename({'valid_time': 'time'})
hourly_ladakh = hourly_ladakh.rename({'pressure_level': 'level'})
print(hourly_ladakh)


## Mapping and visualization

### Step 196

This cell defines reusable helper function(s) `calculate_pwv_pressure`, `one_profile` so later sections can apply the same processing logic consistently.

In [ ]:
#!/usr/bin/env python3
# ─────────────────────────────────────────────────────────────────────────
#  HOURLY PWV – IAO–Hanle, January 2025, using observed pressure
# ─────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator   # only needed if your
                                                  # calculate_pwv_pressure
                                                  # relies on it

# ----------------------------------------------------------------------
# 1. CONFIGURATION
# ----------------------------------------------------------------------
#HANLE = {'lat': 32.78, 'lon': 78.96, 'name': 'IAO–Hanle'}
HOURLY_FILE = 'jan_hourly_ladakh.nc'
PRESSURE_CSV = 'hanle_monthly_surface_data.csv'   # same file as in notebook

# ----------------------------------------------------------------------
# 2. LOAD HOURLY DATA (specific humidity q on pressure levels)
# ----------------------------------------------------------------------
ds = xr.open_dataset(HOURLY_FILE)
ds = ds.rename({'valid_time': 'time', 'pressure_level': 'level'})  # harmonise

# interpolate q to Hanle coordinates (nearest → one 1-D column)
q_hanle = (ds['q']
           .interp(latitude=HANLE['lat'],
                    longitude=HANLE['lon'],
                    method='linear')
           .load())                              # (time, level)

# ----------------------------------------------------------------------
# 3. GET OBSERVED JANUARY SURFACE PRESSURE FOR HANLE
# ----------------------------------------------------------------------
hanle_df = pd.read_csv(PRESSURE_CSV)

# make lookup (month → avg pressure [Pa])
hanle_df['avg_pressure_pa'] = hanle_df[['Pressure_Day_Mean',
                                        'Pressure_Night_Mean']].mean(axis=1) * 100
month_map = {m: i+1 for i, m in enumerate(
             ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])}
hanle_df['month_num'] = hanle_df['Month'].map(month_map)
jan_psurf_pa = float(hanle_df.set_index('month_num').loc[1, 'avg_pressure_pa'])

# broadcast constant Psfc onto the hourly time axis
psurf_da = xr.DataArray(np.full(q_hanle.time.size, jan_psurf_pa),
                        coords={'time': q_hanle.time},
                        dims=['time'],
                        name='sp')

# align just in case (should already match)
q_hanle, psurf_da = xr.align(q_hanle, psurf_da, join='inner')

# ----------------------------------------------------------------------
# 4. PWV CALCULATION  (exactly the same routine you already use)
# ----------------------------------------------------------------------
def calculate_pwv_pressure(q_da: xr.DataArray,
                           levels_da: xr.DataArray,
                           psurf_da: xr.DataArray) -> xr.DataArray:
    """
    Your original trapezoidal PWV integrator (q in kg/kg, P in Pa, g = 9.81).
    """
    g = 9.81
    p_levels_pa = levels_da * 100.0  # hPa → Pa

    def one_profile(q_prof, p_prof, p_sfc_val):
        finite = np.isfinite(q_prof)
        if finite.sum() < 2:
            return np.nan

        p, qv = p_prof[finite], q_prof[finite]
        sorter = np.argsort(p)
        p, qv = p[sorter], qv[sorter]

        # keep layers above surface
        keep = p >= p_sfc_val
        p, qv = p[keep], qv[keep]
        if p.size < 2:
            return np.nan

        # add point at Psfc using PCHIP interpolation
        q_sfc = PchipInterpolator(p, qv, extrapolate=True)(p_sfc_val)
        p_full = np.append(p, p_sfc_val)
        q_full = np.append(qv, q_sfc)
        sorter2 = np.argsort(p_full)
        return np.trapz(q_full[sorter2], p_full[sorter2]) / g   # kg m⁻² ≈ mm

    return xr.apply_ufunc(
        one_profile,
        q_da, p_levels_pa, psurf_da,
        input_core_dims=[['level'], ['level'], []],
        output_core_dims=[[]],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[q_da.dtype]
    )

pwv_hanle = calculate_pwv_vectorized(q_hanle, psurf_da)

# ----------------------------------------------------------------------
# 5. PLOT  (single two-panel layout, matching your style)
# ----------------------------------------------------------------------
plt.rcParams.update({'font.size': 18,
                     'font.family': 'serif',
                     'text.usetex': True,
                     'text.latex.preamble': r'\usepackage{amsmath}'})

fig = plt.figure(figsize=(15, 8))
gs  = plt.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.0)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)

# colours & markers consistent with your style
color  = '#e45756'   # red (fifth colour in your palette)
marker = 'p'         # pentagon (fifth marker)

# top panel: full range
ax1.plot(pwv_hanle.time, pwv_hanle,
         color=color, linewidth=2.5, marker=marker, markersize=6,
         linestyle='-', alpha=0.7, label=HANLE['name'])

# bottom panel: zoom 0.4–1.6 mm
ax2.plot(pwv_hanle.time, pwv_hanle,
         color=color, linewidth=2.5, marker=marker, markersize=6,
         linestyle='-', alpha=0.9)

# 1 mm threshold
for ax in (ax1, ax2):
    ax.axhline(1.0, color='black', linestyle='--', linewidth=2.5)
    ax.set_facecolor('white')
    ax.tick_params(direction='in', top=True, right=True)

ax1.set_ylabel('PWV (mm)', fontsize=22)
ax2.set_ylabel('PWV (mm)', fontsize=22)
ax2.set_xlabel('2025-01 Date (UTC)', fontsize=22)

ax1.legend(loc='upper right', fontsize=18)
ax1.set_ylim(0, float(pwv_hanle.max()) + 0.5)
ax2.set_ylim(0.4, 1.6)

plt.setp(ax1.get_xticklabels(), visible=False)
fig.tight_layout()
#plt.savefig('PWV_vs_Time_IAO-Hanle_Jan2025.pdf', dpi=450, bbox_inches='tight')
plt.show()


## Summary statistics and reporting

### Step 197

This cell computes or evaluates precipitable water vapor (PWV)-related quantities that are central to the site-quality analysis.

In [ ]:
# mean and median of the PWV values
mean_pwv = pwv_hanle.mean().item()
median_pwv = pwv_hanle.median().item()
print(f"Mean PWV: {mean_pwv:.2f} mm")
print(f"Median PWV: {median_pwv:.2f} mm")


## Averaged Elevation Map

## Analysis workflow

### Step 198

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
2+3


## Mapping and visualization

### Step 199

This cell defines reusable helper function(s) `read_dem_as_latlon`, `clip_to_bbox`, `build_era5_edges`, `aligned_edges` so later sections can apply the same processing logic consistently.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import rasterio
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────────────────────
DEM_PATH = "/Users/wavefunction/ASU Dropbox/Tanmay Singh/THz_Mac/output_hh.tif"

# Geographic bounding box (lon_min, lon_max, lat_min, lat_max)
BBOX = (76.0, 80.0, 32.0, 36.0)

# ERA5 grid resolution in degrees
ERA5_RES = 0.25

# Output products
OUT_DIR = Path("./era5_dem_products")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_GRID = OUT_DIR / "era5_0p25_stats_ladakh.csv"
GTIFF_MEAN = OUT_DIR / "era5_0p25_mean_elevation_ladakh.tif"

# ─────────────────────────────────────────────────────────────
# Site definitions
# ─────────────────────────────────────────────────────────────
SITES = [
    { 'name' : 'Hanle',      'lat' : 32.7789, 'lon' : 78.9650, 'elevation' : 4500, 'T_b' : 287.49 },
    { 'name' : 'IAO-Hanle',  'lat' : 32.7789, 'lon' : 78.9650, 'elevation' : 4500, 'T_b' : 287.49 },
    { 'name' : 'Merak',      'lat' : 33.7828, 'lon' : 78.57782, 'elevation' : 4310 },
    { 'name' : 'Site A',     'lat' : 34.25,   'lon' : 78.75,   'elevation' : 4800, 'T_b' : 285.82 },
    { 'name' : 'Site B',     'lat' : 32.5,    'lon' : 79.0,    'elevation' : 4500, 'T_b' : 286.0  },
]

# ─────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────
def read_dem_as_latlon(geotiff_path: str) -> xr.DataArray:
    """
    Load DEM as an xarray.DataArray with CRS awareness.
    Reproject to EPSG:4326 (lon/lat) if needed.
    """
    da = rxr.open_rasterio(geotiff_path, masked=True)  # shape: (band, y, x) typically band=1
    # Squeeze band dimension if present
    if "band" in da.dims and da.sizes["band"] == 1:
        da = da.squeeze("band", drop=True)
    # Ensure CRS
    if da.rio.crs is None:
        raise RuntimeError("Input GeoTIFF lacks a CRS. Please define/set CRS before proceeding.")
    # Reproject to lon/lat
    if da.rio.crs.to_epsg() != 4326:
        da = da.rio.reproject("EPSG:4326")
    return da

def clip_to_bbox(da: xr.DataArray, bbox):
    lon_min, lon_max, lat_min, lat_max = bbox
    return da.rio.clip_box(minx=lon_min, miny=lat_min, maxx=lon_max, maxy=lat_max)

def build_era5_edges(bbox, res):
    lon_min, lon_max, lat_min, lat_max = bbox
    # Edges aligned to multiples of res
    def aligned_edges(vmin, vmax, step):
        start = np.floor(vmin / step) * step
        end   = np.ceil (vmax / step) * step
        # ensure numeric precision cleanliness
        n = int(round((end - start) / step))
        return np.round(start + np.arange(n + 1) * step, 10)

    lon_edges = aligned_edges(lon_min, lon_max, res)
    lat_edges = aligned_edges(lat_min, lat_max, res)
    return lon_edges, lat_edges

def centers_from_edges(edges):
    return (edges[:-1] + edges[1:]) / 2.0

def raster_to_points_dataframe(da: xr.DataArray) -> pd.DataFrame:
    """
    Flatten raster into a DataFrame with columns: lon, lat, elev.
    """
    # da dims should be (y, x) with coordinates x=lon, y=lat
    # Create 2D coordinate grids
    lon2d, lat2d = xr.broadcast(da["x"], da["y"])
    df = pd.DataFrame({
        "lon": lon2d.values.ravel(),
        "lat": lat2d.values.ravel(),
        "elev": da.values.ravel()
    })
    # Drop NaNs (nodata) and masked
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["elev"])
    return df

def aggregate_to_era5(df_pts: pd.DataFrame, lon_edges, lat_edges) -> pd.DataFrame:
    """
    Bin points to ERA5 0.25° grid and compute statistics per cell.
    Returns a DataFrame with one row per cell: lon_center, lat_center, stats...
    """
    # Bin indices via pandas.cut
    df_pts["lon_bin"] = pd.cut(df_pts["lon"], bins=lon_edges, right=False, include_lowest=True)
    df_pts["lat_bin"] = pd.cut(df_pts["lat"], bins=lat_edges, right=False, include_lowest=True)

    # Drop points outside (can happen on exact max edge)
    df_pts = df_pts.dropna(subset=["lon_bin", "lat_bin"])

    # Compute centers for each bin
    lon_centers = centers_from_edges(lon_edges)
    lat_centers = centers_from_edges(lat_edges)
    # Map Interval -> center
    def interval_center_map(edges):
        # build dict {Interval(left, right, closed='left'): center}
        centers = centers_from_edges(edges)
        mapping = {}
        for i in range(len(edges) - 1):
            left, right = edges[i], edges[i+1]
            mapping[pd.Interval(left, right, closed="left")] = centers[i]
        return mapping

    lon_center_map = interval_center_map(lon_edges)
    lat_center_map = interval_center_map(lat_edges)

    df_pts["lon_center"] = df_pts["lon_bin"].map(lon_center_map)
    df_pts["lat_center"] = df_pts["lat_bin"].map(lat_center_map)

    # Groupby and aggregate
    gb = df_pts.groupby(["lat_center", "lon_center"])  # lat first for sensible row ordering
    stats = gb["elev"].agg(['mean', 'std', 'count', 'min', 'max']).reset_index()

    # Sort for consistent 2D reshaping/plotting
    stats = stats.sort_values(["lat_center", "lon_center"]).reset_index(drop=True)
    return stats

def stats_to_2d(stats_df: pd.DataFrame, value_col: str):
    """
    Pivot the stats DF into 2D arrays for plotting (lat x lon).
    Returns: lat_vals, lon_vals, grid2d (2D array with shape [nlat, nlon])
    """
    # Get sorted unique centers
    lat_vals = np.sort(stats_df["lat_center"].unique())
    lon_vals = np.sort(stats_df["lon_center"].unique())
    # Build full grid with NaNs where cells had no samples
    pivot = stats_df.pivot(index="lat_center", columns="lon_center", values=value_col)
    # Reindex to full set to ensure monotonic grids
    pivot = pivot.reindex(index=lat_vals, columns=lon_vals)
    return lat_vals, lon_vals, pivot.values

def write_mean_geotiff(mean_grid, lon_vals, lat_vals, out_path: Path):
    """
    Write the mean grid as a GeoTIFF in EPSG:4326.
    The grid is cell-centered on lon_vals, lat_vals; we compute an affine transform.
    """
    if len(lon_vals) < 2 or len(lat_vals) < 2:
        # Cannot infer resolution/transform robustly with <2 samples
        return

    # Resolution from centers
    dx = np.mean(np.diff(lon_vals))
    dy = np.mean(np.diff(lat_vals))
    # Build edges from centers to construct transform (assuming regular grid)
    lon0 = lon_vals[0] - dx/2
    lat0 = lat_vals[0] - dy/2

    # Note: rasterio expects transform with top-left origin.
    # Our lat_vals are ascending; top-left pixel should have max latitude.
    # Flip vertically for writing (north-up).
    grid_to_write = np.flipud(mean_grid)

    from rasterio.transform import from_origin
    transform = from_origin(lon0, lat_vals[-1] + dy/2, dx, dy)  # top-left x,y; pixel size dx,dy

    profile = {
        "driver": "GTiff",
        "height": grid_to_write.shape[0],
        "width": grid_to_write.shape[1],
        "count": 1,
        "dtype": rasterio.float32,
        "crs": "EPSG:4326",
        "transform": transform,
        "compress": "deflate",
        "nodata": np.nan,
    }
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(grid_to_write.astype(np.float32), 1)

def era5_cell_from_point(lat, lon, res, bbox):
    """
    Compute the ERA5 cell center for a given (lat, lon), aligned with edges at multiples of res.
    """
    lon_min, lon_max, lat_min, lat_max = bbox
    lon_edges, lat_edges = build_era5_edges(bbox, res)
    # find bin index (closed-left)
    lon_idx = np.searchsorted(lon_edges, lon, side="right") - 1
    lat_idx = np.searchsorted(lat_edges, lat, side="right") - 1
    if lon_idx < 0 or lon_idx >= len(lon_edges) - 1 or lat_idx < 0 or lat_idx >= len(lat_edges) - 1:
        return None, None
    lon_center = (lon_edges[lon_idx] + lon_edges[lon_idx + 1]) / 2.0
    lat_center = (lat_edges[lat_idx] + lat_edges[lat_idx + 1]) / 2.0
    return lat_center, lon_center

def lookup_cell_stats(stats_df: pd.DataFrame, lat_center, lon_center):
    row = stats_df[(stats_df["lat_center"] == lat_center) & (stats_df["lon_center"] == lon_center)]
    if row.empty:
        return None
    return row.iloc[0].to_dict()

# ─────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────
def main():
    # Load DEM, reproject to lat/lon, clip
    dem = read_dem_as_latlon(DEM_PATH)
    dem_clip = clip_to_bbox(dem, BBOX)

    # Flatten to point table
    df_pts = raster_to_points_dataframe(dem_clip)

    # Build ERA5 grid edges
    lon_edges, lat_edges = build_era5_edges(BBOX, ERA5_RES)

    # Aggregate stats to ERA5 0.25° cells
    stats_df = aggregate_to_era5(df_pts, lon_edges, lat_edges)

    # Persist stats table
    stats_df.to_csv(CSV_GRID, index=False)
    print(f"[OK] Wrote grid statistics: {CSV_GRID}")

    # 2D mean for plotting
    lat_vals, lon_vals, mean_grid = stats_to_2d(stats_df, "mean")

    # Optional: write GeoTIFF of mean elevation on ERA5 grid
    write_mean_geotiff(mean_grid, lon_vals, lat_vals, GTIFF_MEAN)
    print(f"[OK] Wrote mean grid GeoTIFF: {GTIFF_MEAN}")

    # Plot aggregated mean elevation over the region
    fig, ax = plt.subplots(figsize=(8, 7))
    # Build edges from centers for pcolormesh
    def centers_to_edges(vals):
        d = np.diff(vals)
        d = np.append(d, d[-1])  # extend last interval
        edges = np.concatenate(([vals[0] - d[0] / 2], vals + d / 2))
        return edges

    lon_plot_edges = centers_to_edges(lon_vals)
    lat_plot_edges = centers_to_edges(lat_vals)

    # pcolormesh expects [Y,X] shaped grid; mean_grid is [nlat, nlon]
    m = ax.pcolormesh(lon_plot_edges, lat_plot_edges, mean_grid, shading="auto")
    cb = plt.colorbar(m, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label("Mean Elevation (m)")
    ax.set_xlabel("Longitude (°E)")
    ax.set_ylabel("Latitude (°N)")
    ax.set_title("ERA5 0.25° Aggregated Mean Elevation (Ladakh)")

    # Overlay site markers
    for s in SITES:
        ax.plot(s["lon"], s["lat"], marker="o", markersize=5, linestyle="None")
        ax.text(s["lon"] + 0.03, s["lat"] + 0.03, s["name"], fontsize=9)

    # Bounding box
    ax.set_xlim(BBOX[0], BBOX[1])
    ax.set_ylim(BBOX[2], BBOX[3])

    plt.tight_layout()
    fig.savefig(OUT_DIR / "era5_mean_elevation_map.png", dpi=200)
    print(f"[OK] Saved plot: {OUT_DIR / 'era5_mean_elevation_map.png'}")

    # Report per-site cell stats
    print("\n=== Per-site ERA5 cell statistics (0.25°) ===")
    for s in SITES:
        latc, lonc = era5_cell_from_point(s["lat"], s["lon"], ERA5_RES, BBOX)
        if latc is None:
            print(f"{s['name']}: outside bounding box")
            continue
        st = lookup_cell_stats(stats_df, latc, lonc)
        if st is None:
            print(f"{s['name']}: no DEM samples in cell centered at (lat={latc:.3f}, lon={lonc:.3f})")
            continue
        print(
            f"{s['name']}: cell_center(lat={latc:.3f}, lon={lonc:.3f})  "
            f"mean={st['mean']:.2f} m, std={st['std']:.2f} m, "
            f"min={st['min']:.2f} m, max={st['max']:.2f} m, count={int(st['count'])}"
        )

if __name__ == "__main__":
    main()


### Step 200

This cell defines reusable helper function(s) `read_dem_latlon`, `clip_bbox`, `grid_centers`, `centers_to_edges` so later sections can apply the same processing logic consistently.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# Resample DEM to a target grid (ERA5-like), save outputs, print per-site stats,
# and OVERPLOT the sites on the resampled map.

from pathlib import Path
import numpy as np
import xarray as xr
import rioxarray as rxr
import rasterio
from rasterio.transform import from_origin
import matplotlib.pyplot as plt

# ---------------- Config ----------------
DEM_PATH = "/Users/wavefunction/ASU Dropbox/Tanmay Singh/THz_Mac/output_hh.tif"
BBOX = (76.0, 80.0, 32.0, 36.0)   # (lon_min, lon_max, lat_min, lat_max)
RES = 0.25                        # set 0.25 (ERA5), or 0.025, etc.

OUT = Path("./era5_dem_products"); OUT.mkdir(parents=True, exist_ok=True)
OUT_NC  = OUT / f"elevation_grid_{RES:g}.nc"
OUT_TIF = OUT / f"elevation_grid_{RES:g}_mean.tif"
OUT_PNG = OUT / f"elevation_grid_{RES:g}_mean.png"

SITES = [
    { 'name' : 'Hanle',      'lat' : 32.7789, 'lon' : 78.9650 },
    { 'name' : 'IAO-Hanle',  'lat' : 32.7789, 'lon' : 78.9650 },
    { 'name' : 'NLST-Merak',      'lat' : 33.7828, 'lon' : 78.57782 },
    { 'name' : 'Site A',     'lat' : 34.25,   'lon' : 78.75 },
    { 'name' : 'Site B',     'lat' : 32.5,    'lon' : 79.0 },
]

# ---------------- Utilities ----------------
def read_dem_latlon(path):
    da = rxr.open_rasterio(path, masked=True)
    if "band" in da.dims and da.sizes.get("band", 0) == 1:
        da = da.squeeze("band", drop=True)
    if da.rio.crs is None:
        raise RuntimeError("DEM has no CRS.")
    if da.rio.crs.to_epsg() != 4326:
        da = da.rio.reproject("EPSG:4326")
    return da

def clip_bbox(da, bbox):
    lon_min, lon_max, lat_min, lat_max = bbox
    return da.rio.clip_box(minx=lon_min, miny=lat_min, maxx=lon_max, maxy=lat_max)

def grid_centers(vmin, vmax, res):
    start = np.ceil((vmin - 1e-12)/res)*res
    stop  = np.floor((vmax + 1e-12)/res)*res
    if stop < start:
        return np.array([])
    n = int(round((stop - start)/res)) + 1
    return np.round(start + np.arange(n)*res, 10)

def centers_to_edges(centers, res):
    return np.round(np.concatenate(([centers[0]-res/2], centers+res/2)), 10)

# ---------------- Core resampling (RAM-lean) ----------------
def aggregate_dem_to_grid(dem_clip, lon_c, lat_c, lon_e, lat_e):
    """
    dem_clip: xarray.DataArray with dims (y, x), coords x (lon ASC), y (lat usually DESC).
    Returns: dict of grids (mean, std, min, max, count), all shaped [nlat, nlon] with lat ASC.
    """
    nlon, nlat = lon_c.size, lat_c.size
    lon_vals = dem_clip.x.values                     # ASC
    lat_vals = dem_clip.y.values                     # often DESC
    elev = dem_clip.values                           # 2D (ny, nx)

    # Bin edges with tiny padding to catch edge points
    eps = RES * 1e-6
    lon_edges = lon_e.copy(); lat_edges = lat_e.copy()
    lon_edges[0]-=eps; lon_edges[-1]+=eps; lat_edges[0]-=eps; lat_edges[-1]+=eps

    # Column indices for all x (vectorized)
    col_idx = np.searchsorted(lon_edges, lon_vals, side="right") - 1   # shape (nx,)
    valid_col = (col_idx >= 0) & (col_idx < nlon)

    # Accumulators (lat ASC)
    sum_grid   = np.zeros((nlat, nlon), dtype=np.float64)
    sumsq_grid = np.zeros((nlat, nlon), dtype=np.float64)
    cnt_grid   = np.zeros((nlat, nlon), dtype=np.int64)
    min_grid   = np.full((nlat, nlon), np.inf, dtype=np.float64)
    max_grid   = np.full((nlat, nlon), -np.inf, dtype=np.float64)

    # Iterate each raster row
    for j in range(lat_vals.size):
        # Map this raster row to a target-lat bin index
        row_idx = np.searchsorted(lat_edges, lat_vals[j], side="right") - 1
        if row_idx < 0 or row_idx >= nlat:
            continue

        row = elev[j, :].astype(np.float64)
        m = np.isfinite(row) & valid_col
        if not np.any(m):
            continue

        cols = col_idx[m]
        vals = row[m]

        # Aggregate sums & counts
        np.add.at(sum_grid[row_idx],   cols, vals)
        np.add.at(sumsq_grid[row_idx], cols, vals*vals)
        np.add.at(cnt_grid[row_idx],   cols, 1)

        # Min/Max per unique column
        if cols.size:
            order = np.argsort(cols)
            cols_sorted = cols[order]
            vals_sorted = vals[order]
            ucols, start_idx = np.unique(cols_sorted, return_index=True)
            for k, c in enumerate(ucols):
                s = start_idx[k]
                e = start_idx[k+1] if k+1 < start_idx.size else cols_sorted.size
                vseg = vals_sorted[s:e]
                vmin = float(np.min(vseg)); vmax = float(np.max(vseg))
                if vmin < min_grid[row_idx, c]:
                    min_grid[row_idx, c] = vmin
                if vmax > max_grid[row_idx, c]:
                    max_grid[row_idx, c] = vmax

    # Final stats
    with np.errstate(invalid="ignore", divide="ignore"):
        mean_grid = sum_grid / cnt_grid
        var_grid  = (sumsq_grid / cnt_grid) - mean_grid**2
        std_grid  = np.sqrt(np.maximum(var_grid, 0.0))

    # NaN where empty
    empty = cnt_grid == 0
    mean_grid[empty] = np.nan
    std_grid[empty]  = np.nan
    min_grid[empty]  = np.nan
    max_grid[empty]  = np.nan

    return {
        "mean":  mean_grid,
        "std":   std_grid,
        "min":   min_grid,
        "max":   max_grid,
        "count": cnt_grid.astype(np.int32),
    }

# ---------------- Main ----------------
def main():
    # Read + clip DEM
    dem = read_dem_latlon(DEM_PATH)
    dem = clip_bbox(dem, BBOX)

    # Target grid (ASC)
    lon_c = grid_centers(BBOX[0], BBOX[1], RES)
    lat_c = grid_centers(BBOX[2], BBOX[3], RES)
    if lon_c.size == 0 or lat_c.size == 0:
        raise RuntimeError("No grid centers fall inside BBOX at this resolution.")
    lon_e = centers_to_edges(lon_c, RES)   # ASC
    lat_e = centers_to_edges(lat_c, RES)   # ASC

    # Aggregate (RAM-lean)
    grids = aggregate_dem_to_grid(dem, lon_c, lat_c, lon_e, lat_e)

    # Save NetCDF (latitude ascending to match plotting)
    ds = xr.Dataset(
        data_vars=dict(
            orog_mean=(["latitude","longitude"], grids["mean"].astype("float32")),
            orog_std =( ["latitude","longitude"], grids["std"].astype("float32")),
            orog_min =( ["latitude","longitude"], grids["min"].astype("float32")),
            orog_max =( ["latitude","longitude"], grids["max"].astype("float32")),
            orog_n   =( ["latitude","longitude"], grids["count"].astype("int32")),
        ),
        coords=dict(latitude=lat_c, longitude=lon_c),
        attrs=dict(
            title=f"DEM aggregated to {RES:g}° grid (mean/std/min/max/count)",
            grid_resolution=f"{RES:g} degree",
            source_tif=str(DEM_PATH),
        ),
    )
    ds.to_netcdf(OUT_NC)
    print(f"[OK] wrote {OUT_NC}")

    # Save GeoTIFF of mean (north-up; flip from lat ASC → lat DESC)
    mean_for_tif = np.flipud(grids["mean"]).astype("float32")  # row 0 = max latitude
    transform = from_origin(lon_e[0], lat_e[-1], RES, RES)
    profile = dict(driver="GTiff", height=mean_for_tif.shape[0], width=mean_for_tif.shape[1],
                   count=1, dtype="float32", crs="EPSG:4326", transform=transform,
                   compress="deflate", nodata=np.nan)
    with rasterio.open(OUT_TIF, "w", **profile) as dst:
        dst.write(mean_for_tif, 1)
    print(f"[OK] wrote {OUT_TIF}")

    # Quick plot (lat/lon edges ASC; same orientation as array) + overplot SITES
    lon_plot = np.concatenate(([lon_c[0]-RES/2], lon_c+RES/2))
    lat_plot = np.concatenate(([lat_c[0]-RES/2], lat_c+RES/2))
    fig, ax = plt.subplots(figsize=(8,7))
    pm = ax.pcolormesh(lon_plot, lat_plot, grids["mean"], shading="auto")
    cb = fig.colorbar(pm, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label("Mean elevation (m)")
    ax.set_xlim(BBOX[0], BBOX[1]); ax.set_ylim(BBOX[2], BBOX[3])
    ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
    ax.set_title(f"DEM mean on {RES:g}° grid ")

    # OVERPLOT sites
    for s in SITES:
        ax.plot(s["lon"], s["lat"], marker="o", markersize=5, linestyle="None", markeredgecolor="k")
        ax.text(s["lon"]+0.03, s["lat"]+0.03, s["name"], fontsize=9, ha="left", va="bottom")

    fig.tight_layout(); fig.savefig(OUT_PNG, dpi=180); plt.close(fig)
    print(f"[OK] wrote {OUT_PNG}")

    # -------- Per-site stats (cell center + stats) --------
    print(f"\n=== Per-site grid cell statistics ({RES:g}°) ===")
    for s in SITES:
        # find bin index using edges (ASC)
        lon_idx = int(np.searchsorted(lon_e, s["lon"], side="right") - 1)
        lat_idx = int(np.searchsorted(lat_e, s["lat"], side="right") - 1)
        if (lon_idx < 0 or lon_idx >= lon_c.size or
            lat_idx < 0 or lat_idx >= lat_c.size):
            print(f"{s['name']}: outside bounding box")
            continue
        # cell centers (exact)
        lon_center = float(lon_c[lon_idx])
        lat_center = float(lat_c[lat_idx])

        # fetch stats from ASC arrays
        mean_v = grids["mean"][lat_idx, lon_idx]
        std_v  = grids["std"][lat_idx, lon_idx]
        min_v  = grids["min"][lat_idx, lon_idx]
        max_v  = grids["max"][lat_idx, lon_idx]
        cnt_v  = int(grids["count"][lat_idx, lon_idx])

        if np.isnan(mean_v):
            print(f"{s['name']}: no DEM samples in cell centered at (lat={lat_center:.3f}, lon={lon_center:.3f})")
            continue

        print(
            f"{s['name']}: cell_center(lat={lat_center:.3f}, lon={lon_center:.3f})  "
            f"mean={mean_v:.2f} m, std={std_v:.2f} m, "
            f"min={min_v:.2f} m, max={max_v:.2f} m, count={cnt_v}"
        )

if __name__ == "__main__":
    main()


## Exports and file generation

### Step 201

This cell defines reusable helper function(s) `read_dem_latlon`, `clip_bbox`, `grid_centers`, `centers_to_edges` so later sections can apply the same processing logic consistently.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# Standalone script: aggregate DEM onto an ERA5-like grid and output a per-site
# 0.25°×0.25° statistics table, with sites snapped to the nearest cell centre.

from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr

# ---------------- Config ----------------
DEM_PATH = "/Users/wavefunction/ASU Dropbox/Tanmay Singh/THz_Mac/output_hh.tif"
BBOX = (76.0, 80.0, 32.0, 36.0)   # (lon_min, lon_max, lat_min, lat_max)
RES = 0.25                        # grid resolution in degrees

OUT = Path("./era5_dem_products"); OUT.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT / f"site_stats_{RES:g}deg.csv"

SITES = [
    { 'name' : 'Hanle',        'lat' : 32.7789, 'lon' : 78.9650 },
    { 'name' : 'IAO-Hanle',    'lat' : 32.7789, 'lon' : 78.9650 },
    { 'name' : 'NLST-Merak',   'lat' : 33.7828, 'lon' : 78.57782 },
    { 'name' : 'Site A',       'lat' : 34.25,   'lon' : 78.75 },
    { 'name' : 'Site B',       'lat' : 32.5,    'lon' : 79.0 },
]

# ---------------- Utilities ----------------
def read_dem_latlon(path):
    da = rxr.open_rasterio(path, masked=True)
    if "band" in da.dims and da.sizes.get("band", 0) == 1:
        da = da.squeeze("band", drop=True)
    if da.rio.crs is None:
        raise RuntimeError("DEM has no CRS.")
    if da.rio.crs.to_epsg() != 4326:
        da = da.rio.reproject("EPSG:4326")
    return da

def clip_bbox(da, bbox):
    lon_min, lon_max, lat_min, lat_max = bbox
    return da.rio.clip_box(minx=lon_min, miny=lat_min, maxx=lon_max, maxy=lat_max)

def grid_centers(vmin, vmax, res):
    start = np.ceil((vmin - 1e-12)/res)*res
    stop  = np.floor((vmax + 1e-12)/res)*res
    if stop < start:
        return np.array([])
    n = int(round((stop - start)/res)) + 1
    return np.round(start + np.arange(n)*res, 10)

def centers_to_edges(centers, res):
    return np.round(np.concatenate(([centers[0]-res/2], centers+res/2)), 10)

# ---------------- Core resampling (RAM-lean) ----------------
def aggregate_dem_to_grid(dem_clip, lon_c, lat_c, lon_e, lat_e):
    """
    dem_clip: xarray.DataArray with dims (y, x), coords x (lon ASC), y (lat usually DESC).
    Returns: dict of grids (mean, std, min, max, count), all shaped [nlat, nlon] with lat ASC.
    """
    nlon, nlat = lon_c.size, lat_c.size
    lon_vals = dem_clip.x.values                     # ASC
    lat_vals = dem_clip.y.values                     # often DESC
    elev = dem_clip.values                           # 2D (ny, nx)

    eps = RES * 1e-6
    lon_edges = lon_e.copy(); lat_edges = lat_e.copy()
    lon_edges[0]-=eps; lon_edges[-1]+=eps; lat_edges[0]-=eps; lat_edges[-1]+=eps

    col_idx = np.searchsorted(lon_edges, lon_vals, side="right") - 1   # shape (nx,)
    valid_col = (col_idx >= 0) & (col_idx < nlon)

    sum_grid   = np.zeros((nlat, nlon), dtype=np.float64)
    sumsq_grid = np.zeros((nlat, nlon), dtype=np.float64)
    cnt_grid   = np.zeros((nlat, nlon), dtype=np.int64)
    min_grid   = np.full((nlat, nlon), np.inf, dtype=np.float64)
    max_grid   = np.full((nlat, nlon), -np.inf, dtype=np.float64)

    for j in range(lat_vals.size):
        row_idx = np.searchsorted(lat_edges, lat_vals[j], side="right") - 1
        if row_idx < 0 or row_idx >= nlat:
            continue

        row = elev[j, :].astype(np.float64)
        m = np.isfinite(row) & valid_col
        if not np.any(m):
            continue

        cols = col_idx[m]
        vals = row[m]

        np.add.at(sum_grid[row_idx],   cols, vals)
        np.add.at(sumsq_grid[row_idx], cols, vals*vals)
        np.add.at(cnt_grid[row_idx],   cols, 1)

        if cols.size:
            order = np.argsort(cols)
            cols_sorted = cols[order]
            vals_sorted = vals[order]
            ucols, start_idx = np.unique(cols_sorted, return_index=True)
            for k, c in enumerate(ucols):
                s = start_idx[k]
                e = start_idx[k+1] if k+1 < start_idx.size else cols_sorted.size
                vseg = vals_sorted[s:e]
                vmin = float(np.min(vseg)); vmax = float(np.max(vseg))
                if vmin < min_grid[row_idx, c]:
                    min_grid[row_idx, c] = vmin
                if vmax > max_grid[row_idx, c]:
                    max_grid[row_idx, c] = vmax

    with np.errstate(invalid="ignore", divide="ignore"):
        mean_grid = sum_grid / cnt_grid
        var_grid  = (sumsq_grid / cnt_grid) - mean_grid**2
        std_grid  = np.sqrt(np.maximum(var_grid, 0.0))

    empty = cnt_grid == 0
    mean_grid[empty] = np.nan
    std_grid[empty]  = np.nan
    min_grid[empty]  = np.nan
    max_grid[empty]  = np.nan

    return {
        "mean":  mean_grid,
        "std":   std_grid,
        "min":   min_grid,
        "max":   max_grid,
        "count": cnt_grid.astype(np.int32),
    }

# ---------------- Main ----------------
def main():
    # Read + clip DEM
    dem = read_dem_latlon(DEM_PATH)
    dem = clip_bbox(dem, BBOX)

    # Target grid (ASC)
    lon_c = grid_centers(BBOX[0], BBOX[1], RES)
    lat_c = grid_centers(BBOX[2], BBOX[3], RES)
    if lon_c.size == 0 or lat_c.size == 0:
        raise RuntimeError("No grid centers fall inside BBOX at this resolution.")
    lon_e = centers_to_edges(lon_c, RES)   # ASC
    lat_e = centers_to_edges(lat_c, RES)   # ASC

    # Aggregate once
    grids = aggregate_dem_to_grid(dem, lon_c, lat_c, lon_e, lat_e)

    # ---------- Per-site 0.25°×0.25° stats table (snap sites to nearest cell CENTRE) ----------
    rows = []
    for s in SITES:
        # nearest centre indices
        lon_idx = int(np.argmin(np.abs(lon_c - s["lon"])))
        lat_idx = int(np.argmin(np.abs(lat_c - s["lat"])))

        lon_center = float(lon_c[lon_idx])
        lat_center = float(lat_c[lat_idx])
        lon_left,  lon_right  = lon_center - RES/2, lon_center + RES/2
        lat_bottom, lat_top   = lat_center - RES/2, lat_center + RES/2

        mean_v = float(grids["mean"][lat_idx, lon_idx])
        std_v  = float(grids["std"][lat_idx, lon_idx])
        min_v  = float(grids["min"][lat_idx, lon_idx])
        max_v  = float(grids["max"][lat_idx, lon_idx])
        cnt_v  = int(grids["count"][lat_idx, lon_idx])

        status = "ok"
        if np.isnan(mean_v):
            status = "no_samples"

        rows.append({
            "site": s["name"],
            "input_lat": s["lat"],
            "input_lon": s["lon"],
            "cell_lat_c": lat_center,
            "cell_lon_c": lon_center,
            "box_lat_min": lat_bottom,
            "box_lat_max": lat_top,
            "box_lon_min": lon_left,
            "box_lon_max": lon_right,
            "mean_m": mean_v,
            "std_m": std_v,
            "min_m": min_v,
            "max_m": max_v,
            "n_samples": cnt_v,
            "status": status,
        })

    cols = [
        "site","input_lat","input_lon","cell_lat_c","cell_lon_c",
        "box_lat_min","box_lat_max","box_lon_min","box_lon_max",
        "mean_m","std_m","min_m","max_m","n_samples","status"
    ]
    stats_df = pd.DataFrame(rows, columns=cols)

    # Pretty print and save
    with pd.option_context('display.float_format', '{:0.2f}'.format):
        print(stats_df.to_string(index=False))
    stats_df.to_csv(OUT_CSV, index=False)
    print(f"[OK] wrote {OUT_CSV}")

if __name__ == "__main__":
    main()


## Mapping and visualization

### Step 202

This cell defines reusable helper function(s) `read_dem_latlon`, `clip_bbox`, `grid_centers`, `centers_to_edges` so later sections can apply the same processing logic consistently.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# DEM → ERA-like aggregation + native-resolution 1×4 panels around sites
# Changes per request:
# - Only leftmost subplot shows the Latitude (y) label; others hide y-label.
# - Increased vertical spacing between boxed site names and the plots.
# - Colorbar redesigned (thicker frame, inward ticks, minor ticks, rounded box,
#   extended triangle caps, integer meters, tight alignment with panels).
# - GridSpec used to keep 4 panels and colorbar perfectly aligned.

from pathlib import Path
import numpy as np
import xarray as xr
import rioxarray as rxr
import rasterio
from rasterio.transform import from_origin
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LightSource
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FormatStrFormatter, MultipleLocator, AutoMinorLocator
import matplotlib.patheffects as pe

# ---------------- Config ----------------
DEM_PATH = "/Users/wavefunction/ASU Dropbox/Tanmay Singh/THz_Mac/output_hh.tif"
BBOX = (76.0, 80.0, 32.0, 36.0)   # (lon_min, lon_max, lat_min, lat_max)
RES = 0.25

OUT = Path("./era5_dem_products"); OUT.mkdir(parents=True, exist_ok=True)
OUT_NC  = OUT / f"elevation_grid_{RES:g}.nc"
OUT_TIF = OUT / f"elevation_grid_{RES:g}_mean.tif"
OUT_PNG = OUT / f"elevation_grid_{RES:g}_mean.png"

# Native-resolution multi-panel
PANEL_HALF_DEG = 0.25
OUT_PANEL_PNG  = OUT / f"native_panels_{PANEL_HALF_DEG:.2f}deg_each.png"

SITES = [
    { 'name' : 'Hanle',      'lat' : 32.7789, 'lon' : 78.9650 },  # kept but NOT used
    { 'name' : 'IAO-Hanle',  'lat' : 32.7789, 'lon' : 78.9650 },
    { 'name' : 'Merak',      'lat' : 33.7953, 'lon' : 78.6167 },
    { 'name' : 'Site A',     'lat' : 34.25,   'lon' : 78.75 },
    { 'name' : 'Site B',     'lat' : 32.5,    'lon' : 79.0 },
]


SELECT_FOUR = ["IAO-Hanle", "Merak", "Site A", "Site B"]

# ---------------- Typography & style ----------------
try:
    mpl.rcParams.update({
        "text.usetex": True,
        "font.family": "serif",
        "font.size": 16,
        "axes.titlesize": 20,
        "axes.labelsize": 18,
        "xtick.labelsize": 15,
        "ytick.labelsize": 15,
        "legend.fontsize": 14,
        "figure.titlesize": 22,
        "axes.linewidth": 1.1,
    })
except Exception:
    mpl.rcParams.update({
        "text.usetex": False,
        "font.family": "serif",
        "font.size": 16,
        "axes.titlesize": 20,
        "axes.labelsize": 18,
        "xtick.labelsize": 15,
        "ytick.labelsize": 15,
        "legend.fontsize": 14,
        "figure.titlesize": 22,
        "axes.linewidth": 1.1,
    })

CMAP = "viridis"
CONTOUR_COLOR = "k"
CONTOUR_ALPHA = 0.32
CONTOUR_LEVELS = 14
HILLSHADE_BLEND = 'soft'   # 'overlay' is punchy but can shift hues; 'soft' is cleaner

# ---------------- Utilities ----------------
def read_dem_latlon(path):
    da = rxr.open_rasterio(path, masked=True)
    if "band" in da.dims and da.sizes.get("band", 0) == 1:
        da = da.squeeze("band", drop=True)
    if da.rio.crs is None:
        raise RuntimeError("DEM has no CRS.")
    if da.rio.crs.to_epsg() != 4326:
        da = da.rio.reproject("EPSG:4326")
    return da

def clip_bbox(da, bbox):
    lon_min, lon_max, lat_min, lat_max = bbox
    return da.rio.clip_box(minx=lon_min, miny=lat_min, maxx=lon_max, maxy=lat_max)

def grid_centers(vmin, vmax, res):
    start = np.ceil((vmin - 1e-12)/res)*res
    stop  = np.floor((vmax + 1e-12)/res)*res
    if stop < start:
        return np.array([])
    n = int(round((stop - start)/res)) + 1
    return np.round(start + np.arange(n)*res, 10)

def centers_to_edges(centers, res):
    return np.round(np.concatenate(([centers[0]-res/2], centers+res/2)), 10)

# ---------------- Aggregation (RAM-lean) ----------------
def aggregate_dem_to_grid(dem_clip, lon_c, lat_c, lon_e, lat_e):
    nlon, nlat = lon_c.size, lat_c.size
    lon_vals = dem_clip.x.values
    lat_vals = dem_clip.y.values
    elev = dem_clip.values

    eps = RES * 1e-6
    lon_edges = lon_e.copy(); lat_edges = lat_e.copy()
    lon_edges[0]-=eps; lon_edges[-1]+=eps; lat_edges[0]-=eps; lat_edges[-1]+=eps

    col_idx = np.searchsorted(lon_edges, lon_vals, side="right") - 1
    valid_col = (col_idx >= 0) & (col_idx < nlon)

    sum_grid   = np.zeros((nlat, nlon), dtype=np.float64)
    sumsq_grid = np.zeros((nlat, nlon), dtype=np.float64)
    cnt_grid   = np.zeros((nlat, nlon), dtype=np.int64)
    min_grid   = np.full((nlat, nlon), np.inf, dtype=np.float64)
    max_grid   = np.full((nlat, nlon), -np.inf, dtype=np.float64)

    for j in range(lat_vals.size):
        row_idx = np.searchsorted(lat_edges, lat_vals[j], side="right") - 1
        if row_idx < 0 or row_idx >= nlat: continue
        row = elev[j, :].astype(np.float64)
        m = np.isfinite(row) & valid_col
        if not np.any(m): continue
        cols = col_idx[m]; vals = row[m]
        np.add.at(sum_grid[row_idx],   cols, vals)
        np.add.at(sumsq_grid[row_idx], cols, vals*vals)
        np.add.at(cnt_grid[row_idx],   cols, 1)

        if cols.size:
            order = np.argsort(cols)
            cols_sorted = cols[order]; vals_sorted = vals[order]
            ucols, start_idx = np.unique(cols_sorted, return_index=True)
            for k, c in enumerate(ucols):
                s = start_idx[k]
                e = start_idx[k+1] if k+1 < start_idx.size else cols_sorted.size
                vseg = vals_sorted[s:e]
                vmin = float(np.min(vseg)); vmax = float(np.max(vseg))
                if vmin < min_grid[row_idx, c]: min_grid[row_idx, c] = vmin
                if vmax > max_grid[row_idx, c]: max_grid[row_idx, c] = vmax

    with np.errstate(invalid="ignore", divide="ignore"):
        mean_grid = sum_grid / cnt_grid
        var_grid  = (sumsq_grid / cnt_grid) - mean_grid**2
        std_grid  = np.sqrt(np.maximum(var_grid, 0.0))

    empty = cnt_grid == 0
    for A in (mean_grid, std_grid, min_grid, max_grid): A[empty] = np.nan

    return {
        "mean":  mean_grid,
        "std":   std_grid,
        "min":   min_grid,
        "max":   max_grid,
        "count": cnt_grid.astype(np.int32),
    }

# ---------------- Native-resolution helpers ----------------
def _panel_clip_native(dem_da, lon_c, lat_c, half_deg):
    lon_min = lon_c - half_deg; lon_max = lon_c + half_deg
    lat_min = lat_c - half_deg; lat_max = lat_c + half_deg
    return dem_da.rio.clip_box(minx=lon_min, miny=lat_min, maxx=lon_max, maxy=lat_max)

def _extent_from_da(da):
    xs = da.x.values; ys = da.y.values
    return [float(xs.min()), float(xs.max()), float(ys.min()), float(ys.max())]

def _robust_norm(patches):
    v_all = []
    vmins, vmaxs = [], []
    for _, p in patches:
        v = np.asarray(p.values)
        v = v[np.isfinite(v)]
        if v.size:
            v_all.append(v.ravel())
            vmins.append(np.nanmin(v)); vmaxs.append(np.nanmax(v))
    if not v_all: raise RuntimeError("No valid DEM data in requested windows.")
    v_all = np.concatenate(v_all)
    lo = float(np.nanpercentile(v_all, 2.0))
    hi = float(np.nanpercentile(v_all, 98.0))
    vmin = max(min(vmins), lo)
    vmax = min(max(vmaxs), hi)
    if vmin >= vmax: vmin, vmax = min(vmins), max(vmaxs)
    return Normalize(vmin=vmin, vmax=vmax), (vmin, vmax)

def _add_boxed_title(ax, text):
    # Increased pad to make extra vertical space above the main axes
    ax.set_title(
        rf"\textbf{{{text}}}",
        bbox=dict(boxstyle="round,pad=0.45,rounding_size=0.8",
                  fc="white", ec="0.2", lw=0.9),
        pad=16   # <-- more spacing between title box and plot
    )

def _add_panel_letter(ax, letter):
    ax.text(0.02, 0.975, rf"\textbf{{({letter})}}",
            transform=ax.transAxes, ha="left", va="top")

def _add_scalebar(ax, center_lat_deg, bar_km=20.0, pad=0.035, height=0.008, color="k"):
    km_per_deg_lon = 111.32 * np.cos(np.deg2rad(center_lat_deg))
    bar_deg_lon = bar_km / max(km_per_deg_lon, 1e-9)

    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    x_start = x0 + 0.06 * (x1 - x0)
    x_end   = x_start + bar_deg_lon
    y = y0 + pad * (y1 - y0)
    ax.add_patch(mpl.patches.Rectangle(
        (x_start, y), bar_deg_lon, height*(y1-y0),
        facecolor=color, edgecolor="none", alpha=0.9
    ))
    ax.text((x_start+x_end)/2, y + 2.2*height*(y1-y0),
            rf"$\approx\,{int(bar_km)}\,\mathrm{{km}}$",
            ha="center", va="bottom")

def _add_north_arrow(ax, size=0.05, color="k"):
    ax.annotate("",
        xy=(0.06, 0.90), xytext=(0.06, 0.90 - size),
        xycoords="axes fraction", textcoords="axes fraction",
        arrowprops=dict(arrowstyle="-|>", color=color, lw=1.1))
    ax.text(0.06, 0.905, r"\textbf{N}", ha="center", va="bottom",
            transform=ax.transAxes, color=color)

# ---------------- Native-resolution 1×4 panel plot ----------------
def plot_native_site_panels(dem_da, sites, half_deg, out_png):
    chosen = [next(s for s in sites if s["name"] == q) for q in SELECT_FOUR]
    patches = [(s, _panel_clip_native(dem_da, s["lon"], s["lat"], half_deg)) for s in chosen]
    norm, (vmin, vmax) = _robust_norm(patches)



    fig = plt.figure(figsize=(22, 6.0), constrained_layout=False)
    # ---- GridSpec ----
    gs = GridSpec(nrows=1, ncols=5, width_ratios=[1, 1, 1, 1, 0.05],
                  wspace=0.20, left=0.055, right=0.965, bottom=0.115, top=0.915)
    axs = [fig.add_subplot(gs[0, i]) for i in range(4)]
    cax = fig.add_subplot(gs[0, 4])

    ls = LightSource(azdeg=315, altdeg=45)
    cmap = mpl.cm.get_cmap(CMAP)

    for i, (ax, (s, p)) in enumerate(zip(axs, patches), start=1):
        Z = np.asarray(p.values)
        ext = _extent_from_da(p)

        # hillshade + color
        rgb = ls.shade(Z, cmap=cmap, vert_exag=1.0, vmin=vmin, vmax=vmax, blend_mode=HILLSHADE_BLEND)
        ax.imshow(rgb, origin="upper", extent=ext, interpolation="nearest")

        # contours
        try:
            levels = np.linspace(vmin, vmax, CONTOUR_LEVELS)
            ax.contour(Z, levels=levels, origin="upper", extent=ext,
                       linewidths=0.45, colors=CONTOUR_COLOR, alpha=CONTOUR_ALPHA)
        except Exception:
            pass

        # white center cross with black stroke
        cross, = ax.plot(s["lon"], s["lat"], marker="+", markersize=12,
                         markeredgewidth=2.0, color="white", zorder=6, linestyle="None")
        cross.set_path_effects([pe.Stroke(linewidth=3.0, foreground="black"), pe.Normal()])

        _add_boxed_title(ax, s["name"])
        # _add_panel_letter(ax, "abcd"[i-1])

        # labels and ticks
        ax.set_xlabel(r"$\mathrm{Longitude}\ (^\circ\mathrm{E})$")
        if i == 1:
            ax.set_ylabel(r"$\mathrm{Latitude}\ (^\circ\mathrm{N})$")
        else:
            ax.set_ylabel("")  # remove y-label on others

        # ticks every 0.10°, formatted to 2 decimals
        dl = 0.1
        xt = np.round(np.arange(s["lon"]-half_deg, s["lon"]+half_deg+1e-9, dl), 3)
        yt = np.round(np.arange(s["lat"]-half_deg, s["lat"]+half_deg+1e-9, dl), 3)
        ax.set_xticks(xt); ax.set_yticks(yt)
        ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        ax.tick_params(axis="both", which="both", direction="out", length=5, width=1.0)
        for spine in ax.spines.values(): spine.set_linewidth(1.1)

        _add_north_arrow(ax, size=0.07)
        _add_scalebar(ax, center_lat_deg=s["lat"], bar_km=20.0)

    # Shared, styled colorbar aligned with panel stack
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    cb = fig.colorbar(sm, cax=cax, extend="both", pad=0.01)
    cb.outline.set_linewidth(1.2)
    cb.outline.set_joinstyle("round")
    cb.ax.tick_params(direction="in", length=5, width=1.0, which="major")
    cb.ax.tick_params(direction="in", length=3, width=0.8, which="minor")
    cb.ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    # integer meters on ticks (no decimals)
    cb.formatter = FormatStrFormatter("%d")
    cb.update_ticks()
    cb.set_label(r"$\mathrm{Elevation}\ (\mathrm{m})$", rotation=90, labelpad=10)

    fig.savefig(
        out_png,
        dpi=100,)
    plt.close(fig)
    print(f"[OK] wrote {out_png}")

# ---------------- Main (aggregation part kept intact) ----------------
def main():
    dem_full = read_dem_latlon(DEM_PATH)
    dem = clip_bbox(dem_full, BBOX)

    lon_c = grid_centers(BBOX[0], BBOX[1], RES)
    lat_c = grid_centers(BBOX[2], BBOX[3], RES)
    if lon_c.size == 0 or lat_c.size == 0:
        raise RuntimeError("No grid centers fall inside BBOX at this resolution.")
    lon_e = centers_to_edges(lon_c, RES)
    lat_e = centers_to_edges(lat_c, RES)

    grids = aggregate_dem_to_grid(dem, lon_c, lat_c, lon_e, lat_e)

    # Save NetCDF
    ds = xr.Dataset(
        data_vars=dict(
            orog_mean=(["latitude","longitude"], grids["mean"].astype("float32")),
            orog_std =( ["latitude","longitude"], grids["std"].astype("float32")),
            orog_min =( ["latitude","longitude"], grids["min"].astype("float32")),
            orog_max =( ["latitude","longitude"], grids["max"].astype("float32")),
            orog_n   =( ["latitude","longitude"], grids["count"].astype("int32")),
        ),
        coords=dict(latitude=lat_c, longitude=lon_c),
        attrs=dict(
            title=f"DEM aggregated to {RES:g}° grid (mean/std/min/max/count)",
            grid_resolution=f"{RES:g} degree",
            source_tif=str(DEM_PATH),
        ),
    )
    ds.to_netcdf(OUT_NC); print(f"[OK] wrote {OUT_NC}")

    # Save mean GeoTIFF (flip lat to north-up)
    mean_for_tif = np.flipud(grids["mean"]).astype("float32")
    transform = from_origin(lon_e[0], lat_e[-1], RES, RES)
    profile = dict(driver="GTiff", height=mean_for_tif.shape[0], width=mean_for_tif.shape[1],
                   count=1, dtype="float32", crs="EPSG:4326", transform=transform,
                   compress="deflate", nodata=np.nan)
    with rasterio.open(OUT_TIF, "w", **profile) as dst:
        dst.write(mean_for_tif, 1)
    print(f"[OK] wrote {OUT_TIF}")

    # Quick full-BBOX mean plot
    lon_plot = np.concatenate(([lon_c[0]-RES/2], lon_c+RES/2))
    lat_plot = np.concatenate(([lat_c[0]-RES/2], lat_c+RES/2))
    fig, ax = plt.subplots(figsize=(8.5,7.2))
    pm = ax.pcolormesh(lon_plot, lat_plot, grids["mean"], shading="auto", cmap=CMAP)
    cb = fig.colorbar(pm, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label(r"$\mathrm{Mean\ elevation}\ (\mathrm{m})$")
    ax.set_xlim(BBOX[0], BBOX[1]); ax.set_ylim(BBOX[2], BBOX[3])
    ax.set_xlabel(r"$\mathrm{Longitude}\ (^\circ\mathrm{E})$")
    ax.set_ylabel(r"$\mathrm{Latitude}\ (^\circ\mathrm{N})$")
    ax.set_title(rf"$\mathrm{{DEM\ mean\ on}}\ {RES:g}^\circ\ \mathrm{{grid}}$")
    fig.tight_layout(); fig.savefig(OUT_PNG, dpi=200); plt.close(fig)
    print(f"[OK] wrote {OUT_PNG}")

    # Native-resolution 1×4 panels (IAO-Hanle, Merak, Site A, Site B)
    plot_native_site_panels(dem, SITES, PANEL_HALF_DEG, OUT_PANEL_PNG)

if __name__ == "__main__":
    main()


### Step 203

This cell defines reusable helper function(s) `read_dem_latlon`, `clip_bbox`, `grid_centers`, `centers_to_edges` so later sections can apply the same processing logic consistently.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# DEM → ERA-like aggregation + native-resolution 1×4 panels around sites
# Changes per request:
# - Only leftmost subplot shows the Latitude (y) label; others hide y-label.
# - Increased vertical spacing between boxed site names and the plots.
# - Colorbar redesigned (thicker frame, inward ticks, minor ticks, rounded box,
#   extended triangle caps, integer meters, tight alignment with panels).
# - GridSpec used to keep 4 panels and colorbar perfectly aligned.

from pathlib import Path
import numpy as np
import xarray as xr
import rioxarray as rxr
import rasterio
from rasterio.transform import from_origin
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LightSource
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FormatStrFormatter, MultipleLocator, AutoMinorLocator
import matplotlib.patheffects as pe

# ---------------- Config ----------------
DEM_PATH = "/Users/wavefunction/ASU Dropbox/Tanmay Singh/THz_Mac/output_hh.tif"
BBOX = (76.0, 80.0, 32.0, 36.0)   # (lon_min, lon_max, lat_min, lat_max)
RES = 0.25

OUT = Path("./era5_dem_products"); OUT.mkdir(parents=True, exist_ok=True)
OUT_NC  = OUT / f"elevation_grid_{RES:g}.nc"
OUT_TIF = OUT / f"elevation_grid_{RES:g}_mean.tif"
OUT_PNG = OUT / f"elevation_grid_{RES:g}_mean.png"

# Native-resolution multi-panel
PANEL_HALF_DEG = 0.25
OUT_PANEL_PNG  = OUT / f"native_panels_{PANEL_HALF_DEG:.2f}deg_each.png"

SITES = [
    { 'name' : 'Hanle',      'lat' : 32.7789, 'lon' : 78.9650 },  # kept but NOT used
    { 'name' : 'IAO-Hanle',  'lat' : 32.7789, 'lon' : 78.9650 },
    { 'name' : 'Merak',      'lat' : 33.7953, 'lon' : 78.6167 },
    { 'name' : 'Site A',     'lat' : 34.25,   'lon' : 78.75 },
    { 'name' : 'Site B',     'lat' : 32.5,    'lon' : 79.0 },
]

SELECT_FOUR = ["IAO-Hanle", "Merak", "Site A", "Site B"]

# ---------------- Typography & style ----------------
try:
    mpl.rcParams.update({
        "text.usetex": True,
        "font.family": "serif",
        "font.size": 16,
        "axes.titlesize": 20,
        "axes.labelsize": 18,
        "xtick.labelsize": 15,
        "ytick.labelsize": 15,
        "legend.fontsize": 14,
        "figure.titlesize": 22,
        "axes.linewidth": 1.1,
    })
except Exception:
    mpl.rcParams.update({
        "text.usetex": False,
        "font.family": "serif",
        "font.size": 16,
        "axes.titlesize": 20,
        "axes.labelsize": 18,
        "xtick.labelsize": 15,
        "ytick.labelsize": 15,
        "legend.fontsize": 14,
        "figure.titlesize": 22,
        "axes.linewidth": 1.1,
    })

CMAP = "viridis"
CONTOUR_COLOR = "k"
CONTOUR_ALPHA = 0.32
CONTOUR_LEVELS = 14
HILLSHADE_BLEND = 'soft'   # 'overlay' is punchy but can shift hues; 'soft' is cleaner

# ---------------- Utilities ----------------
def read_dem_latlon(path):
    da = rxr.open_rasterio(path, masked=True)
    if "band" in da.dims and da.sizes.get("band", 0) == 1:
        da = da.squeeze("band", drop=True)
    if da.rio.crs is None:
        raise RuntimeError("DEM has no CRS.")
    if da.rio.crs.to_epsg() != 4326:
        da = da.rio.reproject("EPSG:4326")
    return da

def clip_bbox(da, bbox):
    lon_min, lon_max, lat_min, lat_max = bbox
    return da.rio.clip_box(minx=lon_min, miny=lat_min, maxx=lon_max, maxy=lat_max)

def grid_centers(vmin, vmax, res):
    start = np.ceil((vmin - 1e-12)/res)*res
    stop  = np.floor((vmax + 1e-12)/res)*res
    if stop < start:
        return np.array([])
    n = int(round((stop - start)/res)) + 1
    return np.round(start + np.arange(n)*res, 10)

def centers_to_edges(centers, res):
    return np.round(np.concatenate(([centers[0]-res/2], centers+res/2)), 10)

# ---------------- Aggregation (RAM-lean) ----------------
def aggregate_dem_to_grid(dem_clip, lon_c, lat_c, lon_e, lat_e):
    nlon, nlat = lon_c.size, lat_c.size
    lon_vals = dem_clip.x.values
    lat_vals = dem_clip.y.values
    elev = dem_clip.values

    eps = RES * 1e-6
    lon_edges = lon_e.copy(); lat_edges = lat_e.copy()
    lon_edges[0]-=eps; lon_edges[-1]+=eps; lat_edges[0]-=eps; lat_edges[-1]+=eps

    col_idx = np.searchsorted(lon_edges, lon_vals, side="right") - 1
    valid_col = (col_idx >= 0) & (col_idx < nlon)

    sum_grid   = np.zeros((nlat, nlon), dtype=np.float64)
    sumsq_grid = np.zeros((nlat, nlon), dtype=np.float64)
    cnt_grid   = np.zeros((nlat, nlon), dtype=np.int64)
    min_grid   = np.full((nlat, nlon), np.inf, dtype=np.float64)
    max_grid   = np.full((nlat, nlon), -np.inf, dtype=np.float64)

    for j in range(lat_vals.size):
        row_idx = np.searchsorted(lat_edges, lat_vals[j], side="right") - 1
        if row_idx < 0 or row_idx >= nlat: continue
        row = elev[j, :].astype(np.float64)
        m = np.isfinite(row) & valid_col
        if not np.any(m): continue
        cols = col_idx[m]; vals = row[m]
        np.add.at(sum_grid[row_idx],   cols, vals)
        np.add.at(sumsq_grid[row_idx], cols, vals*vals)
        np.add.at(cnt_grid[row_idx],   cols, 1)

        if cols.size:
            order = np.argsort(cols)
            cols_sorted = cols[order]; vals_sorted = vals[order]
            ucols, start_idx = np.unique(cols_sorted, return_index=True)
            for k, c in enumerate(ucols):
                s = start_idx[k]
                e = start_idx[k+1] if k+1 < start_idx.size else cols_sorted.size
                vseg = vals_sorted[s:e]
                vmin = float(np.min(vseg)); vmax = float(np.max(vseg))
                if vmin < min_grid[row_idx, c]: min_grid[row_idx, c] = vmin
                if vmax > max_grid[row_idx, c]: max_grid[row_idx, c] = vmax

    with np.errstate(invalid="ignore", divide="ignore"):
        mean_grid = sum_grid / cnt_grid
        var_grid  = (sumsq_grid / cnt_grid) - mean_grid**2
        std_grid  = np.sqrt(np.maximum(var_grid, 0.0))

    empty = cnt_grid == 0
    for A in (mean_grid, std_grid, min_grid, max_grid): A[empty] = np.nan

    return {
        "mean":  mean_grid,
        "std":   std_grid,
        "min":   min_grid,
        "max":   max_grid,
        "count": cnt_grid.astype(np.int32),
    }

# ---------------- Native-resolution helpers ----------------
def _panel_clip_native(dem_da, lon_c, lat_c, half_deg):
    lon_min = lon_c - half_deg; lon_max = lon_c + half_deg
    lat_min = lat_c - half_deg; lat_max = lat_c + half_deg
    return dem_da.rio.clip_box(minx=lon_min, miny=lat_min, maxx=lon_max, maxy=lat_max)

def _extent_from_da(da):
    xs = da.x.values; ys = da.y.values
    return [float(xs.min()), float(xs.max()), float(ys.min()), float(ys.max())]

def _robust_norm(patches):
    v_all = []
    vmins, vmaxs = [], []
    for _, p in patches:
        v = np.asarray(p.values)
        v = v[np.isfinite(v)]
        if v.size:
            v_all.append(v.ravel())
            vmins.append(np.nanmin(v)); vmaxs.append(np.nanmax(v))
    if not v_all: raise RuntimeError("No valid DEM data in requested windows.")
    v_all = np.concatenate(v_all)
    lo = float(np.nanpercentile(v_all, 2.0))
    hi = float(np.nanpercentile(v_all, 98.0))
    vmin = max(min(vmins), lo)
    vmax = min(max(vmaxs), hi)
    if vmin >= vmax: vmin, vmax = min(vmins), max(vmaxs)
    return Normalize(vmin=vmin, vmax=vmax), (vmin, vmax)

def _add_boxed_title(ax, text):
    ax.set_title(
        rf"\textbf{{{text}}}",
        bbox=dict(boxstyle="round,pad=0.45,rounding_size=0.8",
                  fc="white", ec="0.2", lw=0.9),
        pad=16
    )

def _add_panel_letter(ax, letter):
    ax.text(0.02, 0.975, rf"\textbf{{({letter})}}",
            transform=ax.transAxes, ha="left", va="top")

def _add_scalebar(ax, center_lat_deg, bar_km=20.0, pad=0.035, height=0.008, color="k"):
    km_per_deg_lon = 111.32 * np.cos(np.deg2rad(center_lat_deg))
    bar_deg_lon = bar_km / max(km_per_deg_lon, 1e-9)

    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    x_start = x0 + 0.06 * (x1 - x0)
    x_end   = x_start + bar_deg_lon
    y = y0 + pad * (y1 - y0)
    ax.add_patch(mpl.patches.Rectangle(
        (x_start, y), bar_deg_lon, height*(y1-y0),
        facecolor=color, edgecolor="none", alpha=0.9
    ))
    ax.text((x_start+x_end)/2, y + 2.2*height*(y1-y0),
            rf"$\approx\,{int(bar_km)}\,\mathrm{{km}}$",
            ha="center", va="bottom")

def _add_north_arrow(ax, size=0.05, color="k"):
    ax.annotate("",
        xy=(0.06, 0.90), xytext=(0.06, 0.90 - size),
        xycoords="axes fraction", textcoords="axes fraction",
        arrowprops=dict(arrowstyle="-|>", color=color, lw=1.1))
    ax.text(0.06, 0.905, r"\textbf{N}", ha="center", va="bottom",
            transform=ax.transAxes, color=color)

# ---------------- Native-resolution 1×4 panel plot ----------------
def plot_native_site_panels(dem_da, sites, half_deg, out_png):
    chosen = [next(s for s in sites if s["name"] == q) for q in SELECT_FOUR]
    patches = [(s, _panel_clip_native(dem_da, s["lon"], s["lat"], half_deg)) for s in chosen]
    norm, (vmin, vmax) = _robust_norm(patches)

    fig = plt.figure(figsize=(22, 6.0), constrained_layout=False)
    # GridSpec: reduce gap between last subplot and colorbar via smaller wspace;
    # keep everything else identical.
    gs = GridSpec(nrows=1, ncols=5, width_ratios=[1, 1, 1, 1, 0.05],
                  wspace=0.20, left=0.055, right=0.965, bottom=0.115, top=0.915)
    axs = [fig.add_subplot(gs[0, i]) for i in range(4)]
    cax = fig.add_subplot(gs[0, 4])

    ls = LightSource(azdeg=315, altdeg=45)
    cmap = mpl.cm.get_cmap(CMAP)

    for i, (ax, (s, p)) in enumerate(zip(axs, patches), start=1):
        Z = np.asarray(p.values)
        ext = _extent_from_da(p)

        rgb = ls.shade(Z, cmap=cmap, vert_exag=1.0, vmin=vmin, vmax=vmax, blend_mode=HILLSHADE_BLEND)
        ax.imshow(rgb, origin="upper", extent=ext, interpolation="nearest")

        try:
            levels = np.linspace(vmin, vmax, CONTOUR_LEVELS)
            ax.contour(Z, levels=levels, origin="upper", extent=ext,
                       linewidths=0.45, colors=CONTOUR_COLOR, alpha=CONTOUR_ALPHA)
        except Exception:
            pass

        cross, = ax.plot(s["lon"], s["lat"], marker="+", markersize=12,
                         markeredgewidth=2.0, color="white", zorder=6, linestyle="None")
        cross.set_path_effects([pe.Stroke(linewidth=3.0, foreground="black"), pe.Normal()])

        _add_boxed_title(ax, s["name"])

        ax.set_xlabel(r"$\mathrm{Longitude}\ (^\circ\mathrm{E})$")
        if i == 1:
            ax.set_ylabel(r"$\mathrm{Latitude}\ (^\circ\mathrm{N})$")
        else:
            ax.set_ylabel("")

        dl = 0.1
        xt = np.round(np.arange(s["lon"]-half_deg, s["lon"]+half_deg+1e-9, dl), 3)
        yt = np.round(np.arange(s["lat"]-half_deg, s["lat"]+half_deg+1e-9, dl), 3)
        ax.set_xticks(xt); ax.set_yticks(yt)
        ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        ax.tick_params(axis="both", which="both", direction="out", length=5, width=1.0)
        for spine in ax.spines.values(): spine.set_linewidth(1.1)

        _add_north_arrow(ax, size=0.07)
        _add_scalebar(ax, center_lat_deg=s["lat"], bar_km=20.0)

    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    cb = fig.colorbar(sm, cax=cax, extend="both", pad=0.01)
    cb.outline.set_linewidth(1.2)
    cb.outline.set_joinstyle("round")
    cb.ax.tick_params(direction="in", length=5, width=1.0, which="major")
    cb.ax.tick_params(direction="in", length=3, width=0.8, which="minor")
    cb.ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    cb.formatter = FormatStrFormatter("%d")
    cb.update_ticks()
    cb.set_label(r"$\mathrm{Elevation}\ (\mathrm{m})$", rotation=90, labelpad=10)

    # Save with extra white space outside the colorbar (while keeping the tighter gap to it)
    fig.savefig(
        out_png,
        dpi=200,
        bbox_inches="tight",
        pad_inches=0.25,
        facecolor="white",
        bbox_extra_artists=[cb.ax],
    )
    plt.close(fig)
    print(f"[OK] wrote {out_png}")

# ---------------- Main (aggregation part kept intact) ----------------
def main():
    dem_full = read_dem_latlon(DEM_PATH)
    dem = clip_bbox(dem_full, BBOX)

    lon_c = grid_centers(BBOX[0], BBOX[1], RES)
    lat_c = grid_centers(BBOX[2], BBOX[3], RES)
    if lon_c.size == 0 or lat_c.size == 0:
        raise RuntimeError("No grid centers fall inside BBOX at this resolution.")
    lon_e = centers_to_edges(lon_c, RES)
    lat_e = centers_to_edges(lat_c, RES)

    grids = aggregate_dem_to_grid(dem, lon_c, lat_c, lon_e, lat_e)

    ds = xr.Dataset(
        data_vars=dict(
            orog_mean=(["latitude","longitude"], grids["mean"].astype("float32")),
            orog_std =( ["latitude","longitude"], grids["std"].astype("float32")),
            orog_min =( ["latitude","longitude"], grids["min"].astype("float32")),
            orog_max =( ["latitude","longitude"], grids["max"].astype("float32")),
            orog_n   =( ["latitude","longitude"], grids["count"].astype("int32")),
        ),
        coords=dict(latitude=lat_c, longitude=lon_c),
        attrs=dict(
            title=f"DEM aggregated to {RES:g}° grid (mean/std/min/max/count)",
            grid_resolution=f"{RES:g} degree",
            source_tif=str(DEM_PATH),
        ),
    )
    ds.to_netcdf(OUT_NC); print(f"[OK] wrote {OUT_NC}")

    mean_for_tif = np.flipud(grids["mean"]).astype("float32")
    transform = from_origin(lon_e[0], lat_e[-1], RES, RES)
    profile = dict(driver="GTiff", height=mean_for_tif.shape[0], width=mean_for_tif.shape[1],
                   count=1, dtype="float32", crs="EPSG:4326", transform=transform,
                   compress="deflate", nodata=np.nan)
    with rasterio.open(OUT_TIF, "w", **profile) as dst:
        dst.write(mean_for_tif, 1)
    print(f"[OK] wrote {OUT_TIF}")

    lon_plot = np.concatenate(([lon_c[0]-RES/2], lon_c+RES/2))
    lat_plot = np.concatenate(([lat_c[0]-RES/2], lat_c+RES/2))
    fig, ax = plt.subplots(figsize=(8.5,7.2))
    pm = ax.pcolormesh(lon_plot, lat_plot, grids["mean"], shading="auto", cmap=CMAP)
    cb = fig.colorbar(pm, ax=ax, fraction=0.046, pad=0.4)  # leave as-is
    cb.set_label(r"$\mathrm{Mean\ elevation}\ (\mathrm{m})$")
    ax.set_xlim(BBOX[0], BBOX[1]); ax.set_ylim(BBOX[2], BBOX[3])
    ax.set_xlabel(r"$\mathrm{Longitude}\ (^\circ\mathrm{E})$")
    ax.set_ylabel(r"$\mathrm{Latitude}\ (^\circ\mathrm{N})$")
    ax.set_title(rf"$\mathrm{{DEM\ mean\ on}}\ {RES:g}^\circ\ \mathrm{{grid}}$")
    fig.tight_layout(); fig.savefig(OUT_PNG, dpi=200); plt.close(fig)
    print(f"[OK] wrote {OUT_PNG}")

    plot_native_site_panels(dem, SITES, PANEL_HALF_DEG, OUT_PANEL_PNG)

if __name__ == "__main__":
    main()


### Step 204

This cell defines reusable helper function(s) `read_dem_latlon`, `clip_bbox`, `grid_centers`, `centers_to_edges` so later sections can apply the same processing logic consistently.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# DEM → ERA-like aggregation + native-resolution 1×4 panels around sites
# Changes per request:
# - Only leftmost subplot shows the Latitude (y) label; others hide y-label.
# - Increased vertical spacing between boxed site names and the plots.
# - Colorbar redesigned (thicker frame, inward ticks, minor ticks, rounded box,
#   extended triangle caps, integer meters, tight alignment with panels).
# - GridSpec used to keep 4 panels and colorbar perfectly aligned.
# - ADD: draw a white 0.25°×0.25° square centered on each site (ERA5 pixel size).

from pathlib import Path
import numpy as np
import xarray as xr
import rioxarray as rxr
import rasterio
from rasterio.transform import from_origin
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LightSource
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FormatStrFormatter, MultipleLocator, AutoMinorLocator
import matplotlib.patheffects as pe

# ---------------- Config ----------------
DEM_PATH = "/Users/wavefunction/ASU Dropbox/Tanmay Singh/THz_Mac/output_hh.tif"
BBOX = (76.0, 80.0, 32.0, 36.0)   # (lon_min, lon_max, lat_min, lat_max)
RES = 0.25

OUT = Path("./era5_dem_products"); OUT.mkdir(parents=True, exist_ok=True)
OUT_NC  = OUT / f"elevation_grid_{RES:g}.nc"
OUT_TIF = OUT / f"elevation_grid_{RES:g}_mean.tif"
OUT_PNG = OUT / f"elevation_grid_{RES:g}_mean.png"

# Native-resolution multi-panel
PANEL_HALF_DEG = 0.25
OUT_PANEL_PNG  = OUT / f"native_panels_{PANEL_HALF_DEG:.2f}deg_each.png"

SITES = [
    { 'name' : 'Hanle',      'lat' : 32.7789, 'lon' : 78.9650 },  # kept but NOT used
    { 'name' : 'IAO-Hanle',  'lat' : 32.7789, 'lon' : 78.9650 },
    { 'name' : 'NLST-Merak',      'lat' : 33.7953, 'lon' : 78.6167 },
    { 'name' : 'Site A',     'lat' : 34.25,   'lon' : 78.75 },
    { 'name' : 'Site B',     'lat' : 32.5,    'lon' : 79.0 },
]

SELECT_FOUR = ["IAO-Hanle", "NLST-Merak", "Site A", "Site B"]

# ---------------- Typography & style ----------------
try:
    mpl.rcParams.update({
        "text.usetex": True,
        "font.family": "serif",
        "font.size": 16,
        "axes.titlesize": 20,
        "axes.labelsize": 18,
        "xtick.labelsize": 15,
        "ytick.labelsize": 15,
        "legend.fontsize": 14,
        "figure.titlesize": 22,
        "axes.linewidth": 1.1,
    })
except Exception:
    mpl.rcParams.update({
        "text.usetex": False,
        "font.family": "serif",
        "font.size": 16,
        "axes.titlesize": 20,
        "axes.labelsize": 18,
        "xtick.labelsize": 15,
        "ytick.labelsize": 15,
        "legend.fontsize": 14,
        "figure.titlesize": 22,
        "axes.linewidth": 1.1,
    })

CMAP = "viridis"
# CMAP = "terrain"    
CONTOUR_COLOR = "k"
CONTOUR_ALPHA = 0.32
CONTOUR_LEVELS = 14
HILLSHADE_BLEND = 'soft'   # 'overlay' is punchy but can shift hues; 'soft' is cleaner

# ---------------- Utilities ----------------
def read_dem_latlon(path):
    da = rxr.open_rasterio(path, masked=True)
    if "band" in da.dims and da.sizes.get("band", 0) == 1:
        da = da.squeeze("band", drop=True)
    if da.rio.crs is None:
        raise RuntimeError("DEM has no CRS.")
    if da.rio.crs.to_epsg() != 4326:
        da = da.rio.reproject("EPSG:4326")
    return da

def clip_bbox(da, bbox):
    lon_min, lon_max, lat_min, lat_max = bbox
    return da.rio.clip_box(minx=lon_min, miny=lat_min, maxx=lon_max, maxy=lat_max)

def grid_centers(vmin, vmax, res):
    start = np.ceil((vmin - 1e-12)/res)*res
    stop  = np.floor((vmax + 1e-12)/res)*res
    if stop < start:
        return np.array([])
    n = int(round((stop - start)/res)) + 1
    return np.round(start + np.arange(n)*res, 10)

def centers_to_edges(centers, res):
    return np.round(np.concatenate(([centers[0]-res/2], centers+res/2)), 10)

# ---------------- Aggregation (RAM-lean) ----------------
def aggregate_dem_to_grid(dem_clip, lon_c, lat_c, lon_e, lat_e):
    nlon, nlat = lon_c.size, lat_c.size
    lon_vals = dem_clip.x.values
    lat_vals = dem_clip.y.values
    elev = dem_clip.values

    eps = RES * 1e-6
    lon_edges = lon_e.copy(); lat_edges = lat_e.copy()
    lon_edges[0]-=eps; lon_edges[-1]+=eps; lat_edges[0]-=eps; lat_edges[-1]+=eps

    col_idx = np.searchsorted(lon_edges, lon_vals, side="right") - 1
    valid_col = (col_idx >= 0) & (col_idx < nlon)

    sum_grid   = np.zeros((nlat, nlon), dtype=np.float64)
    sumsq_grid = np.zeros((nlat, nlon), dtype=np.float64)
    cnt_grid   = np.zeros((nlat, nlon), dtype=np.int64)
    min_grid   = np.full((nlat, nlon), np.inf, dtype=np.float64)
    max_grid   = np.full((nlat, nlon), -np.inf, dtype=np.float64)

    for j in range(lat_vals.size):
        row_idx = np.searchsorted(lat_edges, lat_vals[j], side="right") - 1
        if row_idx < 0 or row_idx >= nlat: continue
        row = elev[j, :].astype(np.float64)
        m = np.isfinite(row) & valid_col
        if not np.any(m): continue
        cols = col_idx[m]; vals = row[m]
        np.add.at(sum_grid[row_idx],   cols, vals)
        np.add.at(sumsq_grid[row_idx], cols, vals*vals)
        np.add.at(cnt_grid[row_idx],   cols, 1)

        if cols.size:
            order = np.argsort(cols)
            cols_sorted = cols[order]; vals_sorted = vals[order]
            ucols, start_idx = np.unique(cols_sorted, return_index=True)
            for k, c in enumerate(ucols):
                s = start_idx[k]
                e = start_idx[k+1] if k+1 < start_idx.size else cols_sorted.size
                vseg = vals_sorted[s:e]
                vmin = float(np.min(vseg)); vmax = float(np.max(vseg))
                if vmin < min_grid[row_idx, c]: min_grid[row_idx, c] = vmin
                if vmax > max_grid[row_idx, c]: max_grid[row_idx, c] = vmax

    with np.errstate(invalid="ignore", divide="ignore"):
        mean_grid = sum_grid / cnt_grid
        var_grid  = (sumsq_grid / cnt_grid) - mean_grid**2
        std_grid  = np.sqrt(np.maximum(var_grid, 0.0))

    empty = cnt_grid == 0
    for A in (mean_grid, std_grid, min_grid, max_grid): A[empty] = np.nan

    return {
        "mean":  mean_grid,
        "std":   std_grid,
        "min":   min_grid,
        "max":   max_grid,
        "count": cnt_grid.astype(np.int32),
    }

# ---------------- Native-resolution helpers ----------------
def _panel_clip_native(dem_da, lon_c, lat_c, half_deg):
    lon_min = lon_c - half_deg; lon_max = lon_c + half_deg
    lat_min = lat_c - half_deg; lat_max = lat_c + half_deg
    return dem_da.rio.clip_box(minx=lon_min, miny=lat_min, maxx=lon_max, maxy=lat_max)

def _extent_from_da(da):
    xs = da.x.values; ys = da.y.values
    return [float(xs.min()), float(xs.max()), float(ys.min()), float(ys.max())]

def _robust_norm(patches):
    v_all = []
    vmins, vmaxs = [], []
    for _, p in patches:
        v = np.asarray(p.values)
        v = v[np.isfinite(v)]
        if v.size:
            v_all.append(v.ravel())
            vmins.append(np.nanmin(v)); vmaxs.append(np.nanmax(v))
    if not v_all: raise RuntimeError("No valid DEM data in requested windows.")
    v_all = np.concatenate(v_all)
    lo = float(np.nanpercentile(v_all, 2.0))
    hi = float(np.nanpercentile(v_all, 98.0))
    vmin = max(min(vmins), lo)
    vmin = 4000 # to fix colorbar issue
    vmax = min(max(vmaxs), hi)
    if vmin >= vmax: vmin, vmax = min(vmins), max(vmaxs)
    return Normalize(vmin=vmin, vmax=vmax), (vmin, vmax)

def _add_boxed_title(ax, text):
    ax.set_title(
        rf"\textbf{{{text}}}",
        bbox=dict(boxstyle="round,pad=0.45,rounding_size=0.8",
                  fc="white", ec="0.2", lw=0.9),
        pad=16
    )

def _add_panel_letter(ax, letter):
    ax.text(0.02, 0.975, rf"\textbf{{({letter})}}",
            transform=ax.transAxes, ha="left", va="top")

def _add_scalebar(ax, center_lat_deg, bar_km=20.0, pad=0.035, height=0.008, color="k"):
    km_per_deg_lon = 111.32 * np.cos(np.deg2rad(center_lat_deg))
    bar_deg_lon = bar_km / max(km_per_deg_lon, 1e-9)

    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    x_start = x0 + 0.06 * (x1 - x0)
    x_end   = x_start + bar_deg_lon
    y = y0 + pad * (y1 - y0)
    ax.add_patch(mpl.patches.Rectangle(
        (x_start, y), bar_deg_lon, height*(y1-y0),
        facecolor=color, edgecolor="none", alpha=0.9
    ))
    ax.text((x_start+x_end)/2, y + 2.2*height*(y1-y0),
            rf"$\approx\,{int(bar_km)}\,\mathrm{{km}}$",
            ha="center", va="bottom")

def _add_north_arrow(ax, size=0.05, color="k"):
    ax.annotate("",
        xy=(0.06, 0.90), xytext=(0.06, 0.90 - size),
        xycoords="axes fraction", textcoords="axes fraction",
        arrowprops=dict(arrowstyle="-|>", color=color, lw=1.1))
    ax.text(0.06, 0.905, r"\textbf{N}", ha="center", va="bottom",
            transform=ax.transAxes, color=color)

# ---------------- Native-resolution 1×4 panel plot ----------------
def plot_native_site_panels(dem_da, sites, half_deg, out_png):
    chosen = [next(s for s in sites if s["name"] == q) for q in SELECT_FOUR]
    patches = [(s, _panel_clip_native(dem_da, s["lon"], s["lat"], half_deg)) for s in chosen]
    norm, (vmin, vmax) = _robust_norm(patches)

    fig = plt.figure(figsize=(22, 6.0), constrained_layout=False)
    gs = GridSpec(nrows=1, ncols=5, width_ratios=[1, 1, 1, 1, 0.05],
                  wspace=0.20, left=0.055, right=0.965, bottom=0.115, top=0.915)
    axs = [fig.add_subplot(gs[0, i]) for i in range(4)]
    cax = fig.add_subplot(gs[0, 4])

    ls = LightSource(azdeg=315, altdeg=45)
    cmap = mpl.cm.get_cmap(CMAP)

    for i, (ax, (s, p)) in enumerate(zip(axs, patches), start=1):
        Z = np.asarray(p.values)
        ext = _extent_from_da(p)

        # hillshade + color
        rgb = ls.shade(Z, cmap=cmap, vert_exag=1.0, vmin=vmin, vmax=vmax, blend_mode=HILLSHADE_BLEND)
        ax.imshow(rgb, origin="upper", extent=ext, interpolation="nearest")

        # contours
        try:
            levels = np.linspace(vmin, vmax, CONTOUR_LEVELS)
            ax.contour(Z, levels=levels, origin="upper", extent=ext,
                       linewidths=0.45, colors=CONTOUR_COLOR, alpha=CONTOUR_ALPHA)
        except Exception:
            pass

        # white center cross with black stroke
        cross, = ax.plot(s["lon"], s["lat"], marker="+", markersize=12,
                         markeredgewidth=2.0, color="white", zorder=6, linestyle="None")
        cross.set_path_effects([pe.Stroke(linewidth=3.0, foreground="black"), pe.Normal()])

        # --- NEW: draw 0.25° × 0.25° square centered at site (ERA5 pixel) ---
        side = 0.25
        half = side / 2.0
        rect = mpl.patches.Rectangle(
            (s["lon"] - half, s["lat"] - half),
            side, side,
            facecolor="none",
            edgecolor="white",
            linewidth=2.0,
            zorder=6
        )
        rect.set_path_effects([pe.Stroke(linewidth=3.2, foreground="black"), pe.Normal()])
        ax.add_patch(rect)
        # ---------------------------------------------------------------------

        _add_boxed_title(ax, s["name"])

        # labels and ticks
        ax.set_xlabel(r"$\mathrm{Longitude}\ (^\circ\mathrm{E})$")
        if i == 1:
            ax.set_ylabel(r"$\mathrm{Latitude}\ (^\circ\mathrm{N})$")
        else:
            ax.set_ylabel("")  # remove y-label on others

        # ticks every 0.10°, formatted to 2 decimals
        dl = 0.1
        xt = np.round(np.arange(s["lon"]-half_deg, s["lon"]+half_deg+1e-9, dl), 3)
        yt = np.round(np.arange(s["lat"]-half_deg, s["lat"]+half_deg+1e-9, dl), 3)
        ax.set_xticks(xt); ax.set_yticks(yt)
        ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        ax.tick_params(axis="both", which="both", direction="out", length=5, width=1.0)
        for spine in ax.spines.values(): spine.set_linewidth(1.1)

        _add_north_arrow(ax, size=0.07)
        _add_scalebar(ax, center_lat_deg=s["lat"], bar_km=20.0)

    # Shared, styled colorbar aligned with panel stack
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    cb = fig.colorbar(sm, cax=cax, extend="both", pad=0.01)
    cb.outline.set_linewidth(1.2)
    cb.outline.set_joinstyle("round")
    cb.ax.tick_params(direction="in", length=5, width=1.0, which="major")
    cb.ax.tick_params(direction="in", length=3, width=0.8, which="minor")
    cb.ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    cb.formatter = FormatStrFormatter("%d")
    cb.update_ticks()
    cb.set_label(r"$\mathrm{Elevation}\ (\mathrm{m})$", rotation=90, labelpad=10)

    fig.savefig(
        out_png,
        dpi=200,
        bbox_inches="tight",
        pad_inches=0.25,
        facecolor="white",
        bbox_extra_artists=[cb.ax],
    )
    plt.close(fig)
    print(f"[OK] wrote {out_png}")

# ---------------- Main (aggregation part kept intact) ----------------
def main():
    dem_full = read_dem_latlon(DEM_PATH)
    dem = clip_bbox(dem_full, BBOX)

    lon_c = grid_centers(BBOX[0], BBOX[1], RES)
    lat_c = grid_centers(BBOX[2], BBOX[3], RES)
    if lon_c.size == 0 or lat_c.size == 0:
        raise RuntimeError("No grid centers fall inside BBOX at this resolution.")
    lon_e = centers_to_edges(lon_c, RES)
    lat_e = centers_to_edges(lat_c, RES)

    grids = aggregate_dem_to_grid(dem, lon_c, lat_c, lon_e, lat_e)

    ds = xr.Dataset(
        data_vars=dict(
            orog_mean=(["latitude","longitude"], grids["mean"].astype("float32")),
            orog_std =( ["latitude","longitude"], grids["std"].astype("float32")),
            orog_min =( ["latitude","longitude"], grids["min"].astype("float32")),
            orog_max =( ["latitude","longitude"], grids["max"].astype("float32")),
            orog_n   =( ["latitude","longitude"], grids["count"].astype("int32")),
        ),
        coords=dict(latitude=lat_c, longitude=lon_c),
        attrs=dict(
            title=f"DEM aggregated to {RES:g}° grid (mean/std/min/max/count)",
            grid_resolution=f"{RES:g} degree",
            source_tif=str(DEM_PATH),
        ),
    )
    ds.to_netcdf(OUT_NC); print(f"[OK] wrote {OUT_NC}")

    mean_for_tif = np.flipud(grids["mean"]).astype("float32")
    transform = from_origin(lon_e[0], lat_e[-1], RES, RES)
    profile = dict(driver="GTiff", height=mean_for_tif.shape[0], width=mean_for_tif.shape[1],
                   count=1, dtype="float32", crs="EPSG:4326", transform=transform,
                   compress="deflate", nodata=np.nan)
    with rasterio.open(OUT_TIF, "w", **profile) as dst:
        dst.write(mean_for_tif, 1)
    print(f"[OK] wrote {OUT_TIF}")

    lon_plot = np.concatenate(([lon_c[0]-RES/2], lon_c+RES/2))
    lat_plot = np.concatenate(([lat_c[0]-RES/2], lat_c+RES/2))
    fig, ax = plt.subplots(figsize=(8.5,7.2))
    pm = ax.pcolormesh(lon_plot, lat_plot, grids["mean"], shading="auto", cmap=CMAP)
    cb = fig.colorbar(pm, ax=ax, fraction=0.046, pad=0.4)  # leave as-is
    cb.set_label(r"$\mathrm{Mean\ elevation}\ (\mathrm{m})$")
    ax.set_xlim(BBOX[0], BBOX[1]); ax.set_ylim(BBOX[2], BBOX[3])
    ax.set_xlabel(r"$\mathrm{Longitude}\ (^\circ\mathrm{E})$")
    ax.set_ylabel(r"$\mathrm{Latitude}\ (^\circ\mathrm{N})$")
    ax.set_title(rf"$\mathrm{{DEM\ mean\ on}}\ {RES:g}^\circ\ \mathrm{{grid}}$")
    fig.tight_layout(); fig.savefig(OUT_PNG, dpi=200); plt.close(fig)
    print(f"[OK] wrote {OUT_PNG}")

    plot_native_site_panels(dem, SITES, PANEL_HALF_DEG, OUT_PANEL_PNG)

if __name__ == "__main__":
    main()


### Step 205

This cell defines reusable helper function(s) `read_dem_latlon`, `clip_bbox`, `grid_centers`, `centers_to_edges` so later sections can apply the same processing logic consistently.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# DEM → ERA-like aggregation + native-resolution 2×2 panels around sites
# Changes per request (this revision):
# - Convert 1×4 site panels to a 2×2 layout.
# - Colorbar on the far right, spanning BOTH rows (full height of the 2×2 panel block).
# - Keeps prior styling: boxed titles, hillshade, contours, crosshair + 0.25° pixel box, north arrow, scalebar.
# - Keeps prior label rule: only the leftmost panel (top-left) has the Latitude label; others hide y-label.

from pathlib import Path
import numpy as np
import xarray as xr
import rioxarray as rxr
import rasterio
from rasterio.transform import from_origin
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LightSource
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FormatStrFormatter, AutoMinorLocator
import matplotlib.patheffects as pe

# ---------------- Config ----------------
DEM_PATH = "/Users/wavefunction/ASU Dropbox/Tanmay Singh/THz_Mac/output_hh.tif"
BBOX = (76.0, 80.0, 32.0, 36.0)   # (lon_min, lon_max, lat_min, lat_max)
RES = 0.25

OUT = Path("./era5_dem_products"); OUT.mkdir(parents=True, exist_ok=True)
OUT_NC  = OUT / f"elevation_grid_{RES:g}.nc"
OUT_TIF = OUT / f"elevation_grid_{RES:g}_mean.tif"
OUT_PNG = OUT / f"elevation_grid_{RES:g}_mean.png"

# Native-resolution multi-panel
PANEL_HALF_DEG = 0.25
OUT_PANEL_PNG  = OUT / f"native_panels_{PANEL_HALF_DEG:.2f}deg_each_2x2.png"

SITES = [
    { 'name' : 'Hanle',      'lat' : 32.7789, 'lon' : 78.9650 },  # kept but NOT used
    { 'name' : 'IAO-Hanle',  'lat' : 32.7789, 'lon' : 78.9650 },
    { 'name' : 'NLST-Merak', 'lat' : 33.7953, 'lon' : 78.6167 },
    { 'name' : 'Site A',     'lat' : 34.25,   'lon' : 78.75 },
    { 'name' : 'Site B',     'lat' : 32.5,    'lon' : 79.0 },
]

SELECT_FOUR = ["IAO-Hanle", "NLST-Merak", "Site A", "Site B"]

# ---------------- Typography & style ----------------
try:
    mpl.rcParams.update({
        "text.usetex": True,
        "font.family": "serif",
        "font.size": 16,
        "axes.titlesize": 20,
        "axes.labelsize": 18,
        "xtick.labelsize": 15,
        "ytick.labelsize": 15,
        "legend.fontsize": 14,
        "figure.titlesize": 22,
        "axes.linewidth": 1.1,
    })
except Exception:
    mpl.rcParams.update({
        "text.usetex": False,
        "font.family": "serif",
        "font.size": 16,
        "axes.titlesize": 20,
        "axes.labelsize": 18,
        "xtick.labelsize": 15,
        "ytick.labelsize": 15,
        "legend.fontsize": 14,
        "figure.titlesize": 22,
        "axes.linewidth": 1.1,
    })

CMAP = "viridis"
CONTOUR_COLOR = "k"
CONTOUR_ALPHA = 0.32
CONTOUR_LEVELS = 14
HILLSHADE_BLEND = 'soft'   # 'overlay' is punchy but can shift hues; 'soft' is cleaner

# ---------------- Utilities ----------------
def read_dem_latlon(path):
    da = rxr.open_rasterio(path, masked=True)
    if "band" in da.dims and da.sizes.get("band", 0) == 1:
        da = da.squeeze("band", drop=True)
    if da.rio.crs is None:
        raise RuntimeError("DEM has no CRS.")
    if da.rio.crs.to_epsg() != 4326:
        da = da.rio.reproject("EPSG:4326")
    return da

def clip_bbox(da, bbox):
    lon_min, lon_max, lat_min, lat_max = bbox
    return da.rio.clip_box(minx=lon_min, miny=lat_min, maxx=lon_max, maxy=lat_max)

def grid_centers(vmin, vmax, res):
    start = np.ceil((vmin - 1e-12)/res)*res
    stop  = np.floor((vmax + 1e-12)/res)*res
    if stop < start:
        return np.array([])
    n = int(round((stop - start)/res)) + 1
    return np.round(start + np.arange(n)*res, 10)

def centers_to_edges(centers, res):
    return np.round(np.concatenate(([centers[0]-res/2], centers+res/2)), 10)

# ---------------- Aggregation (RAM-lean) ----------------
def aggregate_dem_to_grid(dem_clip, lon_c, lat_c, lon_e, lat_e):
    nlon, nlat = lon_c.size, lat_c.size
    lon_vals = dem_clip.x.values
    lat_vals = dem_clip.y.values
    elev = dem_clip.values

    eps = RES * 1e-6
    lon_edges = lon_e.copy(); lat_edges = lat_e.copy()
    lon_edges[0]-=eps; lon_edges[-1]+=eps; lat_edges[0]-=eps; lat_edges[-1]+=eps

    col_idx = np.searchsorted(lon_edges, lon_vals, side="right") - 1
    valid_col = (col_idx >= 0) & (col_idx < nlon)

    sum_grid   = np.zeros((nlat, nlon), dtype=np.float64)
    sumsq_grid = np.zeros((nlat, nlon), dtype=np.float64)
    cnt_grid   = np.zeros((nlat, nlon), dtype=np.int64)
    min_grid   = np.full((nlat, nlon), np.inf, dtype=np.float64)
    max_grid   = np.full((nlat, nlon), -np.inf, dtype=np.float64)

    for j in range(lat_vals.size):
        row_idx = np.searchsorted(lat_edges, lat_vals[j], side="right") - 1
        if row_idx < 0 or row_idx >= nlat:
            continue
        row = elev[j, :].astype(np.float64)
        m = np.isfinite(row) & valid_col
        if not np.any(m):
            continue
        cols = col_idx[m]
        vals = row[m]
        np.add.at(sum_grid[row_idx],   cols, vals)
        np.add.at(sumsq_grid[row_idx], cols, vals*vals)
        np.add.at(cnt_grid[row_idx],   cols, 1)

        if cols.size:
            order = np.argsort(cols)
            cols_sorted = cols[order]
            vals_sorted = vals[order]
            ucols, start_idx = np.unique(cols_sorted, return_index=True)
            for k, c in enumerate(ucols):
                s = start_idx[k]
                e = start_idx[k+1] if k+1 < start_idx.size else cols_sorted.size
                vseg = vals_sorted[s:e]
                vmin = float(np.min(vseg))
                vmax = float(np.max(vseg))
                if vmin < min_grid[row_idx, c]:
                    min_grid[row_idx, c] = vmin
                if vmax > max_grid[row_idx, c]:
                    max_grid[row_idx, c] = vmax

    with np.errstate(invalid="ignore", divide="ignore"):
        mean_grid = sum_grid / cnt_grid
        var_grid  = (sumsq_grid / cnt_grid) - mean_grid**2
        std_grid  = np.sqrt(np.maximum(var_grid, 0.0))

    empty = cnt_grid == 0
    for A in (mean_grid, std_grid, min_grid, max_grid):
        A[empty] = np.nan

    return {
        "mean":  mean_grid,
        "std":   std_grid,
        "min":   min_grid,
        "max":   max_grid,
        "count": cnt_grid.astype(np.int32),
    }

# ---------------- Native-resolution helpers ----------------
def _panel_clip_native(dem_da, lon_c, lat_c, half_deg):
    lon_min = lon_c - half_deg; lon_max = lon_c + half_deg
    lat_min = lat_c - half_deg; lat_max = lat_c + half_deg
    return dem_da.rio.clip_box(minx=lon_min, miny=lat_min, maxx=lon_max, maxy=lat_max)

def _extent_from_da(da):
    xs = da.x.values; ys = da.y.values
    return [float(xs.min()), float(xs.max()), float(ys.min()), float(ys.max())]

def _robust_norm(patches):
    v_all = []
    vmins, vmaxs = [], []
    for _, p in patches:
        v = np.asarray(p.values)
        v = v[np.isfinite(v)]
        if v.size:
            v_all.append(v.ravel())
            vmins.append(np.nanmin(v))
            vmaxs.append(np.nanmax(v))
    if not v_all:
        raise RuntimeError("No valid DEM data in requested windows.")
    v_all = np.concatenate(v_all)
    lo = float(np.nanpercentile(v_all, 2.0))
    hi = float(np.nanpercentile(v_all, 98.0))
    vmin = max(min(vmins), lo)
    vmin = 4000  # to fix colorbar issue
    vmax = min(max(vmaxs), hi)
    if vmin >= vmax:
        vmin, vmax = min(vmins), max(vmaxs)
    return Normalize(vmin=vmin, vmax=vmax), (vmin, vmax)

def _add_boxed_title(ax, text):
    ax.set_title(
        rf"\textbf{{{text}}}",
        bbox=dict(boxstyle="round,pad=0.45,rounding_size=0.8",
                  fc="white", ec="0.2", lw=0.9),
        pad=16
    )

def _add_scalebar(ax, center_lat_deg, bar_km=20.0, pad=0.035, height=0.008, color="k"):
    km_per_deg_lon = 111.32 * np.cos(np.deg2rad(center_lat_deg))
    bar_deg_lon = bar_km / max(km_per_deg_lon, 1e-9)

    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    x_start = x0 + 0.06 * (x1 - x0)
    y = y0 + pad * (y1 - y0)
    ax.add_patch(mpl.patches.Rectangle(
        (x_start, y), bar_deg_lon, height*(y1-y0),
        facecolor=color, edgecolor="none", alpha=0.9
    ))
    ax.text(x_start + 0.5*bar_deg_lon, y + 2.2*height*(y1-y0),
            rf"$\approx\,{int(bar_km)}\,\mathrm{{km}}$",
            ha="center", va="bottom")

def _add_north_arrow(ax, size=0.05, color="k"):
    ax.annotate("",
        xy=(0.06, 0.90), xytext=(0.06, 0.90 - size),
        xycoords="axes fraction", textcoords="axes fraction",
        arrowprops=dict(arrowstyle="-|>", color=color, lw=1.1))
    ax.text(0.06, 0.905, r"\textbf{N}", ha="center", va="bottom",
            transform=ax.transAxes, color=color)

# ---------------- Native-resolution 2×2 panel plot ----------------
def plot_native_site_panels(dem_da, sites, half_deg, out_png):
    chosen = [next(s for s in sites if s["name"] == q) for q in SELECT_FOUR]
    patches = [(s, _panel_clip_native(dem_da, s["lon"], s["lat"], half_deg)) for s in chosen]
    norm, (vmin, vmax) = _robust_norm(patches)

    # 2×2 panels + one colorbar column spanning both rows
    fig = plt.figure(figsize=(14.6, 11.0), constrained_layout=False)
    gs = GridSpec(
        nrows=2, ncols=3,
        width_ratios=[1, 1, 0.06],   # last col is colorbar
        height_ratios=[1, 1],
        wspace=0.22, hspace=0.30,
        left=0.07, right=0.94, bottom=0.08, top=0.94
    )

    axs = np.array([
        [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])],
        [fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])]
    ], dtype=object)
    axs_flat = [axs[0,0], axs[0,1], axs[1,0], axs[1,1]]

    cax = fig.add_subplot(gs[:, 2])  # spans both rows

    ls = LightSource(azdeg=315, altdeg=45)
    cmap = mpl.cm.get_cmap(CMAP)

    for idx, (ax, (s, p)) in enumerate(zip(axs_flat, patches)):
        Z = np.asarray(p.values)
        ext = _extent_from_da(p)

        # hillshade + color
        rgb = ls.shade(Z, cmap=cmap, vert_exag=1.0, vmin=vmin, vmax=vmax, blend_mode=HILLSHADE_BLEND)
        ax.imshow(rgb, origin="upper", extent=ext, interpolation="nearest")

        # contours
        try:
            levels = np.linspace(vmin, vmax, CONTOUR_LEVELS)
            ax.contour(
                Z, levels=levels, origin="upper", extent=ext,
                linewidths=0.45, colors=CONTOUR_COLOR, alpha=CONTOUR_ALPHA
            )
        except Exception:
            pass

        # white center cross with black stroke
        cross, = ax.plot(
            s["lon"], s["lat"], marker="+", markersize=12,
            markeredgewidth=2.0, color="white", zorder=6, linestyle="None"
        )
        cross.set_path_effects([pe.Stroke(linewidth=3.0, foreground="black"), pe.Normal()])

        # 0.25° × 0.25° square centered at site (ERA5 pixel size)
        side = 0.25
        half = side / 2.0
        rect = mpl.patches.Rectangle(
            (s["lon"] - half, s["lat"] - half),
            side, side,
            facecolor="none",
            edgecolor="white",
            linewidth=2.0,
            zorder=6
        )
        rect.set_path_effects([pe.Stroke(linewidth=3.2, foreground="black"), pe.Normal()])
        ax.add_patch(rect)

        _add_boxed_title(ax, s["name"])

        # Axis labels:
        # - Only top-left gets the y-label (per your stated rule).
        # - Only bottom row gets x-labels to reduce clutter in 2×2.
        if idx == 0:
            ax.set_ylabel(r"$\mathrm{Latitude}\ (^\circ\mathrm{N})$")
        else:
            ax.set_ylabel("")

        if idx in (2, 3):
            ax.set_xlabel(r"$\mathrm{Longitude}\ (^\circ\mathrm{E})$")
        else:
            ax.set_xlabel("")

        # ticks every 0.10°, formatted to 2 decimals
        dl = 0.1
        xt = np.round(np.arange(s["lon"]-half_deg, s["lon"]+half_deg+1e-9, dl), 3)
        yt = np.round(np.arange(s["lat"]-half_deg, s["lat"]+half_deg+1e-9, dl), 3)
        ax.set_xticks(xt)
        ax.set_yticks(yt)
        ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        ax.tick_params(axis="both", which="both", direction="out", length=5, width=1.0)
        for spine in ax.spines.values():
            spine.set_linewidth(1.1)

        _add_north_arrow(ax, size=0.07)
        _add_scalebar(ax, center_lat_deg=s["lat"], bar_km=20.0)

    # Shared, styled colorbar aligned with the 2×2 block
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    cb = fig.colorbar(sm, cax=cax, extend="both", pad=0.01)
    cb.outline.set_linewidth(1.2)
    cb.outline.set_joinstyle("round")
    cb.ax.tick_params(direction="in", length=5, width=1.0, which="major")
    cb.ax.tick_params(direction="in", length=3, width=0.8, which="minor")
    cb.ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    cb.formatter = FormatStrFormatter("%d")
    cb.update_ticks()
    cb.set_label(r"$\mathrm{Elevation}\ (\mathrm{m})$", rotation=90, labelpad=10)

    fig.savefig(
        out_png,
        dpi=200,
        bbox_inches="tight",
        pad_inches=0.25,
        facecolor="white",
        bbox_extra_artists=[cb.ax],
    )
    plt.close(fig)
    print(f"[OK] wrote {out_png}")

# ---------------- Main (aggregation part kept intact) ----------------
def main():
    dem_full = read_dem_latlon(DEM_PATH)
    dem = clip_bbox(dem_full, BBOX)

    lon_c = grid_centers(BBOX[0], BBOX[1], RES)
    lat_c = grid_centers(BBOX[2], BBOX[3], RES)
    if lon_c.size == 0 or lat_c.size == 0:
        raise RuntimeError("No grid centers fall inside BBOX at this resolution.")
    lon_e = centers_to_edges(lon_c, RES)
    lat_e = centers_to_edges(lat_c, RES)

    grids = aggregate_dem_to_grid(dem, lon_c, lat_c, lon_e, lat_e)

    ds = xr.Dataset(
        data_vars=dict(
            orog_mean=(["latitude","longitude"], grids["mean"].astype("float32")),
            orog_std =( ["latitude","longitude"], grids["std"].astype("float32")),
            orog_min =( ["latitude","longitude"], grids["min"].astype("float32")),
            orog_max =( ["latitude","longitude"], grids["max"].astype("float32")),
            orog_n   =( ["latitude","longitude"], grids["count"].astype("int32")),
        ),
        coords=dict(latitude=lat_c, longitude=lon_c),
        attrs=dict(
            title=f"DEM aggregated to {RES:g}° grid (mean/std/min/max/count)",
            grid_resolution=f"{RES:g} degree",
            source_tif=str(DEM_PATH),
        ),
    )
    ds.to_netcdf(OUT_NC)
    print(f"[OK] wrote {OUT_NC}")

    mean_for_tif = np.flipud(grids["mean"]).astype("float32")
    transform = from_origin(lon_e[0], lat_e[-1], RES, RES)
    profile = dict(
        driver="GTiff",
        height=mean_for_tif.shape[0],
        width=mean_for_tif.shape[1],
        count=1,
        dtype="float32",
        crs="EPSG:4326",
        transform=transform,
        compress="deflate",
        nodata=np.nan
    )
    with rasterio.open(OUT_TIF, "w", **profile) as dst:
        dst.write(mean_for_tif, 1)
    print(f"[OK] wrote {OUT_TIF}")

    lon_plot = np.concatenate(([lon_c[0]-RES/2], lon_c+RES/2))
    lat_plot = np.concatenate(([lat_c[0]-RES/2], lat_c+RES/2))
    fig, ax = plt.subplots(figsize=(8.5, 7.2))
    pm = ax.pcolormesh(lon_plot, lat_plot, grids["mean"], shading="auto", cmap=CMAP)
    cb = fig.colorbar(pm, ax=ax, fraction=0.046, pad=0.4)
    cb.set_label(r"$\mathrm{Mean\ elevation}\ (\mathrm{m})$")
    ax.set_xlim(BBOX[0], BBOX[1]); ax.set_ylim(BBOX[2], BBOX[3])
    ax.set_xlabel(r"$\mathrm{Longitude}\ (^\circ\mathrm{E})$")
    ax.set_ylabel(r"$\mathrm{Latitude}\ (^\circ\mathrm{N})$")
    ax.set_title(rf"$\mathrm{{DEM\ mean\ on}}\ {RES:g}^\circ\ \mathrm{{grid}}$")
    fig.tight_layout()
    fig.savefig(OUT_PNG, dpi=200)
    plt.close(fig)
    print(f"[OK] wrote {OUT_PNG}")

    plot_native_site_panels(dem, SITES, PANEL_HALF_DEG, OUT_PANEL_PNG)

if __name__ == "__main__":
    main()


### Step 206

This cell defines reusable helper function(s) `read_dem_latlon`, `clip_bbox`, `clip_square`, `panel_clip_native` so later sections can apply the same processing logic consistently.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from pathlib import Path
import numpy as np
import rioxarray as rxr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LightSource
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FormatStrFormatter, AutoMinorLocator, ScalarFormatter
import matplotlib.patheffects as pe
from matplotlib.patches import Patch

# ---------------- Config ----------------
DEM_PATH = "/Users/wavefunction/ASU Dropbox/Tanmay Singh/THz_Mac/output_hh.tif"
BBOX = (76.0, 80.0, 32.0, 36.0)   # lon_min, lon_max, lat_min, lat_max (safety clip)

PANEL_HALF_DEG   = 0.25           # map half-width
ERA5_PIXEL_SIDE  = 0.25           # 0.25° × 0.25° ERA5 pixel square

SQUARE_SIDE = 0.25                # for histograms
HALF        = SQUARE_SIDE / 2.0
THRESH_M    = 4300.0

OUT = Path("./era5_dem_products"); OUT.mkdir(parents=True, exist_ok=True)
OUT_PNG = OUT / f"combined_maps_plus_hist_{SQUARE_SIDE:.2f}deg_thresh{int(THRESH_M)}m.png"
OUT_PDF = OUT / f"combined_maps_plus_hist_{SQUARE_SIDE:.2f}deg_thresh{int(THRESH_M)}m.pdf"

SITES = [
    { 'name' : 'IAO-Hanle',   'lat' : 32.7789, 'lon' : 78.9650 },
    { 'name' : 'NLST-Merak',  'lat' : 33.7953, 'lon' : 78.6167 },
    { 'name' : 'Site A',      'lat' : 34.25,   'lon' : 78.75   },
    { 'name' : 'Site B',      'lat' : 32.5,    'lon' : 79.0    },
]
HATCHED_SITES = {"IAO-Hanle", "NLST-Merak"}

# ---------------- Typography & style ----------------
try:
    mpl.rcParams.update({
        "text.usetex": True,
        "font.family": "serif",
        "font.size": 15,
        "axes.titlesize": 18,
        "axes.labelsize": 16,
        "xtick.labelsize": 13,
        "ytick.labelsize": 13,
        "axes.linewidth": 1.1,
    })
except Exception:
    mpl.rcParams.update({
        "text.usetex": False,
        "font.family": "serif",
        "font.size": 15,
        "axes.titlesize": 18,
        "axes.labelsize": 16,
        "xtick.labelsize": 13,
        "ytick.labelsize": 13,
        "axes.linewidth": 1.1,
    })

CMAP = "viridis"
CONTOUR_COLOR  = "k"
CONTOUR_ALPHA  = 0.32
CONTOUR_LEVELS = 14
HILLSHADE_BLEND = "soft"

# ---------------- Utilities ----------------
def read_dem_latlon(path):
    da = rxr.open_rasterio(path, masked=True)
    if "band" in da.dims and da.sizes.get("band", 0) == 1:
        da = da.squeeze("band", drop=True)
    if da.rio.crs is None:
        raise RuntimeError("DEM has no CRS.")
    if da.rio.crs.to_epsg() != 4326:
        da = da.rio.reproject("EPSG:4326")
    return da

def clip_bbox(da, bbox):
    lon_min, lon_max, lat_min, lat_max = bbox
    return da.rio.clip_box(minx=lon_min, miny=lat_min, maxx=lon_max, maxy=lat_max)

def clip_square(da, lon_c, lat_c, half):
    return da.rio.clip_box(
        minx=lon_c-half, maxx=lon_c+half,
        miny=lat_c-half, maxy=lat_c+half
    )

def panel_clip_native(dem_da, lon_c, lat_c, half_deg):
    return dem_da.rio.clip_box(
        minx=lon_c-half_deg, maxx=lon_c+half_deg,
        miny=lat_c-half_deg, maxy=lat_c+half_deg
    )

def extent_from_da(da):
    xs = da.x.values; ys = da.y.values
    return [float(xs.min()), float(xs.max()), float(ys.min()), float(ys.max())]

def robust_norm(patches):
    v_all, vmins, vmaxs = [], [], []
    for _, p in patches:
        v = np.asarray(p.values)
        v = v[np.isfinite(v)]
        if v.size:
            v_all.append(v.ravel())
            vmins.append(np.nanmin(v)); vmaxs.append(np.nanmax(v))
    if not v_all:
        raise RuntimeError("No valid DEM data in requested windows.")
    v_all = np.concatenate(v_all)
    lo = float(np.nanpercentile(v_all, 2.0))
    hi = float(np.nanpercentile(v_all, 98.0))
    vmin = max(min(vmins), lo)
    vmin = 4000.0
    vmax = min(max(vmaxs), hi)
    if vmin >= vmax:
        vmin, vmax = float(min(vmins)), float(max(vmaxs))
    return Normalize(vmin=vmin, vmax=vmax), (vmin, vmax)

def add_boxed_title(ax, text):
    ax.set_title(
        rf"\textbf{{{text}}}",
        bbox=dict(boxstyle="round,pad=0.40,rounding_size=0.9",
                  fc="white", ec="0.2", lw=0.9),
        pad=18
    )

def add_scalebar(ax, center_lat_deg, bar_km=20.0, pad=0.035, height=0.008, color="k"):
    km_per_deg_lon = 111.32 * np.cos(np.deg2rad(center_lat_deg))
    bar_deg_lon = bar_km / max(km_per_deg_lon, 1e-9)
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    x_start = x0 + 0.06 * (x1 - x0)
    y = y0 + pad * (y1 - y0)
    ax.add_patch(mpl.patches.Rectangle(
        (x_start, y), bar_deg_lon, height*(y1-y0),
        facecolor=color, edgecolor="none", alpha=0.9
    ))
    ax.text(
        x_start + 0.5*bar_deg_lon,
        y + 2.2*height*(y1-y0),
        rf"$\approx\,{int(bar_km)}\,\mathrm{{km}}$",
        ha="center", va="bottom"
    )

def add_north_arrow(ax, size=0.07, color="k"):
    ax.annotate(
        "",
        xy=(0.06, 0.90), xytext=(0.06, 0.90 - size),
        xycoords="axes fraction", textcoords="axes fraction",
        arrowprops=dict(arrowstyle="-|>", color=color, lw=1.1)
    )
    ax.text(0.06, 0.905, r"\textbf{N}", ha="center", va="bottom",
            transform=ax.transAxes, color=color)

# --- histogram helpers ---
def freedman_diaconis_width(x):
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    n = x.size
    if n < 2:
        return np.nan
    iqr = np.subtract(*np.nanpercentile(x, [75, 25]))
    if iqr <= 0:
        return np.nan
    return 2.0 * iqr / np.cbrt(n)

def choose_global_bins(datasets, min_bins=10, max_bins=200):
    widths = []
    data_min, data_max = +np.inf, -np.inf
    for x in datasets:
        x = np.asarray(x); x = x[np.isfinite(x)]
        if x.size == 0:
            continue
        data_min = min(data_min, float(np.nanmin(x)))
        data_max = max(data_max, float(np.nanmax(x)))
        w = freedman_diaconis_width(x)
        if np.isfinite(w) and w > 0:
            widths.append(w)

    if not widths:
        xx = np.concatenate(
            [np.asarray(d)[np.isfinite(d)] for d in datasets if np.asarray(d).size]
        )
        if xx.size < 2:
            w_global = 10.0
        else:
            sigma = float(np.nanstd(xx))
            w_global = 3.5 * sigma / np.cbrt(xx.size)
            if not np.isfinite(w_global) or w_global <= 0:
                w_global = 10.0
    else:
        w_global = float(np.nanmedian(widths))
        if w_global <= 1e-6:
            w_global = 10.0

    span = max(data_max - data_min, 1.0)
    nbins = int(np.clip(np.round(span / w_global), min_bins, max_bins))
    edges = np.linspace(data_min, data_max, nbins + 1)
    return edges

# ---------------- Combined plot ----------------
def plot_combined(dem_da, sites):
    patches = [(s, panel_clip_native(dem_da, s["lon"], s["lat"], PANEL_HALF_DEG))
               for s in sites]
    norm, (vmin, vmax) = robust_norm(patches)
    cmap = mpl.cm.get_cmap(CMAP)
    ls = LightSource(azdeg=315, altdeg=45)

    fig = plt.figure(figsize=(10, 16.0), constrained_layout=False)

        # Keep the SAME map block bounds (as in your code)
    LEFT   = 0.10
    RIGHT  = 0.82
    BOTTOM = 0.06
    TOP    = 0.975

    # Make GridSpec ONLY for the top 2×2 maps; histogram will be positioned explicitly
    gs = GridSpec(
        nrows=2, ncols=2, figure=fig,
        left=LEFT, right=RIGHT,
        bottom=0.52, top=TOP,      # top block only
        wspace=0.075, hspace=0.35
    )

    ax_map = [
        fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]),
        fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1]),
    ]

    # Histogram axis: explicitly spans EXACTLY the same width as the map block (LEFT..RIGHT)
    # This removes the "wasted" right-side margin under the colorbar.
    # SHIFT = 0.06  # adjust between 0.02–0.05 if needed

    # axh = fig.add_axes([LEFT + SHIFT, BOTTOM, RIGHT-LEFT, 0.40])
    
    
    SHIFT = 0.055   # slightly smaller shift
    WIDTH_EXPAND = 0.018  # tiny width increase

    axh = fig.add_axes([
        LEFT + SHIFT,
        BOTTOM,
        (RIGHT - LEFT) + WIDTH_EXPAND,
        0.40
    ])

    # --- maps ---
    for idx, (ax, (s, p)) in enumerate(zip(ax_map, patches)):
        Z = np.asarray(p.values)
        ext = extent_from_da(p)

        rgb = ls.shade(
            Z, cmap=cmap, vert_exag=1.0,
            vmin=vmin, vmax=vmax, blend_mode=HILLSHADE_BLEND
        )
        ax.imshow(rgb, origin="upper", extent=ext, interpolation="nearest")

        try:
            levels = np.linspace(vmin, vmax, CONTOUR_LEVELS)
            ax.contour(
                Z, levels=levels, origin="upper", extent=ext,
                linewidths=0.45, colors=CONTOUR_COLOR, alpha=CONTOUR_ALPHA
            )
        except Exception:
            pass

        cross, = ax.plot(
            s["lon"], s["lat"], marker="+", markersize=11,
            markeredgewidth=2.0, color="white", zorder=6, linestyle="None"
        )
        cross.set_path_effects([
            pe.Stroke(linewidth=3.0, foreground="black"),
            pe.Normal()
        ])

        half_pix = ERA5_PIXEL_SIDE / 2.0
        rect = mpl.patches.Rectangle(
            (s["lon"] - half_pix, s["lat"] - half_pix),
            ERA5_PIXEL_SIDE, ERA5_PIXEL_SIDE,
            facecolor="none", edgecolor="white",
            linewidth=2.0, zorder=6
        )
        rect.set_path_effects([
            pe.Stroke(linewidth=3.2, foreground="black"),
            pe.Normal()
        ])
        ax.add_patch(rect)

        add_boxed_title(ax, s["name"])

        ax.set_xlabel(r"$\mathrm{Longitude}\ (^\circ\mathrm{E})$")
        if idx in (0, 2):
            ax.set_ylabel(r"$\mathrm{Latitude}\ (^\circ\mathrm{N})$")
        else:
            ax.set_ylabel("")
            ax.tick_params(labelleft=False)

        dl = 0.1
        xt = np.round(np.arange(s["lon"]-PANEL_HALF_DEG,
                                s["lon"]+PANEL_HALF_DEG+1e-9, dl), 3)
        yt = np.round(np.arange(s["lat"]-PANEL_HALF_DEG,
                                s["lat"]+PANEL_HALF_DEG+1e-9, dl), 3)
        ax.set_xticks(xt); ax.set_yticks(yt)
        ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        ax.tick_params(axis="both", which="both",
                       direction="out", length=4.5, width=1.0, pad=1.8)
        for sp in ax.spines.values():
            sp.set_linewidth(1.1)

        add_north_arrow(ax, size=0.065)
        add_scalebar(ax, center_lat_deg=s["lat"], bar_km=20.0)
        ax.set_aspect("equal", adjustable="box")

    # --- colorbar: outside the map block, aligned to map rows only (unchanged behavior) ---
    pos = [a.get_position() for a in ax_map]
    y0 = min(p.y0 for p in pos); y1 = max(p.y1 for p in pos)
    cax = fig.add_axes([0.845, y0, 0.028, y1 - y0])
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    cb = fig.colorbar(sm, cax=cax, extend="both")
    cb.outline.set_linewidth(1.2)
    cb.outline.set_joinstyle("round")
    cb.ax.tick_params(direction="in", length=5, width=1.0, which="major")
    cb.ax.tick_params(direction="in", length=3, width=0.8, which="minor")
    cb.ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    cb.formatter = FormatStrFormatter("%d")
    cb.update_ticks()
    cb.set_label(r"$\mathrm{Elevation}\ (\mathrm{m})$", rotation=90, labelpad=10, fontsize=22)

    # ---------------- histograms ----------------
    axh.set_xlabel(r"$\mathrm{Elevation}\ \mathrm{(m)}$", fontsize=24)
    axh.set_ylabel(
        r"$\mathrm{Probability\ density\ (per\ site,\ }z\ge{}"
        + f"{int(THRESH_M)}" + r"\mathrm{\,m)}$",
        fontsize=24
    )

    for side in ["top", "bottom", "left", "right"]:
        axh.spines[side].set_visible(True)
        axh.spines[side].set_linewidth(2.0)

    axh.tick_params(
        axis="both", which="both",
        direction="in", top=True, right=True, labelsize=16
    )
    axh.tick_params(axis="both", which="major", length=9, width=1.6)
    axh.tick_params(axis="both", which="minor", length=5, width=1.2)
    axh.xaxis.set_minor_locator(AutoMinorLocator(2))
    axh.yaxis.set_minor_locator(AutoMinorLocator(2))
    axh.yaxis.set_major_formatter(ScalarFormatter(useMathText=True))

    series, kept_counts, total_counts = {}, {}, {}
    for s in sites:
        sub = clip_square(dem_da, s["lon"], s["lat"], HALF)
        vals = np.asarray(sub.values).ravel()
        vals = vals[np.isfinite(vals)]
        kept = vals[vals >= THRESH_M]
        series[s["name"]] = kept
        kept_counts[s["name"]] = kept.size
        total_counts[s["name"]] = vals.size

    items = [(k, v) for k, v in series.items() if v.size > 0]
    if not items:
        raise RuntimeError(f"No valid DEM samples >= {THRESH_M:.0f} m.")

    edges = choose_global_bins([v for _, v in items])

    color_cycle = mpl.rcParams['axes.prop_cycle'].by_key().get(
        'color', ["C0", "C1", "C2", "C3"]
    )
    linestyles = ["-", "--", "-.", ":"]
    hatches = {"IAO-Hanle": "///", "NLST-Merak": "\\\\\\"}
    shade_alpha = 0.22

    legend_handles, legend_labels = [], []

    for i, (name, vals) in enumerate(items):
        color = color_cycle[i % len(color_cycle)]
        ls_   = linestyles[i % len(linestyles)]
        N_kept = kept_counts[name]
        N_tot  = total_counts[name] if total_counts[name] > 0 else 1
        frac   = N_kept / N_tot

        if name in HATCHED_SITES:
            _, _, patches_hist = axh.hist(
                vals, bins=edges, density=True, histtype="bar",
                edgecolor=color, linewidth=2.0, alpha=1.0
            )
            for p in patches_hist:
                p.set_hatch(hatches[name])
                p.set_facecolor("none")
                p.set_edgecolor(color)
                p.set_linewidth(2.0)
            axh.hist(
                vals, bins=edges, histtype="step",
                linewidth=2.7, color=color, linestyle=ls_,
                density=True
            )
            proxy = Patch(facecolor="none", edgecolor=color,
                          hatch=hatches[name], linewidth=1.8)
        else:
            axh.hist(
                vals, bins=edges, histtype="stepfilled",
                density=True, facecolor=color, edgecolor="none",
                alpha=shade_alpha
            )
            axh.hist(
                vals, bins=edges, histtype="step",
                linewidth=2.7, color=color, linestyle=ls_,
                density=True
            )
            proxy = Patch(facecolor=color, edgecolor="none",
                          alpha=shade_alpha)

        legend_handles.append(proxy)
        legend_labels.append(
            rf"\textbf{{{name}}} ($f_{{\ge {int(THRESH_M)}}}={frac:.2f}$)"
        )

    leg = axh.legend(
        legend_handles, legend_labels,
        loc="center right", frameon=True, fancybox=True,
        framealpha=0.96, borderpad=0.8,
        handlelength=2.4, handletextpad=0.9,
        fontsize=16
    )
    leg.get_frame().set_linewidth(1.5)

    title_txt = (
        r"\textbf{Elevation distributions in }"
        + rf"${SQUARE_SIDE:0.2f}^{{\circ}}\times{SQUARE_SIDE:0.2f}^{{\circ}}$ "
        + r"\textbf{boxes}"
    )
    axh.text(
        0.5, 0.965, title_txt,
        transform=axh.transAxes, ha="center", va="top",
        bbox=dict(boxstyle="round,pad=0.45,rounding_size=0.9",
                  fc="white", ec="0.25", lw=1.4),
        fontsize=20
    )
    axh.text(
        0.5, 0.870, r"\textit{Native DEM resolution }$\sim 30\,\mathrm{m}$",
        transform=axh.transAxes, ha="center", va="top",
        bbox=dict(boxstyle="round,pad=0.32,rounding_size=0.8",
                  fc="white", ec="0.25", lw=1.2),
        fontsize=16
    )

    fig.savefig(OUT_PNG, dpi=100, facecolor="white",
                bbox_inches="tight", pad_inches=0.15)
    fig.savefig(OUT_PDF, dpi=100, facecolor="white",
                bbox_inches="tight", pad_inches=0.15)
    plt.close(fig)
    print(f"[OK] wrote:\n  {OUT_PNG}\n  {OUT_PDF}")

# ---------------- Main ----------------
def main():
    dem_full = read_dem_latlon(DEM_PATH)
    dem = clip_bbox(dem_full, BBOX)
    plot_combined(dem, SITES)

if __name__ == "__main__":
    main()


### Step 207

This cell defines reusable helper function(s) `read_dem_latlon`, `clip_bbox`, `clip_square`, `freedman_diaconis_width` so later sections can apply the same processing logic consistently.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# Elevation histograms from native DEM within 0.25°×0.25° squares
# centered on four sites. Single overlaid plot.
# Updates:
# - Legend uses PROXY PATCHES so hatching appears for IAO-Hanle & NLST-Merak.
# - Title textbox centered at top inside axes.
# - Resolution textbox centered directly below the title textbox.
# - Per-site PDF after threshold; common FD binning; thicker boundaries.

from pathlib import Path
import numpy as np
import xarray as xr
import rioxarray as rxr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator, ScalarFormatter
from matplotlib.patches import Patch

# ---------------- Config ----------------
DEM_PATH = "/Users/wavefunction/ASU Dropbox/Tanmay Singh/THz_Mac/output_hh.tif"
BBOX = (76.0, 80.0, 32.0, 36.0)   # lon_min, lon_max, lat_min, lat_max (safety clip)
SQUARE_SIDE = 0.25                # 0.25° × 0.25° box
HALF = SQUARE_SIDE / 2.0
THRESH_M = 4300.0                 # include ONLY >= THRESH_M

OUT = Path("./era5_dem_products"); OUT.mkdir(parents=True, exist_ok=True)
OUT_PNG = OUT / f"site_histograms_{SQUARE_SIDE:.2f}deg_boxes_thresh{int(THRESH_M)}m.png"
OUT_PDF = OUT / f"site_histograms_{SQUARE_SIDE:.2f}deg_boxes_thresh{int(THRESH_M)}m.pdf"

SITES = [
    { 'name' : 'IAO-Hanle',   'lat' : 32.7789, 'lon' : 78.9650 },
    { 'name' : 'NLST-Merak',  'lat' : 33.7953, 'lon' : 78.6167 },
    { 'name' : 'Site A',      'lat' : 34.25,   'lon' : 78.75   },
    { 'name' : 'Site B',      'lat' : 32.5,    'lon' : 79.0    },
]
HATCHED_SITES = {"IAO-Hanle", "NLST-Merak"}  # only these two get hatching

# ---------------- Typography & style ----------------
try:
    mpl.rcParams.update({
        "text.usetex": True,
        "font.family": "serif",
        "font.size": 22,             # base font larger
        "axes.titlesize": 28,
        "axes.labelsize": 26,
        "xtick.labelsize": 22,
        "ytick.labelsize": 22,
        "legend.fontsize": 22,
        "figure.titlesize": 30,
        "axes.linewidth": 2.2,       # thicker axes boundary
    })
except Exception:
    mpl.rcParams.update({
        "text.usetex": False,
        "font.family": "serif",
        "font.size": 22,
        "axes.titlesize": 28,
        "axes.labelsize": 26,
        "xtick.labelsize": 22,
        "ytick.labelsize": 22,
        "legend.fontsize": 22,
        "figure.titlesize": 30,
        "axes.linewidth": 2.2,
    })

# ---------------- Utilities ----------------
def read_dem_latlon(path):
    da = rxr.open_rasterio(path, masked=True)
    if "band" in da.dims and da.sizes.get("band", 0) == 1:
        da = da.squeeze("band", drop=True)
    if da.rio.crs is None:
        raise RuntimeError("DEM has no CRS.")
    if da.rio.crs.to_epsg() != 4326:
        da = da.rio.reproject("EPSG:4326")
    return da

def clip_bbox(da, bbox):
    lon_min, lon_max, lat_min, lat_max = bbox
    return da.rio.clip_box(minx=lon_min, miny=lat_min, maxx=lon_max, maxy=lat_max)

def clip_square(da, lon_c, lat_c, half):
    return da.rio.clip_box(
        minx=lon_c - half, maxx=lon_c + half,
        miny=lat_c - half, maxy=lat_c + half
    )

def freedman_diaconis_width(x):
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    n = x.size
    if n < 2:
        return np.nan
    iqr = np.subtract(*np.nanpercentile(x, [75, 25]))
    if iqr <= 0:
        return np.nan
    return 2.0 * iqr / np.cbrt(n)

def choose_global_bins(datasets, min_bins=10, max_bins=200):
    widths = []
    data_min, data_max = +np.inf, -np.inf
    for x in datasets:
        x = np.asarray(x); x = x[np.isfinite(x)]
        if x.size == 0:
            continue
        data_min = min(data_min, float(np.nanmin(x)))
        data_max = max(data_max, float(np.nanmax(x)))
        w = freedman_diaconis_width(x)
        if np.isfinite(w) and w > 0:
            widths.append(w)

    if not widths:
        xx = np.concatenate([np.asarray(d)[np.isfinite(d)] for d in datasets if np.asarray(d).size])
        if xx.size < 2:
            w_global = 10.0
        else:
            sigma = float(np.nanstd(xx))
            w_global = 3.5 * sigma / np.cbrt(xx.size)
            if not np.isfinite(w_global) or w_global <= 0:
                w_global = 10.0
    else:
        w_global = float(np.nanmedian(widths))
        if w_global <= 1e-6:
            w_global = 10.0

    span = max(data_max - data_min, 1.0)
    nbins = int(np.clip(np.round(span / w_global), min_bins, max_bins))
    edges = np.linspace(data_min, data_max, nbins + 1)
    return edges

# ---------------- Main ----------------
def main():
    dem_full = read_dem_latlon(DEM_PATH)
    dem = clip_bbox(dem_full, BBOX)

    # Thresholded and total series per site
    series = {}
    kept_counts = {}
    total_counts = {}
    for s in SITES:
        sub = clip_square(dem, s["lon"], s["lat"], HALF)
        vals = np.asarray(sub.values).ravel()
        vals = vals[np.isfinite(vals)]
        kept = vals[vals >= THRESH_M]
        series[s["name"]] = kept
        kept_counts[s["name"]] = kept.size
        total_counts[s["name"]] = vals.size

    items = [(k, v) for k, v in series.items() if v.size > 0]
    if not items:
        raise RuntimeError(f"No valid DEM samples >= {THRESH_M:.0f} m in the specified boxes.")

    # Common bin edges computed AFTER thresholding (so PDFs are comparable)
    edges = choose_global_bins([v for _, v in items])

    # Plot
    fig, ax = plt.subplots(figsize=(14.0, 9.0))
    ax.set_xlabel(r"$\mathrm{Elevation}\ \mathrm{(m)}$")
    ax.set_ylabel(r"$\mathrm{Probability\ density\ (per\ site,\ }z\ge{}"
                  + f"{int(THRESH_M)}" + r"\mathrm{\,m)}$)")

    # Thicker, visible spines
    for side in ["top", "bottom", "left", "right"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(2.2)

    # Inward ticks, major + minor
    ax.tick_params(axis="both", which="both", direction="in", top=True, right=True)
    ax.tick_params(axis="both", which="major", length=10, width=1.8)
    ax.tick_params(axis="both", which="minor", length=6,  width=1.4)
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.set_major_formatter(ScalarFormatter(useMathText=True))

    color_cycle = mpl.rcParams['axes.prop_cycle'].by_key().get('color', ["C0","C1","C2","C3"])
    linestyles = ["-", "--", "-.", ":"]
    hatches = {"IAO-Hanle": "///", "NLST-Merak": "\\\\\\"}  # only these two hatched
    shade_alpha = 0.22

    legend_handles, legend_labels = [], []

    for i, (name, vals) in enumerate(items):
        color = color_cycle[i % len(color_cycle)]
        ls = linestyles[i % len(linestyles)]
        N_kept = kept_counts[name]
        N_tot  = total_counts[name] if total_counts[name] > 0 else 1
        frac   = N_kept / N_tot

        if name in HATCHED_SITES:
            # Hatched bars (transparent fill), plus bold step outline
            n_bar, _, patches = ax.hist(
                vals, bins=edges, density=True, histtype="bar",
                edgecolor=color, linewidth=2.2, alpha=1.0, label=None
            )
            for p in patches:
                p.set_hatch(hatches[name])
                p.set_facecolor("none")
                p.set_edgecolor(color)
                p.set_linewidth(2.2)
            n_step, _, line_patches = ax.hist(
                vals, bins=edges, histtype="step", linewidth=3.0,
                color=color, linestyle=ls, density=True
            )
            # Legend PROXY showing hatch
            proxy = Patch(facecolor="none", edgecolor=color, hatch=hatches[name], linewidth=2.0)
        else:
            # Shaded translucent fill (no hatch), plus bold step outline
            ax.hist(
                vals, bins=edges, histtype="stepfilled", density=True,
                facecolor=color, edgecolor="none", alpha=shade_alpha
            )
            n_step, _, line_patches = ax.hist(
                vals, bins=edges, histtype="step", linewidth=3.0,
                color=color, linestyle=ls, density=True
            )
            # Legend PROXY showing shaded style
            proxy = Patch(facecolor=color, edgecolor="none", alpha=shade_alpha)

        legend_handles.append(proxy)
        legend_labels.append(rf"\textbf{{{name}}} ($f_{{\ge {int(THRESH_M)}}}={frac:.2f}$)")

    # Legend (center-right retained so it doesn't clash with centered title boxes)
    leg = ax.legend(
        legend_handles, legend_labels,
        loc="center right", frameon=True, fancybox=True, framealpha=0.96,
        borderpad=0.9, handlelength=2.8, handletextpad=1.0
    )
    leg.get_frame().set_linewidth(1.6)

    # Title textbox centered at top inside axes
    title_txt = (
        r"\textbf{Elevation distributions in }"
        + rf"${SQUARE_SIDE:0.2f}^{{\circ}}\times{SQUARE_SIDE:0.2f}^{{\circ}}$ "
        + r"\textbf{boxes}"
    )
    ax.text(
        0.5, 0.96, title_txt,
        transform=ax.transAxes, ha="center", va="top",
        bbox=dict(boxstyle="round,pad=0.55,rounding_size=1.0",
                  fc="white", ec="0.25", lw=1.6)
    )

    # Resolution textbox centered directly below the title textbox
    ax.text(
        0.5, 0.885, r"\textit{Native DEM resolution }$\sim 30\,\mathrm{m}$",
        transform=ax.transAxes, ha="center", va="top",
        bbox=dict(boxstyle="round,pad=0.40,rounding_size=0.9",
                  fc="white", ec="0.25", lw=1.2)
    )

    fig.tight_layout()
    fig.savefig(OUT_PNG, dpi=300, facecolor="white", bbox_inches="tight", pad_inches=0.25)
    fig.savefig(OUT_PDF, dpi=300, facecolor="white", bbox_inches="tight", pad_inches=0.25)
    plt.close(fig)
    print(f"[OK] wrote:\n  {OUT_PNG}\n  {OUT_PDF}")

if __name__ == "__main__":
    main()


### Step 208

This cell defines reusable helper function(s) `read_dem_latlon`, `clip_bbox`, `edges_from_centers_1d`, `era5_centers` so later sections can apply the same processing logic consistently.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import rasterio
from rasterio.transform import from_origin
import matplotlib.pyplot as plt

# ---------------- Config ----------------
DEM_PATH = "/Users/wavefunction/ASU Dropbox/Tanmay Singh/THz_Mac/output_hh.tif"
BBOX = (76.0, 80.0, 32.0, 36.0)   # (lon_min, lon_max, lat_min, lat_max)
ERA5_RES = 0.25

OUT = Path("./era5_dem_products"); OUT.mkdir(parents=True, exist_ok=True)
PNG_NATIVE = OUT / "native_resolution_elevation_map.png"
PNG_ERA5   = OUT / "elevation_era5grid_0p25_mean.png"
GTIFF_MEAN = OUT / "era5_0p25_mean_elevation_ladakh.tif"
CSV_GRID   = OUT / "era5_0p25_stats_ladakh.csv"

SITES = [
    { 'name' : 'Hanle',      'lat' : 32.7789, 'lon' : 78.9650,  'elevation' : 4500, 'T_b' : 287.49 },
    { 'name' : 'IAO-Hanle',  'lat' : 32.7789, 'lon' : 78.9650,  'elevation' : 4500, 'T_b' : 287.49 },
    { 'name' : 'Merak',      'lat' : 33.7828, 'lon' : 78.57782, 'elevation' : 4310 },
    { 'name' : 'Site A',     'lat' : 34.25,   'lon' : 78.75,    'elevation' : 4800, 'T_b' : 285.82 },
    { 'name' : 'Site B',     'lat' : 32.5,    'lon' : 79.0,     'elevation' : 4500, 'T_b' : 286.0  },
]

# ---------------- Helpers ----------------
def read_dem_latlon(path):
    da = rxr.open_rasterio(path, masked=True)
    if "band" in da.dims and da.sizes["band"] == 1:
        da = da.squeeze("band", drop=True)
    if da.rio.crs is None:
        raise RuntimeError("DEM has no CRS.")
    if da.rio.crs.to_epsg() != 4326:
        da = da.rio.reproject("EPSG:4326")
    return da

def clip_bbox(da, bbox):
    lon_min, lon_max, lat_min, lat_max = bbox
    return da.rio.clip_box(minx=lon_min, miny=lat_min, maxx=lon_max, maxy=lat_max)

def edges_from_centers_1d(centers):
    if centers.size == 1:
        d = 1.0
        return np.array([centers[0]-d/2, centers[0]+d/2])
    d = np.diff(centers)
    d = np.append(d, d[-1])
    return np.concatenate(([centers[0] - d[0]/2], centers + d/2))

def era5_centers(vmin, vmax, res):
    start = np.ceil((vmin - 1e-12)/res)*res
    stop  = np.floor((vmax + 1e-12)/res)*res
    if stop < start: return np.array([])
    n = int(round((stop - start)/res)) + 1
    return np.round(start + np.arange(n)*res, 10)

def centers_to_edges(centers, res):
    return np.round(np.concatenate(([centers[0]-res/2], centers+res/2)), 10)

def raster_to_points(da):
    lon2d, lat2d = xr.broadcast(da["x"], da["y"])
    df = pd.DataFrame({"lon": lon2d.values.ravel(),
                       "lat": lat2d.values.ravel(),
                       "elev": da.values.ravel()})
    return df.replace([np.inf,-np.inf], np.nan).dropna(subset=["elev"])

def aggregate_to_era5(df_pts, lon_c, lat_c, lon_e, lat_e):
    df_pts["lon_bin"] = pd.cut(df_pts["lon"], lon_e, right=False, include_lowest=True)
    df_pts["lat_bin"] = pd.cut(df_pts["lat"], lat_e, right=False, include_lowest=True)
    df_pts = df_pts.dropna(subset=["lon_bin","lat_bin"])
    lon_map = {pd.Interval(lon_e[i], lon_e[i+1], closed="left"): lon_c[i] for i in range(len(lon_c))}
    lat_map = {pd.Interval(lat_e[i], lat_e[i+1], closed="left"): lat_c[i] for i in range(len(lat_c))}
    df_pts["lon_center"] = df_pts["lon_bin"].map(lon_map)
    df_pts["lat_center"] = df_pts["lat_bin"].map(lat_map)
    stats = (df_pts.groupby(["lat_center","lon_center"])["elev"]
             .agg(mean="mean", std="std", count="count", min="min", max="max")
             .reset_index()
             .sort_values(["lat_center","lon_center"])
             .reset_index(drop=True))
    return stats

def stats_to_grid(stats, lat_c, lon_c, var):
    lat_asc = np.sort(lat_c)
    lon_asc = np.sort(lon_c)
    P = stats.pivot(index="lat_center", columns="lon_center", values=var)
    P = P.reindex(index=lat_asc, columns=lon_asc)
    return lat_asc, lon_asc, P.values

# ---------------- Main ----------------
def main():
    # Read + clip
    dem = read_dem_latlon(DEM_PATH)
    dem_clip = clip_bbox(dem, BBOX)

    # -------- Native-resolution plot (lat ↑ upward) --------
    lon_native_c = dem_clip.x.values                # ascending
    lat_native_c = dem_clip.y.values                # often descending
    elev_native  = dem_clip.values                  # row0 aligns with max(lat) if descending

    lon_native_e = edges_from_centers_1d(lon_native_c)
    lat_native_e = edges_from_centers_1d(lat_native_c)

    if lat_native_c[0] > lat_native_c[-1]:         # descending → flip for plotting
        elev_native_plot = np.flipud(elev_native)
        lat_native_e_plot = lat_native_e[::-1]     # ascending edges for y
    else:
        elev_native_plot = elev_native
        lat_native_e_plot = lat_native_e

    plt.figure(figsize=(8,7))
    plt.pcolormesh(lon_native_e, lat_native_e_plot, elev_native_plot, shading="auto")
    plt.colorbar(label="Elevation (m)")
    plt.xlim(BBOX[0], BBOX[1]); plt.ylim(BBOX[2], BBOX[3])
    plt.xlabel("Longitude (°E)"); plt.ylabel("Latitude (°N)")
    plt.title("Native-resolution Elevation (Ladakh) — origin='lower'")
    for s in SITES:
        plt.plot(s["lon"], s["lat"], "o", ms=4)
        plt.text(s["lon"]+0.03, s["lat"]+0.03, s["name"], fontsize=9)
    plt.tight_layout()
    plt.savefig(PNG_NATIVE, dpi=200)
    plt.close()
    print(f"[OK] saved {PNG_NATIVE}")

    # -------- ERA5-aligned aggregation --------
    lon_c = era5_centers(BBOX[0], BBOX[1], ERA5_RES)   # centers ascending
    lat_c = era5_centers(BBOX[2], BBOX[3], ERA5_RES)   # centers ascending
    lon_e = centers_to_edges(lon_c, ERA5_RES)          # edges ascending
    lat_e = centers_to_edges(lat_c, ERA5_RES)          # edges ascending

    df = raster_to_points(dem_clip)
    stats = aggregate_to_era5(df, lon_c, lat_c, lon_e, lat_e)
    stats.to_csv(CSV_GRID, index=False)
    print(f"[OK] saved {CSV_GRID}")

    lat_asc, lon_asc, mean_grid = stats_to_grid(stats, lat_c, lon_c, "mean")

    # -------- ERA5 plot (lat ↑ upward) --------
    lon_plot_e = np.concatenate(([lon_asc[0]-ERA5_RES/2], lon_asc+ERA5_RES/2))
    lat_plot_e = np.concatenate(([lat_asc[0]-ERA5_RES/2], lat_asc+ERA5_RES/2))

    plt.figure(figsize=(8,7))
    plt.pcolormesh(lon_plot_e, lat_plot_e, mean_grid, shading="auto")
    plt.colorbar(label="Mean elevation (m)")
    plt.xlim(BBOX[0], BBOX[1]); plt.ylim(BBOX[2], BBOX[3])
    plt.xlabel("Longitude (°E)"); plt.ylabel("Latitude (°N)")
    plt.title("DEM mean on ERA5 0.25° grid — origin='lower'")
    for s in SITES:
        plt.plot(s["lon"], s["lat"], "o", ms=5)
        plt.text(s["lon"]+0.03, s["lat"]+0.03, s["name"], fontsize=9)
    plt.tight_layout()
    plt.savefig(PNG_ERA5, dpi=200)
    plt.close()
    print(f"[OK] saved {PNG_ERA5}")

    # -------- GeoTIFF (north-up; lat DESC in file) --------
    mean_for_tif = np.flipud(mean_grid)  # convert from LAT ASC (plot) to LAT DESC (north-up)
    transform = from_origin(lon_plot_e[0], lat_plot_e[-1], ERA5_RES, ERA5_RES)
    profile = dict(driver="GTiff", height=mean_for_tif.shape[0], width=mean_for_tif.shape[1],
                   count=1, dtype="float32", crs="EPSG:4326", transform=transform,
                   compress="deflate", nodata=np.nan)
    with rasterio.open(GTIFF_MEAN, "w", **profile) as dst:
        dst.write(mean_for_tif.astype("float32"), 1)
    print(f"[OK] saved {GTIFF_MEAN}")

    # -------- QA: compare NW cell from native mask vs aggregated grid --------
    latc_NW, lonc_NW = lat_asc[-1], lon_asc[0]
    lat0, lat1 = latc_NW - ERA5_RES/2, latc_NW + ERA5_RES/2
    lon0, lon1 = lonc_NW - ERA5_RES/2, lonc_NW + ERA5_RES/2

    # build masks on native coordinates
    lat_c_arr = lat_native_c
    lon_c_arr = lon_native_c
    lat_mask = (lat_c_arr >= lat0) & (lat_c_arr < lat1)
    lon_mask = (lon_c_arr >= lon0) & (lon_c_arr < lon1)

    # align row order to native array
    if lat_native_c[0] > lat_native_c[-1]:
        # native array rows are in DESC lat; select with the same order
        block = elev_native[lat_mask, :][:, lon_mask]
    else:
        block = elev_native[lat_mask, :][:, lon_mask]

    mean_direct = float(np.nanmean(block))
    mean_grid_val = float(mean_grid[-1, 0])  # LAT ASC → NW cell

    print("[QA] NW cell centers:", (latc_NW, lonc_NW))
    print("[QA] mean from native mask:", mean_direct)
    print("[QA] mean from ERA5 grid:", mean_grid_val)

if __name__ == "__main__":
    main()
